In [ ]:
from pathlib import Path
import pandas as pd
import json
import sys
import platform

# ============================================================
# Experiment 0: DAPT2020-MAST dataset discovery and audit
# ============================================================

DATA_DIR = Path(
    r"."
)

OUTPUT_DIR = DATA_DIR / "outputs"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

csv_files = sorted(DATA_DIR.glob("*.csv"))

print("=" * 70)
print("SparseSentinel-APT: DAPT-2020 Dataset Discovery")
print("=" * 70)
print(f"Python:       {sys.version.split()[0]}")
print(f"Platform:     {platform.platform()}")
print(f"Dataset path: {DATA_DIR}")
print(f"CSV files:    {len(csv_files)}")
print()

if not csv_files:
    raise FileNotFoundError(f"No CSV files were found in: {DATA_DIR}")

file_records = []

for file_path in csv_files:
    # Read only the header and a small sample
    sample = pd.read_csv(file_path, nrows=5, low_memory=False)

    filename_lower = file_path.name.lower()

    if "monday" in filename_lower:
        day = "Monday"
    elif "tuesday" in filename_lower:
        day = "Tuesday"
    elif "wednesday" in filename_lower:
        day = "Wednesday"
    elif "thursday" in filename_lower:
        day = "Thursday"
    elif "friday" in filename_lower:
        day = "Friday"
    else:
        day = "Unknown"

    if "-pvt" in filename_lower:
        network_scope = "private"
    elif "-public" in filename_lower:
        network_scope = "public"
    else:
        network_scope = "general"

    file_records.append({
        "filename": file_path.name,
        "size_mb": round(file_path.stat().st_size / (1024 ** 2), 2),
        "day": day,
        "network_scope": network_scope,
        "number_of_columns": len(sample.columns),
        "columns": list(sample.columns)
    })

summary = pd.DataFrame([
    {
        "filename": record["filename"],
        "size_mb": record["size_mb"],
        "day": record["day"],
        "network_scope": record["network_scope"],
        "number_of_columns": record["number_of_columns"]
    }
    for record in file_records
])

display(summary)

print("\nColumns in the first file:")
for index, column in enumerate(file_records[0]["columns"], start=1):
    print(f"{index:02d}. {column}")

summary.to_csv(
    OUTPUT_DIR / "dapt2020_file_inventory.csv",
    index=False
)

with open(
    OUTPUT_DIR / "dapt2020_schema_inventory.json",
    "w",
    encoding="utf-8"
) as file:
    json.dump(file_records, file, indent=2)

print("\nSaved:")
print(OUTPUT_DIR / "dapt2020_file_inventory.csv")
print(OUTPUT_DIR / "dapt2020_schema_inventory.json")

In [ ]:
# ============================================================
# Cell 2: Full schema, label, timestamp, and leakage audit
# ============================================================

import numpy as np
import pandas as pd
from pathlib import Path
from collections import Counter

DATA_DIR = Path(
    r"."
)
OUTPUT_DIR = DATA_DIR / "outputs"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

csv_files = sorted(DATA_DIR.glob("*.csv"))

# ------------------------------------------------------------
# 1. Check exact schema consistency
# ------------------------------------------------------------

schemas = {}

for file_path in csv_files:
    columns = pd.read_csv(file_path, nrows=0).columns
    columns = [str(c).strip() for c in columns]
    schemas[file_path.name] = columns

reference_file = csv_files[0].name
reference_schema = schemas[reference_file]

schema_rows = []

for filename, columns in schemas.items():
    missing = sorted(set(reference_schema) - set(columns))
    additional = sorted(set(columns) - set(reference_schema))

    schema_rows.append({
        "filename": filename,
        "column_count": len(columns),
        "same_column_names": set(columns) == set(reference_schema),
        "same_column_order": columns == reference_schema,
        "missing_columns": "; ".join(missing),
        "additional_columns": "; ".join(additional)
    })

schema_audit = pd.DataFrame(schema_rows)

print("=" * 80)
print("SCHEMA CONSISTENCY")
print("=" * 80)
display(
    schema_audit[
        [
            "filename",
            "column_count",
            "same_column_names",
            "same_column_order"
        ]
    ]
)

# ------------------------------------------------------------
# 2. Print a compact complete column list
# ------------------------------------------------------------

print("\n" + "=" * 80)
print("COMPLETE COLUMN LIST")
print("=" * 80)

for start in range(0, len(reference_schema), 10):
    stop = min(start + 10, len(reference_schema))

    print(
        f"\nColumns {start + 1:02d}-{stop:02d}:"
    )

    for index in range(start, stop):
        print(f"{index + 1:02d}. {reference_schema[index]}")

# ------------------------------------------------------------
# 3. Identify possible labels and leakage-prone fields
# ------------------------------------------------------------

label_keywords = [
    "label",
    "class",
    "attack",
    "activity",
    "stage",
    "scenario",
    "category",
    "target"
]

metadata_keywords = [
    "flow id",
    "src ip",
    "dst ip",
    "source ip",
    "destination ip",
    "timestamp"
]

candidate_label_columns = [
    column
    for column in reference_schema
    if any(keyword in column.lower() for keyword in label_keywords)
]

candidate_metadata_columns = [
    column
    for column in reference_schema
    if any(keyword == column.lower() for keyword in metadata_keywords)
]

print("\n" + "=" * 80)
print("POTENTIAL LABEL/ANNOTATION COLUMNS")
print("=" * 80)
print(candidate_label_columns)

print("\nPotential identifier/metadata columns:")
print(candidate_metadata_columns)

# ------------------------------------------------------------
# 4. Load and audit each file
# ------------------------------------------------------------

file_audit_records = []
label_distributions = []
all_flow_sources = []
all_dataframes = []

for file_index, file_path in enumerate(csv_files):
    print(f"Auditing {file_index + 1:02d}/{len(csv_files)}: {file_path.name}")

    df = pd.read_csv(file_path, low_memory=False)

    # Normalize accidental whitespace in column names
    df.columns = [str(c).strip() for c in df.columns]

    filename_lower = file_path.name.lower()

    if "monday" in filename_lower:
        day = "Monday"
    elif "tuesday" in filename_lower:
        day = "Tuesday"
    elif "wednesday" in filename_lower:
        day = "Wednesday"
    elif "thursday" in filename_lower:
        day = "Thursday"
    elif "friday" in filename_lower:
        day = "Friday"
    else:
        day = "Unknown"

    if "-pvt" in filename_lower:
        network_scope = "private"
    elif "-public" in filename_lower:
        network_scope = "public"
    else:
        network_scope = "general"

    # Add source metadata. These are audit fields, not model features.
    df["__source_file"] = file_path.name
    df["__day"] = day
    df["__network_scope"] = network_scope
    df["__source_row"] = np.arange(len(df), dtype=np.int64)

    missing_cells = int(df.isna().sum().sum())
    duplicate_rows = int(
        df.drop(
            columns=[
                "__source_file",
                "__day",
                "__network_scope",
                "__source_row"
            ]
        ).duplicated().sum()
    )

    flow_id_column = next(
        (
            column for column in df.columns
            if column.lower() == "flow id"
        ),
        None
    )

    timestamp_column = next(
        (
            column for column in df.columns
            if column.lower() == "timestamp"
        ),
        None
    )

    unique_flow_ids = (
        int(df[flow_id_column].nunique(dropna=True))
        if flow_id_column else np.nan
    )

    duplicate_flow_ids = (
        int(df[flow_id_column].duplicated(keep=False).sum())
        if flow_id_column else np.nan
    )

    timestamp_min = None
    timestamp_max = None
    timestamp_parse_rate = np.nan

    if timestamp_column:
        parsed_timestamp = pd.to_datetime(
            df[timestamp_column],
            errors="coerce",
            dayfirst=True
        )

        timestamp_parse_rate = float(parsed_timestamp.notna().mean())

        if parsed_timestamp.notna().any():
            timestamp_min = str(parsed_timestamp.min())
            timestamp_max = str(parsed_timestamp.max())

    file_audit_records.append({
        "filename": file_path.name,
        "day": day,
        "network_scope": network_scope,
        "rows": len(df),
        "columns_before_metadata": len(df.columns) - 4,
        "missing_cells": missing_cells,
        "duplicate_rows": duplicate_rows,
        "unique_flow_ids": unique_flow_ids,
        "rows_in_repeated_flow_ids": duplicate_flow_ids,
        "timestamp_parse_rate": timestamp_parse_rate,
        "timestamp_min": timestamp_min,
        "timestamp_max": timestamp_max
    })

    # Record label/annotation distributions separately
    for column in candidate_label_columns:
        if column in df.columns:
            counts = (
                df[column]
                .fillna("<MISSING>")
                .astype(str)
                .value_counts(dropna=False)
            )

            for value, count in counts.items():
                label_distributions.append({
                    "filename": file_path.name,
                    "day": day,
                    "column": column,
                    "value": value,
                    "count": int(count)
                })

    if flow_id_column:
        flow_source = df[
            [
                flow_id_column,
                "__source_file",
                "__day",
                "__network_scope"
            ]
        ].copy()

        flow_source.columns = [
            "flow_id",
            "source_file",
            "day",
            "network_scope"
        ]

        all_flow_sources.append(flow_source)

    all_dataframes.append(df)

file_audit = pd.DataFrame(file_audit_records)

print("\n" + "=" * 80)
print("FILE-LEVEL AUDIT")
print("=" * 80)

display(
    file_audit[
        [
            "filename",
            "rows",
            "unique_flow_ids",
            "rows_in_repeated_flow_ids",
            "missing_cells",
            "duplicate_rows",
            "timestamp_parse_rate"
        ]
    ]
)

print("\nTotal records:", f"{file_audit['rows'].sum():,}")

# ------------------------------------------------------------
# 5. Display compact label distributions
# ------------------------------------------------------------

label_distribution_df = pd.DataFrame(label_distributions)

print("\n" + "=" * 80)
print("GLOBAL LABEL/ANNOTATION DISTRIBUTIONS")
print("=" * 80)

if not label_distribution_df.empty:
    global_labels = (
        label_distribution_df
        .groupby(["column", "value"], as_index=False)["count"]
        .sum()
        .sort_values(["column", "count"], ascending=[True, False])
    )

    for column in global_labels["column"].unique():
        print(f"\n{column}:")
        display(
            global_labels[
                global_labels["column"] == column
            ][["value", "count"]].reset_index(drop=True)
        )
else:
    global_labels = pd.DataFrame()
    print("No candidate label columns were detected.")

# ------------------------------------------------------------
# 6. Detect Flow ID overlap across files and days
# ------------------------------------------------------------

print("\n" + "=" * 80)
print("FLOW-ID OVERLAP AUDIT")
print("=" * 80)

if all_flow_sources:
    flow_sources = pd.concat(all_flow_sources, ignore_index=True)

    flow_overlap = (
        flow_sources
        .groupby("flow_id", as_index=False)
        .agg(
            number_of_rows=("flow_id", "size"),
            number_of_files=("source_file", "nunique"),
            number_of_days=("day", "nunique"),
            number_of_scopes=("network_scope", "nunique")
        )
    )

    repeated_across_files = flow_overlap[
        flow_overlap["number_of_files"] > 1
    ]

    repeated_across_days = flow_overlap[
        flow_overlap["number_of_days"] > 1
    ]

    print(f"Unique Flow IDs:              {len(flow_overlap):,}")
    print(f"Flow IDs in multiple files:   {len(repeated_across_files):,}")
    print(f"Flow IDs on multiple days:    {len(repeated_across_days):,}")

    display(
        flow_overlap[
            [
                "number_of_rows",
                "number_of_files",
                "number_of_days",
                "number_of_scopes"
            ]
        ].describe()
    )
else:
    flow_sources = pd.DataFrame()
    flow_overlap = pd.DataFrame()
    print("Flow ID column was not detected.")

# ------------------------------------------------------------
# 7. Numeric infinity audit
# ------------------------------------------------------------

print("\n" + "=" * 80)
print("NON-FINITE NUMERIC VALUES")
print("=" * 80)

combined = pd.concat(all_dataframes, ignore_index=True)

numeric_columns = combined.select_dtypes(
    include=[np.number]
).columns.tolist()

infinite_counts = {}

for column in numeric_columns:
    values = pd.to_numeric(combined[column], errors="coerce")
    count = int(np.isinf(values.to_numpy(dtype=np.float64)).sum())

    if count > 0:
        infinite_counts[column] = count

if infinite_counts:
    infinite_table = pd.DataFrame(
        [
            {"column": column, "infinite_count": count}
            for column, count in infinite_counts.items()
        ]
    ).sort_values("infinite_count", ascending=False)

    display(infinite_table)
else:
    infinite_table = pd.DataFrame()
    print("No positive or negative infinity values detected.")

# ------------------------------------------------------------
# 8. Save audit outputs
# ------------------------------------------------------------

schema_audit.to_csv(
    OUTPUT_DIR / "dapt2020_schema_audit.csv",
    index=False
)

file_audit.to_csv(
    OUTPUT_DIR / "dapt2020_file_audit.csv",
    index=False
)

label_distribution_df.to_csv(
    OUTPUT_DIR / "dapt2020_label_distributions.csv",
    index=False
)

if not flow_overlap.empty:
    flow_overlap.to_csv(
        OUTPUT_DIR / "dapt2020_flow_id_overlap.csv",
        index=False
    )

if not infinite_table.empty:
    infinite_table.to_csv(
        OUTPUT_DIR / "dapt2020_infinite_values.csv",
        index=False
    )

print("\n" + "=" * 80)
print("AUDIT COMPLETE")
print("=" * 80)
print(f"Combined shape: {combined.shape}")
print(f"Numeric columns: {len(numeric_columns)}")
print(f"Outputs saved to: {OUTPUT_DIR}")

In [ ]:
# ============================================================
# Cell 3: Diagnose and repair the Thursday-private schema
# ============================================================

import csv
import re
import unicodedata
from pathlib import Path

import pandas as pd

DATA_DIR = Path(
    r"."
)

REFERENCE_FILE = DATA_DIR / "enp0s3-monday-pvt.pcap_Flow.csv"
PROBLEM_FILE = DATA_DIR / "enp0s3-pvt-thursday.pcap_Flow.csv"

# ------------------------------------------------------------
# 1. Read original headers
# ------------------------------------------------------------

reference_columns = pd.read_csv(
    REFERENCE_FILE,
    nrows=0
).columns.tolist()

problem_columns = pd.read_csv(
    PROBLEM_FILE,
    nrows=0
).columns.tolist()

print("=" * 90)
print("RAW HEADER COMPARISON")
print("=" * 90)

print(f"Reference columns: {len(reference_columns)}")
print(f"Problem columns:   {len(problem_columns)}")

comparison_rows = []

for index, (reference, problem) in enumerate(
    zip(reference_columns, problem_columns),
    start=1
):
    comparison_rows.append({
        "position": index,
        "reference": repr(reference),
        "problem": repr(problem),
        "exact_match": reference == problem
    })

comparison = pd.DataFrame(comparison_rows)

display(comparison)

print(
    "\nExact positional matches:",
    int(comparison["exact_match"].sum()),
    "/",
    len(comparison)
)

# ------------------------------------------------------------
# 2. Show the first two physical lines of each file
# ------------------------------------------------------------

def show_raw_lines(file_path, number_of_lines=2):
    print(f"\nRaw lines from: {file_path.name}")

    with open(
        file_path,
        "r",
        encoding="utf-8-sig",
        errors="replace"
    ) as file:
        for line_number in range(1, number_of_lines + 1):
            line = file.readline()

            print(
                f"Line {line_number}:",
                repr(line[:1000])
            )

show_raw_lines(REFERENCE_FILE)
show_raw_lines(PROBLEM_FILE)

# ------------------------------------------------------------
# 3. Normalize header strings
# ------------------------------------------------------------

def normalize_header(value):
    value = unicodedata.normalize("NFKC", str(value))
    value = value.replace("\ufeff", "")
    value = value.replace("\x00", "")
    value = value.strip()
    value = re.sub(r"\s+", " ", value)
    return value.lower()

normalized_reference = [
    normalize_header(column)
    for column in reference_columns
]

normalized_problem = [
    normalize_header(column)
    for column in problem_columns
]

normalized_matches = [
    reference == problem
    for reference, problem in zip(
        normalized_reference,
        normalized_problem
    )
]

print("\n" + "=" * 90)
print("NORMALIZED HEADER COMPARISON")
print("=" * 90)

print(
    "Normalized positional matches:",
    sum(normalized_matches),
    "/",
    len(normalized_matches)
)

# ------------------------------------------------------------
# 4. Determine whether positional repair is safe
# ------------------------------------------------------------

reference_sample = pd.read_csv(
    REFERENCE_FILE,
    nrows=10,
    low_memory=False
)

problem_sample = pd.read_csv(
    PROBLEM_FILE,
    nrows=10,
    low_memory=False
)

print("\nReference sample shape:", reference_sample.shape)
print("Problem sample shape:  ", problem_sample.shape)

print("\nFirst five reference column names:")
print(reference_columns[:5])

print("\nFirst five problem column names:")
print(problem_columns[:5])

print("\nLast five reference column names:")
print(reference_columns[-5:])

print("\nLast five problem column names:")
print(problem_columns[-5:])

# Compare broad data types by position
type_comparison = []

for index in range(len(reference_columns)):
    reference_series = reference_sample.iloc[:, index]
    problem_series = problem_sample.iloc[:, index]

    type_comparison.append({
        "position": index + 1,
        "reference_column": reference_columns[index],
        "problem_column": problem_columns[index],
        "reference_dtype": str(reference_series.dtype),
        "problem_dtype": str(problem_series.dtype),
        "compatible": (
            pd.api.types.is_numeric_dtype(reference_series)
            ==
            pd.api.types.is_numeric_dtype(problem_series)
        )
    })

type_comparison_df = pd.DataFrame(type_comparison)

print(
    "\nBroad positional dtype compatibility:",
    f"{type_comparison_df['compatible'].mean():.2%}"
)

display(
    type_comparison_df[
        ~type_comparison_df["compatible"]
    ]
)

# ------------------------------------------------------------
# 5. Repair only if positions and data types are compatible
# ------------------------------------------------------------

POSITIONAL_COMPATIBILITY_THRESHOLD = 0.95

if (
    len(reference_columns) == len(problem_columns)
    and type_comparison_df["compatible"].mean()
        >= POSITIONAL_COMPATIBILITY_THRESHOLD
):
    repaired = pd.read_csv(
        PROBLEM_FILE,
        low_memory=False
    )

    # Assign the verified canonical schema by position
    repaired.columns = reference_columns

    repaired_path = (
        DATA_DIR /
        "enp0s3-pvt-thursday.schema_repaired.csv"
    )

    repaired.to_csv(
        repaired_path,
        index=False
    )

    print("\nSchema repair completed.")
    print("Repaired file:", repaired_path)
    print("Repaired shape:", repaired.shape)

    print("\nActivity distribution:")
    display(
        repaired["Activity"]
        .value_counts(dropna=False)
        .rename_axis("Activity")
        .reset_index(name="count")
    )

    print("\nStage distribution:")
    display(
        repaired["Stage"]
        .value_counts(dropna=False)
        .rename_axis("Stage")
        .reset_index(name="count")
    )

else:
    repaired_path = None

    print("\nAUTOMATIC REPAIR STOPPED.")
    print(
        "The columns are not sufficiently compatible for safe "
        "positional renaming."
    )

In [ ]:
# ============================================================
# Cell 4: Correct headerless file and build canonical dataset
# ============================================================

from pathlib import Path
import hashlib
import json

import numpy as np
import pandas as pd

DATA_DIR = Path(
    r"."
)

OUTPUT_DIR = DATA_DIR / "outputs"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

REFERENCE_FILE = DATA_DIR / "enp0s3-monday-pvt.pcap_Flow.csv"
HEADERLESS_FILE = DATA_DIR / "enp0s3-pvt-thursday.pcap_Flow.csv"

# Use the verified 85-column schema
canonical_columns = (
    pd.read_csv(REFERENCE_FILE, nrows=0)
    .columns.astype(str)
    .str.strip()
    .tolist()
)

# Explicitly list the ten original source files.
# This prevents the generated repaired file from being loaded twice.
source_files = [
    DATA_DIR / "enp0s3-monday-pvt.pcap_Flow.csv",
    DATA_DIR / "enp0s3-monday.pcap_Flow.csv",
    DATA_DIR / "enp0s3-public-thursday.pcap_Flow.csv",
    DATA_DIR / "enp0s3-public-tuesday.pcap_Flow.csv",
    DATA_DIR / "enp0s3-public-wednesday.pcap_Flow.csv",
    DATA_DIR / "enp0s3-pvt-thursday.pcap_Flow.csv",
    DATA_DIR / "enp0s3-pvt-tuesday.pcap_Flow.csv",
    DATA_DIR / "enp0s3-pvt-wednesday.pcap_Flow.csv",
    DATA_DIR / "enp0s3-tcpdump-friday.pcap_Flow.csv",
    DATA_DIR / "enp0s3-tcpdump-pvt-friday.pcap_Flow.csv"
]

for file_path in source_files:
    if not file_path.exists():
        raise FileNotFoundError(file_path)

# ------------------------------------------------------------
# 1. Helper functions
# ------------------------------------------------------------

def extract_day(filename):
    name = filename.lower()

    for day in [
        "monday",
        "tuesday",
        "wednesday",
        "thursday",
        "friday"
    ]:
        if day in name:
            return day.capitalize()

    return "Unknown"


def extract_scope(filename):
    name = filename.lower()

    if "-pvt" in name:
        return "private"

    if "-public" in name:
        return "public"

    # Monday and Friday filenames do not consistently contain "public".
    return "general"


def normalize_activity(value):
    value = str(value).strip()

    mapping = {
        "BENIGN": "Benign",
        "Normal": "Benign"
    }

    return mapping.get(value, value)


def normalize_stage(value):
    value = str(value).strip()

    mapping = {
        "BENIGN": "Benign",
        "Normal": "Benign"
    }

    return mapping.get(value, value)


def make_evidence_id(source_file, source_row, flow_id):
    raw_value = (
        f"{source_file}|{source_row}|{flow_id}"
    ).encode("utf-8")

    digest = hashlib.sha256(raw_value).hexdigest()[:20]

    return f"DAPT-EV-{digest.upper()}"


# ------------------------------------------------------------
# 2. Correctly read and save the headerless file
# ------------------------------------------------------------

headerless_df = pd.read_csv(
    HEADERLESS_FILE,
    header=None,
    names=canonical_columns,
    low_memory=False
)

print("=" * 80)
print("HEADERLESS FILE CORRECTION")
print("=" * 80)

print("Corrected shape:", headerless_df.shape)
print("Expected columns:", len(canonical_columns))
print("Actual columns:  ", len(headerless_df.columns))

assert len(headerless_df.columns) == 85
assert list(headerless_df.columns) == canonical_columns
assert len(headerless_df) == 4114, (
    f"Expected 4,114 rows, found {len(headerless_df):,}"
)

corrected_path = (
    OUTPUT_DIR /
    "enp0s3-pvt-thursday.header_corrected.csv"
)

headerless_df.to_csv(
    corrected_path,
    index=False
)

print("Corrected file saved:", corrected_path)

# ------------------------------------------------------------
# 3. Load all ten original files correctly
# ------------------------------------------------------------

frames = []
source_inventory = []

for file_path in source_files:

    if file_path == HEADERLESS_FILE:
        df = pd.read_csv(
            file_path,
            header=None,
            names=canonical_columns,
            low_memory=False
        )
        header_status = "reconstructed"
    else:
        df = pd.read_csv(
            file_path,
            low_memory=False
        )

        df.columns = [
            str(column).strip()
            for column in df.columns
        ]

        header_status = "original"

    if list(df.columns) != canonical_columns:
        raise ValueError(
            f"Schema mismatch remains in {file_path.name}"
        )

    day = extract_day(file_path.name)
    scope = extract_scope(file_path.name)

    df["source_file"] = file_path.name
    df["source_row"] = np.arange(
        len(df),
        dtype=np.int64
    )
    df["day"] = day
    df["network_scope"] = scope
    df["header_status"] = header_status

    source_inventory.append({
        "source_file": file_path.name,
        "rows": len(df),
        "day": day,
        "network_scope": scope,
        "header_status": header_status
    })

    frames.append(df)

master = pd.concat(
    frames,
    ignore_index=True,
    verify_integrity=False
)

# ------------------------------------------------------------
# 4. Normalize labels without using them as input features
# ------------------------------------------------------------

master["Activity_original"] = master["Activity"]
master["Stage_original"] = master["Stage"]

master["activity_label"] = (
    master["Activity"]
    .map(normalize_activity)
)

master["stage_label"] = (
    master["Stage"]
    .map(normalize_stage)
)

master["campaign_label"] = (
    master["stage_label"] != "Benign"
).astype(np.int8)

# These original label columns remain annotation-only.
master.drop(
    columns=["Activity", "Stage"],
    inplace=True
)

# ------------------------------------------------------------
# 5. Parse timestamp consistently
# ------------------------------------------------------------

master["timestamp_parsed"] = pd.to_datetime(
    master["Timestamp"],
    format="%d/%m/%Y %I:%M:%S %p",
    errors="coerce"
)

timestamp_success = master["timestamp_parsed"].notna().mean()

print(
    f"\nTimestamp parse success: "
    f"{timestamp_success:.4%}"
)

if timestamp_success < 0.999:
    failed_timestamps = master.loc[
        master["timestamp_parsed"].isna(),
        ["source_file", "source_row", "Timestamp"]
    ]

    display(failed_timestamps.head(20))

# ------------------------------------------------------------
# 6. Generate immutable evidence identifiers
# ------------------------------------------------------------

master["evidence_id"] = [
    make_evidence_id(source_file, source_row, flow_id)
    for source_file, source_row, flow_id in zip(
        master["source_file"],
        master["source_row"],
        master["Flow ID"]
    )
]

if not master["evidence_id"].is_unique:
    raise ValueError("Evidence identifiers are not unique.")

# ------------------------------------------------------------
# 7. Define annotation and model-input boundaries
# ------------------------------------------------------------

annotation_columns = [
    "Activity_original",
    "Stage_original",
    "activity_label",
    "stage_label",
    "campaign_label"
]

identifier_columns = [
    "Flow ID",
    "Src IP",
    "Dst IP",
    "Timestamp",
    "timestamp_parsed",
    "evidence_id",
    "source_file",
    "source_row",
    "day",
    "network_scope",
    "header_status"
]

excluded_from_model = sorted(
    set(annotation_columns + identifier_columns)
)

model_feature_columns = [
    column
    for column in master.columns
    if column not in excluded_from_model
]

# ------------------------------------------------------------
# 8. Validate the canonical master table
# ------------------------------------------------------------

print("\n" + "=" * 80)
print("CANONICAL DAPT2020 MASTER TABLE")
print("=" * 80)

print("Master shape:", master.shape)
print("Expected rows: 86,691")
print("Model feature columns:", len(model_feature_columns))
print("Unique evidence IDs:", f"{master['evidence_id'].nunique():,}")
print("Unique Flow IDs:", f"{master['Flow ID'].nunique():,}")
print("Missing cells:", f"{master.isna().sum().sum():,}")

assert len(master) == 86691, (
    f"Expected 86,691 records, found {len(master):,}"
)

assert master["evidence_id"].is_unique
assert "Activity" not in model_feature_columns
assert "Stage" not in model_feature_columns
assert "activity_label" not in model_feature_columns
assert "stage_label" not in model_feature_columns
assert "campaign_label" not in model_feature_columns

# ------------------------------------------------------------
# 9. Display corrected distributions
# ------------------------------------------------------------

print("\nActivity distribution:")
activity_distribution = (
    master["activity_label"]
    .value_counts(dropna=False)
    .rename_axis("activity_label")
    .reset_index(name="count")
)

display(activity_distribution)

print("\nStage distribution:")
stage_distribution = (
    master["stage_label"]
    .value_counts(dropna=False)
    .rename_axis("stage_label")
    .reset_index(name="count")
)

display(stage_distribution)

print("\nCampaign distribution:")
campaign_distribution = (
    master["campaign_label"]
    .value_counts(dropna=False)
    .rename_axis("campaign_label")
    .reset_index(name="count")
)

display(campaign_distribution)

# ------------------------------------------------------------
# 10. Check evidence leakage boundary
# ------------------------------------------------------------

leaked_features = [
    column
    for column in model_feature_columns
    if any(
        keyword in column.lower()
        for keyword in [
            "activity",
            "stage",
            "label",
            "target",
            "campaign"
        ]
    )
]

print("\nPotential leakage columns remaining in X:")
print(leaked_features)

if leaked_features:
    raise ValueError(
        "Potential label leakage remains in model features."
    )

# ------------------------------------------------------------
# 11. Save dataset and manifests
# ------------------------------------------------------------

source_inventory_df = pd.DataFrame(source_inventory)

source_inventory_df.to_csv(
    OUTPUT_DIR / "dapt2020_canonical_source_inventory.csv",
    index=False
)

activity_distribution.to_csv(
    OUTPUT_DIR / "dapt2020_activity_distribution.csv",
    index=False
)

stage_distribution.to_csv(
    OUTPUT_DIR / "dapt2020_stage_distribution.csv",
    index=False
)

campaign_distribution.to_csv(
    OUTPUT_DIR / "dapt2020_campaign_distribution.csv",
    index=False
)

master.to_csv(
    OUTPUT_DIR / "dapt2020_canonical_master.csv",
    index=False
)

feature_manifest = {
    "total_records": int(len(master)),
    "canonical_source_files": [
        path.name for path in source_files
    ],
    "model_feature_columns": model_feature_columns,
    "annotation_only_columns": annotation_columns,
    "identifier_metadata_columns": identifier_columns,
    "excluded_from_model": excluded_from_model
}

with open(
    OUTPUT_DIR / "dapt2020_feature_manifest.json",
    "w",
    encoding="utf-8"
) as file:
    json.dump(
        feature_manifest,
        file,
        indent=2
    )

# Optional Parquet output
try:
    parquet_path = (
        OUTPUT_DIR /
        "dapt2020_canonical_master.parquet"
    )

    master.to_parquet(
        parquet_path,
        index=False
    )

    print("\nParquet saved:", parquet_path)

except ImportError:
    print(
        "\nParquet was not saved because pyarrow/fastparquet "
        "is unavailable. The CSV output is sufficient."
    )

print("\nCanonical CSV saved:")
print(OUTPUT_DIR / "dapt2020_canonical_master.csv")

print("\nFeature manifest saved:")
print(OUTPUT_DIR / "dapt2020_feature_manifest.json")

In [ ]:
# ============================================================
# Cell 5: Aspect mapping and Flow-ID group feasibility audit
# ============================================================

from pathlib import Path
import json

import numpy as np
import pandas as pd

DATA_DIR = Path(
    r"."
)

OUTPUT_DIR = DATA_DIR / "outputs"

MASTER_PATH = (
    OUTPUT_DIR /
    "dapt2020_canonical_master.parquet"
)

master = pd.read_parquet(MASTER_PATH)

# ------------------------------------------------------------
# 1. Define the four shared APT aspects
# ------------------------------------------------------------

STAGE_TO_ASPECT = {
    "Reconnaissance": "reconnaissance",
    "Establish Foothold": "initial_access_foothold",
    "Lateral Movement": "lateral_movement",
    "Data Exfiltration": "data_exfiltration"
}

ASPECTS = [
    "reconnaissance",
    "initial_access_foothold",
    "lateral_movement",
    "data_exfiltration"
]

master["aspect_label"] = (
    master["stage_label"]
    .map(STAGE_TO_ASPECT)
    .fillna("benign")
)

print("=" * 80)
print("ASPECT DISTRIBUTION")
print("=" * 80)

aspect_distribution = (
    master["aspect_label"]
    .value_counts(dropna=False)
    .rename_axis("aspect")
    .reset_index(name="records")
)

display(aspect_distribution)

# ------------------------------------------------------------
# 2. Construct one profile per Flow ID
# ------------------------------------------------------------

def sorted_unique(values):
    return sorted(
        {
            str(value)
            for value in values
            if pd.notna(value)
        }
    )


flow_profile = (
    master
    .groupby("Flow ID", sort=False)
    .agg(
        number_of_records=("evidence_id", "size"),
        first_timestamp=("timestamp_parsed", "min"),
        last_timestamp=("timestamp_parsed", "max"),
        number_of_days=("day", "nunique"),
        number_of_files=("source_file", "nunique"),
        number_of_stages=("stage_label", "nunique"),
        number_of_activities=("activity_label", "nunique"),
        number_of_aspects=("aspect_label", "nunique"),
        contains_attack=("campaign_label", "max"),
        stages=("stage_label", sorted_unique),
        activities=("activity_label", sorted_unique),
        aspects=("aspect_label", sorted_unique)
    )
    .reset_index()
)

flow_profile["duration_seconds"] = (
    flow_profile["last_timestamp"]
    - flow_profile["first_timestamp"]
).dt.total_seconds()

flow_profile["mixed_stage"] = (
    flow_profile["number_of_stages"] > 1
)

flow_profile["mixed_aspect"] = (
    flow_profile["number_of_aspects"] > 1
)

# ------------------------------------------------------------
# 3. Add group-level aspect indicators
# ------------------------------------------------------------

for aspect in ASPECTS:
    flow_profile[f"has_{aspect}"] = (
        flow_profile["aspects"]
        .apply(lambda values: aspect in values)
        .astype(np.int8)
    )

flow_profile["has_benign"] = (
    flow_profile["aspects"]
    .apply(lambda values: "benign" in values)
    .astype(np.int8)
)

# ------------------------------------------------------------
# 4. Summarize group sizes
# ------------------------------------------------------------

print("\n" + "=" * 80)
print("FLOW-ID GROUP SIZE SUMMARY")
print("=" * 80)

display(
    flow_profile[
        [
            "number_of_records",
            "number_of_days",
            "number_of_files",
            "number_of_stages",
            "number_of_aspects",
            "duration_seconds"
        ]
    ].describe(
        percentiles=[
            0.50,
            0.75,
            0.90,
            0.95,
            0.99
        ]
    )
)

# ------------------------------------------------------------
# 5. Count Flow IDs supporting each aspect
# ------------------------------------------------------------

group_aspect_records = []

for aspect in ["benign"] + ASPECTS:
    column = (
        "has_benign"
        if aspect == "benign"
        else f"has_{aspect}"
    )

    selected_groups = flow_profile[
        flow_profile[column] == 1
    ]

    selected_flow_ids = set(
        selected_groups["Flow ID"]
    )

    number_of_records = int(
        master["Flow ID"]
        .isin(selected_flow_ids)
        .sum()
    )

    exact_aspect_records = int(
        (master["aspect_label"] == aspect).sum()
    )

    group_aspect_records.append({
        "aspect": aspect,
        "flow_id_groups": len(selected_groups),
        "exact_aspect_records": exact_aspect_records,
        "all_records_in_selected_groups": number_of_records
    })

group_aspect_summary = pd.DataFrame(
    group_aspect_records
)

print("\n" + "=" * 80)
print("ASPECT SUPPORT BY FLOW-ID GROUP")
print("=" * 80)

display(group_aspect_summary)

# ------------------------------------------------------------
# 6. Examine mixed-stage and mixed-aspect Flow IDs
# ------------------------------------------------------------

print("\n" + "=" * 80)
print("MIXED-GROUP AUDIT")
print("=" * 80)

print(
    "Total Flow ID groups:",
    f"{len(flow_profile):,}"
)

print(
    "Mixed-stage groups:",
    f"{flow_profile['mixed_stage'].sum():,}"
)

print(
    "Mixed-aspect groups:",
    f"{flow_profile['mixed_aspect'].sum():,}"
)

mixed_groups = (
    flow_profile[
        flow_profile["mixed_aspect"]
    ]
    .sort_values(
        "number_of_records",
        ascending=False
    )
)

display(
    mixed_groups[
        [
            "Flow ID",
            "number_of_records",
            "number_of_days",
            "stages",
            "activities",
            "aspects"
        ]
    ].head(20)
)

# ------------------------------------------------------------
# 7. Analyze the rare data-exfiltration class
# ------------------------------------------------------------

exfiltration_records = master[
    master["aspect_label"] == "data_exfiltration"
].copy()

exfiltration_groups = (
    exfiltration_records
    .groupby("Flow ID", sort=False)
    .agg(
        records=("evidence_id", "size"),
        first_timestamp=("timestamp_parsed", "min"),
        last_timestamp=("timestamp_parsed", "max"),
        days=("day", sorted_unique),
        files=("source_file", sorted_unique),
        activities=("activity_label", sorted_unique)
    )
    .reset_index()
    .sort_values("records", ascending=False)
)

print("\n" + "=" * 80)
print("DATA-EXFILTRATION GROUPS")
print("=" * 80)

print(
    "Exfiltration records:",
    f"{len(exfiltration_records):,}"
)

print(
    "Exfiltration Flow ID groups:",
    f"{len(exfiltration_groups):,}"
)

display(exfiltration_groups)

# ------------------------------------------------------------
# 8. Sequence-length feasibility for T = 8
# ------------------------------------------------------------

SEQUENCE_LENGTH = 8

flow_profile["full_windows_T8"] = (
    flow_profile["number_of_records"]
    // SEQUENCE_LENGTH
)

flow_profile["requires_padding_T8"] = (
    flow_profile["number_of_records"]
    < SEQUENCE_LENGTH
)

print("\n" + "=" * 80)
print("T=8 SEQUENCE FEASIBILITY")
print("=" * 80)

print(
    "Groups with at least 8 records:",
    f"{(flow_profile['number_of_records'] >= 8).sum():,}"
)

print(
    "Groups requiring padding:",
    f"{flow_profile['requires_padding_T8'].sum():,}"
)

print(
    "Percentage requiring padding:",
    f"{flow_profile['requires_padding_T8'].mean():.2%}"
)

print(
    "Non-overlapping full T=8 windows:",
    f"{flow_profile['full_windows_T8'].sum():,}"
)

# ------------------------------------------------------------
# 9. Save the audit artifacts
# ------------------------------------------------------------

flow_profile_for_csv = flow_profile.copy()

for column in ["stages", "activities", "aspects"]:
    flow_profile_for_csv[column] = (
        flow_profile_for_csv[column]
        .apply(json.dumps)
    )

mixed_groups_for_csv = mixed_groups.copy()

for column in ["stages", "activities", "aspects"]:
    mixed_groups_for_csv[column] = (
        mixed_groups_for_csv[column]
        .apply(json.dumps)
    )

exfiltration_groups_for_csv = exfiltration_groups.copy()

for column in ["days", "files", "activities"]:
    exfiltration_groups_for_csv[column] = (
        exfiltration_groups_for_csv[column]
        .apply(json.dumps)
    )

flow_profile_for_csv.to_csv(
    OUTPUT_DIR / "dapt2020_flow_group_profile.csv",
    index=False
)

group_aspect_summary.to_csv(
    OUTPUT_DIR / "dapt2020_group_aspect_summary.csv",
    index=False
)

mixed_groups_for_csv.to_csv(
    OUTPUT_DIR / "dapt2020_mixed_flow_groups.csv",
    index=False
)

exfiltration_groups_for_csv.to_csv(
    OUTPUT_DIR / "dapt2020_exfiltration_groups.csv",
    index=False
)

master.to_parquet(
    OUTPUT_DIR /
    "dapt2020_canonical_master_with_aspects.parquet",
    index=False
)

print("\nSaved group and aspect audit artifacts.")

In [ ]:
# ============================================================
# Replacement Cell 6:
# Nested stratified and Flow-ID-disjoint splitting
# ============================================================

from pathlib import Path
import json

import numpy as np
import pandas as pd
import sklearn
from sklearn.model_selection import StratifiedGroupKFold

DATA_DIR = Path(
    r"."
)

OUTPUT_DIR = DATA_DIR / "outputs"

MASTER_PATH = (
    OUTPUT_DIR /
    "dapt2020_canonical_master_with_aspects.parquet"
)

master = pd.read_parquet(MASTER_PATH)

SEED = 42

STAGES = [
    "Benign",
    "Reconnaissance",
    "Establish Foothold",
    "Lateral Movement",
    "Data Exfiltration"
]

print("=" * 80)
print("NESTED STRATIFIED GROUP SPLIT")
print("=" * 80)
print("scikit-learn version:", sklearn.__version__)

# ------------------------------------------------------------
# 1. Restore annotation-review status
# ------------------------------------------------------------

master["annotation_status"] = "weak_verified"

master.loc[
    (
        master["stage_label"] == "Data Exfiltration"
    )
    &
    (
        master["activity_label"] != "Data Exfiltration"
    ),
    "annotation_status"
] = "requires_expert_review"

# ------------------------------------------------------------
# 2. Helper for evaluating a candidate holdout fold
# ------------------------------------------------------------

def candidate_fold_score(
    dataframe,
    candidate_indices,
    target_fraction
):
    """
    Lower is better.

    The score jointly evaluates:
    1. total partition size;
    2. stage-distribution similarity;
    3. presence of every stage.
    """

    candidate = dataframe.iloc[candidate_indices]

    global_counts = (
        dataframe["stage_label"]
        .value_counts()
        .reindex(STAGES, fill_value=0)
        .astype(float)
    )

    candidate_counts = (
        candidate["stage_label"]
        .value_counts()
        .reindex(STAGES, fill_value=0)
        .astype(float)
    )

    expected_counts = global_counts * target_fraction

    relative_error = (
        (candidate_counts - expected_counts).abs()
        / expected_counts.clip(lower=1.0)
    )

    actual_fraction = len(candidate) / len(dataframe)

    size_error = abs(
        actual_fraction - target_fraction
    ) / target_fraction

    missing_stages = int(
        (candidate_counts == 0).sum()
    )

    # Missing any stage receives a strong penalty.
    score = (
        relative_error.mean()
        + 2.0 * size_error
        + 100.0 * missing_stages
    )

    return {
        "score": float(score),
        "actual_fraction": float(actual_fraction),
        "missing_stages": missing_stages,
        "counts": candidate_counts.to_dict()
    }


def select_best_fold(
    dataframe,
    number_of_splits,
    seed
):
    """
    Generate stratified group folds and select the fold closest
    to the target class and sample distribution.
    """

    splitter = StratifiedGroupKFold(
        n_splits=number_of_splits,
        shuffle=True,
        random_state=seed
    )

    y = dataframe["stage_label"].to_numpy()
    groups = dataframe["Flow ID"].to_numpy()

    candidates = []

    for fold_number, (_, holdout_indices) in enumerate(
        splitter.split(
            X=np.zeros(len(dataframe)),
            y=y,
            groups=groups
        )
    ):
        evaluation = candidate_fold_score(
            dataframe=dataframe,
            candidate_indices=holdout_indices,
            target_fraction=1.0 / number_of_splits
        )

        candidates.append({
            "fold_number": fold_number,
            "indices": holdout_indices,
            **evaluation
        })

    candidate_summary = pd.DataFrame(
        [
            {
                "fold_number": candidate["fold_number"],
                "score": candidate["score"],
                "actual_fraction": candidate["actual_fraction"],
                "missing_stages": candidate["missing_stages"],
                **{
                    f"count_{stage}":
                    candidate["counts"][stage]
                    for stage in STAGES
                }
            }
            for candidate in candidates
        ]
    )

    # Prefer folds containing every stage.
    valid_candidates = [
        candidate
        for candidate in candidates
        if candidate["missing_stages"] == 0
    ]

    if not valid_candidates:
        raise RuntimeError(
            "No candidate fold contains every stage. "
            "The rare-class split requires manual review."
        )

    selected = min(
        valid_candidates,
        key=lambda candidate: candidate["score"]
    )

    return (
        selected["indices"],
        candidate_summary,
        selected["fold_number"]
    )

# ------------------------------------------------------------
# 3. Outer seven-fold split: reserve one fold for testing
# ------------------------------------------------------------

test_indices, outer_summary, selected_test_fold = (
    select_best_fold(
        dataframe=master,
        number_of_splits=7,
        seed=SEED
    )
)

master["split"] = "development"
master.iloc[
    test_indices,
    master.columns.get_loc("split")
] = "test"

development = (
    master[
        master["split"] == "development"
    ]
    .copy()
    .reset_index()
    .rename(columns={"index": "master_index"})
)

print("\nOuter-fold candidates:")
display(outer_summary)

print("Selected test fold:", selected_test_fold)

# ------------------------------------------------------------
# 4. Inner six-fold split: reserve one fold for validation
# ------------------------------------------------------------

validation_local_indices, inner_summary, selected_val_fold = (
    select_best_fold(
        dataframe=development,
        number_of_splits=6,
        seed=SEED + 1
    )
)

validation_master_indices = (
    development
    .iloc[validation_local_indices]
    ["master_index"]
    .to_numpy()
)

master.loc[
    validation_master_indices,
    "split"
] = "validation"

master.loc[
    master["split"] == "development",
    "split"
] = "train"

print("\nInner-fold candidates:")
display(inner_summary)

print("Selected validation fold:", selected_val_fold)

# ------------------------------------------------------------
# 5. Verify Flow-ID disjointness
# ------------------------------------------------------------

split_names = [
    "train",
    "validation",
    "test"
]

split_flow_ids = {
    split_name: set(
        master.loc[
            master["split"] == split_name,
            "Flow ID"
        ]
    )
    for split_name in split_names
}

overlap_matrix = pd.DataFrame(
    index=split_names,
    columns=split_names,
    dtype=int
)

for first_split in split_names:
    for second_split in split_names:
        if first_split == second_split:
            overlap_matrix.loc[
                first_split,
                second_split
            ] = len(split_flow_ids[first_split])
        else:
            overlap_matrix.loc[
                first_split,
                second_split
            ] = len(
                split_flow_ids[first_split]
                &
                split_flow_ids[second_split]
            )

print("\n" + "=" * 80)
print("FLOW-ID OVERLAP MATRIX")
print("=" * 80)

display(overlap_matrix)

assert (
    len(
        split_flow_ids["train"]
        &
        split_flow_ids["validation"]
    )
    == 0
)

assert (
    len(
        split_flow_ids["train"]
        &
        split_flow_ids["test"]
    )
    == 0
)

assert (
    len(
        split_flow_ids["validation"]
        &
        split_flow_ids["test"]
    )
    == 0
)

# ------------------------------------------------------------
# 6. Record-level stage distribution
# ------------------------------------------------------------

stage_counts = (
    master
    .groupby(
        ["split", "stage_label"]
    )
    .size()
    .unstack(fill_value=0)
    .reindex(
        index=split_names,
        columns=STAGES,
        fill_value=0
    )
)

stage_counts["total"] = (
    stage_counts.sum(axis=1)
)

stage_counts["percentage"] = (
    100.0
    * stage_counts["total"]
    / len(master)
)

print("\n" + "=" * 80)
print("RECORD DISTRIBUTION BY SPLIT")
print("=" * 80)

display(stage_counts)

# Require every stage in every partition
for split_name in split_names:
    for stage in STAGES:
        assert stage_counts.loc[
            split_name,
            stage
        ] > 0, (
            f"{stage} is absent from {split_name}."
        )

# ------------------------------------------------------------
# 7. Group distribution
# ------------------------------------------------------------

group_counts = (
    master[
        ["Flow ID", "split"]
    ]
    .drop_duplicates()
    ["split"]
    .value_counts()
    .reindex(split_names)
    .rename_axis("split")
    .reset_index(name="flow_id_groups")
)

print("\nFlow-ID groups by split:")
display(group_counts)

# ------------------------------------------------------------
# 8. Activity and aspect distributions
# ------------------------------------------------------------

aspect_counts = (
    master
    .groupby(
        ["split", "aspect_label"]
    )
    .size()
    .unstack(fill_value=0)
)

print("\nAspect distribution:")
display(aspect_counts)

review_counts = (
    master
    .groupby(
        ["split", "annotation_status"]
    )
    .size()
    .unstack(fill_value=0)
)

print("\nAnnotation-review distribution:")
display(review_counts)

# ------------------------------------------------------------
# 9. Save the corrected split
# ------------------------------------------------------------

split_path = (
    OUTPUT_DIR /
    "dapt2020_group_disjoint_split.parquet"
)

master.to_parquet(
    split_path,
    index=False
)

stage_counts.to_csv(
    OUTPUT_DIR /
    "dapt2020_split_stage_distribution.csv"
)

group_counts.to_csv(
    OUTPUT_DIR /
    "dapt2020_split_group_distribution.csv",
    index=False
)

outer_summary.to_csv(
    OUTPUT_DIR /
    "dapt2020_outer_fold_candidates.csv",
    index=False
)

inner_summary.to_csv(
    OUTPUT_DIR /
    "dapt2020_inner_fold_candidates.csv",
    index=False
)

flow_assignments = (
    master[
        ["Flow ID", "split"]
    ]
    .drop_duplicates()
    .sort_values(
        ["split", "Flow ID"]
    )
)

flow_assignments.to_csv(
    OUTPUT_DIR /
    "dapt2020_flow_id_split_assignments.csv",
    index=False
)

split_manifest = {
    "seed": SEED,
    "method": "nested_stratified_group_kfold",
    "outer_folds": 7,
    "selected_test_fold": int(
        selected_test_fold
    ),
    "inner_folds": 6,
    "selected_validation_fold": int(
        selected_val_fold
    ),
    "approximate_ratios": {
        "train": 5 / 7,
        "validation": 1 / 7,
        "test": 1 / 7
    },
    "group_key": "Flow ID",
    "group_overlap": {
        "train_validation": 0,
        "train_test": 0,
        "validation_test": 0
    },
    "exfiltration_limitation": (
        "Only 15 records, all captured on Friday; "
        "per-class results are exploratory."
    ),
    "window_construction": {
        "performed_after_split": True,
        "candidate_evidence_count": 32
    }
}

with open(
    OUTPUT_DIR /
    "dapt2020_split_manifest.json",
    "w",
    encoding="utf-8"
) as file:
    json.dump(
        split_manifest,
        file,
        indent=2
    )

print("\nCorrected split saved:")
print(split_path)

In [ ]:
# ============================================================
# Cell 7: Construct temporal evidence windows (N=32)
# ============================================================

from pathlib import Path
import hashlib
import json
import math

import numpy as np
import pandas as pd

DATA_DIR = Path(
    r"."
)

OUTPUT_DIR = DATA_DIR / "outputs"

SPLIT_PATH = (
    OUTPUT_DIR /
    "dapt2020_group_disjoint_split.parquet"
)

master = pd.read_parquet(SPLIT_PATH)

WINDOW_SIZE = 32

STRIDE_BY_SPLIT = {
    "train": 16,
    "validation": 32,
    "test": 32
}

ASPECTS = [
    "reconnaissance",
    "initial_access_foothold",
    "lateral_movement",
    "data_exfiltration"
]

SPLIT_ORDER = [
    "train",
    "validation",
    "test"
]

# ------------------------------------------------------------
# 1. Basic validation
# ------------------------------------------------------------

required_columns = [
    "evidence_id",
    "Flow ID",
    "timestamp_parsed",
    "day",
    "split",
    "aspect_label",
    "stage_label",
    "activity_label",
    "network_scope",
    "annotation_status"
]

missing_columns = [
    column
    for column in required_columns
    if column not in master.columns
]

if missing_columns:
    raise ValueError(
        f"Missing required columns: {missing_columns}"
    )

master["timestamp_parsed"] = pd.to_datetime(
    master["timestamp_parsed"],
    errors="raise"
)

assert master["evidence_id"].is_unique
assert master["split"].isin(SPLIT_ORDER).all()

# ------------------------------------------------------------
# 2. Deterministic window identifier
# ------------------------------------------------------------

def make_window_id(
    split_name,
    day,
    start_timestamp,
    first_evidence_id,
    ordinal
):
    raw_value = (
        f"{split_name}|{day}|{start_timestamp}|"
        f"{first_evidence_id}|{ordinal}"
    ).encode("utf-8")

    digest = hashlib.sha256(
        raw_value
    ).hexdigest()[:16].upper()

    split_code = {
        "train": "TR",
        "validation": "VA",
        "test": "TE"
    }[split_name]

    day_code = day[:3].upper()

    return (
        f"DAPT-W-{split_code}-{day_code}-{digest}"
    )

# ------------------------------------------------------------
# 3. Generate window boundaries
# ------------------------------------------------------------

def generate_window_positions(
    number_of_records,
    window_size,
    stride
):
    """
    Training:
        Uses overlapping windows and includes a final full tail window.

    Validation/test:
        Uses strictly non-overlapping windows. The final window may
        contain fewer than window_size records and is padded later.
    """

    if number_of_records <= 0:
        return []

    # Strictly non-overlapping evaluation windows
    if stride == window_size:
        return [
            (
                start,
                min(start + window_size, number_of_records)
            )
            for start in range(
                0,
                number_of_records,
                window_size
            )
        ]

    # Overlapping training windows
    if number_of_records <= window_size:
        return [(0, number_of_records)]

    starts = list(
        range(
            0,
            number_of_records - window_size + 1,
            stride
        )
    )

    # Add one final full-length window to cover the tail.
    final_start = number_of_records - window_size

    if starts[-1] != final_start:
        starts.append(final_start)

    return [
        (
            start,
            start + window_size
        )
        for start in starts
    ]

# ------------------------------------------------------------
# 4. Build windows separately inside every split and day
# ------------------------------------------------------------

window_records = []
membership_records = []

for split_name in SPLIT_ORDER:

    split_data = master[
        master["split"] == split_name
    ].copy()

    stride = STRIDE_BY_SPLIT[split_name]

    for day, day_data in split_data.groupby(
        "day",
        sort=True
    ):
        day_data = (
            day_data
            .sort_values(
                [
                    "timestamp_parsed",
                    "source_file",
                    "source_row",
                    "evidence_id"
                ]
            )
            .reset_index(drop=True)
        )

        positions = generate_window_positions(
            number_of_records=len(day_data),
            window_size=WINDOW_SIZE,
            stride=stride
        )

        for ordinal, (start, stop) in enumerate(
            positions
        ):
            window = (
                day_data
                .iloc[start:stop]
                .copy()
                .reset_index(drop=True)
            )

            first_row = window.iloc[0]
            last_row = window.iloc[-1]

            window_id = make_window_id(
                split_name=split_name,
                day=day,
                start_timestamp=first_row[
                    "timestamp_parsed"
                ],
                first_evidence_id=first_row[
                    "evidence_id"
                ],
                ordinal=ordinal
            )

            evidence_count = len(window)
            padding_count = (
                WINDOW_SIZE - evidence_count
            )

            attack_count = int(
                (window["aspect_label"] != "benign").sum()
            )

            benign_count = int(
                (window["aspect_label"] == "benign").sum()
            )

            aspect_counts = {
                aspect: int(
                    (
                        window["aspect_label"]
                        == aspect
                    ).sum()
                )
                for aspect in ASPECTS
            }

            present_aspects = [
                aspect
                for aspect, count
                in aspect_counts.items()
                if count > 0
            ]

            campaign_label = int(
                attack_count > 0
            )

            review_count = int(
                (
                    window["annotation_status"]
                    == "requires_expert_review"
                ).sum()
            )

            window_records.append({
                "window_id": window_id,
                "split": split_name,
                "day": day,
                "window_ordinal": ordinal,
                "start_timestamp": first_row[
                    "timestamp_parsed"
                ],
                "end_timestamp": last_row[
                    "timestamp_parsed"
                ],
                "duration_seconds": float(
                    (
                        last_row["timestamp_parsed"]
                        - first_row["timestamp_parsed"]
                    ).total_seconds()
                ),
                "evidence_count": evidence_count,
                "padding_count": padding_count,
                "stride": stride,
                "unique_flow_ids": int(
                    window["Flow ID"].nunique()
                ),
                "network_scopes": json.dumps(
                    sorted(
                        window["network_scope"]
                        .astype(str)
                        .unique()
                        .tolist()
                    )
                ),
                "attack_evidence_count": attack_count,
                "benign_evidence_count": benign_count,
                "campaign_label": campaign_label,
                "present_aspects": json.dumps(
                    present_aspects
                ),
                "reconnaissance_count":
                    aspect_counts["reconnaissance"],
                "initial_access_foothold_count":
                    aspect_counts[
                        "initial_access_foothold"
                    ],
                "lateral_movement_count":
                    aspect_counts[
                        "lateral_movement"
                    ],
                "data_exfiltration_count":
                    aspect_counts[
                        "data_exfiltration"
                    ],
                "requires_expert_review_count":
                    review_count
            })

            for position, (_, evidence) in enumerate(
                window.iterrows()
            ):
                membership_records.append({
                    "window_id": window_id,
                    "split": split_name,
                    "day": day,
                    "position": position,
                    "attention_mask": 1,
                    "evidence_id":
                        evidence["evidence_id"],
                    "Flow ID":
                        evidence["Flow ID"],
                    "timestamp_parsed":
                        evidence["timestamp_parsed"],
                    "network_scope":
                        evidence["network_scope"],
                    "aspect_label":
                        evidence["aspect_label"],
                    "stage_label":
                        evidence["stage_label"],
                    "activity_label":
                        evidence["activity_label"],
                    "annotation_status":
                        evidence["annotation_status"]
                })

# ------------------------------------------------------------
# 5. Create window and membership tables
# ------------------------------------------------------------

windows = pd.DataFrame(window_records)
membership = pd.DataFrame(membership_records)

assert windows["window_id"].is_unique

print("=" * 80)
print("TEMPORAL WINDOW CONSTRUCTION")
print("=" * 80)

print("Window size:", WINDOW_SIZE)
print("Total windows:", f"{len(windows):,}")
print("Membership rows:", f"{len(membership):,}")

# ------------------------------------------------------------
# 6. Window distribution
# ------------------------------------------------------------

window_split_summary = (
    windows
    .groupby("split")
    .agg(
        windows=("window_id", "size"),
        attack_windows=("campaign_label", "sum"),
        mean_evidence_count=(
            "evidence_count",
            "mean"
        ),
        mean_unique_flow_ids=(
            "unique_flow_ids",
            "mean"
        ),
        mean_duration_seconds=(
            "duration_seconds",
            "mean"
        ),
        windows_requiring_review=(
            "requires_expert_review_count",
            lambda values: int(
                (values > 0).sum()
            )
        )
    )
    .reindex(SPLIT_ORDER)
)

window_split_summary[
    "benign_windows"
] = (
    window_split_summary["windows"]
    - window_split_summary["attack_windows"]
)

print("\nWindow distribution:")
display(window_split_summary)

# ------------------------------------------------------------
# 7. Per-aspect window coverage
# ------------------------------------------------------------

aspect_window_records = []

for split_name in SPLIT_ORDER:
    split_windows = windows[
        windows["split"] == split_name
    ]

    for aspect in ASPECTS:
        count_column = f"{aspect}_count"

        aspect_window_records.append({
            "split": split_name,
            "aspect": aspect,
            "supporting_windows": int(
                (
                    split_windows[count_column] > 0
                ).sum()
            ),
            "supporting_evidence": int(
                split_windows[count_column].sum()
            )
        })

aspect_window_summary = pd.DataFrame(
    aspect_window_records
)

print("\nPer-aspect window coverage:")
display(aspect_window_summary)

# ------------------------------------------------------------
# 8. Evidence reuse audit
# ------------------------------------------------------------

evidence_reuse = (
    membership
    .groupby(
        ["split", "evidence_id"]
    )
    .size()
    .rename("window_appearances")
    .reset_index()
)

reuse_summary = (
    evidence_reuse
    .groupby("split")
    ["window_appearances"]
    .agg(
        [
            "count",
            "min",
            "median",
            "mean",
            "max"
        ]
    )
    .reindex(SPLIT_ORDER)
)

print("\nEvidence reuse:")
display(reuse_summary)

# Evaluation evidence should appear exactly once
for split_name in [
    "validation",
    "test"
]:
    split_reuse = evidence_reuse[
        evidence_reuse["split"] == split_name
    ]

    assert (
        split_reuse["window_appearances"].max()
        == 1
    )

    assert (
        split_reuse["window_appearances"].min()
        == 1
    )

# Every source evidence record must be represented
for split_name in SPLIT_ORDER:
    expected_ids = set(
        master.loc[
            master["split"] == split_name,
            "evidence_id"
        ]
    )

    observed_ids = set(
        membership.loc[
            membership["split"] == split_name,
            "evidence_id"
        ]
    )

    assert expected_ids == observed_ids

# ------------------------------------------------------------
# 9. Verify no Flow ID crosses split boundaries
# ------------------------------------------------------------

membership_flow_splits = (
    membership
    .groupby("Flow ID")
    ["split"]
    .nunique()
)

assert membership_flow_splits.max() == 1

print(
    "\nMaximum partitions per Flow ID:",
    membership_flow_splits.max()
)

# ------------------------------------------------------------
# 10. Produce aspect-pair candidate labels
# ------------------------------------------------------------

pair_records = []

for _, window in windows.iterrows():

    for aspect in ASPECTS:
        support_count = int(
            window[f"{aspect}_count"]
        )

        other_attack_count = int(
            window["attack_evidence_count"]
            - support_count
        )

        benign_count = int(
            window["benign_evidence_count"]
        )

        if support_count > 0:
            weak_stance = "SUPPORTING"
            insufficient_context = False

        elif other_attack_count > 0:
            weak_stance = "NEUTRAL"
            insufficient_context = False

        else:
            # Benign-only windows do not automatically contradict
            # an attack aspect.
            weak_stance = "NEUTRAL"
            insufficient_context = True

        pair_records.append({
            "window_id": window["window_id"],
            "split": window["split"],
            "day": window["day"],
            "aspect": aspect,
            "weak_stance": weak_stance,
            "support_count": support_count,
            "other_attack_count": other_attack_count,
            "benign_count": benign_count,
            "insufficient_context":
                insufficient_context,
            "contradiction_status":
                "not_yet_annotated",
            "requires_expert_review": bool(
                window[
                    "requires_expert_review_count"
                ] > 0
            )
        })

aspect_pairs = pd.DataFrame(pair_records)

print("\nWeak stance-pair distribution:")
display(
    aspect_pairs
    .groupby(
        [
            "split",
            "weak_stance",
            "insufficient_context"
        ]
    )
    .size()
    .rename("pairs")
    .reset_index()
)

# ------------------------------------------------------------
# 11. Save artifacts
# ------------------------------------------------------------

windows.to_parquet(
    OUTPUT_DIR /
    "dapt2020_mast_windows.parquet",
    index=False
)

membership.to_parquet(
    OUTPUT_DIR /
    "dapt2020_mast_window_membership.parquet",
    index=False
)

aspect_pairs.to_parquet(
    OUTPUT_DIR /
    "dapt2020_mast_aspect_pairs.parquet",
    index=False
)

windows.to_csv(
    OUTPUT_DIR /
    "dapt2020_mast_windows.csv",
    index=False
)

aspect_pairs.to_csv(
    OUTPUT_DIR /
    "dapt2020_mast_aspect_pairs.csv",
    index=False
)

window_split_summary.to_csv(
    OUTPUT_DIR /
    "dapt2020_mast_window_summary.csv"
)

aspect_window_summary.to_csv(
    OUTPUT_DIR /
    "dapt2020_mast_aspect_coverage.csv",
    index=False
)

window_manifest = {
    "window_size": WINDOW_SIZE,
    "stride": STRIDE_BY_SPLIT,
    "grouping": [
        "split",
        "day"
    ],
    "ordering": [
        "timestamp_parsed",
        "source_file",
        "source_row",
        "evidence_id"
    ],
    "evaluation_windows_overlap": False,
    "training_windows_overlap": True,
    "stance_policy": {
        "SUPPORTING": (
            "At least one exact aspect-stage match."
        ),
        "NEUTRAL": (
            "No exact aspect match."
        ),
        "CONTRADICTING": (
            "Reserved for expert-verified benign alternatives; "
            "not automatically inferred."
        ),
        "INSUFFICIENT_CONTEXT": (
            "Flag for benign-only windows without decisive "
            "aspect evidence."
        )
    }
}

with open(
    OUTPUT_DIR /
    "dapt2020_mast_window_manifest.json",
    "w",
    encoding="utf-8"
) as file:
    json.dump(
        window_manifest,
        file,
        indent=2
    )

print("\nSaved:")
print(
    OUTPUT_DIR /
    "dapt2020_mast_windows.parquet"
)
print(
    OUTPUT_DIR /
    "dapt2020_mast_window_membership.parquet"
)
print(
    OUTPUT_DIR /
    "dapt2020_mast_aspect_pairs.parquet"
)

In [ ]:
# Validation and test must contain each source record exactly once
for split_name in ["validation", "test"]:

    expected_records = int(
        (master["split"] == split_name).sum()
    )

    observed_memberships = int(
        (membership["split"] == split_name).sum()
    )

    assert observed_memberships == expected_records, (
        f"{split_name}: expected {expected_records:,} memberships, "
        f"found {observed_memberships:,}"
    )

# Training may reuse evidence because stride 16 creates overlap
train_source_records = int(
    (master["split"] == "train").sum()
)

train_memberships = int(
    (membership["split"] == "train").sum()
)

assert train_memberships >= train_source_records

print("\nMembership verification:")
print(
    "Train:",
    f"{train_memberships:,} memberships from "
    f"{train_source_records:,} source records"
)
print(
    "Validation:",
    f"{(membership['split'] == 'validation').sum():,} memberships"
)
print(
    "Test:",
    f"{(membership['split'] == 'test').sum():,} memberships"
)

In [ ]:
# ============================================================
# Cell 8: Leakage-safe deterministic event verbalization
# ============================================================

from pathlib import Path
import ipaddress
import json

import numpy as np
import pandas as pd

DATA_DIR = Path(
    r"."
)

OUTPUT_DIR = DATA_DIR / "outputs"

MASTER_PATH = (
    OUTPUT_DIR /
    "dapt2020_group_disjoint_split.parquet"
)

MEMBERSHIP_PATH = (
    OUTPUT_DIR /
    "dapt2020_mast_window_membership.parquet"
)

master = pd.read_parquet(MASTER_PATH)
membership = pd.read_parquet(MEMBERSHIP_PATH)

# ------------------------------------------------------------
# 1. Model-safe fields used by the verbalizer
# ------------------------------------------------------------

VERBALIZATION_NUMERIC_FIELDS = {
    "duration": "Flow Duration",
    "forward_packets": "Total Fwd Packet",
    "backward_packets": "Total Bwd packets",
    "forward_bytes": "Total Length of Fwd Packet",
    "backward_bytes": "Total Length of Bwd Packet",
    "flow_bytes_per_second": "Flow Bytes/s",
    "flow_packets_per_second": "Flow Packets/s"
}

for semantic_name, column in (
    VERBALIZATION_NUMERIC_FIELDS.items()
):
    if column not in master.columns:
        raise ValueError(
            f"Required column is missing: {column}"
        )

# ------------------------------------------------------------
# 2. Fit numerical thresholds using training data only
# ------------------------------------------------------------

QUANTILES = [
    0.25,
    0.50,
    0.75,
    0.90
]

train_data = master[
    master["split"] == "train"
]

bucket_thresholds = {}

for semantic_name, column in (
    VERBALIZATION_NUMERIC_FIELDS.items()
):
    values = pd.to_numeric(
        train_data[column],
        errors="coerce"
    ).replace(
        [np.inf, -np.inf],
        np.nan
    ).dropna()

    thresholds = (
        values
        .quantile(QUANTILES)
        .to_numpy(dtype=float)
    )

    bucket_thresholds[semantic_name] = (
        thresholds.tolist()
    )

print("=" * 80)
print("TRAIN-ONLY VERBALIZATION THRESHOLDS")
print("=" * 80)

threshold_table = pd.DataFrame(
    [
        {
            "semantic_field": field,
            "q25": values[0],
            "q50": values[1],
            "q75": values[2],
            "q90": values[3]
        }
        for field, values
        in bucket_thresholds.items()
    ]
)

display(threshold_table)

# ------------------------------------------------------------
# 3. Value abstraction functions
# ------------------------------------------------------------

BUCKET_NAMES = [
    "very low",
    "low",
    "moderate",
    "high",
    "very high"
]

def bucketize(value, semantic_name):
    value = pd.to_numeric(
        pd.Series([value]),
        errors="coerce"
    ).iloc[0]

    if pd.isna(value):
        return "unknown"

    thresholds = np.asarray(
        bucket_thresholds[semantic_name],
        dtype=float
    )

    bucket_index = int(
        np.searchsorted(
            thresholds,
            float(value),
            side="right"
        )
    )

    bucket_index = min(
        bucket_index,
        len(BUCKET_NAMES) - 1
    )

    return BUCKET_NAMES[bucket_index]


def is_rfc1918(value):
    try:
        address = ipaddress.ip_address(
            str(value).strip()
        )

        private_networks = [
            ipaddress.ip_network("10.0.0.0/8"),
            ipaddress.ip_network("172.16.0.0/12"),
            ipaddress.ip_network("192.168.0.0/16")
        ]

        return any(
            address in network
            for network in private_networks
        )

    except ValueError:
        return False


def endpoint_role(value):
    return (
        "internal"
        if is_rfc1918(value)
        else "external"
    )


def protocol_name(value):
    try:
        value = int(float(value))
    except (TypeError, ValueError):
        return "unknown-protocol"

    return {
        0: "IP",
        1: "ICMP",
        6: "TCP",
        17: "UDP"
    }.get(
        value,
        f"protocol-{value}"
    )


SERVICE_PORTS = {
    20: "FTP-data",
    21: "FTP",
    22: "SSH",
    23: "Telnet",
    25: "SMTP",
    53: "DNS",
    80: "HTTP",
    110: "POP3",
    123: "NTP",
    135: "RPC",
    139: "NetBIOS",
    143: "IMAP",
    389: "LDAP",
    443: "HTTPS",
    445: "SMB",
    1433: "MSSQL",
    1521: "Oracle",
    3306: "MySQL",
    3389: "RDP",
    5432: "PostgreSQL",
    8080: "HTTP-alt"
}


def service_description(value):
    try:
        port = int(float(value))
    except (TypeError, ValueError):
        return "an unknown destination service"

    if port in SERVICE_PORTS:
        return f"the {SERVICE_PORTS[port]} service"

    if port == 0:
        return "a non-application endpoint"

    if 1 <= port <= 1023:
        return "an uncommon system service"

    if 1024 <= port <= 49151:
        return "a registered application service"

    if 49152 <= port <= 65535:
        return "an ephemeral destination port"

    return "an invalid or unknown port"

# ------------------------------------------------------------
# 4. Deterministic event verbalizer
# ------------------------------------------------------------

def verbalize_event(row):
    source_role = endpoint_role(
        row["Src IP"]
    )

    destination_role = endpoint_role(
        row["Dst IP"]
    )

    direction = (
        f"{source_role}-to-{destination_role}"
    )

    protocol = protocol_name(
        row["Protocol"]
    )

    service = service_description(
        row["Dst Port"]
    )

    duration = bucketize(
        row["Flow Duration"],
        "duration"
    )

    forward_packets = bucketize(
        row["Total Fwd Packet"],
        "forward_packets"
    )

    backward_packets = bucketize(
        row["Total Bwd packets"],
        "backward_packets"
    )

    forward_bytes = bucketize(
        row["Total Length of Fwd Packet"],
        "forward_bytes"
    )

    backward_bytes = bucketize(
        row["Total Length of Bwd Packet"],
        "backward_bytes"
    )

    byte_rate = bucketize(
        row["Flow Bytes/s"],
        "flow_bytes_per_second"
    )

    packet_rate = bucketize(
        row["Flow Packets/s"],
        "flow_packets_per_second"
    )

    return (
        f"An {direction} {protocol} flow contacted {service}. "
        f"The connection had {duration} duration, "
        f"{forward_packets} forward-packet volume, and "
        f"{backward_packets} backward-packet volume. "
        f"It carried {forward_bytes} forward-byte volume and "
        f"{backward_bytes} backward-byte volume, with a "
        f"{byte_rate} byte rate and {packet_rate} packet rate."
    )

master["verbalized_event"] = (
    master.apply(
        verbalize_event,
        axis=1
    )
)

# ------------------------------------------------------------
# 5. Add modality-availability masks
# ------------------------------------------------------------

# DAPT-2020 provides numerical network and temporal evidence.
# The deterministic verbalization supplies an NLP view.
# Host, command, and external CTI are naturally unavailable.

master["mask_network"] = np.int8(1)
master["mask_temporal"] = np.int8(1)
master["mask_verbalized_text"] = np.int8(1)
master["mask_host"] = np.int8(0)
master["mask_command"] = np.int8(0)
master["mask_cti"] = np.int8(0)

# ------------------------------------------------------------
# 6. Strict verbalization leakage audit
# ------------------------------------------------------------

forbidden_strings = set(
    master["activity_label"]
    .astype(str)
    .str.lower()
    .unique()
    .tolist()
)

forbidden_strings.update(
    master["stage_label"]
    .astype(str)
    .str.lower()
    .unique()
    .tolist()
)

forbidden_strings.update(
    [
        "benign",
        "reconnaissance",
        "foothold",
        "lateral movement",
        "data exfiltration",
        "network scan",
        "directory bruteforce",
        "account discovery",
        "sql injection",
        "privilege escalation"
    ]
)

detected_leakage_terms = []

lower_text = (
    master["verbalized_event"]
    .str.lower()
)

for forbidden_string in sorted(
    forbidden_strings
):
    if (
        forbidden_string
        and lower_text.str.contains(
            forbidden_string,
            regex=False
        ).any()
    ):
        detected_leakage_terms.append(
            forbidden_string
        )

print("\n" + "=" * 80)
print("VERBALIZATION LEAKAGE AUDIT")
print("=" * 80)

print(
    "Detected forbidden label terms:",
    detected_leakage_terms
)

assert not detected_leakage_terms

# Raw IPs must not appear in generated text
sample_raw_ips = pd.concat(
    [
        master["Src IP"],
        master["Dst IP"]
    ]
).dropna().astype(str).unique()

ip_leak_count = 0

for ip_value in sample_raw_ips:
    if lower_text.str.contains(
        ip_value.lower(),
        regex=False
    ).any():
        ip_leak_count += 1

print(
    "Raw IP values found in text:",
    ip_leak_count
)

assert ip_leak_count == 0

# Every generated direction begins with either "internal" or
# "external", so the grammatically correct article is "An".
invalid_article_count = int(
    lower_text.str.startswith(
        ("a internal", "a external")
    ).sum()
)

print(
    "Invalid A internal/external articles:",
    invalid_article_count
)

assert invalid_article_count == 0

# ------------------------------------------------------------
# 7. Attach text to window membership
# ------------------------------------------------------------

evidence_text_table = master[
    [
        "evidence_id",
        "verbalized_event",
        "mask_network",
        "mask_temporal",
        "mask_verbalized_text",
        "mask_host",
        "mask_command",
        "mask_cti"
    ]
]

membership_enriched = membership.merge(
    evidence_text_table,
    on="evidence_id",
    how="left",
    validate="many_to_one"
)

assert (
    membership_enriched[
        "verbalized_event"
    ].notna().all()
)

# ------------------------------------------------------------
# 8. Audit verbalization diversity
# ------------------------------------------------------------

verbalization_summary = pd.DataFrame(
    {
        "records": [len(master)],
        "unique_verbalizations": [
            master[
                "verbalized_event"
            ].nunique()
        ],
        "mean_characters": [
            master[
                "verbalized_event"
            ].str.len().mean()
        ],
        "minimum_characters": [
            master[
                "verbalized_event"
            ].str.len().min()
        ],
        "maximum_characters": [
            master[
                "verbalized_event"
            ].str.len().max()
        ]
    }
)

print("\nVerbalization summary:")
display(verbalization_summary)

print("\nExample verbalizations:")

example_columns = [
    "evidence_id",
    "split",
    "day",
    "network_scope",
    "verbalized_event"
]

display(
    master[
        example_columns
    ]
    .sample(
        n=10,
        random_state=42
    )
    .reset_index(drop=True)
)

print("\nModality masks:")

mask_columns = [
    "mask_network",
    "mask_temporal",
    "mask_verbalized_text",
    "mask_host",
    "mask_command",
    "mask_cti"
]

display(
    master[
        mask_columns
    ]
    .drop_duplicates()
)

# ------------------------------------------------------------
# 9. Save outputs
# ------------------------------------------------------------

master.to_parquet(
    OUTPUT_DIR /
    "dapt2020_mast_evidence.parquet",
    index=False
)

membership_enriched.to_parquet(
    OUTPUT_DIR /
    "dapt2020_mast_window_membership_enriched.parquet",
    index=False
)

threshold_table.to_csv(
    OUTPUT_DIR /
    "dapt2020_verbalization_thresholds.csv",
    index=False
)

verbalization_summary.to_csv(
    OUTPUT_DIR /
    "dapt2020_verbalization_summary.csv",
    index=False
)

with open(
    OUTPUT_DIR /
    "dapt2020_verbalization_config.json",
    "w",
    encoding="utf-8"
) as file:
    json.dump(
        {
            "fit_partition": "train",
            "quantiles": QUANTILES,
            "bucket_names": BUCKET_NAMES,
            "thresholds": bucket_thresholds,
            "raw_ip_in_text": False,
            "exact_activity_or_stage_in_text": False,
            "modality_masks": {
                "network": 1,
                "temporal": 1,
                "verbalized_text": 1,
                "host": 0,
                "command": 0,
                "cti": 0
            }
        },
        file,
        indent=2
    )

print("\nSaved:")
print(
    OUTPUT_DIR /
    "dapt2020_mast_evidence.parquet"
)
print(
    OUTPUT_DIR /
    "dapt2020_mast_window_membership_enriched.parquet"
)


In [ ]:
# ============================================================
# Cell 9: Prepare AM-JEPA numerical and temporal tensors
# ============================================================

from pathlib import Path
import json
import pickle

import numpy as np
import pandas as pd
from sklearn.preprocessing import RobustScaler

DATA_DIR = Path(
    r"."
)

OUTPUT_DIR = DATA_DIR / "outputs"
TENSOR_DIR = OUTPUT_DIR / "am_jepa_tensors"
TENSOR_DIR.mkdir(parents=True, exist_ok=True)

EVIDENCE_PATH = (
    OUTPUT_DIR /
    "dapt2020_mast_evidence.parquet"
)

WINDOW_PATH = (
    OUTPUT_DIR /
    "dapt2020_mast_windows.parquet"
)

MEMBERSHIP_PATH = (
    OUTPUT_DIR /
    "dapt2020_mast_window_membership_enriched.parquet"
)

FEATURE_MANIFEST_PATH = (
    OUTPUT_DIR /
    "dapt2020_feature_manifest.json"
)

WINDOW_SIZE = 32

ASPECTS = [
    "reconnaissance",
    "initial_access_foothold",
    "lateral_movement",
    "data_exfiltration"
]

ASPECT_TO_ID = {
    aspect: index
    for index, aspect in enumerate(ASPECTS)
}

EVIDENCE_ASPECT_TO_ID = {
    "benign": 0,
    "reconnaissance": 1,
    "initial_access_foothold": 2,
    "lateral_movement": 3,
    "data_exfiltration": 4
}

SPLIT_TO_ID = {
    "train": 0,
    "validation": 1,
    "test": 2
}

# ------------------------------------------------------------
# 1. Load artifacts
# ------------------------------------------------------------

evidence = pd.read_parquet(EVIDENCE_PATH)
windows = pd.read_parquet(WINDOW_PATH)
membership = pd.read_parquet(MEMBERSHIP_PATH)

with open(
    FEATURE_MANIFEST_PATH,
    "r",
    encoding="utf-8"
) as file:
    feature_manifest = json.load(file)

feature_columns = (
    feature_manifest["model_feature_columns"]
)

print("=" * 80)
print("AM-JEPA TENSOR PREPARATION")
print("=" * 80)

print("Evidence records:", f"{len(evidence):,}")
print("Windows:", f"{len(windows):,}")
print("Membership rows:", f"{len(membership):,}")
print("Numerical features:", len(feature_columns))

assert len(feature_columns) == 79
assert evidence["evidence_id"].is_unique
assert windows["window_id"].is_unique

# ------------------------------------------------------------
# 2. Convert all model features to finite numerical values
# ------------------------------------------------------------

numeric_features = evidence[
    feature_columns
].copy()

conversion_failures = {}

for column in feature_columns:
    original_nonmissing = (
        numeric_features[column].notna().sum()
    )

    numeric_features[column] = pd.to_numeric(
        numeric_features[column],
        errors="coerce"
    )

    converted_nonmissing = (
        numeric_features[column].notna().sum()
    )

    failures = int(
        original_nonmissing - converted_nonmissing
    )

    if failures > 0:
        conversion_failures[column] = failures

print("\nConversion failures:")
print(conversion_failures)

if conversion_failures:
    raise ValueError(
        "Some model features contain unexpected nonnumeric values."
    )

numeric_features = numeric_features.replace(
    [np.inf, -np.inf],
    np.nan
)

# ------------------------------------------------------------
# 3. Fit imputation values on training data only
# ------------------------------------------------------------

train_mask = (
    evidence["split"] == "train"
).to_numpy()

train_features = numeric_features.loc[
    train_mask
]

train_medians = (
    train_features
    .median(axis=0)
)

all_missing_train_columns = (
    train_medians[
        train_medians.isna()
    ].index.tolist()
)

print(
    "\nColumns entirely missing in training:",
    all_missing_train_columns
)

if all_missing_train_columns:
    raise ValueError(
        "At least one feature is completely unavailable "
        "in the training partition."
    )

numeric_features = (
    numeric_features
    .fillna(train_medians)
)

assert not numeric_features.isna().any().any()

# ------------------------------------------------------------
# 4. Fit RobustScaler on training data only
# ------------------------------------------------------------

scaler = RobustScaler(
    quantile_range=(5.0, 95.0),
    with_centering=True,
    with_scaling=True
)

scaler.fit(
    numeric_features.loc[
        train_mask,
        feature_columns
    ].to_numpy(dtype=np.float64)
)

scaled_features = scaler.transform(
    numeric_features[
        feature_columns
    ].to_numpy(dtype=np.float64)
)

# Limit extreme outliers after train-only scaling
scaled_features = np.clip(
    scaled_features,
    -10.0,
    10.0
).astype(np.float32)

assert np.isfinite(scaled_features).all()

print(
    "Scaled evidence matrix:",
    scaled_features.shape
)

# ------------------------------------------------------------
# 5. Build evidence lookup tables
# ------------------------------------------------------------

evidence_ids = (
    evidence["evidence_id"]
    .astype(str)
    .to_numpy()
)

evidence_index = {
    evidence_id: index
    for index, evidence_id
    in enumerate(evidence_ids)
}

evidence_aspect_ids = (
    evidence["aspect_label"]
    .map(EVIDENCE_ASPECT_TO_ID)
    .to_numpy(dtype=np.int64)
)

if (evidence_aspect_ids < 0).any():
    raise ValueError(
        "Unknown evidence aspect was detected."
    )

evidence_timestamps = (
    pd.to_datetime(
        evidence["timestamp_parsed"]
    )
    .astype("int64")
    .to_numpy()
    / 1e9
)

# ------------------------------------------------------------
# 6. Sort windows and memberships deterministically
# ------------------------------------------------------------

windows = (
    windows
    .sort_values(
        [
            "split",
            "day",
            "start_timestamp",
            "window_id"
        ]
    )
    .reset_index(drop=True)
)

membership = (
    membership
    .sort_values(
        [
            "window_id",
            "position"
        ]
    )
    .reset_index(drop=True)
)

membership_groups = {
    window_id: group
    for window_id, group
    in membership.groupby(
        "window_id",
        sort=False
    )
}

# ------------------------------------------------------------
# 7. Allocate window tensors
# ------------------------------------------------------------

number_of_windows = len(windows)
number_of_features = len(feature_columns)

X_network = np.zeros(
    (
        number_of_windows,
        WINDOW_SIZE,
        number_of_features
    ),
    dtype=np.float32
)

X_temporal = np.zeros(
    (
        number_of_windows,
        WINDOW_SIZE,
        4
    ),
    dtype=np.float32
)

attention_mask = np.zeros(
    (
        number_of_windows,
        WINDOW_SIZE
    ),
    dtype=np.int8
)

evidence_aspect_targets = np.full(
    (
        number_of_windows,
        WINDOW_SIZE
    ),
    fill_value=-100,
    dtype=np.int64
)

aspect_presence_targets = np.zeros(
    (
        number_of_windows,
        len(ASPECTS)
    ),
    dtype=np.float32
)

campaign_targets = np.zeros(
    number_of_windows,
    dtype=np.int64
)

split_ids = np.zeros(
    number_of_windows,
    dtype=np.int8
)

window_ids = []
window_evidence_ids = np.full(
    (
        number_of_windows,
        WINDOW_SIZE
    ),
    fill_value="",
    dtype="<U32"
)

# ------------------------------------------------------------
# 8. Populate tensors
# ------------------------------------------------------------

for window_index, window_row in (
    windows.iterrows()
):
    window_id = window_row["window_id"]

    if window_id not in membership_groups:
        raise KeyError(
            f"No membership records for {window_id}"
        )

    group = membership_groups[window_id]

    if len(group) > WINDOW_SIZE:
        raise ValueError(
            f"{window_id} exceeds WINDOW_SIZE."
        )

    positions = (
        group["position"]
        .to_numpy(dtype=np.int64)
    )

    if not np.array_equal(
        positions,
        np.arange(len(group))
    ):
        raise ValueError(
            f"Invalid positions in {window_id}"
        )

    row_indices = np.asarray(
        [
            evidence_index[evidence_id]
            for evidence_id
            in group["evidence_id"]
        ],
        dtype=np.int64
    )

    evidence_count = len(row_indices)

    X_network[
        window_index,
        :evidence_count
    ] = scaled_features[row_indices]

    attention_mask[
        window_index,
        :evidence_count
    ] = 1

    evidence_aspect_targets[
        window_index,
        :evidence_count
    ] = evidence_aspect_ids[row_indices]

    current_evidence_ids = (
        group["evidence_id"]
        .astype(str)
        .to_numpy()
    )

    window_evidence_ids[
        window_index,
        :evidence_count
    ] = current_evidence_ids

    timestamps = (
        evidence_timestamps[row_indices]
    )

    start_timestamp = timestamps[0]

    relative_seconds = (
        timestamps - start_timestamp
    )

    previous_delta = np.diff(
        timestamps,
        prepend=timestamps[0]
    )

    # Negative deltas should not occur after sorting.
    relative_seconds = np.maximum(
        relative_seconds,
        0.0
    )

    previous_delta = np.maximum(
        previous_delta,
        0.0
    )

    # Temporal features:
    # 0: log time from window start
    # 1: log time since previous evidence
    # 2: normalized position
    # 3: valid-evidence indicator
    X_temporal[
        window_index,
        :evidence_count,
        0
    ] = np.log1p(
        relative_seconds
    ).astype(np.float32)

    X_temporal[
        window_index,
        :evidence_count,
        1
    ] = np.log1p(
        previous_delta
    ).astype(np.float32)

    X_temporal[
        window_index,
        :evidence_count,
        2
    ] = (
        np.arange(
            evidence_count,
            dtype=np.float32
        )
        / max(WINDOW_SIZE - 1, 1)
    )

    X_temporal[
        window_index,
        :evidence_count,
        3
    ] = 1.0

    for aspect_index, aspect in enumerate(
        ASPECTS
    ):
        aspect_presence_targets[
            window_index,
            aspect_index
        ] = float(
            window_row[
                f"{aspect}_count"
            ] > 0
        )

    campaign_targets[window_index] = int(
        window_row["campaign_label"]
    )

    split_ids[window_index] = SPLIT_TO_ID[
        window_row["split"]
    ]

    window_ids.append(window_id)

# ------------------------------------------------------------
# 9. Tensor integrity checks
# ------------------------------------------------------------

assert np.isfinite(X_network).all()
assert np.isfinite(X_temporal).all()

assert np.array_equal(
    attention_mask.sum(axis=1),
    windows["evidence_count"].to_numpy()
)

assert (
    evidence_aspect_targets[
        attention_mask == 0
    ]
    == -100
).all()

assert (
    X_network[
        attention_mask == 0
    ]
    == 0
).all()

assert (
    X_temporal[
        attention_mask == 0
    ]
    == 0
).all()

print("\nTensor shapes:")
print("X_network:", X_network.shape)
print("X_temporal:", X_temporal.shape)
print("attention_mask:", attention_mask.shape)
print(
    "aspect_presence_targets:",
    aspect_presence_targets.shape
)
print("campaign_targets:", campaign_targets.shape)

# ------------------------------------------------------------
# 10. Partition indices
# ------------------------------------------------------------

train_indices = np.flatnonzero(
    split_ids == SPLIT_TO_ID["train"]
)

validation_indices = np.flatnonzero(
    split_ids == SPLIT_TO_ID["validation"]
)

test_indices = np.flatnonzero(
    split_ids == SPLIT_TO_ID["test"]
)

print("\nWindow indices:")
print("Train:", len(train_indices))
print("Validation:", len(validation_indices))
print("Test:", len(test_indices))

assert len(train_indices) == (
    windows["split"] == "train"
).sum()

assert len(validation_indices) == (
    windows["split"] == "validation"
).sum()

assert len(test_indices) == (
    windows["split"] == "test"
).sum()

# ------------------------------------------------------------
# 11. Save arrays
# ------------------------------------------------------------

np.save(
    TENSOR_DIR / "X_network.npy",
    X_network
)

np.save(
    TENSOR_DIR / "X_temporal.npy",
    X_temporal
)

np.save(
    TENSOR_DIR / "attention_mask.npy",
    attention_mask
)

np.save(
    TENSOR_DIR /
    "evidence_aspect_targets.npy",
    evidence_aspect_targets
)

np.save(
    TENSOR_DIR /
    "aspect_presence_targets.npy",
    aspect_presence_targets
)

np.save(
    TENSOR_DIR /
    "campaign_targets.npy",
    campaign_targets
)

np.save(
    TENSOR_DIR / "split_ids.npy",
    split_ids
)

np.save(
    TENSOR_DIR / "train_indices.npy",
    train_indices
)

np.save(
    TENSOR_DIR /
    "validation_indices.npy",
    validation_indices
)

np.save(
    TENSOR_DIR / "test_indices.npy",
    test_indices
)

np.save(
    TENSOR_DIR / "window_ids.npy",
    np.asarray(window_ids)
)

np.save(
    TENSOR_DIR /
    "window_evidence_ids.npy",
    window_evidence_ids
)

np.save(
    TENSOR_DIR /
    "evidence_ids.npy",
    evidence_ids
)

# Save text separately to avoid duplication inside windows
evidence_text = evidence[
    [
        "evidence_id",
        "split",
        "verbalized_event",
        "mask_network",
        "mask_temporal",
        "mask_verbalized_text",
        "mask_host",
        "mask_command",
        "mask_cti"
    ]
].copy()

evidence_text.to_parquet(
    TENSOR_DIR /
    "evidence_text_and_masks.parquet",
    index=False
)

# ------------------------------------------------------------
# 12. Save preprocessing objects and manifest
# ------------------------------------------------------------

with open(
    TENSOR_DIR / "robust_scaler.pkl",
    "wb"
) as file:
    pickle.dump(
        {
            "scaler": scaler,
            "train_medians": train_medians,
            "feature_columns": feature_columns,
            "clip_range": [-10.0, 10.0]
        },
        file
    )

tensor_manifest = {
    "window_size": WINDOW_SIZE,
    "number_of_windows": number_of_windows,
    "number_of_network_features":
        number_of_features,
    "number_of_temporal_features": 4,
    "network_shape": list(
        X_network.shape
    ),
    "temporal_shape": list(
        X_temporal.shape
    ),
    "attention_mask_shape": list(
        attention_mask.shape
    ),
    "aspects": ASPECTS,
    "aspect_to_id": ASPECT_TO_ID,
    "evidence_aspect_to_id":
        EVIDENCE_ASPECT_TO_ID,
    "split_to_id": SPLIT_TO_ID,
    "split_windows": {
        "train": int(len(train_indices)),
        "validation": int(
            len(validation_indices)
        ),
        "test": int(len(test_indices))
    },
    "preprocessing": {
        "imputation": "training_median",
        "scaling": "RobustScaler",
        "quantile_range": [5.0, 95.0],
        "clipping": [-10.0, 10.0],
        "fit_partition": "train"
    }
}

with open(
    TENSOR_DIR / "tensor_manifest.json",
    "w",
    encoding="utf-8"
) as file:
    json.dump(
        tensor_manifest,
        file,
        indent=2
    )

print("\nSaved AM-JEPA tensors to:")
print(TENSOR_DIR)

In [ ]:
import sys
import subprocess

print("Installing into:", sys.executable)

subprocess.check_call([
    sys.executable,
    "-m",
    "pip",
    "install",
    "sentence-transformers==3.4.1"
])

In [ ]:
import os
import ssl
import sys
import certifi

# Force the PyTorch backend
os.environ["USE_TF"] = "0"
os.environ["USE_TORCH"] = "1"
os.environ["TRANSFORMERS_NO_TF"] = "1"

# Force Python/Hugging Face to use certifi's valid CA bundle
certificate_bundle = certifi.where()

os.environ["SSL_CERT_FILE"] = certificate_bundle
os.environ["REQUESTS_CA_BUNDLE"] = certificate_bundle
os.environ["CURL_CA_BUNDLE"] = certificate_bundle

print("Python:", sys.executable)
print("OpenSSL:", ssl.OPENSSL_VERSION)
print("Certificate bundle:", certificate_bundle)
print(
    "Certificate exists:",
    os.path.exists(certificate_bundle)
)
print(
    "Certificate size:",
    os.path.getsize(certificate_bundle)
)

In [ ]:
import sys
import subprocess

subprocess.check_call([
    sys.executable,
    "-m",
    "pip",
    "install",
    "--upgrade",
    "--force-reinstall",
    "certifi"
])

In [ ]:
import os
import sys
import ssl
import certifi

os.environ["USE_TF"] = "0"
os.environ["USE_TORCH"] = "1"
os.environ["TRANSFORMERS_NO_TF"] = "1"

ca_bundle = certifi.where()

os.environ["SSL_CERT_FILE"] = ca_bundle
os.environ["REQUESTS_CA_BUNDLE"] = ca_bundle
os.environ["CURL_CA_BUNDLE"] = ca_bundle

print("Python:", sys.executable)
print("OpenSSL:", ssl.OPENSSL_VERSION)
print("CA bundle:", ca_bundle)
print("Exists:", os.path.isfile(ca_bundle))
print("Size:", os.path.getsize(ca_bundle))

In [ ]:
# ============================================================
# Cell 10: Offline leakage-safe text encoder
# TF-IDF + Truncated SVD
# ============================================================

from pathlib import Path
import json
import pickle
import sys

import numpy as np
import pandas as pd
import sklearn

from sklearn.decomposition import TruncatedSVD
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.preprocessing import normalize

DATA_DIR = Path(
    r"."
)

OUTPUT_DIR = DATA_DIR / "outputs"
TENSOR_DIR = OUTPUT_DIR / "am_jepa_tensors"

TEXT_PATH = (
    TENSOR_DIR /
    "evidence_text_and_masks.parquet"
)

WINDOW_EVIDENCE_PATH = (
    TENSOR_DIR /
    "window_evidence_ids.npy"
)

ATTENTION_MASK_PATH = (
    TENSOR_DIR /
    "attention_mask.npy"
)

SEED = 42
WINDOW_SIZE = 32
MAX_TFIDF_FEATURES = 4096
TARGET_TEXT_DIMENSION = 128

print("=" * 80)
print("OFFLINE AM-JEPA TEXT ENCODING")
print("=" * 80)

print("Python:", sys.version.split()[0])
print("scikit-learn:", sklearn.__version__)
print("Encoder: train-only TF-IDF + Truncated SVD")

# ------------------------------------------------------------
# 1. Load evidence text and window IDs
# ------------------------------------------------------------

evidence_text = pd.read_parquet(
    TEXT_PATH
)

window_evidence_ids = np.load(
    WINDOW_EVIDENCE_PATH,
    allow_pickle=False
)

attention_mask = np.load(
    ATTENTION_MASK_PATH,
    allow_pickle=False
)

required_columns = [
    "evidence_id",
    "split",
    "verbalized_event",
    "mask_network",
    "mask_temporal",
    "mask_verbalized_text",
    "mask_host",
    "mask_command",
    "mask_cti"
]

missing_columns = [
    column
    for column in required_columns
    if column not in evidence_text.columns
]

if missing_columns:
    raise ValueError(
        f"Missing columns: {missing_columns}"
    )

assert evidence_text["evidence_id"].is_unique

assert window_evidence_ids.shape == (
    4701,
    WINDOW_SIZE
)

assert attention_mask.shape == (
    4701,
    WINDOW_SIZE
)

print("Evidence records:", f"{len(evidence_text):,}")
print(
    "Window evidence-ID shape:",
    window_evidence_ids.shape
)

# ------------------------------------------------------------
# 2. Separate training text before fitting
# ------------------------------------------------------------

training_texts = (
    evidence_text.loc[
        evidence_text["split"] == "train",
        "verbalized_event"
    ]
    .drop_duplicates()
    .sort_values()
    .tolist()
)

all_unique_texts = (
    evidence_text["verbalized_event"]
    .drop_duplicates()
    .sort_values()
    .reset_index(drop=True)
)

print(
    "Unique training texts:",
    f"{len(training_texts):,}"
)

print(
    "All unique texts:",
    f"{len(all_unique_texts):,}"
)

# ------------------------------------------------------------
# 3. Fit TF-IDF using training text only
# ------------------------------------------------------------

vectorizer = TfidfVectorizer(
    lowercase=True,
    strip_accents="unicode",
    analyzer="word",
    ngram_range=(1, 2),
    min_df=2,
    max_features=MAX_TFIDF_FEATURES,
    sublinear_tf=True,
    norm="l2",
    dtype=np.float32
)

training_tfidf = vectorizer.fit_transform(
    training_texts
)

all_tfidf = vectorizer.transform(
    all_unique_texts.tolist()
)

print(
    "\nTraining TF-IDF shape:",
    training_tfidf.shape
)

print(
    "All-text TF-IDF shape:",
    all_tfidf.shape
)

vocabulary_size = len(
    vectorizer.vocabulary_
)

print("Vocabulary size:", vocabulary_size)

if vocabulary_size < 2:
    raise ValueError(
        "TF-IDF vocabulary is unexpectedly small."
    )

# ------------------------------------------------------------
# 4. Fit SVD using training text only
# ------------------------------------------------------------

svd_dimension = min(
    TARGET_TEXT_DIMENSION,
    training_tfidf.shape[0] - 1,
    training_tfidf.shape[1] - 1
)

if svd_dimension < 2:
    raise ValueError(
        "Insufficient TF-IDF rank for SVD."
    )

svd = TruncatedSVD(
    n_components=svd_dimension,
    algorithm="randomized",
    n_iter=10,
    random_state=SEED
)

svd.fit(training_tfidf)

all_embeddings = svd.transform(
    all_tfidf
).astype(np.float32)

# L2-normalize nonzero text embeddings
all_embeddings = normalize(
    all_embeddings,
    norm="l2",
    axis=1
).astype(np.float32)

assert np.isfinite(
    all_embeddings
).all()

print(
    "\nText embedding matrix:",
    all_embeddings.shape
)

print(
    "Explained variance ratio:",
    float(
        svd.explained_variance_ratio_.sum()
    )
)

# ------------------------------------------------------------
# 5. Create text vocabulary and padding embedding
# ------------------------------------------------------------

text_to_id = {
    text: index + 1
    for index, text
    in enumerate(all_unique_texts)
}

text_vocabulary = pd.DataFrame(
    {
        "text_id": np.arange(
            1,
            len(all_unique_texts) + 1,
            dtype=np.int32
        ),
        "verbalized_event":
            all_unique_texts.to_numpy()
    }
)

embedding_lookup = np.zeros(
    (
        len(all_unique_texts) + 1,
        svd_dimension
    ),
    dtype=np.float32
)

embedding_lookup[1:] = all_embeddings

assert np.all(
    embedding_lookup[0] == 0
)

# ------------------------------------------------------------
# 6. Map every evidence record to a text ID
# ------------------------------------------------------------

evidence_text["text_id"] = (
    evidence_text["verbalized_event"]
    .map(text_to_id)
    .astype(np.int32)
)

evidence_to_text_id = dict(
    zip(
        evidence_text["evidence_id"],
        evidence_text["text_id"]
    )
)

window_text_ids = np.zeros(
    window_evidence_ids.shape,
    dtype=np.int32
)

unknown_evidence_ids = []

for window_index in range(
    window_evidence_ids.shape[0]
):
    for position in range(
        window_evidence_ids.shape[1]
    ):
        evidence_id = window_evidence_ids[
            window_index,
            position
        ]

        if evidence_id == "":
            continue

        text_id = evidence_to_text_id.get(
            evidence_id
        )

        if text_id is None:
            unknown_evidence_ids.append(
                evidence_id
            )
        else:
            window_text_ids[
                window_index,
                position
            ] = text_id

if unknown_evidence_ids:
    raise ValueError(
        "Missing text IDs for evidence records: "
        f"{unknown_evidence_ids[:10]}"
    )

# ------------------------------------------------------------
# 7. Validate padding alignment
# ------------------------------------------------------------

assert np.array_equal(
    window_text_ids == 0,
    attention_mask == 0
)

valid_text_ids = window_text_ids[
    attention_mask == 1
]

assert valid_text_ids.min() >= 1
assert valid_text_ids.max() <= len(
    all_unique_texts
)

embedding_norms = np.linalg.norm(
    embedding_lookup[1:],
    axis=1
)

nonzero_norms = embedding_norms[
    embedding_norms > 0
]

print(
    "\nValid text positions:",
    f"{len(valid_text_ids):,}"
)

print(
    "Padding positions:",
    f"{(window_text_ids == 0).sum():,}"
)

print(
    "Zero text embeddings:",
    int((embedding_norms == 0).sum())
)

if len(nonzero_norms) > 0:
    print(
        "Mean nonzero embedding norm:",
        float(nonzero_norms.mean())
    )

# ------------------------------------------------------------
# 8. Check validation/test vocabulary coverage
# ------------------------------------------------------------

feature_names = set(
    vectorizer.get_feature_names_out()
)

coverage_records = []

for split_name in [
    "train",
    "validation",
    "test"
]:
    split_texts = (
        evidence_text.loc[
            evidence_text["split"] == split_name,
            "verbalized_event"
        ]
        .drop_duplicates()
    )

    transformed = vectorizer.transform(
        split_texts.tolist()
    )

    nonempty_rows = np.asarray(
        transformed.getnnz(axis=1) > 0
    )

    coverage_records.append({
        "split": split_name,
        "unique_texts": len(split_texts),
        "texts_with_known_terms": int(
            nonempty_rows.sum()
        ),
        "coverage": float(
            nonempty_rows.mean()
        )
    })

coverage_table = pd.DataFrame(
    coverage_records
)

print("\nText vocabulary coverage:")
display(coverage_table)

assert (
    coverage_table["coverage"] > 0.99
).all()

# ------------------------------------------------------------
# 9. Save compact text artifacts
# ------------------------------------------------------------

np.save(
    TENSOR_DIR /
    "text_embedding_lookup.npy",
    embedding_lookup
)

np.save(
    TENSOR_DIR /
    "window_text_ids.npy",
    window_text_ids
)

text_vocabulary.to_parquet(
    TENSOR_DIR /
    "text_vocabulary.parquet",
    index=False
)

evidence_text[
    [
        "evidence_id",
        "split",
        "text_id",
        "mask_network",
        "mask_temporal",
        "mask_verbalized_text",
        "mask_host",
        "mask_command",
        "mask_cti"
    ]
].to_parquet(
    TENSOR_DIR /
    "evidence_text_id_map.parquet",
    index=False
)

coverage_table.to_csv(
    TENSOR_DIR /
    "text_vocabulary_coverage.csv",
    index=False
)

with open(
    TENSOR_DIR /
    "tfidf_svd_text_encoder.pkl",
    "wb"
) as file:
    pickle.dump(
        {
            "vectorizer": vectorizer,
            "svd": svd,
            "fit_partition": "train",
            "seed": SEED
        },
        file
    )

text_manifest = {
    "encoder": "TF-IDF + TruncatedSVD",
    "role": "offline_initial_text_baseline",
    "fit_partition": "train",
    "tfidf": {
        "ngram_range": [1, 2],
        "min_df": 2,
        "maximum_features":
            MAX_TFIDF_FEATURES,
        "actual_vocabulary_size":
            vocabulary_size,
        "sublinear_tf": True
    },
    "svd": {
        "dimension": int(
            svd_dimension
        ),
        "explained_variance_ratio": float(
            svd.explained_variance_ratio_.sum()
        ),
        "random_state": SEED
    },
    "padding_text_id": 0,
    "unique_texts": int(
        len(all_unique_texts)
    ),
    "embedding_lookup_shape": list(
        embedding_lookup.shape
    ),
    "window_text_ids_shape": list(
        window_text_ids.shape
    ),
    "pretrained_encoder_status": (
        "deferred_due_to_local "
        "SentenceTransformers SSL conflict"
    )
}

with open(
    TENSOR_DIR /
    "text_embedding_manifest.json",
    "w",
    encoding="utf-8"
) as file:
    json.dump(
        text_manifest,
        file,
        indent=2
    )

print("\nSaved:")
print(
    TENSOR_DIR /
    "text_embedding_lookup.npy"
)
print(
    TENSOR_DIR /
    "window_text_ids.npy"
)
print(
    TENSOR_DIR /
    "tfidf_svd_text_encoder.pkl"
)

In [ ]:
# ============================================================
# Cell 11: Pooled multimodal linear baseline
# ============================================================

from pathlib import Path
import json
import pickle
import random

import numpy as np
import pandas as pd

from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score,
    average_precision_score,
    balanced_accuracy_score,
    classification_report,
    f1_score,
    matthews_corrcoef,
    precision_score,
    recall_score,
    roc_auc_score
)
from sklearn.multiclass import OneVsRestClassifier
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

# ------------------------------------------------------------
# 1. Configuration
# ------------------------------------------------------------

SEED = 42

random.seed(SEED)
np.random.seed(SEED)

DATA_DIR = Path(
    r"."
)

OUTPUT_DIR = DATA_DIR / "outputs"
TENSOR_DIR = OUTPUT_DIR / "am_jepa_tensors"
BASELINE_DIR = OUTPUT_DIR / "baseline_results"

BASELINE_DIR.mkdir(
    parents=True,
    exist_ok=True
)

ASPECTS = [
    "reconnaissance",
    "initial_access_foothold",
    "lateral_movement",
    "data_exfiltration"
]

# ------------------------------------------------------------
# 2. Load tensors
# ------------------------------------------------------------

X_network = np.load(
    TENSOR_DIR / "X_network.npy"
)

X_temporal = np.load(
    TENSOR_DIR / "X_temporal.npy"
)

attention_mask = np.load(
    TENSOR_DIR / "attention_mask.npy"
)

text_embedding_lookup = np.load(
    TENSOR_DIR / "text_embedding_lookup.npy"
)

window_text_ids = np.load(
    TENSOR_DIR / "window_text_ids.npy"
)

aspect_targets = np.load(
    TENSOR_DIR / "aspect_presence_targets.npy"
)

campaign_targets = np.load(
    TENSOR_DIR / "campaign_targets.npy"
)

train_indices = np.load(
    TENSOR_DIR / "train_indices.npy"
)

validation_indices = np.load(
    TENSOR_DIR / "validation_indices.npy"
)

test_indices = np.load(
    TENSOR_DIR / "test_indices.npy"
)

print("=" * 80)
print("POOLED MULTIMODAL BASELINE")
print("=" * 80)

print("Network:", X_network.shape)
print("Temporal:", X_temporal.shape)
print("Text lookup:", text_embedding_lookup.shape)
print("Text IDs:", window_text_ids.shape)
print("Mask:", attention_mask.shape)

# ------------------------------------------------------------
# 3. Masked mean-pooling
# ------------------------------------------------------------

def masked_mean(values, mask):
    mask_float = mask.astype(
        np.float32
    )[..., None]

    numerator = (
        values * mask_float
    ).sum(axis=1)

    denominator = np.maximum(
        mask_float.sum(axis=1),
        1.0
    )

    return numerator / denominator


network_pooled = masked_mean(
    X_network,
    attention_mask
)

temporal_pooled = masked_mean(
    X_temporal,
    attention_mask
)

window_text_embeddings = (
    text_embedding_lookup[
        window_text_ids
    ]
)

text_pooled = masked_mean(
    window_text_embeddings,
    attention_mask
)

X_pooled = np.concatenate(
    [
        network_pooled,
        temporal_pooled,
        text_pooled
    ],
    axis=1
).astype(np.float32)

assert np.isfinite(X_pooled).all()

print(
    "\nPooled representation:",
    X_pooled.shape
)

# ------------------------------------------------------------
# 4. Partition arrays
# ------------------------------------------------------------

X_train = X_pooled[train_indices]
X_validation = X_pooled[
    validation_indices
]
X_test = X_pooled[test_indices]

y_campaign_train = campaign_targets[
    train_indices
]
y_campaign_validation = campaign_targets[
    validation_indices
]
y_campaign_test = campaign_targets[
    test_indices
]

y_aspect_train = aspect_targets[
    train_indices
]
y_aspect_validation = aspect_targets[
    validation_indices
]
y_aspect_test = aspect_targets[
    test_indices
]

print("\nPartitions:")
print("Train:", X_train.shape)
print("Validation:", X_validation.shape)
print("Test:", X_test.shape)

# ------------------------------------------------------------
# 5. Campaign-detection baseline
# ------------------------------------------------------------

campaign_model = Pipeline(
    [
        (
            "scaler",
            StandardScaler()
        ),
        (
            "classifier",
            LogisticRegression(
                solver="liblinear",
                class_weight="balanced",
                max_iter=3000,
                random_state=SEED
            )
        )
    ]
)

campaign_model.fit(
    X_train,
    y_campaign_train
)

# ------------------------------------------------------------
# 6. Campaign evaluation
# ------------------------------------------------------------

def evaluate_campaign(
    model,
    features,
    labels,
    split_name
):
    probabilities = model.predict_proba(
        features
    )[:, 1]

    predictions = (
        probabilities >= 0.5
    ).astype(np.int64)

    metrics = {
        "split": split_name,
        "accuracy": accuracy_score(
            labels,
            predictions
        ),
        "balanced_accuracy":
            balanced_accuracy_score(
                labels,
                predictions
            ),
        "precision": precision_score(
            labels,
            predictions,
            zero_division=0
        ),
        "recall": recall_score(
            labels,
            predictions,
            zero_division=0
        ),
        "f1": f1_score(
            labels,
            predictions,
            zero_division=0
        ),
        "mcc": matthews_corrcoef(
            labels,
            predictions
        ),
        "auroc": roc_auc_score(
            labels,
            probabilities
        ),
        "auprc": average_precision_score(
            labels,
            probabilities
        )
    }

    return metrics, probabilities, predictions


campaign_validation_metrics, _, _ = (
    evaluate_campaign(
        campaign_model,
        X_validation,
        y_campaign_validation,
        "validation"
    )
)

campaign_test_metrics, (
    campaign_test_probabilities
), campaign_test_predictions = (
    evaluate_campaign(
        campaign_model,
        X_test,
        y_campaign_test,
        "test"
    )
)

campaign_results = pd.DataFrame(
    [
        campaign_validation_metrics,
        campaign_test_metrics
    ]
)

print("\nCampaign-detection results:")
display(campaign_results)

print("\nCampaign test classification report:")
print(
    classification_report(
        y_campaign_test,
        campaign_test_predictions,
        digits=4,
        zero_division=0
    )
)

# ------------------------------------------------------------
# 7. Multilabel aspect-presence baseline
# ------------------------------------------------------------

aspect_base_estimator = Pipeline(
    [
        (
            "scaler",
            StandardScaler()
        ),
        (
            "classifier",
            LogisticRegression(
                solver="liblinear",
                class_weight="balanced",
                max_iter=3000,
                random_state=SEED
            )
        )
    ]
)

aspect_model = OneVsRestClassifier(
    aspect_base_estimator,
    n_jobs=-1
)

aspect_model.fit(
    X_train,
    y_aspect_train
)

# ------------------------------------------------------------
# 8. Aspect evaluation
# ------------------------------------------------------------

def evaluate_aspects(
    model,
    features,
    labels,
    split_name
):
    probabilities = model.predict_proba(
        features
    )

    predictions = (
        probabilities >= 0.5
    ).astype(np.int64)

    rows = []

    for aspect_index, aspect in enumerate(
        ASPECTS
    ):
        y_true = labels[
            :,
            aspect_index
        ]

        y_probability = probabilities[
            :,
            aspect_index
        ]

        y_prediction = predictions[
            :,
            aspect_index
        ]

        positives = int(
            y_true.sum()
        )

        row = {
            "split": split_name,
            "aspect": aspect,
            "positive_windows": positives,
            "precision": precision_score(
                y_true,
                y_prediction,
                zero_division=0
            ),
            "recall": recall_score(
                y_true,
                y_prediction,
                zero_division=0
            ),
            "f1": f1_score(
                y_true,
                y_prediction,
                zero_division=0
            ),
            "mcc": matthews_corrcoef(
                y_true,
                y_prediction
            )
        }

        if len(np.unique(y_true)) == 2:
            row["auroc"] = roc_auc_score(
                y_true,
                y_probability
            )

            row["auprc"] = (
                average_precision_score(
                    y_true,
                    y_probability
                )
            )
        else:
            row["auroc"] = np.nan
            row["auprc"] = np.nan

        rows.append(row)

    macro_metrics = {
        "split": split_name,
        "aspect": "macro",
        "positive_windows": int(
            labels.sum()
        ),
        "precision": precision_score(
            labels,
            predictions,
            average="macro",
            zero_division=0
        ),
        "recall": recall_score(
            labels,
            predictions,
            average="macro",
            zero_division=0
        ),
        "f1": f1_score(
            labels,
            predictions,
            average="macro",
            zero_division=0
        ),
        "mcc": np.nan,
        "auroc": roc_auc_score(
            labels,
            probabilities,
            average="macro"
        ),
        "auprc": average_precision_score(
            labels,
            probabilities,
            average="macro"
        )
    }

    rows.append(macro_metrics)

    return (
        pd.DataFrame(rows),
        probabilities,
        predictions
    )


aspect_validation_results, _, _ = (
    evaluate_aspects(
        aspect_model,
        X_validation,
        y_aspect_validation,
        "validation"
    )
)

aspect_test_results, (
    aspect_test_probabilities
), aspect_test_predictions = (
    evaluate_aspects(
        aspect_model,
        X_test,
        y_aspect_test,
        "test"
    )
)

aspect_results = pd.concat(
    [
        aspect_validation_results,
        aspect_test_results
    ],
    ignore_index=True
)

print("\nAspect-presence results:")
display(aspect_results)

# ------------------------------------------------------------
# 9. Save predictions and models
# ------------------------------------------------------------

window_ids = np.load(
    TENSOR_DIR / "window_ids.npy"
)

test_predictions = pd.DataFrame(
    {
        "window_id":
            window_ids[test_indices],
        "campaign_true":
            y_campaign_test,
        "campaign_probability":
            campaign_test_probabilities,
        "campaign_prediction":
            campaign_test_predictions
    }
)

for aspect_index, aspect in enumerate(
    ASPECTS
):
    test_predictions[
        f"{aspect}_true"
    ] = y_aspect_test[
        :,
        aspect_index
    ]

    test_predictions[
        f"{aspect}_probability"
    ] = aspect_test_probabilities[
        :,
        aspect_index
    ]

    test_predictions[
        f"{aspect}_prediction"
    ] = aspect_test_predictions[
        :,
        aspect_index
    ]

campaign_results.to_csv(
    BASELINE_DIR /
    "pooled_campaign_results.csv",
    index=False
)

aspect_results.to_csv(
    BASELINE_DIR /
    "pooled_aspect_results.csv",
    index=False
)

test_predictions.to_csv(
    BASELINE_DIR /
    "pooled_test_predictions.csv",
    index=False
)

np.save(
    BASELINE_DIR /
    "pooled_multimodal_features.npy",
    X_pooled
)

with open(
    BASELINE_DIR /
    "pooled_campaign_model.pkl",
    "wb"
) as file:
    pickle.dump(
        campaign_model,
        file
    )

with open(
    BASELINE_DIR /
    "pooled_aspect_model.pkl",
    "wb"
) as file:
    pickle.dump(
        aspect_model,
        file
    )

baseline_manifest = {
    "seed": SEED,
    "representation": (
        "masked mean pooling of network, "
        "temporal, and offline text embeddings"
    ),
    "representation_dimension": int(
        X_pooled.shape[1]
    ),
    "campaign_classifier":
        "balanced logistic regression",
    "aspect_classifier":
        "one-vs-rest balanced logistic regression",
    "decision_threshold": 0.5,
    "test_evaluated": True,
    "warning": (
        "Data-exfiltration metrics are exploratory "
        "because the test set contains only two "
        "supporting windows."
    )
}

with open(
    BASELINE_DIR /
    "pooled_baseline_manifest.json",
    "w",
    encoding="utf-8"
) as file:
    json.dump(
        baseline_manifest,
        file,
        indent=2
    )

print("\nSaved baseline results to:")
print(BASELINE_DIR)

In [ ]:
# =============================================================================
# CELL 12: VALIDATION-ONLY DECISION-THRESHOLD CALIBRATION
# =============================================================================

from pathlib import Path

import numpy as np
import pandas as pd

from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    matthews_corrcoef,
    roc_auc_score,
    average_precision_score,
    confusion_matrix,
)


# -----------------------------------------------------------------------------
# Configuration
# -----------------------------------------------------------------------------

DATA_DIR = Path(
    r"."
)

BASELINE_OUTPUT_DIR = (
    DATA_DIR
    / "outputs"
    / "baseline_results"
)

BASELINE_OUTPUT_DIR.mkdir(
    parents=True,
    exist_ok=True,
)

THRESHOLD_GRID = np.linspace(
    0.01,
    0.99,
    99,
)


# IMPORTANT:
# This order must match the columns in aspect_presence_targets.npy.
ASPECT_NAMES = [
    "reconnaissance",
    "initial_access_foothold",
    "lateral_movement",
    "data_exfiltration",
]


# -----------------------------------------------------------------------------
# Input verification
# -----------------------------------------------------------------------------

required_variables = [
    "campaign_model",
    "aspect_model",
    "X_validation",
    "X_test",
    "y_campaign_validation",
    "y_campaign_test",
    "y_aspect_validation",
    "y_aspect_test",
]

missing_variables = [
    variable_name
    for variable_name in required_variables
    if variable_name not in globals()
]

if missing_variables:
    raise NameError(
        "The following variables are missing:\n"
        + "\n".join(missing_variables)
        + "\n\nRun the pooled multimodal baseline cell first."
    )

X_validation = np.asarray(X_validation)
X_test = np.asarray(X_test)

y_campaign_validation = np.asarray(
    y_campaign_validation
).reshape(-1)

y_campaign_test = np.asarray(
    y_campaign_test
).reshape(-1)

y_aspect_validation = np.asarray(
    y_aspect_validation
)

y_aspect_test = np.asarray(
    y_aspect_test
)

assert y_aspect_validation.ndim == 2
assert y_aspect_test.ndim == 2

assert (
    y_aspect_validation.shape[1]
    == len(ASPECT_NAMES)
), (
    "Validation aspect-target columns do not match "
    "ASPECT_NAMES."
)

assert (
    y_aspect_test.shape[1]
    == len(ASPECT_NAMES)
), (
    "Test aspect-target columns do not match "
    "ASPECT_NAMES."
)

assert (
    X_validation.shape[0]
    == y_campaign_validation.shape[0]
    == y_aspect_validation.shape[0]
)

assert (
    X_test.shape[0]
    == y_campaign_test.shape[0]
    == y_aspect_test.shape[0]
)

print("=" * 80)
print("VALIDATION-ONLY THRESHOLD CALIBRATION")
print("=" * 80)

print("Aspect order:", ASPECT_NAMES)
print("Validation representation:", X_validation.shape)
print("Test representation:", X_test.shape)
print("Validation aspect targets:", y_aspect_validation.shape)
print("Test aspect targets:", y_aspect_test.shape)


# -----------------------------------------------------------------------------
# Metric helpers
# -----------------------------------------------------------------------------

def safe_probability_metric(
    metric_function,
    y_true,
    probabilities,
):
    """
    Return NaN if a probability metric cannot be computed,
    for example when only one target class is present.
    """
    try:
        return float(
            metric_function(
                y_true,
                probabilities,
            )
        )
    except ValueError:
        return np.nan


def evaluate_binary_predictions(
    y_true,
    probabilities,
    threshold,
):
    """
    Evaluate binary predictions at a specified decision threshold.
    """
    y_true = np.asarray(y_true).astype(np.int64)
    probabilities = np.asarray(probabilities)

    predictions = (
        probabilities >= threshold
    ).astype(np.int64)

    tn, fp, fn, tp = confusion_matrix(
        y_true,
        predictions,
        labels=[0, 1],
    ).ravel()

    return {
        "threshold": float(threshold),
        "accuracy": accuracy_score(
            y_true,
            predictions,
        ),
        "balanced_accuracy": balanced_accuracy_score(
            y_true,
            predictions,
        ),
        "precision": precision_score(
            y_true,
            predictions,
            zero_division=0,
        ),
        "recall": recall_score(
            y_true,
            predictions,
            zero_division=0,
        ),
        "f1": f1_score(
            y_true,
            predictions,
            zero_division=0,
        ),
        "mcc": matthews_corrcoef(
            y_true,
            predictions,
        ),
        "auroc": safe_probability_metric(
            roc_auc_score,
            y_true,
            probabilities,
        ),
        "auprc": safe_probability_metric(
            average_precision_score,
            y_true,
            probabilities,
        ),
        "tn": int(tn),
        "fp": int(fp),
        "fn": int(fn),
        "tp": int(tp),
    }


def select_validation_threshold(
    y_true,
    probabilities,
    objective="mcc",
):
    """
    Select a threshold using validation data only.

    Tie-breaking order:
      1. highest selected objective;
      2. highest balanced accuracy;
      3. highest F1;
      4. threshold closest to 0.5.
    """
    y_true = np.asarray(y_true).astype(np.int64)
    probabilities = np.asarray(probabilities)

    rows = []

    for threshold in THRESHOLD_GRID:
        predictions = (
            probabilities >= threshold
        ).astype(np.int64)

        rows.append(
            {
                "threshold": float(threshold),
                "precision": precision_score(
                    y_true,
                    predictions,
                    zero_division=0,
                ),
                "recall": recall_score(
                    y_true,
                    predictions,
                    zero_division=0,
                ),
                "f1": f1_score(
                    y_true,
                    predictions,
                    zero_division=0,
                ),
                "balanced_accuracy":
                    balanced_accuracy_score(
                        y_true,
                        predictions,
                    ),
                "mcc": matthews_corrcoef(
                    y_true,
                    predictions,
                ),
            }
        )

    search_table = pd.DataFrame(rows)

    search_table["distance_from_0.5"] = (
        search_table["threshold"] - 0.5
    ).abs()

    best_row = (
        search_table
        .sort_values(
            by=[
                objective,
                "balanced_accuracy",
                "f1",
                "distance_from_0.5",
            ],
            ascending=[
                False,
                False,
                False,
                True,
            ],
        )
        .iloc[0]
    )

    return (
        float(best_row["threshold"]),
        search_table,
    )


# -----------------------------------------------------------------------------
# Campaign probability prediction
# -----------------------------------------------------------------------------

campaign_validation_probabilities = (
    campaign_model.predict_proba(
        X_validation
    )[:, 1]
)

campaign_test_probabilities = (
    campaign_model.predict_proba(
        X_test
    )[:, 1]
)


# -----------------------------------------------------------------------------
# Campaign threshold selection using validation only
# -----------------------------------------------------------------------------

(
    campaign_threshold,
    campaign_threshold_search,
) = select_validation_threshold(
    y_true=y_campaign_validation,
    probabilities=campaign_validation_probabilities,
    objective="mcc",
)

campaign_fixed_result = (
    evaluate_binary_predictions(
        y_true=y_campaign_test,
        probabilities=campaign_test_probabilities,
        threshold=0.5,
    )
)

campaign_calibrated_result = (
    evaluate_binary_predictions(
        y_true=y_campaign_test,
        probabilities=campaign_test_probabilities,
        threshold=campaign_threshold,
    )
)

campaign_results = pd.DataFrame(
    [
        {
            "split": "test",
            "decision_rule": "fixed_0.5",
            **campaign_fixed_result,
        },
        {
            "split": "test",
            "decision_rule":
                "validation_calibrated_mcc",
            **campaign_calibrated_result,
        },
    ]
)


# -----------------------------------------------------------------------------
# Aspect probability prediction
# -----------------------------------------------------------------------------

aspect_validation_probabilities = np.asarray(
    aspect_model.predict_proba(
        X_validation
    )
)

aspect_test_probabilities = np.asarray(
    aspect_model.predict_proba(
        X_test
    )
)

assert (
    aspect_validation_probabilities.shape
    == y_aspect_validation.shape
), (
    "Validation probability and target shapes differ: "
    f"{aspect_validation_probabilities.shape} versus "
    f"{y_aspect_validation.shape}"
)

assert (
    aspect_test_probabilities.shape
    == y_aspect_test.shape
), (
    "Test probability and target shapes differ: "
    f"{aspect_test_probabilities.shape} versus "
    f"{y_aspect_test.shape}"
)


# -----------------------------------------------------------------------------
# Per-aspect threshold selection
# -----------------------------------------------------------------------------

aspect_threshold_rows = []
aspect_test_rows = []

for aspect_index, aspect_name in enumerate(
    ASPECT_NAMES
):
    validation_labels = (
        y_aspect_validation[:, aspect_index]
        .astype(np.int64)
    )

    test_labels = (
        y_aspect_test[:, aspect_index]
        .astype(np.int64)
    )

    validation_probabilities = (
        aspect_validation_probabilities[
            :,
            aspect_index,
        ]
    )

    test_probabilities = (
        aspect_test_probabilities[
            :,
            aspect_index,
        ]
    )

    validation_positive_count = int(
        validation_labels.sum()
    )

    test_positive_count = int(
        test_labels.sum()
    )

    # A reliable threshold should not be estimated from
    # extremely few validation positives.
    if validation_positive_count >= 5:
        (
            selected_threshold,
            threshold_search,
        ) = select_validation_threshold(
            y_true=validation_labels,
            probabilities=validation_probabilities,
            objective="mcc",
        )

        calibration_status = (
            "validation_calibrated_mcc"
        )

        threshold_search.to_csv(
            BASELINE_OUTPUT_DIR
            / (
                f"{aspect_name}"
                "_threshold_search.csv"
            ),
            index=False,
        )

    else:
        selected_threshold = 0.5

        calibration_status = (
            "fixed_0.5_insufficient_validation_positives"
        )

    aspect_threshold_rows.append(
        {
            "aspect_index": aspect_index,
            "aspect": aspect_name,
            "validation_positive_windows":
                validation_positive_count,
            "test_positive_windows":
                test_positive_count,
            "selected_threshold":
                selected_threshold,
            "calibration_status":
                calibration_status,
        }
    )

    fixed_result = evaluate_binary_predictions(
        y_true=test_labels,
        probabilities=test_probabilities,
        threshold=0.5,
    )

    selected_result = evaluate_binary_predictions(
        y_true=test_labels,
        probabilities=test_probabilities,
        threshold=selected_threshold,
    )

    aspect_test_rows.append(
        {
            "split": "test",
            "aspect": aspect_name,
            "positive_windows":
                test_positive_count,
            "decision_rule": "fixed_0.5",
            **fixed_result,
        }
    )

    aspect_test_rows.append(
        {
            "split": "test",
            "aspect": aspect_name,
            "positive_windows":
                test_positive_count,
            "decision_rule":
                calibration_status,
            **selected_result,
        }
    )


aspect_thresholds = pd.DataFrame(
    aspect_threshold_rows
)

aspect_calibrated_results = pd.DataFrame(
    aspect_test_rows
)


# -----------------------------------------------------------------------------
# Macro summary
# -----------------------------------------------------------------------------

calibrated_aspect_rows = (
    aspect_calibrated_results[
        aspect_calibrated_results[
            "decision_rule"
        ] != "fixed_0.5"
    ]
)

fixed_aspect_rows = (
    aspect_calibrated_results[
        aspect_calibrated_results[
            "decision_rule"
        ] == "fixed_0.5"
    ]
)

aspect_macro_summary = pd.DataFrame(
    [
        {
            "decision_rule": "fixed_0.5",
            "macro_precision":
                fixed_aspect_rows["precision"].mean(),
            "macro_recall":
                fixed_aspect_rows["recall"].mean(),
            "macro_f1":
                fixed_aspect_rows["f1"].mean(),
            "macro_mcc":
                fixed_aspect_rows["mcc"].mean(),
            "macro_auroc":
                fixed_aspect_rows["auroc"].mean(),
            "macro_auprc":
                fixed_aspect_rows["auprc"].mean(),
        },
        {
            "decision_rule":
                "validation_selected",
            "macro_precision":
                calibrated_aspect_rows[
                    "precision"
                ].mean(),
            "macro_recall":
                calibrated_aspect_rows[
                    "recall"
                ].mean(),
            "macro_f1":
                calibrated_aspect_rows[
                    "f1"
                ].mean(),
            "macro_mcc":
                calibrated_aspect_rows[
                    "mcc"
                ].mean(),
            "macro_auroc":
                calibrated_aspect_rows[
                    "auroc"
                ].mean(),
            "macro_auprc":
                calibrated_aspect_rows[
                    "auprc"
                ].mean(),
        },
    ]
)


# -----------------------------------------------------------------------------
# Save results
# -----------------------------------------------------------------------------

campaign_threshold_search.to_csv(
    BASELINE_OUTPUT_DIR
    / "campaign_threshold_search.csv",
    index=False,
)

campaign_results.to_csv(
    BASELINE_OUTPUT_DIR
    / "campaign_threshold_calibration_results.csv",
    index=False,
)

aspect_thresholds.to_csv(
    BASELINE_OUTPUT_DIR
    / "aspect_selected_thresholds.csv",
    index=False,
)

aspect_calibrated_results.to_csv(
    BASELINE_OUTPUT_DIR
    / "aspect_threshold_calibration_results.csv",
    index=False,
)

aspect_macro_summary.to_csv(
    BASELINE_OUTPUT_DIR
    / "aspect_threshold_macro_summary.csv",
    index=False,
)


# -----------------------------------------------------------------------------
# Display results
# -----------------------------------------------------------------------------

print("\nSelected campaign threshold:")
print(round(campaign_threshold, 4))

print("\nCampaign test comparison:")
display(campaign_results.round(6))

print("\nSelected aspect thresholds:")
display(aspect_thresholds)

print("\nPer-aspect test comparison:")
display(aspect_calibrated_results.round(6))

print("\nAspect macro comparison:")
display(aspect_macro_summary.round(6))

print("\nImportant interpretation:")
print(
    "- All thresholds were selected using validation data only."
)

print(
    "- The test set was used once for final evaluation."
)

print(
    "- Data-exfiltration retains threshold 0.5 because "
    "validation contains fewer than five positive windows."
)

print("\nSaved calibration results to:")
print(BASELINE_OUTPUT_DIR)

# =============================================================================
# CELL 13: MODALITY ABLATION STUDY
# =============================================================================

In [ ]:
# =============================================================================
# CELL 13: STANDARDIZED MODALITY ABLATION
# =============================================================================

from pathlib import Path
import time
import warnings

import numpy as np
import pandas as pd

from sklearn.exceptions import ConvergenceWarning
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    matthews_corrcoef,
    roc_auc_score,
    average_precision_score,
)
from sklearn.multiclass import OneVsRestClassifier
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

warnings.filterwarnings(
    "ignore",
    category=ConvergenceWarning,
)


# -----------------------------------------------------------------------------
# Configuration
# -----------------------------------------------------------------------------

SEED = 42

DATA_DIR = Path(
    r"."
)

TENSOR_DIR = (
    DATA_DIR
    / "outputs"
    / "am_jepa_tensors"
)

# Preserve the previous unstandardized results.
OUTPUT_DIR = (
    DATA_DIR
    / "outputs"
    / "modality_ablation_scaled"
)

OUTPUT_DIR.mkdir(
    parents=True,
    exist_ok=True,
)

ASPECT_NAMES = [
    "reconnaissance",
    "initial_access_foothold",
    "lateral_movement",
    "data_exfiltration",
]

THRESHOLD_GRID = np.linspace(
    0.01,
    0.99,
    99,
)


# -----------------------------------------------------------------------------
# Flexible tensor loader
# -----------------------------------------------------------------------------

def load_first_existing(directory, candidate_names):
    for filename in candidate_names:
        path = directory / filename

        if path.exists():
            print(f"Loaded: {path.name}")
            return np.load(
                path,
                allow_pickle=False,
            )

    raise FileNotFoundError(
        "None of the following files were found:\n"
        + "\n".join(candidate_names)
    )


# -----------------------------------------------------------------------------
# Load prepared tensors
# -----------------------------------------------------------------------------

X_network = load_first_existing(
    TENSOR_DIR,
    [
        "X_network.npy",
        "x_network.npy",
    ],
).astype(np.float32)

X_temporal = load_first_existing(
    TENSOR_DIR,
    [
        "X_temporal.npy",
        "x_temporal.npy",
    ],
).astype(np.float32)

attention_mask = load_first_existing(
    TENSOR_DIR,
    [
        "attention_mask.npy",
        "window_attention_mask.npy",
    ],
).astype(np.float32)

text_embedding_lookup = load_first_existing(
    TENSOR_DIR,
    [
        "text_embedding_lookup.npy",
    ],
).astype(np.float32)

window_text_ids = load_first_existing(
    TENSOR_DIR,
    [
        "window_text_ids.npy",
    ],
).astype(np.int64)

aspect_targets = load_first_existing(
    TENSOR_DIR,
    [
        "aspect_presence_targets.npy",
        "y_aspect.npy",
    ],
).astype(np.int64)

campaign_targets = load_first_existing(
    TENSOR_DIR,
    [
        "campaign_targets.npy",
        "y_campaign.npy",
    ],
).astype(np.int64)

train_indices = load_first_existing(
    TENSOR_DIR,
    [
        "train_indices.npy",
        "train_idx.npy",
    ],
).astype(np.int64)

validation_indices = load_first_existing(
    TENSOR_DIR,
    [
        "validation_indices.npy",
        "validation_idx.npy",
        "val_indices.npy",
        "val_idx.npy",
    ],
).astype(np.int64)

test_indices = load_first_existing(
    TENSOR_DIR,
    [
        "test_indices.npy",
        "test_idx.npy",
    ],
).astype(np.int64)


# -----------------------------------------------------------------------------
# Structural verification
# -----------------------------------------------------------------------------

number_of_windows = X_network.shape[0]
window_size = X_network.shape[1]

assert X_temporal.shape[:2] == (
    number_of_windows,
    window_size,
)

assert attention_mask.shape == (
    number_of_windows,
    window_size,
)

assert window_text_ids.shape == (
    number_of_windows,
    window_size,
)

assert aspect_targets.shape == (
    number_of_windows,
    len(ASPECT_NAMES),
)

assert campaign_targets.shape == (
    number_of_windows,
)

assert window_text_ids.min() >= 0

assert window_text_ids.max() < len(
    text_embedding_lookup
)

all_indices = np.concatenate(
    [
        train_indices,
        validation_indices,
        test_indices,
    ]
)

assert len(all_indices) == number_of_windows

assert len(np.unique(all_indices)) == (
    number_of_windows
)

print("\nTensor shapes:")
print("Network:", X_network.shape)
print("Temporal:", X_temporal.shape)
print("Text lookup:", text_embedding_lookup.shape)
print("Text IDs:", window_text_ids.shape)
print("Mask:", attention_mask.shape)

print("\nPartitions:")
print("Train:", len(train_indices))
print("Validation:", len(validation_indices))
print("Test:", len(test_indices))


# -----------------------------------------------------------------------------
# Mask-aware window pooling
# -----------------------------------------------------------------------------

def masked_mean_pool(sequence_tensor, mask):
    expanded_mask = (
        mask.astype(np.float32)[..., None]
    )

    masked_sum = (
        sequence_tensor * expanded_mask
    ).sum(axis=1)

    valid_counts = expanded_mask.sum(axis=1)

    valid_counts = np.clip(
        valid_counts,
        a_min=1.0,
        a_max=None,
    )

    return (
        masked_sum / valid_counts
    ).astype(np.float32)


X_text = text_embedding_lookup[
    window_text_ids
]

network_pooled = masked_mean_pool(
    X_network,
    attention_mask,
)

temporal_pooled = masked_mean_pool(
    X_temporal,
    attention_mask,
)

text_pooled = masked_mean_pool(
    X_text,
    attention_mask,
)

for name, representation in {
    "network": network_pooled,
    "temporal": temporal_pooled,
    "text": text_pooled,
}.items():
    assert np.isfinite(
        representation
    ).all(), (
        f"{name} representation contains "
        "NaN or infinite values."
    )

print("\nPooled representations:")
print("Network:", network_pooled.shape)
print("Temporal:", temporal_pooled.shape)
print("Text:", text_pooled.shape)


# -----------------------------------------------------------------------------
# Modality configurations
# -----------------------------------------------------------------------------

MODALITY_CONFIGURATIONS = {
    "network_only":
        network_pooled,

    "temporal_only":
        temporal_pooled,

    "text_only":
        text_pooled,

    "network_temporal":
        np.concatenate(
            [
                network_pooled,
                temporal_pooled,
            ],
            axis=1,
        ),

    "network_text":
        np.concatenate(
            [
                network_pooled,
                text_pooled,
            ],
            axis=1,
        ),

    "full_multimodal":
        np.concatenate(
            [
                network_pooled,
                temporal_pooled,
                text_pooled,
            ],
            axis=1,
        ),
}


# -----------------------------------------------------------------------------
# Metric functions
# -----------------------------------------------------------------------------

def safe_probability_metric(
    metric_function,
    y_true,
    probabilities,
):
    try:
        return float(
            metric_function(
                y_true,
                probabilities,
            )
        )
    except ValueError:
        return np.nan


def calculate_binary_metrics(
    y_true,
    probabilities,
    threshold,
):
    predictions = (
        probabilities >= threshold
    ).astype(np.int64)

    return {
        "threshold": float(threshold),

        "accuracy": accuracy_score(
            y_true,
            predictions,
        ),

        "balanced_accuracy":
            balanced_accuracy_score(
                y_true,
                predictions,
            ),

        "precision": precision_score(
            y_true,
            predictions,
            zero_division=0,
        ),

        "recall": recall_score(
            y_true,
            predictions,
            zero_division=0,
        ),

        "f1": f1_score(
            y_true,
            predictions,
            zero_division=0,
        ),

        "mcc": matthews_corrcoef(
            y_true,
            predictions,
        ),

        "auroc": safe_probability_metric(
            roc_auc_score,
            y_true,
            probabilities,
        ),

        "auprc": safe_probability_metric(
            average_precision_score,
            y_true,
            probabilities,
        ),
    }


def select_validation_threshold(
    y_true,
    probabilities,
):
    search_rows = []

    for threshold in THRESHOLD_GRID:
        predictions = (
            probabilities >= threshold
        ).astype(np.int64)

        search_rows.append(
            {
                "threshold":
                    float(threshold),

                "mcc":
                    matthews_corrcoef(
                        y_true,
                        predictions,
                    ),

                "balanced_accuracy":
                    balanced_accuracy_score(
                        y_true,
                        predictions,
                    ),

                "f1":
                    f1_score(
                        y_true,
                        predictions,
                        zero_division=0,
                    ),

                "distance_from_0.5":
                    abs(threshold - 0.5),
            }
        )

    search_table = pd.DataFrame(
        search_rows
    )

    best_row = (
        search_table
        .sort_values(
            by=[
                "mcc",
                "balanced_accuracy",
                "f1",
                "distance_from_0.5",
            ],
            ascending=[
                False,
                False,
                False,
                True,
            ],
        )
        .iloc[0]
    )

    return float(
        best_row["threshold"]
    )


# -----------------------------------------------------------------------------
# Leakage-safe standardized classifier
# -----------------------------------------------------------------------------

def build_logistic_model():
    """
    StandardScaler is fitted only when fit() is called on the
    training partition. Validation and test statistics cannot
    influence feature standardization.
    """
    return Pipeline(
        steps=[
            (
                "standardizer",
                StandardScaler(),
            ),
            (
                "classifier",
                LogisticRegression(
                    class_weight="balanced",
                    solver="lbfgs",
                    max_iter=5000,
                    random_state=SEED,
                ),
            ),
        ]
    )


# -----------------------------------------------------------------------------
# Partition targets
# -----------------------------------------------------------------------------

y_campaign_train = campaign_targets[
    train_indices
]

y_campaign_validation = campaign_targets[
    validation_indices
]

y_campaign_test = campaign_targets[
    test_indices
]

y_aspect_train = aspect_targets[
    train_indices
]

y_aspect_validation = aspect_targets[
    validation_indices
]

y_aspect_test = aspect_targets[
    test_indices
]


# -----------------------------------------------------------------------------
# Run modality ablations
# -----------------------------------------------------------------------------

campaign_rows = []
aspect_rows = []
runtime_rows = []

for configuration_name, representation in (
    MODALITY_CONFIGURATIONS.items()
):
    print("\n" + "=" * 80)
    print(
        "Configuration:",
        configuration_name,
    )
    print(
        "Feature dimension:",
        representation.shape[1],
    )

    X_train_configuration = representation[
        train_indices
    ]

    X_validation_configuration = representation[
        validation_indices
    ]

    X_test_configuration = representation[
        test_indices
    ]

    # -----------------------------------------------------------------
    # Campaign classification
    # -----------------------------------------------------------------

    campaign_start = time.perf_counter()

    campaign_classifier = (
        build_logistic_model()
    )

    campaign_classifier.fit(
        X_train_configuration,
        y_campaign_train,
    )

    campaign_training_seconds = (
        time.perf_counter()
        - campaign_start
    )

    campaign_validation_probabilities = (
        campaign_classifier.predict_proba(
            X_validation_configuration
        )[:, 1]
    )

    campaign_test_probabilities = (
        campaign_classifier.predict_proba(
            X_test_configuration
        )[:, 1]
    )

    campaign_threshold = (
        select_validation_threshold(
            y_campaign_validation,
            campaign_validation_probabilities,
        )
    )

    campaign_metrics = (
        calculate_binary_metrics(
            y_campaign_test,
            campaign_test_probabilities,
            campaign_threshold,
        )
    )

    campaign_rows.append(
        {
            "configuration":
                configuration_name,

            "feature_dimension":
                representation.shape[1],

            "standardization":
                "train_only",

            **campaign_metrics,
        }
    )

    # -----------------------------------------------------------------
    # Aspect-presence classification
    # -----------------------------------------------------------------

    aspect_start = time.perf_counter()

    aspect_classifier = (
        OneVsRestClassifier(
            build_logistic_model(),
            n_jobs=-1,
        )
    )

    aspect_classifier.fit(
        X_train_configuration,
        y_aspect_train,
    )

    aspect_training_seconds = (
        time.perf_counter()
        - aspect_start
    )

    aspect_validation_probabilities = (
        aspect_classifier.predict_proba(
            X_validation_configuration
        )
    )

    aspect_test_probabilities = (
        aspect_classifier.predict_proba(
            X_test_configuration
        )
    )

    for aspect_index, aspect_name in enumerate(
        ASPECT_NAMES
    ):
        validation_labels = (
            y_aspect_validation[
                :,
                aspect_index,
            ]
        )

        test_labels = (
            y_aspect_test[
                :,
                aspect_index,
            ]
        )

        validation_probabilities = (
            aspect_validation_probabilities[
                :,
                aspect_index,
            ]
        )

        test_probabilities = (
            aspect_test_probabilities[
                :,
                aspect_index,
            ]
        )

        validation_positive_count = int(
            validation_labels.sum()
        )

        if validation_positive_count >= 5:
            aspect_threshold = (
                select_validation_threshold(
                    validation_labels,
                    validation_probabilities,
                )
            )

            threshold_status = (
                "validation_calibrated"
            )

        else:
            aspect_threshold = 0.5

            threshold_status = (
                "fixed_insufficient_positives"
            )

        aspect_metrics = (
            calculate_binary_metrics(
                test_labels,
                test_probabilities,
                aspect_threshold,
            )
        )

        aspect_rows.append(
            {
                "configuration":
                    configuration_name,

                "feature_dimension":
                    representation.shape[1],

                "standardization":
                    "train_only",

                "aspect":
                    aspect_name,

                "validation_positive_windows":
                    validation_positive_count,

                "test_positive_windows":
                    int(test_labels.sum()),

                "threshold_status":
                    threshold_status,

                **aspect_metrics,
            }
        )

    runtime_rows.append(
        {
            "configuration":
                configuration_name,

            "feature_dimension":
                representation.shape[1],

            "campaign_training_seconds":
                campaign_training_seconds,

            "aspect_training_seconds":
                aspect_training_seconds,

            "total_training_seconds":
                campaign_training_seconds
                + aspect_training_seconds,
        }
    )

    print(
        "Campaign threshold:",
        round(campaign_threshold, 4),
    )

    print(
        "Campaign F1:",
        round(campaign_metrics["f1"], 4),
    )

    print(
        "Campaign AUROC:",
        round(campaign_metrics["auroc"], 4),
    )


# -----------------------------------------------------------------------------
# Build result tables
# -----------------------------------------------------------------------------

campaign_ablation = pd.DataFrame(
    campaign_rows
)

aspect_ablation = pd.DataFrame(
    aspect_rows
)

runtime_ablation = pd.DataFrame(
    runtime_rows
)


# All four aspects
aspect_macro_all = (
    aspect_ablation
    .groupby(
        [
            "configuration",
            "feature_dimension",
        ],
        as_index=False,
    )
    .agg(
        macro_precision=(
            "precision",
            "mean",
        ),
        macro_recall=(
            "recall",
            "mean",
        ),
        macro_f1=(
            "f1",
            "mean",
        ),
        macro_mcc=(
            "mcc",
            "mean",
        ),
        macro_auroc=(
            "auroc",
            "mean",
        ),
        macro_auprc=(
            "auprc",
            "mean",
        ),
    )
)


# Core three aspects, excluding exploratory exfiltration
core_three_results = aspect_ablation[
    aspect_ablation["aspect"]
    != "data_exfiltration"
]

aspect_macro_core = (
    core_three_results
    .groupby(
        [
            "configuration",
            "feature_dimension",
        ],
        as_index=False,
    )
    .agg(
        core3_macro_precision=(
            "precision",
            "mean",
        ),
        core3_macro_recall=(
            "recall",
            "mean",
        ),
        core3_macro_f1=(
            "f1",
            "mean",
        ),
        core3_macro_mcc=(
            "mcc",
            "mean",
        ),
        core3_macro_auroc=(
            "auroc",
            "mean",
        ),
        core3_macro_auprc=(
            "auprc",
            "mean",
        ),
    )
)


summary_table = (
    campaign_ablation
    .merge(
        aspect_macro_all,
        on=[
            "configuration",
            "feature_dimension",
        ],
        how="left",
    )
    .merge(
        aspect_macro_core,
        on=[
            "configuration",
            "feature_dimension",
        ],
        how="left",
    )
    .merge(
        runtime_ablation,
        on=[
            "configuration",
            "feature_dimension",
        ],
        how="left",
    )
)

summary_table = (
    summary_table
    .sort_values(
        by=[
            "f1",
            "core3_macro_f1",
        ],
        ascending=False,
    )
    .reset_index(drop=True)
)

summary_table.insert(
    0,
    "campaign_f1_rank",
    summary_table["f1"]
    .rank(
        ascending=False,
        method="min",
    )
    .astype(int),
)


# -----------------------------------------------------------------------------
# Save all results
# -----------------------------------------------------------------------------

campaign_ablation.to_csv(
    OUTPUT_DIR
    / "campaign_modality_ablation_scaled.csv",
    index=False,
)

aspect_ablation.to_csv(
    OUTPUT_DIR
    / "aspect_modality_ablation_scaled.csv",
    index=False,
)

aspect_macro_all.to_csv(
    OUTPUT_DIR
    / "aspect_macro_all_four_scaled.csv",
    index=False,
)

aspect_macro_core.to_csv(
    OUTPUT_DIR
    / "aspect_macro_core_three_scaled.csv",
    index=False,
)

runtime_ablation.to_csv(
    OUTPUT_DIR
    / "modality_ablation_runtime_scaled.csv",
    index=False,
)

summary_table.to_csv(
    OUTPUT_DIR
    / "modality_ablation_summary_scaled.csv",
    index=False,
)


# -----------------------------------------------------------------------------
# Display results
# -----------------------------------------------------------------------------

print("\n" + "=" * 80)
print("STANDARDIZED MODALITY ABLATION SUMMARY")
print("=" * 80)

summary_columns = [
    "campaign_f1_rank",
    "configuration",
    "feature_dimension",
    "threshold",
    "accuracy",
    "balanced_accuracy",
    "precision",
    "recall",
    "f1",
    "mcc",
    "auroc",
    "auprc",
    "macro_f1",
    "macro_mcc",
    "core3_macro_f1",
    "core3_macro_mcc",
    "total_training_seconds",
]

display(
    summary_table[
        summary_columns
    ].round(6)
)

print("\nPer-aspect results:")

display(
    aspect_ablation[
        [
            "configuration",
            "aspect",
            "threshold",
            "precision",
            "recall",
            "f1",
            "mcc",
            "auroc",
            "auprc",
        ]
    ].round(6)
)

best_campaign_row = summary_table.iloc[0]

best_core3_row = (
    summary_table
    .sort_values(
        by="core3_macro_f1",
        ascending=False,
    )
    .iloc[0]
)

print("\nBest campaign configuration:")
print(
    best_campaign_row["configuration"],
    "| F1:",
    round(best_campaign_row["f1"], 6),
    "| AUROC:",
    round(best_campaign_row["auroc"], 6),
)

print("\nBest core-three aspect configuration:")
print(
    best_core3_row["configuration"],
    "| Core-three macro-F1:",
    round(
        best_core3_row["core3_macro_f1"],
        6,
    ),
)

print("\nMethodological notes:")
print(
    "- Every scaler was fitted using training windows only."
)

print(
    "- Every decision threshold was selected using "
    "validation windows only."
)

print(
    "- The test partition was not used for scaling, "
    "training, or threshold selection."
)

print(
    "- Data-exfiltration retains threshold 0.5 because "
    "validation contains only one positive window."
)

print("\nSaved standardized ablation results to:")
print(OUTPUT_DIR)

In [ ]:
# =============================================================================
# CELL 14: TEMPORAL-FEATURE DISTRIBUTION-SHIFT AUDIT
# =============================================================================

from pathlib import Path

import numpy as np
import pandas as pd

from scipy.stats import ks_2samp


# -----------------------------------------------------------------------------
# Configuration
# -----------------------------------------------------------------------------

DATA_DIR = Path(
    r"."
)

TENSOR_DIR = (
    DATA_DIR
    / "outputs"
    / "am_jepa_tensors"
)

OUTPUT_DIR = (
    DATA_DIR
    / "outputs"
    / "temporal_audit"
)

OUTPUT_DIR.mkdir(
    parents=True,
    exist_ok=True,
)

ASPECT_NAMES = [
    "reconnaissance",
    "initial_access_foothold",
    "lateral_movement",
    "data_exfiltration",
]

# Replace these names later if the tensor-preparation cell used
# more specific temporal feature names.
TEMPORAL_FEATURE_NAMES = [
    "temporal_feature_1",
    "temporal_feature_2",
    "temporal_feature_3",
    "temporal_feature_4",
]

PSI_NUMBER_OF_BINS = 10
EPSILON = 1e-6


# -----------------------------------------------------------------------------
# Flexible tensor loader
# -----------------------------------------------------------------------------

def load_first_existing(directory, candidate_names):
    for filename in candidate_names:
        path = directory / filename

        if path.exists():
            print(f"Loaded: {path.name}")

            return np.load(
                path,
                allow_pickle=False,
            )

    raise FileNotFoundError(
        "None of these files were found:\n"
        + "\n".join(candidate_names)
    )


# -----------------------------------------------------------------------------
# Load tensors
# -----------------------------------------------------------------------------

X_temporal = load_first_existing(
    TENSOR_DIR,
    [
        "X_temporal.npy",
        "x_temporal.npy",
    ],
).astype(np.float64)

attention_mask = load_first_existing(
    TENSOR_DIR,
    [
        "attention_mask.npy",
        "window_attention_mask.npy",
    ],
).astype(bool)

campaign_targets = load_first_existing(
    TENSOR_DIR,
    [
        "campaign_targets.npy",
        "y_campaign.npy",
    ],
).astype(np.int64)

aspect_targets = load_first_existing(
    TENSOR_DIR,
    [
        "aspect_presence_targets.npy",
        "y_aspect.npy",
    ],
).astype(np.int64)

train_indices = load_first_existing(
    TENSOR_DIR,
    [
        "train_indices.npy",
        "train_idx.npy",
    ],
).astype(np.int64)

validation_indices = load_first_existing(
    TENSOR_DIR,
    [
        "validation_indices.npy",
        "validation_idx.npy",
        "val_indices.npy",
        "val_idx.npy",
    ],
).astype(np.int64)

test_indices = load_first_existing(
    TENSOR_DIR,
    [
        "test_indices.npy",
        "test_idx.npy",
    ],
).astype(np.int64)


# -----------------------------------------------------------------------------
# Structural verification
# -----------------------------------------------------------------------------

number_of_windows = X_temporal.shape[0]
window_size = X_temporal.shape[1]
number_of_temporal_features = X_temporal.shape[2]

assert attention_mask.shape == (
    number_of_windows,
    window_size,
)

assert campaign_targets.shape == (
    number_of_windows,
)

assert aspect_targets.shape == (
    number_of_windows,
    len(ASPECT_NAMES),
)

if len(TEMPORAL_FEATURE_NAMES) != (
    number_of_temporal_features
):
    TEMPORAL_FEATURE_NAMES = [
        f"temporal_feature_{index + 1}"
        for index in range(
            number_of_temporal_features
        )
    ]

partition_indices = {
    "train": train_indices,
    "validation": validation_indices,
    "test": test_indices,
}

all_indices = np.concatenate(
    list(partition_indices.values())
)

assert len(np.unique(all_indices)) == len(
    all_indices
)

assert len(all_indices) == number_of_windows

print("\nTemporal tensor:", X_temporal.shape)
print("Attention mask:", attention_mask.shape)
print("Feature names:", TEMPORAL_FEATURE_NAMES)


# -----------------------------------------------------------------------------
# Mask-aware pooling
# -----------------------------------------------------------------------------

def masked_mean_pool(sequence_tensor, mask):
    expanded_mask = mask.astype(
        np.float64
    )[..., None]

    feature_sum = (
        sequence_tensor * expanded_mask
    ).sum(axis=1)

    valid_counts = expanded_mask.sum(
        axis=1
    )

    valid_counts = np.clip(
        valid_counts,
        a_min=1.0,
        a_max=None,
    )

    return feature_sum / valid_counts


temporal_pooled = masked_mean_pool(
    X_temporal,
    attention_mask,
)

assert temporal_pooled.shape == (
    number_of_windows,
    number_of_temporal_features,
)

print(
    "Pooled temporal representation:",
    temporal_pooled.shape,
)


# -----------------------------------------------------------------------------
# Statistical helpers
# -----------------------------------------------------------------------------

def finite_values(values):
    values = np.asarray(
        values,
        dtype=np.float64,
    )

    return values[
        np.isfinite(values)
    ]


def safe_correlation(x, y):
    """
    Pearson correlation between a continuous feature and
    a binary target, equivalent to point-biserial correlation.
    """
    x = np.asarray(x, dtype=np.float64)
    y = np.asarray(y, dtype=np.float64)

    valid = np.isfinite(x) & np.isfinite(y)

    x = x[valid]
    y = y[valid]

    if len(x) < 2:
        return np.nan

    if np.std(x) == 0:
        return np.nan

    if np.std(y) == 0:
        return np.nan

    return float(
        np.corrcoef(x, y)[0, 1]
    )


def calculate_psi(
    train_values,
    comparison_values,
    number_of_bins=10,
):
    """
    Population Stability Index using quantile boundaries
    estimated from training data only.
    """
    train_values = finite_values(
        train_values
    )

    comparison_values = finite_values(
        comparison_values
    )

    if (
        len(train_values) == 0
        or len(comparison_values) == 0
    ):
        return np.nan, 0

    if np.unique(train_values).size < 2:
        return np.nan, 0

    quantiles = np.linspace(
        0.0,
        1.0,
        number_of_bins + 1,
    )

    bin_edges = np.quantile(
        train_values,
        quantiles,
    )

    bin_edges = np.unique(bin_edges)

    if len(bin_edges) < 3:
        return np.nan, len(bin_edges) - 1

    # Ensure all future values fall inside the end bins.
    bin_edges[0] = -np.inf
    bin_edges[-1] = np.inf

    train_counts, _ = np.histogram(
        train_values,
        bins=bin_edges,
    )

    comparison_counts, _ = np.histogram(
        comparison_values,
        bins=bin_edges,
    )

    train_proportions = (
        train_counts / train_counts.sum()
    )

    comparison_proportions = (
        comparison_counts
        / comparison_counts.sum()
    )

    train_proportions = np.clip(
        train_proportions,
        EPSILON,
        None,
    )

    comparison_proportions = np.clip(
        comparison_proportions,
        EPSILON,
        None,
    )

    psi = np.sum(
        (
            comparison_proportions
            - train_proportions
        )
        * np.log(
            comparison_proportions
            / train_proportions
        )
    )

    return float(psi), len(bin_edges) - 1


def interpret_psi(psi):
    if np.isnan(psi):
        return "undefined"

    if psi < 0.10:
        return "small_shift"

    if psi < 0.25:
        return "moderate_shift"

    return "large_shift"


def interpret_ks(ks_statistic):
    if np.isnan(ks_statistic):
        return "undefined"

    if ks_statistic < 0.10:
        return "small_difference"

    if ks_statistic < 0.20:
        return "moderate_difference"

    return "large_difference"


# -----------------------------------------------------------------------------
# Window-level descriptive statistics
# -----------------------------------------------------------------------------

window_summary_rows = []

for split_name, indices in (
    partition_indices.items()
):
    split_values = temporal_pooled[
        indices
    ]

    for feature_index, feature_name in enumerate(
        TEMPORAL_FEATURE_NAMES
    ):
        values = split_values[
            :,
            feature_index,
        ]

        finite = finite_values(values)

        window_summary_rows.append(
            {
                "level": "window_pooled",
                "split": split_name,
                "feature_index":
                    feature_index,
                "feature": feature_name,
                "windows": len(values),
                "finite_values": len(finite),
                "missing_values":
                    int(
                        np.isnan(values).sum()
                    ),
                "infinite_values":
                    int(
                        np.isinf(values).sum()
                    ),
                "unique_values":
                    int(
                        np.unique(finite).size
                    ),
                "is_constant":
                    bool(
                        np.unique(finite).size
                        <= 1
                    ),
                "mean":
                    (
                        np.mean(finite)
                        if len(finite)
                        else np.nan
                    ),
                "std":
                    (
                        np.std(finite)
                        if len(finite)
                        else np.nan
                    ),
                "minimum":
                    (
                        np.min(finite)
                        if len(finite)
                        else np.nan
                    ),
                "q25":
                    (
                        np.quantile(
                            finite,
                            0.25,
                        )
                        if len(finite)
                        else np.nan
                    ),
                "median":
                    (
                        np.median(finite)
                        if len(finite)
                        else np.nan
                    ),
                "q75":
                    (
                        np.quantile(
                            finite,
                            0.75,
                        )
                        if len(finite)
                        else np.nan
                    ),
                "maximum":
                    (
                        np.max(finite)
                        if len(finite)
                        else np.nan
                    ),
            }
        )

window_summary = pd.DataFrame(
    window_summary_rows
)


# -----------------------------------------------------------------------------
# Evidence-position descriptive statistics
# -----------------------------------------------------------------------------

event_summary_rows = []

for split_name, indices in (
    partition_indices.items()
):
    split_tensor = X_temporal[
        indices
    ]

    split_mask = attention_mask[
        indices
    ]

    for feature_index, feature_name in enumerate(
        TEMPORAL_FEATURE_NAMES
    ):
        values = split_tensor[
            :,
            :,
            feature_index,
        ][split_mask]

        finite = finite_values(values)

        event_summary_rows.append(
            {
                "level": "valid_event_position",
                "split": split_name,
                "feature_index":
                    feature_index,
                "feature": feature_name,
                "valid_positions":
                    len(values),
                "finite_values":
                    len(finite),
                "missing_values":
                    int(
                        np.isnan(values).sum()
                    ),
                "infinite_values":
                    int(
                        np.isinf(values).sum()
                    ),
                "unique_values":
                    int(
                        np.unique(finite).size
                    ),
                "is_constant":
                    bool(
                        np.unique(finite).size
                        <= 1
                    ),
                "mean":
                    (
                        np.mean(finite)
                        if len(finite)
                        else np.nan
                    ),
                "std":
                    (
                        np.std(finite)
                        if len(finite)
                        else np.nan
                    ),
                "minimum":
                    (
                        np.min(finite)
                        if len(finite)
                        else np.nan
                    ),
                "q25":
                    (
                        np.quantile(
                            finite,
                            0.25,
                        )
                        if len(finite)
                        else np.nan
                    ),
                "median":
                    (
                        np.median(finite)
                        if len(finite)
                        else np.nan
                    ),
                "q75":
                    (
                        np.quantile(
                            finite,
                            0.75,
                        )
                        if len(finite)
                        else np.nan
                    ),
                "maximum":
                    (
                        np.max(finite)
                        if len(finite)
                        else np.nan
                    ),
            }
        )

event_summary = pd.DataFrame(
    event_summary_rows
)


# -----------------------------------------------------------------------------
# Window-level train-to-validation/test shift
# -----------------------------------------------------------------------------

window_shift_rows = []

for feature_index, feature_name in enumerate(
    TEMPORAL_FEATURE_NAMES
):
    train_values = temporal_pooled[
        train_indices,
        feature_index,
    ]

    for comparison_name, comparison_indices in {
        "validation": validation_indices,
        "test": test_indices,
    }.items():
        comparison_values = temporal_pooled[
            comparison_indices,
            feature_index,
        ]

        train_finite = finite_values(
            train_values
        )

        comparison_finite = finite_values(
            comparison_values
        )

        if (
            len(train_finite) > 0
            and len(comparison_finite) > 0
        ):
            ks_result = ks_2samp(
                train_finite,
                comparison_finite,
                alternative="two-sided",
                method="auto",
            )

            ks_statistic = float(
                ks_result.statistic
            )

            ks_pvalue = float(
                ks_result.pvalue
            )

        else:
            ks_statistic = np.nan
            ks_pvalue = np.nan

        psi, effective_bins = calculate_psi(
            train_finite,
            comparison_finite,
            number_of_bins=PSI_NUMBER_OF_BINS,
        )

        train_std = (
            np.std(train_finite)
            if len(train_finite)
            else np.nan
        )

        standardized_mean_difference = (
            (
                np.mean(comparison_finite)
                - np.mean(train_finite)
            )
            / train_std
            if (
                len(train_finite)
                and len(comparison_finite)
                and train_std > 0
            )
            else np.nan
        )

        window_shift_rows.append(
            {
                "level": "window_pooled",
                "feature_index":
                    feature_index,
                "feature": feature_name,
                "reference_split": "train",
                "comparison_split":
                    comparison_name,
                "ks_statistic":
                    ks_statistic,
                "ks_pvalue":
                    ks_pvalue,
                "ks_interpretation":
                    interpret_ks(
                        ks_statistic
                    ),
                "psi":
                    psi,
                "psi_effective_bins":
                    effective_bins,
                "psi_interpretation":
                    interpret_psi(psi),
                "standardized_mean_difference":
                    standardized_mean_difference,
            }
        )

window_shift = pd.DataFrame(
    window_shift_rows
)


# -----------------------------------------------------------------------------
# Event-level train-to-validation/test shift
# -----------------------------------------------------------------------------

event_shift_rows = []

for feature_index, feature_name in enumerate(
    TEMPORAL_FEATURE_NAMES
):
    train_event_values = X_temporal[
        train_indices,
        :,
        feature_index,
    ][
        attention_mask[
            train_indices
        ]
    ]

    for comparison_name, comparison_indices in {
        "validation": validation_indices,
        "test": test_indices,
    }.items():
        comparison_event_values = X_temporal[
            comparison_indices,
            :,
            feature_index,
        ][
            attention_mask[
                comparison_indices
            ]
        ]

        train_finite = finite_values(
            train_event_values
        )

        comparison_finite = finite_values(
            comparison_event_values
        )

        ks_result = ks_2samp(
            train_finite,
            comparison_finite,
            alternative="two-sided",
            method="auto",
        )

        psi, effective_bins = calculate_psi(
            train_finite,
            comparison_finite,
            number_of_bins=PSI_NUMBER_OF_BINS,
        )

        train_std = np.std(
            train_finite
        )

        standardized_mean_difference = (
            (
                np.mean(comparison_finite)
                - np.mean(train_finite)
            )
            / train_std
            if train_std > 0
            else np.nan
        )

        event_shift_rows.append(
            {
                "level":
                    "valid_event_position",
                "feature_index":
                    feature_index,
                "feature":
                    feature_name,
                "reference_split":
                    "train",
                "comparison_split":
                    comparison_name,
                "ks_statistic":
                    float(
                        ks_result.statistic
                    ),
                "ks_pvalue":
                    float(
                        ks_result.pvalue
                    ),
                "ks_interpretation":
                    interpret_ks(
                        float(
                            ks_result.statistic
                        )
                    ),
                "psi":
                    psi,
                "psi_effective_bins":
                    effective_bins,
                "psi_interpretation":
                    interpret_psi(psi),
                "standardized_mean_difference":
                    standardized_mean_difference,
            }
        )

event_shift = pd.DataFrame(
    event_shift_rows
)


# -----------------------------------------------------------------------------
# Correlation with campaign and aspect targets
# -----------------------------------------------------------------------------

correlation_rows = []

for split_name, indices in (
    partition_indices.items()
):
    split_temporal = temporal_pooled[
        indices
    ]

    split_campaign = campaign_targets[
        indices
    ]

    split_aspects = aspect_targets[
        indices
    ]

    for feature_index, feature_name in enumerate(
        TEMPORAL_FEATURE_NAMES
    ):
        feature_values = split_temporal[
            :,
            feature_index,
        ]

        correlation_rows.append(
            {
                "split": split_name,
                "feature_index":
                    feature_index,
                "feature":
                    feature_name,
                "target":
                    "campaign",
                "positive_windows":
                    int(
                        split_campaign.sum()
                    ),
                "correlation":
                    safe_correlation(
                        feature_values,
                        split_campaign,
                    ),
            }
        )

        for aspect_index, aspect_name in enumerate(
            ASPECT_NAMES
        ):
            labels = split_aspects[
                :,
                aspect_index,
            ]

            correlation_rows.append(
                {
                    "split": split_name,
                    "feature_index":
                        feature_index,
                    "feature":
                        feature_name,
                    "target":
                        aspect_name,
                    "positive_windows":
                        int(labels.sum()),
                    "correlation":
                        safe_correlation(
                            feature_values,
                            labels,
                        ),
                }
            )

correlation_table = pd.DataFrame(
    correlation_rows
)


# -----------------------------------------------------------------------------
# Correlation stability across partitions
# -----------------------------------------------------------------------------

correlation_pivot = (
    correlation_table
    .pivot_table(
        index=[
            "feature_index",
            "feature",
            "target",
        ],
        columns="split",
        values="correlation",
    )
    .reset_index()
)

for required_column in [
    "train",
    "validation",
    "test",
]:
    if required_column not in (
        correlation_pivot.columns
    ):
        correlation_pivot[
            required_column
        ] = np.nan

correlation_pivot[
    "train_validation_difference"
] = (
    correlation_pivot["validation"]
    - correlation_pivot["train"]
).abs()

correlation_pivot[
    "train_test_difference"
] = (
    correlation_pivot["test"]
    - correlation_pivot["train"]
).abs()

correlation_pivot[
    "validation_sign_flip"
] = (
    np.sign(correlation_pivot["train"])
    != np.sign(
        correlation_pivot["validation"]
    )
)

correlation_pivot[
    "test_sign_flip"
] = (
    np.sign(correlation_pivot["train"])
    != np.sign(
        correlation_pivot["test"]
    )
)


# -----------------------------------------------------------------------------
# Overall audit flags
# -----------------------------------------------------------------------------

audit_flags = []

for _, row in window_shift.iterrows():
    if row["psi_interpretation"] == (
        "large_shift"
    ):
        audit_flags.append(
            {
                "severity": "high",
                "scope": "window_distribution",
                "feature": row["feature"],
                "comparison_split":
                    row["comparison_split"],
                "reason":
                    f"PSI={row['psi']:.4f}",
            }
        )

    if row["ks_interpretation"] == (
        "large_difference"
    ):
        audit_flags.append(
            {
                "severity": "high",
                "scope": "window_distribution",
                "feature": row["feature"],
                "comparison_split":
                    row["comparison_split"],
                "reason":
                    (
                        "KS="
                        f"{row['ks_statistic']:.4f}"
                    ),
            }
        )

for _, row in correlation_pivot.iterrows():
    if bool(row["test_sign_flip"]):
        audit_flags.append(
            {
                "severity": "warning",
                "scope": "target_correlation",
                "feature": row["feature"],
                "comparison_split": "test",
                "reason":
                    (
                        "Correlation sign changed "
                        f"for target {row['target']}"
                    ),
            }
        )

audit_flags = pd.DataFrame(
    audit_flags,
    columns=[
        "severity",
        "scope",
        "feature",
        "comparison_split",
        "reason",
    ],
)


# -----------------------------------------------------------------------------
# Save audit files
# -----------------------------------------------------------------------------

window_summary.to_csv(
    OUTPUT_DIR
    / "temporal_window_summary.csv",
    index=False,
)

event_summary.to_csv(
    OUTPUT_DIR
    / "temporal_event_summary.csv",
    index=False,
)

window_shift.to_csv(
    OUTPUT_DIR
    / "temporal_window_shift.csv",
    index=False,
)

event_shift.to_csv(
    OUTPUT_DIR
    / "temporal_event_shift.csv",
    index=False,
)

correlation_table.to_csv(
    OUTPUT_DIR
    / "temporal_target_correlations.csv",
    index=False,
)

correlation_pivot.to_csv(
    OUTPUT_DIR
    / "temporal_correlation_stability.csv",
    index=False,
)

audit_flags.to_csv(
    OUTPUT_DIR
    / "temporal_audit_flags.csv",
    index=False,
)


# -----------------------------------------------------------------------------
# Display results
# -----------------------------------------------------------------------------

print("\n" + "=" * 80)
print("TEMPORAL WINDOW-LEVEL SUMMARY")
print("=" * 80)

display(
    window_summary[
        [
            "split",
            "feature",
            "windows",
            "unique_values",
            "is_constant",
            "mean",
            "std",
            "minimum",
            "median",
            "maximum",
        ]
    ].round(6)
)

print("\n" + "=" * 80)
print("TRAIN-TO-VALIDATION/TEST TEMPORAL SHIFT")
print("=" * 80)

display(
    window_shift[
        [
            "feature",
            "comparison_split",
            "ks_statistic",
            "ks_pvalue",
            "ks_interpretation",
            "psi",
            "psi_interpretation",
            "standardized_mean_difference",
        ]
    ].round(6)
)

print("\n" + "=" * 80)
print("TEMPORAL TARGET-CORRELATION STABILITY")
print("=" * 80)

display(
    correlation_pivot[
        [
            "feature",
            "target",
            "train",
            "validation",
            "test",
            "train_validation_difference",
            "train_test_difference",
            "validation_sign_flip",
            "test_sign_flip",
        ]
    ].round(6)
)

print("\n" + "=" * 80)
print("TEMPORAL AUDIT FLAGS")
print("=" * 80)

if len(audit_flags):
    display(audit_flags)
else:
    print("No major temporal-distribution flags detected.")

print("\nInterpretation:")
print("PSI < 0.10: small shift")
print("PSI 0.10-0.25: moderate shift")
print("PSI >= 0.25: large shift")
print("KS < 0.10: small distribution difference")
print("KS 0.10-0.20: moderate distribution difference")
print("KS >= 0.20: large distribution difference")

print("\nSaved temporal audit to:")
print(OUTPUT_DIR)

In [ ]:
print("=" * 80)
print("TEMPORAL FEATURE DEFINITIONS REQUIRED")
print("=" * 80)

print("\nFeature order currently used:")
for index, name in enumerate(
    TEMPORAL_FEATURE_NAMES
):
    print(
        f"{index}: {name}"
    )

print("\nWindow-level distribution summary:")
display(
    window_summary[
        [
            "split",
            "feature_index",
            "feature",
            "mean",
            "std",
            "minimum",
            "q25",
            "median",
            "q75",
            "maximum",
            "unique_values",
        ]
    ].round(6)
)

print("\nComplete shift results:")
display(
    window_shift[
        [
            "feature_index",
            "feature",
            "comparison_split",
            "ks_statistic",
            "psi",
            "standardized_mean_difference",
        ]
    ].round(6)
)

In [ ]:
# =============================================================================
# CELL 15: TEMPORAL SOURCE-SCHEMA DISCOVERY
# =============================================================================

from pathlib import Path
import pandas as pd

DATA_DIR = Path(
    r"."
)

OUTPUT_DIR = DATA_DIR / "outputs"

membership_path = (
    OUTPUT_DIR
    / "dapt2020_mast_window_membership_enriched.parquet"
)

evidence_path = (
    OUTPUT_DIR
    / "dapt2020_mast_evidence.parquet"
)

windows_path = (
    OUTPUT_DIR
    / "dapt2020_mast_windows.parquet"
)

membership = pd.read_parquet(
    membership_path
)

evidence = pd.read_parquet(
    evidence_path
)

windows = pd.read_parquet(
    windows_path
)

print("=" * 80)
print("TEMPORAL SOURCE-SCHEMA DISCOVERY")
print("=" * 80)

print("\nMembership shape:")
print(membership.shape)

print("\nMembership columns:")
for index, column in enumerate(
    membership.columns
):
    print(f"{index:02d}. {column}")

print("\nEvidence shape:")
print(evidence.shape)

print("\nEvidence columns:")
for index, column in enumerate(
    evidence.columns
):
    print(f"{index:02d}. {column}")

print("\nWindow shape:")
print(windows.shape)

print("\nWindow columns:")
for index, column in enumerate(
    windows.columns
):
    print(f"{index:02d}. {column}")

candidate_terms = [
    "time",
    "timestamp",
    "position",
    "order",
    "index",
    "window",
    "evidence",
]

membership_candidates = [
    column
    for column in membership.columns
    if any(
        term in column.lower()
        for term in candidate_terms
    )
]

evidence_candidates = [
    column
    for column in evidence.columns
    if any(
        term in column.lower()
        for term in candidate_terms
    )
]

print("\nPotential membership temporal columns:")
print(membership_candidates)

print("\nPotential evidence temporal columns:")
print(evidence_candidates)

print("\nMembership sample:")
display(
    membership[
        membership_candidates
    ].head(10)
)

print("\nEvidence sample:")
display(
    evidence[
        evidence_candidates
    ].head(10)
)

In [ ]:
# =============================================================================
# CELL 16: RECONSTRUCT SPLIT-INVARIANT TEMPORAL FEATURES
# =============================================================================

from pathlib import Path
import json

import numpy as np
import pandas as pd
from scipy.stats import ks_2samp


# -----------------------------------------------------------------------------
# Configuration
# -----------------------------------------------------------------------------

DATA_DIR = Path(
    r"."
)

OUTPUT_DIR = DATA_DIR / "outputs"

TENSOR_DIR = (
    OUTPUT_DIR
    / "am_jepa_tensors"
)

MEMBERSHIP_PATH = (
    OUTPUT_DIR
    / "dapt2020_mast_window_membership_enriched.parquet"
)

WINDOWS_PATH = (
    OUTPUT_DIR
    / "dapt2020_mast_windows.parquet"
)

STABLE_TEMPORAL_PATH = (
    TENSOR_DIR
    / "X_temporal_stable.npy"
)

STABLE_MANIFEST_PATH = (
    TENSOR_DIR
    / "temporal_stable_manifest.json"
)

STABLE_AUDIT_PATH = (
    OUTPUT_DIR
    / "temporal_audit"
    / "stable_temporal_shift.csv"
)

STABLE_AUDIT_PATH.parent.mkdir(
    parents=True,
    exist_ok=True,
)

WINDOW_SIZE = 32
EPSILON = 1e-8

TEMPORAL_FEATURE_NAMES = [
    "relative_elapsed_time",
    "normalized_log_interarrival",
    "hour_sine",
    "hour_cosine",
]


# -----------------------------------------------------------------------------
# Load source data and existing tensors
# -----------------------------------------------------------------------------

membership = pd.read_parquet(
    MEMBERSHIP_PATH
)

windows = pd.read_parquet(
    WINDOWS_PATH
)

attention_mask = np.load(
    TENSOR_DIR / "attention_mask.npy",
    allow_pickle=False,
).astype(bool)

train_indices = np.load(
    TENSOR_DIR / "train_indices.npy",
    allow_pickle=False,
).astype(np.int64)

validation_indices = np.load(
    TENSOR_DIR / "validation_indices.npy",
    allow_pickle=False,
).astype(np.int64)

test_indices = np.load(
    TENSOR_DIR / "test_indices.npy",
    allow_pickle=False,
).astype(np.int64)

number_of_windows = attention_mask.shape[0]

assert attention_mask.shape == (
    number_of_windows,
    WINDOW_SIZE,
)

assert len(windows) == number_of_windows

required_membership_columns = {
    "window_id",
    "split",
    "position",
    "timestamp_parsed",
    "attention_mask",
}

missing_columns = (
    required_membership_columns
    - set(membership.columns)
)

if missing_columns:
    raise KeyError(
        "Missing membership columns: "
        f"{sorted(missing_columns)}"
    )


# -----------------------------------------------------------------------------
# Recover the exact tensor window order
# -----------------------------------------------------------------------------

window_id_file_candidates = [
    TENSOR_DIR / "window_ids.npy",
    TENSOR_DIR / "window_id.npy",
]

window_ids = None

for candidate_path in (
    window_id_file_candidates
):
    if candidate_path.exists():
        try:
            window_ids = np.load(
                candidate_path,
                allow_pickle=False,
            )
        except ValueError:
            window_ids = np.load(
                candidate_path,
                allow_pickle=True,
            )

        window_ids = np.asarray(
            window_ids
        ).astype(str)

        print(
            "Window order loaded from:",
            candidate_path.name,
        )

        break

if window_ids is None:
    if "window_id" not in windows.columns:
        raise KeyError(
            "The windows table does not contain "
            "'window_id', and no window_ids.npy "
            "file was found."
        )

    window_ids = (
        windows["window_id"]
        .astype(str)
        .to_numpy()
    )

    print(
        "Window order recovered from:",
        WINDOWS_PATH.name,
    )

assert len(window_ids) == number_of_windows
assert len(np.unique(window_ids)) == number_of_windows


# -----------------------------------------------------------------------------
# Clean and validate membership
# -----------------------------------------------------------------------------

membership = membership.copy()

membership["window_id"] = (
    membership["window_id"].astype(str)
)

membership["position"] = pd.to_numeric(
    membership["position"],
    errors="raise",
).astype(np.int64)

membership["timestamp_parsed"] = (
    pd.to_datetime(
        membership["timestamp_parsed"],
        errors="coerce",
    )
)

if membership[
    "timestamp_parsed"
].isna().any():
    invalid_count = int(
        membership[
            "timestamp_parsed"
        ].isna().sum()
    )

    raise ValueError(
        f"{invalid_count} membership timestamps "
        "could not be parsed."
    )

if membership["window_id"].duplicated(
    keep=False
).any():
    duplicate_positions = (
        membership.duplicated(
            subset=[
                "window_id",
                "position",
            ],
            keep=False,
        )
    )

    if duplicate_positions.any():
        raise ValueError(
            "Duplicate window-position pairs found: "
            f"{int(duplicate_positions.sum())}"
        )

assert membership["position"].min() >= 0
assert membership["position"].max() < WINDOW_SIZE

membership_window_ids = set(
    membership["window_id"].unique()
)

tensor_window_ids = set(window_ids)

missing_membership_windows = (
    tensor_window_ids
    - membership_window_ids
)

unexpected_membership_windows = (
    membership_window_ids
    - tensor_window_ids
)

if missing_membership_windows:
    raise ValueError(
        "Tensor windows missing from membership: "
        f"{len(missing_membership_windows)}"
    )

if unexpected_membership_windows:
    raise ValueError(
        "Membership contains unexpected windows: "
        f"{len(unexpected_membership_windows)}"
    )


# -----------------------------------------------------------------------------
# Map window IDs to tensor rows
# -----------------------------------------------------------------------------

window_to_index = {
    window_id: index
    for index, window_id in enumerate(
        window_ids
    )
}

membership["tensor_window_index"] = (
    membership["window_id"]
    .map(window_to_index)
    .astype(np.int64)
)


# -----------------------------------------------------------------------------
# Verify that window order agrees with partition indices
# -----------------------------------------------------------------------------

window_split_lookup = (
    membership[
        [
            "window_id",
            "split",
        ]
    ]
    .drop_duplicates()
)

split_counts_per_window = (
    window_split_lookup
    .groupby("window_id")["split"]
    .nunique()
)

assert split_counts_per_window.max() == 1

window_split_map = dict(
    zip(
        window_split_lookup["window_id"],
        window_split_lookup["split"],
    )
)

split_by_tensor_index = np.asarray(
    [
        window_split_map[window_id]
        for window_id in window_ids
    ]
)

assert np.all(
    split_by_tensor_index[
        train_indices
    ] == "train"
)

assert np.all(
    split_by_tensor_index[
        validation_indices
    ] == "validation"
)

assert np.all(
    split_by_tensor_index[
        test_indices
    ] == "test"
)

print("\nWindow-order and split verification passed.")


# -----------------------------------------------------------------------------
# Initialize stable temporal tensor
# -----------------------------------------------------------------------------

X_temporal_stable = np.zeros(
    (
        number_of_windows,
        WINDOW_SIZE,
        len(TEMPORAL_FEATURE_NAMES),
    ),
    dtype=np.float32,
)

reconstructed_mask = np.zeros(
    (
        number_of_windows,
        WINDOW_SIZE,
    ),
    dtype=bool,
)


# -----------------------------------------------------------------------------
# Construct stable within-window temporal features
# -----------------------------------------------------------------------------

for window_id, group in membership.groupby(
    "window_id",
    sort=False,
):
    tensor_index = window_to_index[
        window_id
    ]

    group = group.sort_values(
        "position"
    )

    positions = group[
        "position"
    ].to_numpy(dtype=np.int64)

    timestamps = (
        group["timestamp_parsed"]
        .astype("int64")
        .to_numpy(dtype=np.float64)
        / 1e9
    )

    # Safety check: events should already be temporally ordered.
    if np.any(np.diff(timestamps) < 0):
        raise ValueError(
            f"Non-monotonic timestamps in {window_id}"
        )

    number_of_events = len(group)

    # -----------------------------------------------------------------
    # Feature 1: elapsed position within the current window
    # -----------------------------------------------------------------

    elapsed_seconds = (
        timestamps - timestamps[0]
    )

    total_duration = elapsed_seconds[-1]

    if total_duration > 0:
        relative_elapsed = (
            elapsed_seconds
            / total_duration
        )
    elif number_of_events > 1:
        relative_elapsed = np.linspace(
            0.0,
            1.0,
            number_of_events,
        )
    else:
        relative_elapsed = np.zeros(
            number_of_events,
            dtype=np.float64,
        )

    # -----------------------------------------------------------------
    # Feature 2: log interarrival normalized by the largest
    # within-window log gap
    # -----------------------------------------------------------------

    interarrival_seconds = np.diff(
        timestamps,
        prepend=timestamps[0],
    )

    interarrival_seconds = np.maximum(
        interarrival_seconds,
        0.0,
    )

    log_interarrival = np.log1p(
        interarrival_seconds
    )

    maximum_log_gap = log_interarrival.max()

    if maximum_log_gap > 0:
        normalized_log_interarrival = (
            log_interarrival
            / maximum_log_gap
        )
    else:
        normalized_log_interarrival = (
            np.zeros(
                number_of_events,
                dtype=np.float64,
            )
        )

    # -----------------------------------------------------------------
    # Features 3-4: cyclic time of day
    # -----------------------------------------------------------------

    timestamp_series = group[
        "timestamp_parsed"
    ]

    fractional_hour = (
        timestamp_series.dt.hour.to_numpy(
            dtype=np.float64
        )
        + timestamp_series.dt.minute.to_numpy(
            dtype=np.float64
        )
        / 60.0
        + timestamp_series.dt.second.to_numpy(
            dtype=np.float64
        )
        / 3600.0
    )

    hour_angle = (
        2.0
        * np.pi
        * fractional_hour
        / 24.0
    )

    hour_sine = np.sin(hour_angle)
    hour_cosine = np.cos(hour_angle)

    window_features = np.column_stack(
        [
            relative_elapsed,
            normalized_log_interarrival,
            hour_sine,
            hour_cosine,
        ]
    ).astype(np.float32)

    if not np.isfinite(
        window_features
    ).all():
        raise ValueError(
            f"Non-finite temporal values in {window_id}"
        )

    X_temporal_stable[
        tensor_index,
        positions,
        :,
    ] = window_features

    reconstructed_mask[
        tensor_index,
        positions,
    ] = True


# -----------------------------------------------------------------------------
# Alignment verification
# -----------------------------------------------------------------------------

mask_difference_count = int(
    np.count_nonzero(
        reconstructed_mask
        != attention_mask
    )
)

if mask_difference_count:
    raise AssertionError(
        "Reconstructed mask differs from the saved "
        f"attention mask at {mask_difference_count} positions."
    )

valid_temporal_values = (
    X_temporal_stable[
        reconstructed_mask
    ]
)

assert np.isfinite(
    valid_temporal_values
).all()

assert (
    valid_temporal_values[:, 0].min()
    >= -EPSILON
)

assert (
    valid_temporal_values[:, 0].max()
    <= 1.0 + EPSILON
)

assert (
    valid_temporal_values[:, 1].min()
    >= -EPSILON
)

assert (
    valid_temporal_values[:, 1].max()
    <= 1.0 + EPSILON
)

assert (
    valid_temporal_values[:, 2].min()
    >= -1.0 - EPSILON
)

assert (
    valid_temporal_values[:, 2].max()
    <= 1.0 + EPSILON
)

assert (
    valid_temporal_values[:, 3].min()
    >= -1.0 - EPSILON
)

assert (
    valid_temporal_values[:, 3].max()
    <= 1.0 + EPSILON
)


# -----------------------------------------------------------------------------
# Save without overwriting the original temporal tensor
# -----------------------------------------------------------------------------

np.save(
    STABLE_TEMPORAL_PATH,
    X_temporal_stable,
)

manifest = {
    "tensor_file":
        STABLE_TEMPORAL_PATH.name,

    "shape":
        list(X_temporal_stable.shape),

    "dtype":
        str(X_temporal_stable.dtype),

    "feature_order":
        TEMPORAL_FEATURE_NAMES,

    "feature_definitions": {
        "relative_elapsed_time":
            (
                "(timestamp_i - timestamp_first) / "
                "(timestamp_last - timestamp_first)"
            ),

        "normalized_log_interarrival":
            (
                "log1p(timestamp_i - timestamp_previous) / "
                "maximum within-window log1p gap"
            ),

        "hour_sine":
            "sin(2*pi*fractional_hour/24)",

        "hour_cosine":
            "cos(2*pi*fractional_hour/24)",
    },

    "normalization_scope":
        "within_window",

    "source":
        str(MEMBERSHIP_PATH),

    "original_temporal_tensor_preserved":
        True,
}

with open(
    STABLE_MANIFEST_PATH,
    "w",
    encoding="utf-8",
) as file:
    json.dump(
        manifest,
        file,
        indent=2,
    )


# -----------------------------------------------------------------------------
# Immediate window-level stability audit
# -----------------------------------------------------------------------------

def masked_mean_pool(
    sequence_tensor,
    mask,
):
    expanded_mask = (
        mask.astype(np.float64)[..., None]
    )

    numerator = (
        sequence_tensor
        * expanded_mask
    ).sum(axis=1)

    denominator = np.clip(
        expanded_mask.sum(axis=1),
        a_min=1.0,
        a_max=None,
    )

    return numerator / denominator


def calculate_psi(
    reference_values,
    comparison_values,
    bins=10,
):
    reference_values = np.asarray(
        reference_values,
        dtype=np.float64,
    )

    comparison_values = np.asarray(
        comparison_values,
        dtype=np.float64,
    )

    quantile_edges = np.quantile(
        reference_values,
        np.linspace(
            0.0,
            1.0,
            bins + 1,
        ),
    )

    quantile_edges = np.unique(
        quantile_edges
    )

    if len(quantile_edges) < 3:
        return np.nan

    quantile_edges[0] = -np.inf
    quantile_edges[-1] = np.inf

    reference_counts, _ = np.histogram(
        reference_values,
        bins=quantile_edges,
    )

    comparison_counts, _ = np.histogram(
        comparison_values,
        bins=quantile_edges,
    )

    reference_proportions = (
        reference_counts
        / reference_counts.sum()
    )

    comparison_proportions = (
        comparison_counts
        / comparison_counts.sum()
    )

    reference_proportions = np.clip(
        reference_proportions,
        1e-6,
        None,
    )

    comparison_proportions = np.clip(
        comparison_proportions,
        1e-6,
        None,
    )

    return float(
        np.sum(
            (
                comparison_proportions
                - reference_proportions
            )
            * np.log(
                comparison_proportions
                / reference_proportions
            )
        )
    )


stable_temporal_pooled = masked_mean_pool(
    X_temporal_stable,
    attention_mask,
)

stable_audit_rows = []

for feature_index, feature_name in enumerate(
    TEMPORAL_FEATURE_NAMES
):
    train_values = stable_temporal_pooled[
        train_indices,
        feature_index,
    ]

    for split_name, split_indices in {
        "validation": validation_indices,
        "test": test_indices,
    }.items():
        comparison_values = (
            stable_temporal_pooled[
                split_indices,
                feature_index,
            ]
        )

        ks_result = ks_2samp(
            train_values,
            comparison_values,
        )

        psi = calculate_psi(
            train_values,
            comparison_values,
        )

        stable_audit_rows.append(
            {
                "feature":
                    feature_name,

                "comparison_split":
                    split_name,

                "train_mean":
                    train_values.mean(),

                "comparison_mean":
                    comparison_values.mean(),

                "ks_statistic":
                    float(
                        ks_result.statistic
                    ),

                "ks_pvalue":
                    float(
                        ks_result.pvalue
                    ),

                "psi":
                    psi,

                "shift_status":
                    (
                        "large"
                        if (
                            ks_result.statistic
                            >= 0.20
                            or (
                                np.isfinite(psi)
                                and psi >= 0.25
                            )
                        )
                        else (
                            "moderate"
                            if (
                                ks_result.statistic
                                >= 0.10
                                or (
                                    np.isfinite(psi)
                                    and psi >= 0.10
                                )
                            )
                            else "small"
                        )
                    ),
            }
        )

stable_audit = pd.DataFrame(
    stable_audit_rows
)

stable_audit.to_csv(
    STABLE_AUDIT_PATH,
    index=False,
)


# -----------------------------------------------------------------------------
# Display
# -----------------------------------------------------------------------------

print("\n" + "=" * 80)
print("STABLE TEMPORAL TENSOR CONSTRUCTION")
print("=" * 80)

print("Tensor shape:", X_temporal_stable.shape)

print(
    "Valid positions:",
    int(reconstructed_mask.sum()),
)

print(
    "Padding positions:",
    int(
        reconstructed_mask.size
        - reconstructed_mask.sum()
    ),
)

print(
    "Mask differences:",
    mask_difference_count,
)

print("\nValid-value ranges:")

for feature_index, feature_name in enumerate(
    TEMPORAL_FEATURE_NAMES
):
    values = valid_temporal_values[
        :,
        feature_index,
    ]

    print(
        f"{feature_name}: "
        f"min={values.min():.6f}, "
        f"mean={values.mean():.6f}, "
        f"max={values.max():.6f}"
    )

print("\nStable temporal shift audit:")
display(
    stable_audit.round(6)
)

print("\nSaved stable tensor:")
print(STABLE_TEMPORAL_PATH)

print("\nSaved feature manifest:")
print(STABLE_MANIFEST_PATH)

print("\nOriginal X_temporal.npy was preserved.")

In [ ]:
# =============================================================================
# CELL 17: FINAL STABLE TEMPORAL TENSOR
# =============================================================================

from pathlib import Path
import json

import numpy as np
import pandas as pd
from scipy.stats import ks_2samp


# -----------------------------------------------------------------------------
# Paths
# -----------------------------------------------------------------------------

DATA_DIR = Path(
    r"."
)

TENSOR_DIR = (
    DATA_DIR
    / "outputs"
    / "am_jepa_tensors"
)

AUDIT_DIR = (
    DATA_DIR
    / "outputs"
    / "temporal_audit"
)

AUDIT_DIR.mkdir(
    parents=True,
    exist_ok=True,
)

INPUT_PATH = (
    TENSOR_DIR
    / "X_temporal_stable.npy"
)

OUTPUT_PATH = (
    TENSOR_DIR
    / "X_temporal_final.npy"
)

MANIFEST_PATH = (
    TENSOR_DIR
    / "temporal_final_manifest.json"
)

AUDIT_PATH = (
    AUDIT_DIR
    / "final_temporal_shift.csv"
)


# -----------------------------------------------------------------------------
# Load stable tensor and partitions
# -----------------------------------------------------------------------------

X_temporal_stable = np.load(
    INPUT_PATH,
    allow_pickle=False,
).astype(np.float32)

attention_mask = np.load(
    TENSOR_DIR / "attention_mask.npy",
    allow_pickle=False,
).astype(bool)

train_indices = np.load(
    TENSOR_DIR / "train_indices.npy",
    allow_pickle=False,
).astype(np.int64)

validation_indices = np.load(
    TENSOR_DIR / "validation_indices.npy",
    allow_pickle=False,
).astype(np.int64)

test_indices = np.load(
    TENSOR_DIR / "test_indices.npy",
    allow_pickle=False,
).astype(np.int64)


# -----------------------------------------------------------------------------
# Retain validation-stable features
# -----------------------------------------------------------------------------

# Original stable tensor:
# 0 = relative_elapsed_time
# 1 = normalized_log_interarrival
# 2 = hour_sine
# 3 = hour_cosine
#
# Feature 1 is removed because validation PSI=0.5002 and KS=0.2040.

retained_feature_indices = [
    0,
    2,
    3,
]

TEMPORAL_FEATURE_NAMES = [
    "relative_elapsed_time",
    "hour_sine",
    "hour_cosine",
]

X_temporal_final = X_temporal_stable[
    :,
    :,
    retained_feature_indices,
].copy()

assert X_temporal_final.shape == (
    X_temporal_stable.shape[0],
    X_temporal_stable.shape[1],
    3,
)

assert np.isfinite(
    X_temporal_final
).all()

assert attention_mask.shape == (
    X_temporal_final.shape[0],
    X_temporal_final.shape[1],
)


# -----------------------------------------------------------------------------
# Mask-aware window pooling
# -----------------------------------------------------------------------------

def masked_mean_pool(
    sequence_tensor,
    mask,
):
    expanded_mask = (
        mask.astype(np.float64)[..., None]
    )

    numerator = (
        sequence_tensor
        * expanded_mask
    ).sum(axis=1)

    denominator = np.clip(
        expanded_mask.sum(axis=1),
        a_min=1.0,
        a_max=None,
    )

    return numerator / denominator


temporal_pooled = masked_mean_pool(
    X_temporal_final,
    attention_mask,
)


# -----------------------------------------------------------------------------
# PSI function
# -----------------------------------------------------------------------------

def calculate_psi(
    reference_values,
    comparison_values,
    number_of_bins=10,
):
    reference_values = np.asarray(
        reference_values,
        dtype=np.float64,
    )

    comparison_values = np.asarray(
        comparison_values,
        dtype=np.float64,
    )

    edges = np.quantile(
        reference_values,
        np.linspace(
            0.0,
            1.0,
            number_of_bins + 1,
        ),
    )

    edges = np.unique(edges)

    if len(edges) < 3:
        return np.nan

    edges[0] = -np.inf
    edges[-1] = np.inf

    reference_counts, _ = np.histogram(
        reference_values,
        bins=edges,
    )

    comparison_counts, _ = np.histogram(
        comparison_values,
        bins=edges,
    )

    reference_proportions = (
        reference_counts
        / reference_counts.sum()
    )

    comparison_proportions = (
        comparison_counts
        / comparison_counts.sum()
    )

    reference_proportions = np.clip(
        reference_proportions,
        1e-6,
        None,
    )

    comparison_proportions = np.clip(
        comparison_proportions,
        1e-6,
        None,
    )

    return float(
        np.sum(
            (
                comparison_proportions
                - reference_proportions
            )
            * np.log(
                comparison_proportions
                / reference_proportions
            )
        )
    )


# -----------------------------------------------------------------------------
# Final temporal audit
# -----------------------------------------------------------------------------

audit_rows = []

for feature_index, feature_name in enumerate(
    TEMPORAL_FEATURE_NAMES
):
    train_values = temporal_pooled[
        train_indices,
        feature_index,
    ]

    for split_name, split_indices in {
        "validation": validation_indices,
        "test": test_indices,
    }.items():
        comparison_values = temporal_pooled[
            split_indices,
            feature_index,
        ]

        ks_result = ks_2samp(
            train_values,
            comparison_values,
            alternative="two-sided",
            method="auto",
        )

        psi = calculate_psi(
            train_values,
            comparison_values,
        )

        if (
            ks_result.statistic >= 0.20
            or (
                np.isfinite(psi)
                and psi >= 0.25
            )
        ):
            shift_status = "large"

        elif (
            ks_result.statistic >= 0.10
            or (
                np.isfinite(psi)
                and psi >= 0.10
            )
        ):
            shift_status = "moderate"

        else:
            shift_status = "small"

        audit_rows.append(
            {
                "feature":
                    feature_name,

                "comparison_split":
                    split_name,

                "train_mean":
                    train_values.mean(),

                "comparison_mean":
                    comparison_values.mean(),

                "ks_statistic":
                    float(
                        ks_result.statistic
                    ),

                "ks_pvalue":
                    float(
                        ks_result.pvalue
                    ),

                "psi":
                    psi,

                "shift_status":
                    shift_status,
            }
        )

final_audit = pd.DataFrame(
    audit_rows
)


# -----------------------------------------------------------------------------
# Save final tensor and documentation
# -----------------------------------------------------------------------------

np.save(
    OUTPUT_PATH,
    X_temporal_final,
)

manifest = {
    "tensor_file":
        OUTPUT_PATH.name,

    "shape":
        list(X_temporal_final.shape),

    "dtype":
        str(X_temporal_final.dtype),

    "feature_order":
        TEMPORAL_FEATURE_NAMES,

    "retained_source_indices":
        retained_feature_indices,

    "removed_feature":
        "normalized_log_interarrival",

    "removal_basis":
        (
            "Validation-only distribution audit: "
            "PSI=0.500157 and KS=0.203997."
        ),

    "feature_definitions": {
        "relative_elapsed_time":
            (
                "(timestamp_i - timestamp_first) / "
                "(timestamp_last - timestamp_first)"
            ),

        "hour_sine":
            "sin(2*pi*fractional_hour/24)",

        "hour_cosine":
            "cos(2*pi*fractional_hour/24)",
    },

    "selection_policy":
        (
            "Feature removal was determined from "
            "training-to-validation shift. Test labels "
            "were not used."
        ),

    "original_tensors_preserved": [
        "X_temporal.npy",
        "X_temporal_stable.npy",
    ],
}

with open(
    MANIFEST_PATH,
    "w",
    encoding="utf-8",
) as file:
    json.dump(
        manifest,
        file,
        indent=2,
    )

final_audit.to_csv(
    AUDIT_PATH,
    index=False,
)


# -----------------------------------------------------------------------------
# Display
# -----------------------------------------------------------------------------

print("=" * 80)
print("FINAL TEMPORAL TENSOR")
print("=" * 80)

print("Input shape:", X_temporal_stable.shape)
print("Final shape:", X_temporal_final.shape)

print("\nRetained features:")
for index, feature_name in enumerate(
    TEMPORAL_FEATURE_NAMES
):
    print(f"{index}: {feature_name}")

print(
    "\nRemoved feature: "
    "normalized_log_interarrival"
)

print(
    "Removal basis: validation PSI=0.500157, "
    "validation KS=0.203997"
)

print("\nFinal temporal shift audit:")
display(
    final_audit.round(6)
)

print("\nSaved final tensor:")
print(OUTPUT_PATH)

print("\nSaved manifest:")
print(MANIFEST_PATH)

print("\nOriginal temporal tensors were preserved.")

In [ ]:
# =============================================================================
# CELL 18: FINAL-TEMPORAL STANDARDIZED MODALITY ABLATION
# =============================================================================

from pathlib import Path
import time
import warnings

import numpy as np
import pandas as pd

from sklearn.exceptions import ConvergenceWarning
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    matthews_corrcoef,
    roc_auc_score,
    average_precision_score,
)
from sklearn.multiclass import OneVsRestClassifier
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

warnings.filterwarnings(
    "ignore",
    category=ConvergenceWarning,
)


# -----------------------------------------------------------------------------
# Configuration
# -----------------------------------------------------------------------------

SEED = 42

DATA_DIR = Path(
    r"."
)

TENSOR_DIR = (
    DATA_DIR
    / "outputs"
    / "am_jepa_tensors"
)

OUTPUT_DIR = (
    DATA_DIR
    / "outputs"
    / "modality_ablation_final_temporal"
)

PREVIOUS_RESULTS_PATH = (
    DATA_DIR
    / "outputs"
    / "modality_ablation_scaled"
    / "modality_ablation_summary_scaled.csv"
)

OUTPUT_DIR.mkdir(
    parents=True,
    exist_ok=True,
)

ASPECT_NAMES = [
    "reconnaissance",
    "initial_access_foothold",
    "lateral_movement",
    "data_exfiltration",
]

THRESHOLD_GRID = np.linspace(
    0.01,
    0.99,
    99,
)

CONTROL_CONFIGURATIONS = [
    "network_only",
    "text_only",
    "network_text",
]


# -----------------------------------------------------------------------------
# Flexible tensor loader
# -----------------------------------------------------------------------------

def load_first_existing(
    directory,
    candidate_names,
):
    for filename in candidate_names:
        path = directory / filename

        if path.exists():
            print(f"Loaded: {path.name}")

            return np.load(
                path,
                allow_pickle=False,
            )

    raise FileNotFoundError(
        "None of these files were found:\n"
        + "\n".join(candidate_names)
    )


# -----------------------------------------------------------------------------
# Load tensors
# -----------------------------------------------------------------------------

X_network = load_first_existing(
    TENSOR_DIR,
    [
        "X_network.npy",
        "x_network.npy",
    ],
).astype(np.float32)

# The final validation-stable tensor is prioritized.
X_temporal = load_first_existing(
    TENSOR_DIR,
    [
        "X_temporal_final.npy",
        "X_temporal_stable.npy",
        "X_temporal.npy",
    ],
).astype(np.float32)

attention_mask = load_first_existing(
    TENSOR_DIR,
    [
        "attention_mask.npy",
        "window_attention_mask.npy",
    ],
).astype(np.float32)

text_embedding_lookup = load_first_existing(
    TENSOR_DIR,
    [
        "text_embedding_lookup.npy",
    ],
).astype(np.float32)

window_text_ids = load_first_existing(
    TENSOR_DIR,
    [
        "window_text_ids.npy",
    ],
).astype(np.int64)

aspect_targets = load_first_existing(
    TENSOR_DIR,
    [
        "aspect_presence_targets.npy",
        "y_aspect.npy",
    ],
).astype(np.int64)

campaign_targets = load_first_existing(
    TENSOR_DIR,
    [
        "campaign_targets.npy",
        "y_campaign.npy",
    ],
).astype(np.int64)

train_indices = load_first_existing(
    TENSOR_DIR,
    [
        "train_indices.npy",
        "train_idx.npy",
    ],
).astype(np.int64)

validation_indices = load_first_existing(
    TENSOR_DIR,
    [
        "validation_indices.npy",
        "validation_idx.npy",
        "val_indices.npy",
        "val_idx.npy",
    ],
).astype(np.int64)

test_indices = load_first_existing(
    TENSOR_DIR,
    [
        "test_indices.npy",
        "test_idx.npy",
    ],
).astype(np.int64)


# -----------------------------------------------------------------------------
# Structural verification
# -----------------------------------------------------------------------------

number_of_windows = X_network.shape[0]
window_size = X_network.shape[1]

assert X_temporal.shape[:2] == (
    number_of_windows,
    window_size,
)

assert X_temporal.shape[2] == 3, (
    "Expected final temporal dimension 3, "
    f"but received {X_temporal.shape[2]}."
)

assert attention_mask.shape == (
    number_of_windows,
    window_size,
)

assert window_text_ids.shape == (
    number_of_windows,
    window_size,
)

assert aspect_targets.shape == (
    number_of_windows,
    len(ASPECT_NAMES),
)

assert campaign_targets.shape == (
    number_of_windows,
)

assert window_text_ids.min() >= 0

assert window_text_ids.max() < len(
    text_embedding_lookup
)

all_indices = np.concatenate(
    [
        train_indices,
        validation_indices,
        test_indices,
    ]
)

assert len(all_indices) == number_of_windows
assert len(np.unique(all_indices)) == number_of_windows

print("\nTensor shapes:")
print("Network:", X_network.shape)
print("Final temporal:", X_temporal.shape)
print("Text lookup:", text_embedding_lookup.shape)
print("Text IDs:", window_text_ids.shape)
print("Mask:", attention_mask.shape)

print("\nPartitions:")
print("Train:", len(train_indices))
print("Validation:", len(validation_indices))
print("Test:", len(test_indices))


# -----------------------------------------------------------------------------
# Mask-aware mean pooling
# -----------------------------------------------------------------------------

def masked_mean_pool(
    sequence_tensor,
    mask,
):
    expanded_mask = (
        mask.astype(np.float32)[..., None]
    )

    numerator = (
        sequence_tensor
        * expanded_mask
    ).sum(axis=1)

    denominator = np.clip(
        expanded_mask.sum(axis=1),
        a_min=1.0,
        a_max=None,
    )

    return (
        numerator / denominator
    ).astype(np.float32)


X_text = text_embedding_lookup[
    window_text_ids
]

network_pooled = masked_mean_pool(
    X_network,
    attention_mask,
)

temporal_pooled = masked_mean_pool(
    X_temporal,
    attention_mask,
)

text_pooled = masked_mean_pool(
    X_text,
    attention_mask,
)

for representation_name, representation in {
    "network": network_pooled,
    "final_temporal": temporal_pooled,
    "text": text_pooled,
}.items():
    if not np.isfinite(
        representation
    ).all():
        raise ValueError(
            f"{representation_name} contains "
            "non-finite values."
        )

print("\nPooled representations:")
print("Network:", network_pooled.shape)
print("Final temporal:", temporal_pooled.shape)
print("Text:", text_pooled.shape)


# -----------------------------------------------------------------------------
# Modality configurations
# -----------------------------------------------------------------------------

MODALITY_CONFIGURATIONS = {
    "network_only":
        network_pooled,

    "temporal_only":
        temporal_pooled,

    "text_only":
        text_pooled,

    "network_temporal":
        np.concatenate(
            [
                network_pooled,
                temporal_pooled,
            ],
            axis=1,
        ),

    "network_text":
        np.concatenate(
            [
                network_pooled,
                text_pooled,
            ],
            axis=1,
        ),

    "full_multimodal":
        np.concatenate(
            [
                network_pooled,
                temporal_pooled,
                text_pooled,
            ],
            axis=1,
        ),
}


# -----------------------------------------------------------------------------
# Metric helpers
# -----------------------------------------------------------------------------

def safe_probability_metric(
    metric_function,
    y_true,
    probabilities,
):
    try:
        return float(
            metric_function(
                y_true,
                probabilities,
            )
        )
    except ValueError:
        return np.nan


def calculate_binary_metrics(
    y_true,
    probabilities,
    threshold,
):
    predictions = (
        probabilities >= threshold
    ).astype(np.int64)

    return {
        "threshold":
            float(threshold),

        "accuracy":
            accuracy_score(
                y_true,
                predictions,
            ),

        "balanced_accuracy":
            balanced_accuracy_score(
                y_true,
                predictions,
            ),

        "precision":
            precision_score(
                y_true,
                predictions,
                zero_division=0,
            ),

        "recall":
            recall_score(
                y_true,
                predictions,
                zero_division=0,
            ),

        "f1":
            f1_score(
                y_true,
                predictions,
                zero_division=0,
            ),

        "mcc":
            matthews_corrcoef(
                y_true,
                predictions,
            ),

        "auroc":
            safe_probability_metric(
                roc_auc_score,
                y_true,
                probabilities,
            ),

        "auprc":
            safe_probability_metric(
                average_precision_score,
                y_true,
                probabilities,
            ),
    }


def select_validation_threshold(
    y_true,
    probabilities,
):
    rows = []

    for threshold in THRESHOLD_GRID:
        predictions = (
            probabilities >= threshold
        ).astype(np.int64)

        rows.append(
            {
                "threshold":
                    float(threshold),

                "mcc":
                    matthews_corrcoef(
                        y_true,
                        predictions,
                    ),

                "balanced_accuracy":
                    balanced_accuracy_score(
                        y_true,
                        predictions,
                    ),

                "f1":
                    f1_score(
                        y_true,
                        predictions,
                        zero_division=0,
                    ),

                "distance_from_0.5":
                    abs(threshold - 0.5),
            }
        )

    search_table = pd.DataFrame(rows)

    best_row = (
        search_table
        .sort_values(
            by=[
                "mcc",
                "balanced_accuracy",
                "f1",
                "distance_from_0.5",
            ],
            ascending=[
                False,
                False,
                False,
                True,
            ],
        )
        .iloc[0]
    )

    return float(
        best_row["threshold"]
    )


# -----------------------------------------------------------------------------
# Leakage-safe standardized model
# -----------------------------------------------------------------------------

def build_logistic_model():
    return Pipeline(
        steps=[
            (
                "standardizer",
                StandardScaler(),
            ),
            (
                "classifier",
                LogisticRegression(
                    class_weight="balanced",
                    solver="lbfgs",
                    max_iter=5000,
                    random_state=SEED,
                ),
            ),
        ]
    )


# -----------------------------------------------------------------------------
# Partition targets
# -----------------------------------------------------------------------------

y_campaign_train = campaign_targets[
    train_indices
]

y_campaign_validation = campaign_targets[
    validation_indices
]

y_campaign_test = campaign_targets[
    test_indices
]

y_aspect_train = aspect_targets[
    train_indices
]

y_aspect_validation = aspect_targets[
    validation_indices
]

y_aspect_test = aspect_targets[
    test_indices
]


# -----------------------------------------------------------------------------
# Train and evaluate all configurations
# -----------------------------------------------------------------------------

campaign_rows = []
aspect_rows = []
runtime_rows = []

for configuration_name, representation in (
    MODALITY_CONFIGURATIONS.items()
):
    print("\n" + "=" * 80)
    print("Configuration:", configuration_name)
    print(
        "Feature dimension:",
        representation.shape[1],
    )

    X_train_configuration = representation[
        train_indices
    ]

    X_validation_configuration = representation[
        validation_indices
    ]

    X_test_configuration = representation[
        test_indices
    ]

    # -----------------------------------------------------------------
    # Campaign classifier
    # -----------------------------------------------------------------

    campaign_start = time.perf_counter()

    campaign_classifier = (
        build_logistic_model()
    )

    campaign_classifier.fit(
        X_train_configuration,
        y_campaign_train,
    )

    campaign_training_seconds = (
        time.perf_counter()
        - campaign_start
    )

    campaign_validation_probabilities = (
        campaign_classifier.predict_proba(
            X_validation_configuration
        )[:, 1]
    )

    campaign_test_probabilities = (
        campaign_classifier.predict_proba(
            X_test_configuration
        )[:, 1]
    )

    campaign_threshold = (
        select_validation_threshold(
            y_campaign_validation,
            campaign_validation_probabilities,
        )
    )

    campaign_metrics = (
        calculate_binary_metrics(
            y_campaign_test,
            campaign_test_probabilities,
            campaign_threshold,
        )
    )

    campaign_rows.append(
        {
            "configuration":
                configuration_name,

            "feature_dimension":
                representation.shape[1],

            "temporal_tensor":
                "X_temporal_final.npy",

            "standardization":
                "train_only",

            **campaign_metrics,
        }
    )

    # -----------------------------------------------------------------
    # Aspect classifiers
    # -----------------------------------------------------------------

    aspect_start = time.perf_counter()

    aspect_classifier = (
        OneVsRestClassifier(
            build_logistic_model(),
            n_jobs=-1,
        )
    )

    aspect_classifier.fit(
        X_train_configuration,
        y_aspect_train,
    )

    aspect_training_seconds = (
        time.perf_counter()
        - aspect_start
    )

    aspect_validation_probabilities = (
        aspect_classifier.predict_proba(
            X_validation_configuration
        )
    )

    aspect_test_probabilities = (
        aspect_classifier.predict_proba(
            X_test_configuration
        )
    )

    for aspect_index, aspect_name in enumerate(
        ASPECT_NAMES
    ):
        validation_labels = (
            y_aspect_validation[
                :,
                aspect_index,
            ]
        )

        test_labels = (
            y_aspect_test[
                :,
                aspect_index,
            ]
        )

        validation_probabilities = (
            aspect_validation_probabilities[
                :,
                aspect_index,
            ]
        )

        test_probabilities = (
            aspect_test_probabilities[
                :,
                aspect_index,
            ]
        )

        validation_positive_count = int(
            validation_labels.sum()
        )

        if validation_positive_count >= 5:
            aspect_threshold = (
                select_validation_threshold(
                    validation_labels,
                    validation_probabilities,
                )
            )

            threshold_status = (
                "validation_calibrated"
            )

        else:
            aspect_threshold = 0.5

            threshold_status = (
                "fixed_insufficient_positives"
            )

        aspect_metrics = (
            calculate_binary_metrics(
                test_labels,
                test_probabilities,
                aspect_threshold,
            )
        )

        aspect_rows.append(
            {
                "configuration":
                    configuration_name,

                "feature_dimension":
                    representation.shape[1],

                "temporal_tensor":
                    "X_temporal_final.npy",

                "standardization":
                    "train_only",

                "aspect":
                    aspect_name,

                "validation_positive_windows":
                    validation_positive_count,

                "test_positive_windows":
                    int(test_labels.sum()),

                "threshold_status":
                    threshold_status,

                **aspect_metrics,
            }
        )

    runtime_rows.append(
        {
            "configuration":
                configuration_name,

            "feature_dimension":
                representation.shape[1],

            "campaign_training_seconds":
                campaign_training_seconds,

            "aspect_training_seconds":
                aspect_training_seconds,

            "total_training_seconds":
                campaign_training_seconds
                + aspect_training_seconds,
        }
    )

    print(
        "Campaign threshold:",
        round(campaign_threshold, 4),
    )

    print(
        "Campaign F1:",
        round(campaign_metrics["f1"], 4),
    )

    print(
        "Campaign AUROC:",
        round(campaign_metrics["auroc"], 4),
    )


# -----------------------------------------------------------------------------
# Construct result tables
# -----------------------------------------------------------------------------

campaign_ablation = pd.DataFrame(
    campaign_rows
)

aspect_ablation = pd.DataFrame(
    aspect_rows
)

runtime_ablation = pd.DataFrame(
    runtime_rows
)


# All four aspects
aspect_macro_all = (
    aspect_ablation
    .groupby(
        [
            "configuration",
            "feature_dimension",
        ],
        as_index=False,
    )
    .agg(
        macro_precision=(
            "precision",
            "mean",
        ),
        macro_recall=(
            "recall",
            "mean",
        ),
        macro_f1=(
            "f1",
            "mean",
        ),
        macro_mcc=(
            "mcc",
            "mean",
        ),
        macro_auroc=(
            "auroc",
            "mean",
        ),
        macro_auprc=(
            "auprc",
            "mean",
        ),
    )
)


# Core-three aspects, excluding exploratory exfiltration
core_three_results = aspect_ablation[
    aspect_ablation["aspect"]
    != "data_exfiltration"
]

aspect_macro_core = (
    core_three_results
    .groupby(
        [
            "configuration",
            "feature_dimension",
        ],
        as_index=False,
    )
    .agg(
        core3_macro_precision=(
            "precision",
            "mean",
        ),
        core3_macro_recall=(
            "recall",
            "mean",
        ),
        core3_macro_f1=(
            "f1",
            "mean",
        ),
        core3_macro_mcc=(
            "mcc",
            "mean",
        ),
        core3_macro_auroc=(
            "auroc",
            "mean",
        ),
        core3_macro_auprc=(
            "auprc",
            "mean",
        ),
    )
)


summary_table = (
    campaign_ablation
    .merge(
        aspect_macro_all,
        on=[
            "configuration",
            "feature_dimension",
        ],
        how="left",
    )
    .merge(
        aspect_macro_core,
        on=[
            "configuration",
            "feature_dimension",
        ],
        how="left",
    )
    .merge(
        runtime_ablation,
        on=[
            "configuration",
            "feature_dimension",
        ],
        how="left",
    )
)

summary_table = (
    summary_table
    .sort_values(
        by=[
            "f1",
            "core3_macro_f1",
        ],
        ascending=False,
    )
    .reset_index(drop=True)
)

summary_table.insert(
    0,
    "campaign_f1_rank",
    summary_table["f1"]
    .rank(
        ascending=False,
        method="min",
    )
    .astype(int),
)


# -----------------------------------------------------------------------------
# Reproducibility-control comparison
# -----------------------------------------------------------------------------

control_rows = []

if PREVIOUS_RESULTS_PATH.exists():
    previous_results = pd.read_csv(
        PREVIOUS_RESULTS_PATH
    )

    metrics_to_compare = [
        "threshold",
        "accuracy",
        "balanced_accuracy",
        "precision",
        "recall",
        "f1",
        "mcc",
        "auroc",
        "auprc",
        "macro_f1",
        "macro_mcc",
        "core3_macro_f1",
        "core3_macro_mcc",
    ]

    for configuration_name in (
        CONTROL_CONFIGURATIONS
    ):
        previous_row = (
            previous_results[
                previous_results[
                    "configuration"
                ] == configuration_name
            ]
            .iloc[0]
        )

        current_row = (
            summary_table[
                summary_table[
                    "configuration"
                ] == configuration_name
            ]
            .iloc[0]
        )

        maximum_difference = 0.0

        for metric in metrics_to_compare:
            difference = abs(
                float(current_row[metric])
                - float(previous_row[metric])
            )

            maximum_difference = max(
                maximum_difference,
                difference,
            )

        control_rows.append(
            {
                "configuration":
                    configuration_name,

                "maximum_absolute_difference":
                    maximum_difference,

                "reproducibility_passed":
                    maximum_difference < 1e-10,
            }
        )

    control_verification = pd.DataFrame(
        control_rows
    )

    if not control_verification[
        "reproducibility_passed"
    ].all():
        raise AssertionError(
            "A non-temporal control configuration changed. "
            "Review the pipeline before accepting results."
        )

else:
    control_verification = pd.DataFrame(
        {
            "configuration":
                CONTROL_CONFIGURATIONS,

            "maximum_absolute_difference":
                np.nan,

            "reproducibility_passed":
                False,
        }
    )

    print(
        "\nWarning: previous scaled summary was not "
        "found, so automatic control comparison "
        "could not be performed."
    )


# -----------------------------------------------------------------------------
# Save results
# -----------------------------------------------------------------------------

campaign_ablation.to_csv(
    OUTPUT_DIR
    / "campaign_ablation_final_temporal.csv",
    index=False,
)

aspect_ablation.to_csv(
    OUTPUT_DIR
    / "aspect_ablation_final_temporal.csv",
    index=False,
)

aspect_macro_all.to_csv(
    OUTPUT_DIR
    / "aspect_macro_all_final_temporal.csv",
    index=False,
)

aspect_macro_core.to_csv(
    OUTPUT_DIR
    / "aspect_macro_core_final_temporal.csv",
    index=False,
)

runtime_ablation.to_csv(
    OUTPUT_DIR
    / "runtime_final_temporal.csv",
    index=False,
)

summary_table.to_csv(
    OUTPUT_DIR
    / "modality_ablation_summary_final_temporal.csv",
    index=False,
)

control_verification.to_csv(
    OUTPUT_DIR
    / "non_temporal_control_verification.csv",
    index=False,
)


# -----------------------------------------------------------------------------
# Display
# -----------------------------------------------------------------------------

print("\n" + "=" * 80)
print("FINAL-TEMPORAL MODALITY ABLATION SUMMARY")
print("=" * 80)

summary_columns = [
    "campaign_f1_rank",
    "configuration",
    "feature_dimension",
    "threshold",
    "accuracy",
    "balanced_accuracy",
    "precision",
    "recall",
    "f1",
    "mcc",
    "auroc",
    "auprc",
    "macro_f1",
    "macro_mcc",
    "core3_macro_f1",
    "core3_macro_mcc",
    "total_training_seconds",
]

display(
    summary_table[
        summary_columns
    ].round(6)
)

print("\nPer-aspect results:")

display(
    aspect_ablation[
        [
            "configuration",
            "aspect",
            "threshold",
            "precision",
            "recall",
            "f1",
            "mcc",
            "auroc",
            "auprc",
        ]
    ].round(6)
)

print("\nNon-temporal reproducibility controls:")

display(
    control_verification
)

best_campaign_row = (
    summary_table.iloc[0]
)

best_core3_row = (
    summary_table
    .sort_values(
        by="core3_macro_f1",
        ascending=False,
    )
    .iloc[0]
)

print("\nBest campaign configuration:")
print(
    best_campaign_row["configuration"],
    "| F1:",
    round(
        best_campaign_row["f1"],
        6,
    ),
    "| AUROC:",
    round(
        best_campaign_row["auroc"],
        6,
    ),
)

print("\nBest core-three aspect configuration:")
print(
    best_core3_row["configuration"],
    "| Core-three macro-F1:",
    round(
        best_core3_row[
            "core3_macro_f1"
        ],
        6,
    ),
)

print("\nSaved final-temporal ablation results to:")
print(OUTPUT_DIR)

In [ ]:
# =============================================================================
# CELL 19: AM-JEPA DATASET AND NON-OVERLAPPING FUTURE-WINDOW PAIRS
# =============================================================================

from pathlib import Path
import json
import random

import numpy as np
import pandas as pd
import torch
from torch.utils.data import Dataset, DataLoader


# -----------------------------------------------------------------------------
# Reproducibility
# -----------------------------------------------------------------------------

SEED = 42

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False


# -----------------------------------------------------------------------------
# Paths and configuration
# -----------------------------------------------------------------------------

DATA_DIR = Path(
    r"."
)

OUTPUT_DIR = DATA_DIR / "outputs"

TENSOR_DIR = (
    OUTPUT_DIR
    / "am_jepa_tensors"
)

AMJEPA_DIR = (
    OUTPUT_DIR
    / "am_jepa_experiment"
)

AMJEPA_DIR.mkdir(
    parents=True,
    exist_ok=True,
)

MEMBERSHIP_PATH = (
    OUTPUT_DIR
    / "dapt2020_mast_window_membership_enriched.parquet"
)

WINDOWS_PATH = (
    OUTPUT_DIR
    / "dapt2020_mast_windows.parquet"
)

BATCH_SIZE = 64
NUMBER_OF_WORKERS = 0  # Recommended for Windows notebooks.


# -----------------------------------------------------------------------------
# Load AM-JEPA tensors
# -----------------------------------------------------------------------------

X_network = np.load(
    TENSOR_DIR / "X_network.npy",
    allow_pickle=False,
).astype(np.float32)

X_temporal = np.load(
    TENSOR_DIR / "X_temporal_final.npy",
    allow_pickle=False,
).astype(np.float32)

attention_mask = np.load(
    TENSOR_DIR / "attention_mask.npy",
    allow_pickle=False,
).astype(bool)

text_embedding_lookup = np.load(
    TENSOR_DIR / "text_embedding_lookup.npy",
    allow_pickle=False,
).astype(np.float32)

window_text_ids = np.load(
    TENSOR_DIR / "window_text_ids.npy",
    allow_pickle=False,
).astype(np.int64)

campaign_targets = np.load(
    TENSOR_DIR / "campaign_targets.npy",
    allow_pickle=False,
).astype(np.int64)

aspect_targets = np.load(
    TENSOR_DIR / "aspect_presence_targets.npy",
    allow_pickle=False,
).astype(np.int64)

train_indices = np.load(
    TENSOR_DIR / "train_indices.npy",
    allow_pickle=False,
).astype(np.int64)

validation_indices = np.load(
    TENSOR_DIR / "validation_indices.npy",
    allow_pickle=False,
).astype(np.int64)

test_indices = np.load(
    TENSOR_DIR / "test_indices.npy",
    allow_pickle=False,
).astype(np.int64)


# -----------------------------------------------------------------------------
# Recover tensor window order
# -----------------------------------------------------------------------------

window_ids_path = (
    TENSOR_DIR / "window_ids.npy"
)

try:
    window_ids = np.load(
        window_ids_path,
        allow_pickle=False,
    )
except ValueError:
    window_ids = np.load(
        window_ids_path,
        allow_pickle=True,
    )

window_ids = np.asarray(
    window_ids
).astype(str)


# -----------------------------------------------------------------------------
# Validate tensor structure
# -----------------------------------------------------------------------------

number_of_windows = X_network.shape[0]
window_size = X_network.shape[1]

assert X_network.shape == (
    number_of_windows,
    window_size,
    79,
)

assert X_temporal.shape == (
    number_of_windows,
    window_size,
    3,
)

assert attention_mask.shape == (
    number_of_windows,
    window_size,
)

assert window_text_ids.shape == (
    number_of_windows,
    window_size,
)

assert campaign_targets.shape == (
    number_of_windows,
)

assert aspect_targets.shape == (
    number_of_windows,
    4,
)

assert len(window_ids) == number_of_windows

assert window_text_ids.min() >= 0

assert window_text_ids.max() < len(
    text_embedding_lookup
)

all_partition_indices = np.concatenate(
    [
        train_indices,
        validation_indices,
        test_indices,
    ]
)

assert len(all_partition_indices) == (
    number_of_windows
)

assert len(
    np.unique(all_partition_indices)
) == number_of_windows

print("=" * 80)
print("AM-JEPA INPUT VERIFICATION")
print("=" * 80)

print("Network:", X_network.shape)
print("Temporal:", X_temporal.shape)
print("Text lookup:", text_embedding_lookup.shape)
print("Text IDs:", window_text_ids.shape)
print("Attention mask:", attention_mask.shape)

print("\nPartitions:")
print("Train:", len(train_indices))
print("Validation:", len(validation_indices))
print("Test:", len(test_indices))


# -----------------------------------------------------------------------------
# Load window membership
# -----------------------------------------------------------------------------

membership = pd.read_parquet(
    MEMBERSHIP_PATH,
    columns=[
        "window_id",
        "split",
        "day",
        "position",
        "evidence_id",
        "timestamp_parsed",
    ],
)

membership["window_id"] = (
    membership["window_id"].astype(str)
)

membership["evidence_id"] = (
    membership["evidence_id"].astype(str)
)

membership["timestamp_parsed"] = (
    pd.to_datetime(
        membership["timestamp_parsed"],
        errors="coerce",
    )
)

if membership[
    "timestamp_parsed"
].isna().any():
    raise ValueError(
        "Membership contains invalid timestamps."
    )

window_to_tensor_index = {
    window_id: index
    for index, window_id in enumerate(
        window_ids
    )
}

membership["tensor_index"] = (
    membership["window_id"]
    .map(window_to_tensor_index)
)

if membership[
    "tensor_index"
].isna().any():
    raise ValueError(
        "Some membership windows were not found "
        "in window_ids.npy."
    )

membership["tensor_index"] = (
    membership["tensor_index"]
    .astype(np.int64)
)


# -----------------------------------------------------------------------------
# Construct window metadata
# -----------------------------------------------------------------------------

window_metadata = (
    membership
    .groupby(
        "window_id",
        as_index=False,
    )
    .agg(
        split=("split", "first"),
        day=("day", "first"),
        start_timestamp=(
            "timestamp_parsed",
            "min",
        ),
        end_timestamp=(
            "timestamp_parsed",
            "max",
        ),
        evidence_count=(
            "evidence_id",
            "size",
        ),
        tensor_index=(
            "tensor_index",
            "first",
        ),
    )
)

assert len(window_metadata) == (
    number_of_windows
)

assert (
    window_metadata["tensor_index"]
    .nunique()
    == number_of_windows
)

window_metadata = (
    window_metadata
    .sort_values("tensor_index")
    .reset_index(drop=True)
)


# -----------------------------------------------------------------------------
# Evidence sets for overlap checking
# -----------------------------------------------------------------------------

window_evidence_sets = {
    window_id: set(
        group["evidence_id"].tolist()
    )
    for window_id, group in membership.groupby(
        "window_id",
        sort=False,
    )
}


# -----------------------------------------------------------------------------
# Build earliest non-overlapping future-window targets
# -----------------------------------------------------------------------------

future_window_index = np.full(
    number_of_windows,
    fill_value=-1,
    dtype=np.int64,
)

future_pair_rows = []

grouped_metadata = window_metadata.groupby(
    [
        "split",
        "day",
    ],
    sort=False,
)

for (
    split_name,
    day_name,
), group in grouped_metadata:
    ordered_group = (
        group
        .sort_values(
            by=[
                "start_timestamp",
                "end_timestamp",
                "tensor_index",
            ]
        )
        .reset_index(drop=True)
    )

    group_records = (
        ordered_group.to_dict(
            orient="records"
        )
    )

    for current_position, current_row in enumerate(
        group_records
    ):
        current_window_id = (
            current_row["window_id"]
        )

        current_evidence = (
            window_evidence_sets[
                current_window_id
            ]
        )

        selected_future_row = None

        for future_position in range(
            current_position + 1,
            len(group_records),
        ):
            candidate_row = group_records[
                future_position
            ]

            candidate_window_id = (
                candidate_row["window_id"]
            )

            candidate_evidence = (
                window_evidence_sets[
                    candidate_window_id
                ]
            )

            overlap_count = len(
                current_evidence.intersection(
                    candidate_evidence
                )
            )

            chronological = (
                candidate_row["start_timestamp"]
                >= current_row["start_timestamp"]
            )

            if (
                overlap_count == 0
                and chronological
            ):
                selected_future_row = (
                    candidate_row
                )

                break

        if selected_future_row is None:
            continue

        current_index = int(
            current_row["tensor_index"]
        )

        selected_future_index = int(
            selected_future_row[
                "tensor_index"
            ]
        )

        future_window_index[
            current_index
        ] = selected_future_index

        time_gap_seconds = (
            selected_future_row[
                "start_timestamp"
            ]
            - current_row[
                "end_timestamp"
            ]
        ).total_seconds()

        future_pair_rows.append(
            {
                "split":
                    split_name,

                "day":
                    day_name,

                "current_window_id":
                    current_window_id,

                "current_tensor_index":
                    current_index,

                "future_window_id":
                    selected_future_row[
                        "window_id"
                    ],

                "future_tensor_index":
                    selected_future_index,

                "current_start":
                    current_row[
                        "start_timestamp"
                    ],

                "current_end":
                    current_row[
                        "end_timestamp"
                    ],

                "future_start":
                    selected_future_row[
                        "start_timestamp"
                    ],

                "future_end":
                    selected_future_row[
                        "end_timestamp"
                    ],

                "evidence_overlap":
                    0,

                "time_gap_seconds":
                    time_gap_seconds,
            }
        )

future_pairs = pd.DataFrame(
    future_pair_rows
)


# -----------------------------------------------------------------------------
# Future-pair verification
# -----------------------------------------------------------------------------

assert len(future_pairs) > 0

assert (
    future_pairs["evidence_overlap"] == 0
).all()

assert (
    future_pairs[
        "current_tensor_index"
    ].duplicated().sum()
    == 0
)

for row in future_pairs.itertuples(
    index=False
):
    current_set = window_evidence_sets[
        row.current_window_id
    ]

    future_set = window_evidence_sets[
        row.future_window_id
    ]

    assert current_set.isdisjoint(
        future_set
    )

    current_split = window_metadata.loc[
        window_metadata["tensor_index"]
        == row.current_tensor_index,
        "split",
    ].iloc[0]

    future_split = window_metadata.loc[
        window_metadata["tensor_index"]
        == row.future_tensor_index,
        "split",
    ].iloc[0]

    assert current_split == future_split


# -----------------------------------------------------------------------------
# Save future-pair artifacts
# -----------------------------------------------------------------------------

np.save(
    AMJEPA_DIR
    / "future_window_index.npy",
    future_window_index,
)

future_pairs.to_parquet(
    AMJEPA_DIR
    / "future_window_pairs.parquet",
    index=False,
)

future_pairs.to_csv(
    AMJEPA_DIR
    / "future_window_pairs.csv",
    index=False,
)


# -----------------------------------------------------------------------------
# Partition-specific pretraining indices
# -----------------------------------------------------------------------------

train_index_set = set(
    train_indices.tolist()
)

validation_index_set = set(
    validation_indices.tolist()
)

test_index_set = set(
    test_indices.tolist()
)

pretraining_train_indices = np.asarray(
    [
        index
        for index in train_indices
        if future_window_index[index] >= 0
        and future_window_index[index]
        in train_index_set
    ],
    dtype=np.int64,
)

pretraining_validation_indices = np.asarray(
    [
        index
        for index in validation_indices
        if future_window_index[index] >= 0
        and future_window_index[index]
        in validation_index_set
    ],
    dtype=np.int64,
)

# The test partition is deliberately excluded from
# self-supervised pretraining and model selection.
pretraining_test_indices = np.asarray(
    [
        index
        for index in test_indices
        if future_window_index[index] >= 0
        and future_window_index[index]
        in test_index_set
    ],
    dtype=np.int64,
)

assert len(
    set(pretraining_train_indices)
    .intersection(
        pretraining_validation_indices
    )
) == 0

assert len(
    set(pretraining_train_indices)
    .intersection(
        pretraining_test_indices
    )
) == 0

np.save(
    AMJEPA_DIR
    / "pretraining_train_indices.npy",
    pretraining_train_indices,
)

np.save(
    AMJEPA_DIR
    / "pretraining_validation_indices.npy",
    pretraining_validation_indices,
)

np.save(
    AMJEPA_DIR
    / "pretraining_test_indices.npy",
    pretraining_test_indices,
)


# -----------------------------------------------------------------------------
# PyTorch dataset
# -----------------------------------------------------------------------------

class AMJEPAPairDataset(Dataset):
    """
    Returns a current investigation window and its earliest
    non-overlapping future window.

    Campaign and aspect labels are included only for later
    frozen-probe evaluation. They are not consumed by the
    self-supervised AM-JEPA objective.
    """

    def __init__(
        self,
        indices,
        network,
        temporal,
        text_ids,
        text_lookup,
        mask,
        future_index,
        campaign_labels,
        aspect_labels,
    ):
        self.indices = np.asarray(
            indices,
            dtype=np.int64,
        )

        self.network = network
        self.temporal = temporal
        self.text_ids = text_ids
        self.text_lookup = text_lookup
        self.mask = mask
        self.future_index = future_index
        self.campaign_labels = (
            campaign_labels
        )
        self.aspect_labels = aspect_labels

    def __len__(self):
        return len(self.indices)

    def _get_window(self, index):
        text_embeddings = self.text_lookup[
            self.text_ids[index]
        ]

        return {
            "network":
                torch.from_numpy(
                    self.network[index]
                ).float(),

            "temporal":
                torch.from_numpy(
                    self.temporal[index]
                ).float(),

            "text":
                torch.from_numpy(
                    text_embeddings
                ).float(),

            "attention_mask":
                torch.from_numpy(
                    self.mask[index]
                ).bool(),
        }

    def __getitem__(self, item):
        current_index = int(
            self.indices[item]
        )

        future_index = int(
            self.future_index[
                current_index
            ]
        )

        if future_index < 0:
            raise RuntimeError(
                "Dataset received a window without "
                "a valid future target."
            )

        current = self._get_window(
            current_index
        )

        future = self._get_window(
            future_index
        )

        return {
            "current_network":
                current["network"],

            "current_temporal":
                current["temporal"],

            "current_text":
                current["text"],

            "current_attention_mask":
                current["attention_mask"],

            "future_network":
                future["network"],

            "future_temporal":
                future["temporal"],

            "future_text":
                future["text"],

            "future_attention_mask":
                future["attention_mask"],

            # Kept only for later probing/auditing.
            "campaign_target":
                torch.tensor(
                    self.campaign_labels[
                        current_index
                    ],
                    dtype=torch.long,
                ),

            "aspect_target":
                torch.from_numpy(
                    self.aspect_labels[
                        current_index
                    ]
                ).float(),

            "current_index":
                torch.tensor(
                    current_index,
                    dtype=torch.long,
                ),

            "future_index":
                torch.tensor(
                    future_index,
                    dtype=torch.long,
                ),
        }


# -----------------------------------------------------------------------------
# Create pretraining datasets and data loaders
# -----------------------------------------------------------------------------

train_pretraining_dataset = (
    AMJEPAPairDataset(
        indices=pretraining_train_indices,
        network=X_network,
        temporal=X_temporal,
        text_ids=window_text_ids,
        text_lookup=text_embedding_lookup,
        mask=attention_mask,
        future_index=future_window_index,
        campaign_labels=campaign_targets,
        aspect_labels=aspect_targets,
    )
)

validation_pretraining_dataset = (
    AMJEPAPairDataset(
        indices=pretraining_validation_indices,
        network=X_network,
        temporal=X_temporal,
        text_ids=window_text_ids,
        text_lookup=text_embedding_lookup,
        mask=attention_mask,
        future_index=future_window_index,
        campaign_labels=campaign_targets,
        aspect_labels=aspect_targets,
    )
)

pin_memory = torch.cuda.is_available()

train_generator = torch.Generator()
train_generator.manual_seed(SEED)

train_pretraining_loader = DataLoader(
    train_pretraining_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=NUMBER_OF_WORKERS,
    pin_memory=pin_memory,
    drop_last=False,
    generator=train_generator,
)

validation_pretraining_loader = DataLoader(
    validation_pretraining_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=NUMBER_OF_WORKERS,
    pin_memory=pin_memory,
    drop_last=False,
)


# -----------------------------------------------------------------------------
# Batch verification
# -----------------------------------------------------------------------------

sample_batch = next(
    iter(train_pretraining_loader)
)

assert sample_batch[
    "current_network"
].shape[1:] == (
    window_size,
    79,
)

assert sample_batch[
    "current_temporal"
].shape[1:] == (
    window_size,
    3,
)

assert sample_batch[
    "current_text"
].shape[1:] == (
    window_size,
    128,
)

assert sample_batch[
    "current_attention_mask"
].shape[1:] == (
    window_size,
)

assert sample_batch[
    "future_network"
].shape[1:] == (
    window_size,
    79,
)


# -----------------------------------------------------------------------------
# Save experiment manifest
# -----------------------------------------------------------------------------

experiment_manifest = {
    "seed": SEED,
    "window_size": window_size,
    "batch_size": BATCH_SIZE,

    "network_dimension":
        int(X_network.shape[2]),

    "temporal_dimension":
        int(X_temporal.shape[2]),

    "text_dimension":
        int(text_embedding_lookup.shape[1]),

    "total_windows":
        int(number_of_windows),

    "training_windows":
        int(len(train_indices)),

    "validation_windows":
        int(len(validation_indices)),

    "test_windows":
        int(len(test_indices)),

    "pretraining_train_pairs":
        int(
            len(pretraining_train_indices)
        ),

    "pretraining_validation_pairs":
        int(
            len(
                pretraining_validation_indices
            )
        ),

    "pretraining_test_pairs":
        int(
            len(pretraining_test_indices)
        ),

    "future_pair_policy":
        (
            "Earliest chronologically later window "
            "from the same split and day having zero "
            "shared evidence identifiers."
        ),

    "test_used_for_pretraining":
        False,

    "labels_used_for_pretraining":
        False,
}

with open(
    AMJEPA_DIR
    / "data_manifest.json",
    "w",
    encoding="utf-8",
) as file:
    json.dump(
        experiment_manifest,
        file,
        indent=2,
    )


# -----------------------------------------------------------------------------
# Display
# -----------------------------------------------------------------------------

print("\n" + "=" * 80)
print("NON-OVERLAPPING FUTURE-WINDOW PAIRS")
print("=" * 80)

pair_summary = (
    future_pairs
    .groupby("split")
    .agg(
        pairs=(
            "current_window_id",
            "size",
        ),
        mean_gap_seconds=(
            "time_gap_seconds",
            "mean",
        ),
        median_gap_seconds=(
            "time_gap_seconds",
            "median",
        ),
        minimum_gap_seconds=(
            "time_gap_seconds",
            "min",
        ),
        maximum_gap_seconds=(
            "time_gap_seconds",
            "max",
        ),
    )
    .reset_index()
)

display(
    pair_summary.round(4)
)

print("\nPretraining pairs:")
print(
    "Train:",
    len(pretraining_train_indices),
)

print(
    "Validation:",
    len(pretraining_validation_indices),
)

print(
    "Test held out:",
    len(pretraining_test_indices),
)

print("\nSample batch:")
print(
    "Current network:",
    tuple(
        sample_batch[
            "current_network"
        ].shape
    ),
)

print(
    "Current temporal:",
    tuple(
        sample_batch[
            "current_temporal"
        ].shape
    ),
)

print(
    "Current text:",
    tuple(
        sample_batch[
            "current_text"
        ].shape
    ),
)

print(
    "Future network:",
    tuple(
        sample_batch[
            "future_network"
        ].shape
    ),
)

print("\nSaved AM-JEPA data artifacts to:")
print(AMJEPA_DIR)

In [ ]:
# =============================================================================
# CELL 20: AM-JEPA SELF-SUPERVISED PRETRAINING
# =============================================================================

from copy import deepcopy
from pathlib import Path
import json
import math
import random
import time

import numpy as np
import pandas as pd

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.nn.utils.rnn import (
    pack_padded_sequence,
    pad_packed_sequence,
)


# -----------------------------------------------------------------------------
# Configuration
# -----------------------------------------------------------------------------

SEED = 42

DATA_DIR = Path(
    r"."
)

AMJEPA_DIR = (
    DATA_DIR
    / "outputs"
    / "am_jepa_experiment"
)

CHECKPOINT_PATH = (
    AMJEPA_DIR
    / "am_jepa_pretrained_best.pt"
)

HISTORY_PATH = (
    AMJEPA_DIR
    / "am_jepa_pretraining_history.csv"
)

CONFIG_PATH = (
    AMJEPA_DIR
    / "am_jepa_model_config.json"
)

LATENT_DIMENSION = 128
HIDDEN_DIMENSION = 192
NUMBER_OF_GRU_LAYERS = 2
DROPOUT_RATE = 0.10

EVIDENCE_MASK_RATE = 0.30
MODALITY_DROP_RATE = 0.25

LEARNING_RATE = 1e-3
WEIGHT_DECAY = 1e-4

MAXIMUM_EPOCHS = 60
EARLY_STOPPING_PATIENCE = 10
GRADIENT_CLIP_NORM = 1.0

EMA_START = 0.990
EMA_END = 0.9995

LAMBDA_EVIDENCE = 1.0
LAMBDA_MODALITY = 1.0
LAMBDA_FUTURE = 0.5
LAMBDA_VARIANCE = 0.05

MINIMUM_IMPROVEMENT = 1e-5


# -----------------------------------------------------------------------------
# Reproducibility
# -----------------------------------------------------------------------------

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

DEVICE = torch.device(
    "cuda"
    if torch.cuda.is_available()
    else "cpu"
)

print("=" * 80)
print("AM-JEPA SELF-SUPERVISED PRETRAINING")
print("=" * 80)

print("Device:", DEVICE)

if DEVICE.type == "cuda":
    print(
        "GPU:",
        torch.cuda.get_device_name(0),
    )

    print(
        "CUDA memory:",
        round(
            torch.cuda.get_device_properties(
                0
            ).total_memory
            / 1024**3,
            2,
        ),
        "GB",
    )


# -----------------------------------------------------------------------------
# Utility functions
# -----------------------------------------------------------------------------

def masked_mean(
    sequence,
    attention_mask,
):
    expanded_mask = (
        attention_mask
        .to(sequence.dtype)
        .unsqueeze(-1)
    )

    numerator = (
        sequence * expanded_mask
    ).sum(dim=1)

    denominator = (
        expanded_mask
        .sum(dim=1)
        .clamp_min(1.0)
    )

    return numerator / denominator


def masked_cosine_loss(
    prediction,
    target,
    selection_mask,
):
    """
    Cosine prediction loss over selected sequence positions.
    """
    selected_prediction = prediction[
        selection_mask
    ]

    selected_target = target[
        selection_mask
    ]

    if selected_prediction.numel() == 0:
        return prediction.sum() * 0.0

    selected_prediction = F.normalize(
        selected_prediction,
        dim=-1,
    )

    selected_target = F.normalize(
        selected_target.detach(),
        dim=-1,
    )

    return (
        1.0
        - (
            selected_prediction
            * selected_target
        ).sum(dim=-1)
    ).mean()


def vector_cosine_loss(
    prediction,
    target,
):
    prediction = F.normalize(
        prediction,
        dim=-1,
    )

    target = F.normalize(
        target.detach(),
        dim=-1,
    )

    return (
        1.0
        - (
            prediction * target
        ).sum(dim=-1)
    ).mean()


def variance_regularization(
    representation,
):
    """
    Prevents degenerate constant representations without
    reconstructing the original observations.
    """
    if representation.shape[0] < 2:
        return representation.sum() * 0.0

    standard_deviation = torch.sqrt(
        representation.var(
            dim=0,
            unbiased=False,
        )
        + 1e-4
    )

    return F.relu(
        1.0 - standard_deviation
    ).mean()


def current_ema_coefficient(
    global_step,
    total_steps,
):
    if total_steps <= 1:
        return EMA_END

    progress = (
        global_step
        / (total_steps - 1)
    )

    cosine_progress = (
        1.0
        - math.cos(
            math.pi * progress
        )
    ) / 2.0

    return (
        EMA_START
        + (
            EMA_END - EMA_START
        )
        * cosine_progress
    )


# -----------------------------------------------------------------------------
# Modality projection block
# -----------------------------------------------------------------------------

class ProjectionEncoder(nn.Module):
    def __init__(
        self,
        input_dimension,
        latent_dimension,
        hidden_dimension,
        dropout_rate,
    ):
        super().__init__()

        self.layers = nn.Sequential(
            nn.LayerNorm(
                input_dimension
            ),
            nn.Linear(
                input_dimension,
                hidden_dimension,
            ),
            nn.GELU(),
            nn.Dropout(dropout_rate),
            nn.Linear(
                hidden_dimension,
                latent_dimension,
            ),
            nn.LayerNorm(
                latent_dimension
            ),
        )

    def forward(self, inputs):
        return self.layers(inputs)


# -----------------------------------------------------------------------------
# Multimodal causal encoder
# -----------------------------------------------------------------------------

class MultimodalCausalEncoder(nn.Module):
    def __init__(
        self,
        network_dimension=79,
        temporal_dimension=3,
        text_dimension=128,
        latent_dimension=128,
        hidden_dimension=192,
        number_of_layers=2,
        dropout_rate=0.10,
    ):
        super().__init__()

        self.latent_dimension = (
            latent_dimension
        )

        self.network_projection = (
            ProjectionEncoder(
                input_dimension=
                    network_dimension,
                latent_dimension=
                    latent_dimension,
                hidden_dimension=
                    hidden_dimension,
                dropout_rate=
                    dropout_rate,
            )
        )

        self.temporal_projection = (
            ProjectionEncoder(
                input_dimension=
                    temporal_dimension,
                latent_dimension=
                    latent_dimension,
                hidden_dimension=
                    hidden_dimension,
                dropout_rate=
                    dropout_rate,
            )
        )

        self.text_projection = (
            ProjectionEncoder(
                input_dimension=
                    text_dimension,
                latent_dimension=
                    latent_dimension,
                hidden_dimension=
                    hidden_dimension,
                dropout_rate=
                    dropout_rate,
            )
        )

        self.modality_embeddings = (
            nn.Parameter(
                torch.randn(
                    3,
                    latent_dimension,
                )
                * 0.02
            )
        )

        self.modality_gate = nn.Sequential(
            nn.Linear(
                latent_dimension,
                latent_dimension // 2,
            ),
            nn.GELU(),
            nn.Linear(
                latent_dimension // 2,
                1,
            ),
        )

        self.evidence_mask_token = (
            nn.Parameter(
                torch.randn(
                    1,
                    1,
                    latent_dimension,
                )
                * 0.02
            )
        )

        self.causal_encoder = nn.GRU(
            input_size=latent_dimension,
            hidden_size=latent_dimension,
            num_layers=number_of_layers,
            batch_first=True,
            dropout=(
                dropout_rate
                if number_of_layers > 1
                else 0.0
            ),
            bidirectional=False,
        )

        self.output_normalization = (
            nn.LayerNorm(
                latent_dimension
            )
        )

    def forward(
        self,
        network,
        temporal,
        text,
        attention_mask,
        modality_keep_mask=None,
        evidence_mask=None,
    ):
        batch_size = network.shape[0]
        sequence_length = network.shape[1]

        network_latent = (
            self.network_projection(
                network
            )
            + self.modality_embeddings[
                0
            ]
        )

        temporal_latent = (
            self.temporal_projection(
                temporal
            )
            + self.modality_embeddings[
                1
            ]
        )

        text_latent = (
            self.text_projection(
                text
            )
            + self.modality_embeddings[
                2
            ]
        )

        modality_latents = torch.stack(
            [
                network_latent,
                temporal_latent,
                text_latent,
            ],
            dim=2,
        )

        if modality_keep_mask is None:
            modality_keep_mask = (
                attention_mask
                .unsqueeze(-1)
                .expand(
                    -1,
                    -1,
                    3,
                )
            )

        gate_logits = (
            self.modality_gate(
                modality_latents
            )
            .squeeze(-1)
        )

        gate_logits = gate_logits.masked_fill(
            ~modality_keep_mask,
            -1e4,
        )

        # Padding positions have no available modality.
        # Set their logits to zero to avoid invalid softmax.
        gate_logits = torch.where(
            attention_mask.unsqueeze(-1),
            gate_logits,
            torch.zeros_like(
                gate_logits
            ),
        )

        modality_weights = F.softmax(
            gate_logits,
            dim=-1,
        )

        modality_weights = (
            modality_weights
            * modality_keep_mask.to(
                modality_weights.dtype
            )
        )

        weight_sum = (
            modality_weights.sum(
                dim=-1,
                keepdim=True,
            )
            .clamp_min(1e-8)
        )

        modality_weights = (
            modality_weights
            / weight_sum
        )

        fused_evidence = (
            modality_latents
            * modality_weights.unsqueeze(-1)
        ).sum(dim=2)

        if evidence_mask is not None:
            mask_token = (
                self.evidence_mask_token
                .expand(
                    batch_size,
                    sequence_length,
                    -1,
                )
            )

            fused_evidence = torch.where(
                evidence_mask.unsqueeze(-1),
                mask_token,
                fused_evidence,
            )

        fused_evidence = (
            fused_evidence
            * attention_mask
            .unsqueeze(-1)
            .to(fused_evidence.dtype)
        )

        lengths = (
            attention_mask
            .sum(dim=1)
            .clamp_min(1)
            .to(torch.long)
        )

        packed_evidence = (
            pack_padded_sequence(
                fused_evidence,
                lengths.cpu(),
                batch_first=True,
                enforce_sorted=False,
            )
        )

        packed_output, _ = (
            self.causal_encoder(
                packed_evidence
            )
        )

        sequence_output, _ = (
            pad_packed_sequence(
                packed_output,
                batch_first=True,
                total_length=
                    sequence_length,
            )
        )

        sequence_output = (
            self.output_normalization(
                sequence_output
            )
        )

        sequence_output = (
            sequence_output
            * attention_mask
            .unsqueeze(-1)
            .to(
                sequence_output.dtype
            )
        )

        pooled_output = masked_mean(
            sequence_output,
            attention_mask,
        )

        return {
            "sequence":
                sequence_output,

            "pooled":
                pooled_output,

            "modality_weights":
                modality_weights,
        }


# -----------------------------------------------------------------------------
# AM-JEPA model
# -----------------------------------------------------------------------------

class AMJEPA(nn.Module):
    def __init__(
        self,
        network_dimension=79,
        temporal_dimension=3,
        text_dimension=128,
        latent_dimension=128,
        hidden_dimension=192,
        number_of_layers=2,
        dropout_rate=0.10,
    ):
        super().__init__()

        self.context_encoder = (
            MultimodalCausalEncoder(
                network_dimension=
                    network_dimension,
                temporal_dimension=
                    temporal_dimension,
                text_dimension=
                    text_dimension,
                latent_dimension=
                    latent_dimension,
                hidden_dimension=
                    hidden_dimension,
                number_of_layers=
                    number_of_layers,
                dropout_rate=
                    dropout_rate,
            )
        )

        self.target_encoder = deepcopy(
            self.context_encoder
        )

        for parameter in (
            self.target_encoder.parameters()
        ):
            parameter.requires_grad = False

        self.evidence_predictor = (
            nn.Sequential(
                nn.Linear(
                    latent_dimension,
                    hidden_dimension,
                ),
                nn.GELU(),
                nn.Linear(
                    hidden_dimension,
                    latent_dimension,
                ),
                nn.LayerNorm(
                    latent_dimension
                ),
            )
        )

        self.modality_predictor = (
            nn.Sequential(
                nn.Linear(
                    latent_dimension,
                    hidden_dimension,
                ),
                nn.GELU(),
                nn.Linear(
                    hidden_dimension,
                    latent_dimension,
                ),
                nn.LayerNorm(
                    latent_dimension
                ),
            )
        )

        self.future_predictor = (
            nn.Sequential(
                nn.Linear(
                    latent_dimension,
                    hidden_dimension,
                ),
                nn.GELU(),
                nn.Linear(
                    hidden_dimension,
                    latent_dimension,
                ),
                nn.LayerNorm(
                    latent_dimension
                ),
            )
        )

    @torch.no_grad()
    def update_target_encoder(
        self,
        ema_coefficient,
    ):
        context_parameters = dict(
            self.context_encoder
            .named_parameters()
        )

        for name, target_parameter in (
            self.target_encoder
            .named_parameters()
        ):
            context_parameter = (
                context_parameters[name]
            )

            target_parameter.data.mul_(
                ema_coefficient
            ).add_(
                context_parameter.data,
                alpha=(
                    1.0
                    - ema_coefficient
                ),
            )

    def forward_context(
        self,
        network,
        temporal,
        text,
        attention_mask,
        modality_keep_mask,
        evidence_mask,
    ):
        return self.context_encoder(
            network=network,
            temporal=temporal,
            text=text,
            attention_mask=
                attention_mask,
            modality_keep_mask=
                modality_keep_mask,
            evidence_mask=
                evidence_mask,
        )

    @torch.no_grad()
    def forward_target(
        self,
        network,
        temporal,
        text,
        attention_mask,
    ):
        self.target_encoder.eval()

        return self.target_encoder(
            network=network,
            temporal=temporal,
            text=text,
            attention_mask=
                attention_mask,
            modality_keep_mask=None,
            evidence_mask=None,
        )


# -----------------------------------------------------------------------------
# Corruption-mask generation
# -----------------------------------------------------------------------------

def generate_corruption_masks(
    attention_mask,
    evidence_mask_rate,
    modality_drop_rate,
    generator=None,
):
    batch_size, sequence_length = (
        attention_mask.shape
    )

    device = attention_mask.device

    evidence_random = torch.rand(
        (
            batch_size,
            sequence_length,
        ),
        device=device,
        generator=generator,
    )

    evidence_mask = (
        evidence_random
        < evidence_mask_rate
    ) & attention_mask

    # Guarantee at least one masked evidence item
    # per nonempty window.
    for batch_index in range(
        batch_size
    ):
        if not evidence_mask[
            batch_index
        ].any():
            valid_positions = torch.nonzero(
                attention_mask[
                    batch_index
                ],
                as_tuple=False,
            ).flatten()

            selected_position = (
                valid_positions[
                    torch.randint(
                        low=0,
                        high=len(
                            valid_positions
                        ),
                        size=(1,),
                        device=device,
                        generator=generator,
                    )
                ]
            )

            evidence_mask[
                batch_index,
                selected_position,
            ] = True

    modality_random = torch.rand(
        (
            batch_size,
            sequence_length,
            3,
        ),
        device=device,
        generator=generator,
    )

    modality_keep_mask = (
        modality_random
        >= modality_drop_rate
    )

    modality_keep_mask = (
        modality_keep_mask
        & attention_mask.unsqueeze(-1)
    )

    all_modalities_dropped = (
        attention_mask
        & (
            ~modality_keep_mask.any(
                dim=-1
            )
        )
    )

    dropped_coordinates = (
        torch.nonzero(
            all_modalities_dropped,
            as_tuple=False,
        )
    )

    if len(dropped_coordinates) > 0:
        replacement_modalities = (
            torch.randint(
                low=0,
                high=3,
                size=(
                    len(
                        dropped_coordinates
                    ),
                ),
                device=device,
                generator=generator,
            )
        )

        modality_keep_mask[
            dropped_coordinates[:, 0],
            dropped_coordinates[:, 1],
            replacement_modalities,
        ] = True

    modality_corruption_mask = (
        attention_mask
        & (
            ~modality_keep_mask.all(
                dim=-1
            )
        )
        & (~evidence_mask)
    )

    return (
        evidence_mask,
        modality_keep_mask,
        modality_corruption_mask,
    )


# -----------------------------------------------------------------------------
# Batch device transfer
# -----------------------------------------------------------------------------

def move_batch_to_device(batch):
    tensor_keys = [
        "current_network",
        "current_temporal",
        "current_text",
        "current_attention_mask",
        "future_network",
        "future_temporal",
        "future_text",
        "future_attention_mask",
    ]

    return {
        key: batch[key].to(
            DEVICE,
            non_blocking=True,
        )
        for key in tensor_keys
    }


# -----------------------------------------------------------------------------
# Train/validation epoch
# -----------------------------------------------------------------------------

def run_pretraining_epoch(
    model,
    data_loader,
    optimizer=None,
    scheduler=None,
    global_step=0,
    total_steps=1,
    training=True,
):
    if training:
        model.context_encoder.train()
        model.evidence_predictor.train()
        model.modality_predictor.train()
        model.future_predictor.train()
    else:
        model.context_encoder.eval()
        model.evidence_predictor.eval()
        model.modality_predictor.eval()
        model.future_predictor.eval()

    model.target_encoder.eval()

    accumulated = {
        "total_loss": 0.0,
        "evidence_loss": 0.0,
        "modality_loss": 0.0,
        "future_loss": 0.0,
        "variance_loss": 0.0,
        "latent_std": 0.0,
        "mean_evidence_mask_rate":
            0.0,
        "mean_modality_corruption_rate":
            0.0,
    }

    number_of_batches = 0

    if not training:
        if DEVICE.type == "cuda":
            validation_generator = (
                torch.Generator(
                    device="cuda"
                )
            )
        else:
            validation_generator = (
                torch.Generator()
            )

        validation_generator.manual_seed(
            20260917
        )

    else:
        validation_generator = None

    gradient_context = (
        torch.enable_grad()
        if training
        else torch.no_grad()
    )

    with gradient_context:
        for batch in data_loader:
            tensors = (
                move_batch_to_device(
                    batch
                )
            )

            current_mask = tensors[
                "current_attention_mask"
            ]

            (
                evidence_mask,
                modality_keep_mask,
                modality_corruption_mask,
            ) = generate_corruption_masks(
                attention_mask=
                    current_mask,
                evidence_mask_rate=
                    EVIDENCE_MASK_RATE,
                modality_drop_rate=
                    MODALITY_DROP_RATE,
                generator=
                    validation_generator,
            )

            if training:
                optimizer.zero_grad(
                    set_to_none=True
                )

            context_output = (
                model.forward_context(
                    network=tensors[
                        "current_network"
                    ],
                    temporal=tensors[
                        "current_temporal"
                    ],
                    text=tensors[
                        "current_text"
                    ],
                    attention_mask=
                        current_mask,
                    modality_keep_mask=
                        modality_keep_mask,
                    evidence_mask=
                        evidence_mask,
                )
            )

            with torch.no_grad():
                current_target = (
                    model.forward_target(
                        network=tensors[
                            "current_network"
                        ],
                        temporal=tensors[
                            "current_temporal"
                        ],
                        text=tensors[
                            "current_text"
                        ],
                        attention_mask=
                            current_mask,
                    )
                )

                future_target = (
                    model.forward_target(
                        network=tensors[
                            "future_network"
                        ],
                        temporal=tensors[
                            "future_temporal"
                        ],
                        text=tensors[
                            "future_text"
                        ],
                        attention_mask=tensors[
                            "future_attention_mask"
                        ],
                    )
                )

            evidence_prediction = (
                model.evidence_predictor(
                    context_output[
                        "sequence"
                    ]
                )
            )

            modality_prediction = (
                model.modality_predictor(
                    context_output[
                        "sequence"
                    ]
                )
            )

            future_prediction = (
                model.future_predictor(
                    context_output[
                        "pooled"
                    ]
                )
            )

            evidence_loss = (
                masked_cosine_loss(
                    prediction=
                        evidence_prediction,
                    target=current_target[
                        "sequence"
                    ],
                    selection_mask=
                        evidence_mask,
                )
            )

            modality_loss = (
                masked_cosine_loss(
                    prediction=
                        modality_prediction,
                    target=current_target[
                        "sequence"
                    ],
                    selection_mask=
                        modality_corruption_mask,
                )
            )

            future_loss = (
                vector_cosine_loss(
                    prediction=
                        future_prediction,
                    target=future_target[
                        "pooled"
                    ],
                )
            )

            variance_loss = (
                variance_regularization(
                    context_output[
                        "pooled"
                    ]
                )
            )

            total_loss = (
                LAMBDA_EVIDENCE
                * evidence_loss
                + LAMBDA_MODALITY
                * modality_loss
                + LAMBDA_FUTURE
                * future_loss
                + LAMBDA_VARIANCE
                * variance_loss
            )

            if not torch.isfinite(
                total_loss
            ):
                raise FloatingPointError(
                    "Non-finite AM-JEPA loss "
                    "detected."
                )

            if training:
                total_loss.backward()

                torch.nn.utils.clip_grad_norm_(
                    [
                        parameter
                        for parameter
                        in model.parameters()
                        if parameter.requires_grad
                    ],
                    max_norm=
                        GRADIENT_CLIP_NORM,
                )

                optimizer.step()

                ema_coefficient = (
                    current_ema_coefficient(
                        global_step,
                        total_steps,
                    )
                )

                model.update_target_encoder(
                    ema_coefficient
                )

                if scheduler is not None:
                    scheduler.step()

                global_step += 1

            valid_count = (
                current_mask.sum()
                .clamp_min(1)
            )

            evidence_mask_rate = (
                evidence_mask.sum()
                / valid_count
            )

            modality_corruption_rate = (
                modality_corruption_mask.sum()
                / valid_count
            )

            latent_standard_deviation = (
                context_output[
                    "pooled"
                ]
                .std(
                    dim=0,
                    unbiased=False,
                )
                .mean()
            )

            accumulated[
                "total_loss"
            ] += float(
                total_loss.detach().cpu()
            )

            accumulated[
                "evidence_loss"
            ] += float(
                evidence_loss.detach().cpu()
            )

            accumulated[
                "modality_loss"
            ] += float(
                modality_loss.detach().cpu()
            )

            accumulated[
                "future_loss"
            ] += float(
                future_loss.detach().cpu()
            )

            accumulated[
                "variance_loss"
            ] += float(
                variance_loss.detach().cpu()
            )

            accumulated[
                "latent_std"
            ] += float(
                latent_standard_deviation
                .detach()
                .cpu()
            )

            accumulated[
                "mean_evidence_mask_rate"
            ] += float(
                evidence_mask_rate
                .detach()
                .cpu()
            )

            accumulated[
                "mean_modality_corruption_rate"
            ] += float(
                modality_corruption_rate
                .detach()
                .cpu()
            )

            number_of_batches += 1

    for key in accumulated:
        accumulated[key] /= max(
            number_of_batches,
            1,
        )

    return (
        accumulated,
        global_step,
    )


# -----------------------------------------------------------------------------
# Initialize model and optimizer
# -----------------------------------------------------------------------------

model = AMJEPA(
    network_dimension=79,
    temporal_dimension=3,
    text_dimension=128,
    latent_dimension=
        LATENT_DIMENSION,
    hidden_dimension=
        HIDDEN_DIMENSION,
    number_of_layers=
        NUMBER_OF_GRU_LAYERS,
    dropout_rate=
        DROPOUT_RATE,
).to(DEVICE)

trainable_parameters = [
    parameter
    for parameter in model.parameters()
    if parameter.requires_grad
]

optimizer = torch.optim.AdamW(
    trainable_parameters,
    lr=LEARNING_RATE,
    weight_decay=WEIGHT_DECAY,
)

total_training_steps = (
    MAXIMUM_EPOCHS
    * len(train_pretraining_loader)
)

scheduler = (
    torch.optim.lr_scheduler.CosineAnnealingLR(
        optimizer,
        T_max=max(
            total_training_steps,
            1,
        ),
        eta_min=1e-5,
    )
)

parameter_counts = {
    "total": sum(
        parameter.numel()
        for parameter in model.parameters()
    ),

    "trainable": sum(
        parameter.numel()
        for parameter in model.parameters()
        if parameter.requires_grad
    ),

    "target_encoder": sum(
        parameter.numel()
        for parameter
        in model.target_encoder.parameters()
    ),
}

print("\nModel parameters:")
print(parameter_counts)


# -----------------------------------------------------------------------------
# Save configuration
# -----------------------------------------------------------------------------

model_configuration = {
    "seed": SEED,
    "network_dimension": 79,
    "temporal_dimension": 3,
    "text_dimension": 128,
    "latent_dimension":
        LATENT_DIMENSION,
    "hidden_dimension":
        HIDDEN_DIMENSION,
    "gru_layers":
        NUMBER_OF_GRU_LAYERS,
    "dropout_rate":
        DROPOUT_RATE,
    "evidence_mask_rate":
        EVIDENCE_MASK_RATE,
    "modality_drop_rate":
        MODALITY_DROP_RATE,
    "learning_rate":
        LEARNING_RATE,
    "weight_decay":
        WEIGHT_DECAY,
    "maximum_epochs":
        MAXIMUM_EPOCHS,
    "early_stopping_patience":
        EARLY_STOPPING_PATIENCE,
    "ema_start":
        EMA_START,
    "ema_end":
        EMA_END,
    "lambda_evidence":
        LAMBDA_EVIDENCE,
    "lambda_modality":
        LAMBDA_MODALITY,
    "lambda_future":
        LAMBDA_FUTURE,
    "lambda_variance":
        LAMBDA_VARIANCE,
    "parameter_counts":
        parameter_counts,
    "test_used_during_pretraining":
        False,
}

with open(
    CONFIG_PATH,
    "w",
    encoding="utf-8",
) as file:
    json.dump(
        model_configuration,
        file,
        indent=2,
    )


# -----------------------------------------------------------------------------
# Pretraining loop
# -----------------------------------------------------------------------------

history_rows = []

best_validation_loss = np.inf
best_epoch = 0
epochs_without_improvement = 0
global_step = 0

training_start_time = time.perf_counter()

for epoch in range(
    1,
    MAXIMUM_EPOCHS + 1,
):
    epoch_start_time = (
        time.perf_counter()
    )

    (
        training_metrics,
        global_step,
    ) = run_pretraining_epoch(
        model=model,
        data_loader=
            train_pretraining_loader,
        optimizer=optimizer,
        scheduler=scheduler,
        global_step=global_step,
        total_steps=
            total_training_steps,
        training=True,
    )

    (
        validation_metrics,
        _,
    ) = run_pretraining_epoch(
        model=model,
        data_loader=
            validation_pretraining_loader,
        optimizer=None,
        scheduler=None,
        global_step=global_step,
        total_steps=
            total_training_steps,
        training=False,
    )

    epoch_seconds = (
        time.perf_counter()
        - epoch_start_time
    )

    learning_rate = (
        optimizer.param_groups[0]["lr"]
    )

    ema_coefficient = (
        current_ema_coefficient(
            global_step,
            total_training_steps,
        )
    )

    history_row = {
        "epoch": epoch,
        "learning_rate":
            learning_rate,
        "ema_coefficient":
            ema_coefficient,
        "epoch_seconds":
            epoch_seconds,
    }

    for metric_name, value in (
        training_metrics.items()
    ):
        history_row[
            f"train_{metric_name}"
        ] = value

    for metric_name, value in (
        validation_metrics.items()
    ):
        history_row[
            f"validation_{metric_name}"
        ] = value

    history_rows.append(
        history_row
    )

    improved = (
        validation_metrics[
            "total_loss"
        ]
        < (
            best_validation_loss
            - MINIMUM_IMPROVEMENT
        )
    )

    if improved:
        best_validation_loss = (
            validation_metrics[
                "total_loss"
            ]
        )

        best_epoch = epoch
        epochs_without_improvement = 0

        torch.save(
            {
                "epoch": epoch,
                "model_state_dict":
                    model.state_dict(),
                "optimizer_state_dict":
                    optimizer.state_dict(),
                "validation_loss":
                    best_validation_loss,
                "configuration":
                    model_configuration,
            },
            CHECKPOINT_PATH,
        )

    else:
        epochs_without_improvement += 1

    print(
        f"Epoch {epoch:02d} | "
        f"train={training_metrics['total_loss']:.5f} | "
        f"val={validation_metrics['total_loss']:.5f} | "
        f"evidence={validation_metrics['evidence_loss']:.5f} | "
        f"modality={validation_metrics['modality_loss']:.5f} | "
        f"future={validation_metrics['future_loss']:.5f} | "
        f"latent_std={validation_metrics['latent_std']:.5f} | "
        f"lr={learning_rate:.6f} | "
        f"{epoch_seconds:.2f}s"
    )

    pd.DataFrame(
        history_rows
    ).to_csv(
        HISTORY_PATH,
        index=False,
    )

    if (
        epochs_without_improvement
        >= EARLY_STOPPING_PATIENCE
    ):
        print(
            "\nEarly stopping triggered."
        )
        break


# -----------------------------------------------------------------------------
# Restore best checkpoint
# -----------------------------------------------------------------------------

checkpoint = torch.load(
    CHECKPOINT_PATH,
    map_location=DEVICE,
    weights_only=False,
)

model.load_state_dict(
    checkpoint["model_state_dict"]
)

model.context_encoder.eval()
model.target_encoder.eval()
model.evidence_predictor.eval()
model.modality_predictor.eval()
model.future_predictor.eval()

total_training_seconds = (
    time.perf_counter()
    - training_start_time
)

history = pd.DataFrame(
    history_rows
)

best_history_row = history.loc[
    history["epoch"] == best_epoch
].iloc[0]


# -----------------------------------------------------------------------------
# Final reporting
# -----------------------------------------------------------------------------

print("\n" + "=" * 80)
print("AM-JEPA PRETRAINING COMPLETED")
print("=" * 80)

print("Best epoch:", best_epoch)

print(
    "Best validation loss:",
    round(
        best_validation_loss,
        6,
    ),
)

print(
    "Validation evidence loss:",
    round(
        best_history_row[
            "validation_evidence_loss"
        ],
        6,
    ),
)

print(
    "Validation modality loss:",
    round(
        best_history_row[
            "validation_modality_loss"
        ],
        6,
    ),
)

print(
    "Validation future loss:",
    round(
        best_history_row[
            "validation_future_loss"
        ],
        6,
    ),
)

print(
    "Validation latent std:",
    round(
        best_history_row[
            "validation_latent_std"
        ],
        6,
    ),
)

print(
    "Training time:",
    round(
        total_training_seconds,
        2,
    ),
    "seconds",
)

print("\nBest checkpoint:")
print(CHECKPOINT_PATH)

print("\nTraining history:")
print(HISTORY_PATH)

print("\nNo campaign or aspect labels were used in pretraining.")
print("The test partition was not evaluated.")

In [ ]:
# =============================================================================
# CELL 21: FROZEN AM-JEPA REPRESENTATION AND LINEAR-PROBE EVALUATION
# =============================================================================

from pathlib import Path
import json

import numpy as np
import pandas as pd
import torch

from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    matthews_corrcoef,
    roc_auc_score,
    average_precision_score,
)
from sklearn.multiclass import OneVsRestClassifier
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler


# -----------------------------------------------------------------------------
# Configuration
# -----------------------------------------------------------------------------

SEED = 42
EXTRACTION_BATCH_SIZE = 256

DATA_DIR = Path(
    r"."
)

TENSOR_DIR = (
    DATA_DIR
    / "outputs"
    / "am_jepa_tensors"
)

AMJEPA_DIR = (
    DATA_DIR
    / "outputs"
    / "am_jepa_experiment"
)

CHECKPOINT_PATH = (
    AMJEPA_DIR
    / "am_jepa_pretrained_best.pt"
)

REPRESENTATION_PATH = (
    AMJEPA_DIR
    / "am_jepa_frozen_representations.npy"
)

MODALITY_WEIGHT_PATH = (
    AMJEPA_DIR
    / "am_jepa_modality_weights.npy"
)

CAMPAIGN_RESULT_PATH = (
    AMJEPA_DIR
    / "am_jepa_campaign_linear_probe.csv"
)

ASPECT_RESULT_PATH = (
    AMJEPA_DIR
    / "am_jepa_aspect_linear_probe.csv"
)

SUMMARY_PATH = (
    AMJEPA_DIR
    / "am_jepa_linear_probe_summary.csv"
)

ASPECT_NAMES = [
    "reconnaissance",
    "initial_access_foothold",
    "lateral_movement",
    "data_exfiltration",
]

THRESHOLD_GRID = np.linspace(
    0.01,
    0.99,
    99,
)


# -----------------------------------------------------------------------------
# Verify that the pretrained model exists
# -----------------------------------------------------------------------------

if "model" not in globals():
    raise NameError(
        "The AM-JEPA model is not in memory. "
        "Run Cell 20 before this cell."
    )

if not CHECKPOINT_PATH.exists():
    raise FileNotFoundError(
        f"Checkpoint not found: {CHECKPOINT_PATH}"
    )

checkpoint = torch.load(
    CHECKPOINT_PATH,
    map_location=DEVICE,
    weights_only=False,
)

model.load_state_dict(
    checkpoint["model_state_dict"]
)

selected_pretraining_epoch = int(
    checkpoint["epoch"]
)

print("=" * 80)
print("FROZEN AM-JEPA LINEAR-PROBE EVALUATION")
print("=" * 80)

print(
    "Loaded pretraining epoch:",
    selected_pretraining_epoch,
)

print(
    "Checkpoint validation loss:",
    checkpoint["validation_loss"],
)


# -----------------------------------------------------------------------------
# Ensure all required arrays are available
# -----------------------------------------------------------------------------

required_arrays = [
    "X_network",
    "X_temporal",
    "attention_mask",
    "text_embedding_lookup",
    "window_text_ids",
    "campaign_targets",
    "aspect_targets",
    "train_indices",
    "validation_indices",
    "test_indices",
]

missing_arrays = [
    name
    for name in required_arrays
    if name not in globals()
]

if missing_arrays:
    raise NameError(
        "Missing arrays:\n"
        + "\n".join(missing_arrays)
        + "\nRun Cell 19 before this cell."
    )


# -----------------------------------------------------------------------------
# Frozen representation extraction
# -----------------------------------------------------------------------------

model.context_encoder.eval()

for parameter in (
    model.context_encoder.parameters()
):
    parameter.requires_grad = False

number_of_windows = X_network.shape[0]
latent_dimension = (
    model.context_encoder.latent_dimension
)

frozen_representations = np.zeros(
    (
        number_of_windows,
        latent_dimension,
    ),
    dtype=np.float32,
)

# Mean modality weights for network, temporal, and text.
mean_modality_weights = np.zeros(
    (
        number_of_windows,
        3,
    ),
    dtype=np.float32,
)

with torch.no_grad():
    for batch_start in range(
        0,
        number_of_windows,
        EXTRACTION_BATCH_SIZE,
    ):
        batch_end = min(
            batch_start
            + EXTRACTION_BATCH_SIZE,
            number_of_windows,
        )

        batch_indices = np.arange(
            batch_start,
            batch_end,
        )

        batch_network = torch.from_numpy(
            X_network[batch_indices]
        ).float().to(DEVICE)

        batch_temporal = torch.from_numpy(
            X_temporal[batch_indices]
        ).float().to(DEVICE)

        batch_text = torch.from_numpy(
            text_embedding_lookup[
                window_text_ids[
                    batch_indices
                ]
            ]
        ).float().to(DEVICE)

        batch_mask = torch.from_numpy(
            attention_mask[
                batch_indices
            ]
        ).bool().to(DEVICE)

        encoder_output = (
            model.context_encoder(
                network=batch_network,
                temporal=batch_temporal,
                text=batch_text,
                attention_mask=batch_mask,
                modality_keep_mask=None,
                evidence_mask=None,
            )
        )

        representations = (
            encoder_output["pooled"]
            .detach()
            .cpu()
            .numpy()
        )

        modality_weights = (
            encoder_output[
                "modality_weights"
            ]
        )

        expanded_mask = (
            batch_mask
            .to(
                modality_weights.dtype
            )
            .unsqueeze(-1)
        )

        pooled_modality_weights = (
            (
                modality_weights
                * expanded_mask
            ).sum(dim=1)
            / expanded_mask
            .sum(dim=1)
            .clamp_min(1.0)
        )

        frozen_representations[
            batch_start:batch_end
        ] = representations

        mean_modality_weights[
            batch_start:batch_end
        ] = (
            pooled_modality_weights
            .detach()
            .cpu()
            .numpy()
        )


# -----------------------------------------------------------------------------
# Representation-collapse audit
# -----------------------------------------------------------------------------

representation_standard_deviation = (
    frozen_representations[
        train_indices
    ].std(axis=0)
)

mean_dimension_std = float(
    representation_standard_deviation.mean()
)

minimum_dimension_std = float(
    representation_standard_deviation.min()
)

near_constant_dimensions = int(
    (
        representation_standard_deviation
        < 1e-3
    ).sum()
)

unique_representations = int(
    np.unique(
        np.round(
            frozen_representations,
            decimals=6,
        ),
        axis=0,
    ).shape[0]
)

if not np.isfinite(
    frozen_representations
).all():
    raise FloatingPointError(
        "Frozen representations contain "
        "NaN or infinite values."
    )

np.save(
    REPRESENTATION_PATH,
    frozen_representations,
)

np.save(
    MODALITY_WEIGHT_PATH,
    mean_modality_weights,
)

print("\nRepresentation shape:")
print(frozen_representations.shape)

print(
    "Mean training dimension std:",
    round(mean_dimension_std, 6),
)

print(
    "Minimum training dimension std:",
    round(minimum_dimension_std, 6),
)

print(
    "Near-constant dimensions:",
    near_constant_dimensions,
)

print(
    "Unique representations:",
    unique_representations,
    "/",
    number_of_windows,
)


# -----------------------------------------------------------------------------
# Metric helpers
# -----------------------------------------------------------------------------

def safe_probability_metric(
    metric_function,
    y_true,
    probabilities,
):
    try:
        return float(
            metric_function(
                y_true,
                probabilities,
            )
        )
    except ValueError:
        return np.nan


def calculate_metrics(
    y_true,
    probabilities,
    threshold,
):
    predictions = (
        probabilities >= threshold
    ).astype(np.int64)

    return {
        "threshold": float(threshold),

        "accuracy":
            accuracy_score(
                y_true,
                predictions,
            ),

        "balanced_accuracy":
            balanced_accuracy_score(
                y_true,
                predictions,
            ),

        "precision":
            precision_score(
                y_true,
                predictions,
                zero_division=0,
            ),

        "recall":
            recall_score(
                y_true,
                predictions,
                zero_division=0,
            ),

        "f1":
            f1_score(
                y_true,
                predictions,
                zero_division=0,
            ),

        "mcc":
            matthews_corrcoef(
                y_true,
                predictions,
            ),

        "auroc":
            safe_probability_metric(
                roc_auc_score,
                y_true,
                probabilities,
            ),

        "auprc":
            safe_probability_metric(
                average_precision_score,
                y_true,
                probabilities,
            ),
    }


def select_validation_threshold(
    y_true,
    probabilities,
):
    rows = []

    for threshold in THRESHOLD_GRID:
        predictions = (
            probabilities >= threshold
        ).astype(np.int64)

        rows.append(
            {
                "threshold":
                    float(threshold),

                "mcc":
                    matthews_corrcoef(
                        y_true,
                        predictions,
                    ),

                "balanced_accuracy":
                    balanced_accuracy_score(
                        y_true,
                        predictions,
                    ),

                "f1":
                    f1_score(
                        y_true,
                        predictions,
                        zero_division=0,
                    ),

                "distance_from_0.5":
                    abs(threshold - 0.5),
            }
        )

    search_table = pd.DataFrame(rows)

    best_row = (
        search_table
        .sort_values(
            by=[
                "mcc",
                "balanced_accuracy",
                "f1",
                "distance_from_0.5",
            ],
            ascending=[
                False,
                False,
                False,
                True,
            ],
        )
        .iloc[0]
    )

    return float(
        best_row["threshold"]
    )


def build_linear_probe():
    return Pipeline(
        steps=[
            (
                "standardizer",
                StandardScaler(),
            ),
            (
                "classifier",
                LogisticRegression(
                    class_weight="balanced",
                    solver="lbfgs",
                    max_iter=5000,
                    random_state=SEED,
                ),
            ),
        ]
    )


# -----------------------------------------------------------------------------
# Partition representations and labels
# -----------------------------------------------------------------------------

X_probe_train = frozen_representations[
    train_indices
]

X_probe_validation = frozen_representations[
    validation_indices
]

X_probe_test = frozen_representations[
    test_indices
]

y_campaign_train = campaign_targets[
    train_indices
]

y_campaign_validation = campaign_targets[
    validation_indices
]

y_campaign_test = campaign_targets[
    test_indices
]

y_aspect_train = aspect_targets[
    train_indices
]

y_aspect_validation = aspect_targets[
    validation_indices
]

y_aspect_test = aspect_targets[
    test_indices
]


# -----------------------------------------------------------------------------
# Campaign linear probe
# -----------------------------------------------------------------------------

campaign_probe = build_linear_probe()

campaign_probe.fit(
    X_probe_train,
    y_campaign_train,
)

campaign_validation_probabilities = (
    campaign_probe.predict_proba(
        X_probe_validation
    )[:, 1]
)

campaign_test_probabilities = (
    campaign_probe.predict_proba(
        X_probe_test
    )[:, 1]
)

campaign_threshold = (
    select_validation_threshold(
        y_campaign_validation,
        campaign_validation_probabilities,
    )
)

campaign_metrics = calculate_metrics(
    y_campaign_test,
    campaign_test_probabilities,
    campaign_threshold,
)

campaign_results = pd.DataFrame(
    [
        {
            "representation":
                "frozen_am_jepa_context",

            "pretraining_epoch":
                selected_pretraining_epoch,

            "dimension":
                latent_dimension,

            **campaign_metrics,
        }
    ]
)


# -----------------------------------------------------------------------------
# Aspect linear probes
# -----------------------------------------------------------------------------

aspect_probe = OneVsRestClassifier(
    build_linear_probe(),
    n_jobs=-1,
)

aspect_probe.fit(
    X_probe_train,
    y_aspect_train,
)

aspect_validation_probabilities = (
    aspect_probe.predict_proba(
        X_probe_validation
    )
)

aspect_test_probabilities = (
    aspect_probe.predict_proba(
        X_probe_test
    )
)

aspect_rows = []

for aspect_index, aspect_name in enumerate(
    ASPECT_NAMES
):
    validation_labels = (
        y_aspect_validation[
            :,
            aspect_index,
        ]
    )

    test_labels = (
        y_aspect_test[
            :,
            aspect_index,
        ]
    )

    if int(
        validation_labels.sum()
    ) >= 5:
        aspect_threshold = (
            select_validation_threshold(
                validation_labels,
                aspect_validation_probabilities[
                    :,
                    aspect_index,
                ],
            )
        )

        threshold_status = (
            "validation_calibrated"
        )

    else:
        aspect_threshold = 0.5

        threshold_status = (
            "fixed_insufficient_positives"
        )

    aspect_metrics = calculate_metrics(
        test_labels,
        aspect_test_probabilities[
            :,
            aspect_index,
        ],
        aspect_threshold,
    )

    aspect_rows.append(
        {
            "representation":
                "frozen_am_jepa_context",

            "pretraining_epoch":
                selected_pretraining_epoch,

            "aspect":
                aspect_name,

            "validation_positive_windows":
                int(
                    validation_labels.sum()
                ),

            "test_positive_windows":
                int(
                    test_labels.sum()
                ),

            "threshold_status":
                threshold_status,

            **aspect_metrics,
        }
    )

aspect_results = pd.DataFrame(
    aspect_rows
)


# -----------------------------------------------------------------------------
# Aspect macro summaries
# -----------------------------------------------------------------------------

core_three_results = aspect_results[
    aspect_results["aspect"]
    != "data_exfiltration"
]

aspect_summary = {
    "four_aspect_macro_f1":
        aspect_results["f1"].mean(),

    "four_aspect_macro_mcc":
        aspect_results["mcc"].mean(),

    "four_aspect_macro_auroc":
        aspect_results["auroc"].mean(),

    "four_aspect_macro_auprc":
        aspect_results["auprc"].mean(),

    "core3_macro_f1":
        core_three_results["f1"].mean(),

    "core3_macro_mcc":
        core_three_results["mcc"].mean(),

    "core3_macro_auroc":
        core_three_results["auroc"].mean(),

    "core3_macro_auprc":
        core_three_results["auprc"].mean(),
}


# -----------------------------------------------------------------------------
# Compare against the strongest frozen-input baselines
# -----------------------------------------------------------------------------

BASELINE_CAMPAIGN_F1 = 0.834437
BASELINE_CAMPAIGN_AUROC = 0.928660
BASELINE_CAMPAIGN_AUPRC = 0.930799
BASELINE_CORE3_MACRO_F1 = 0.785982

comparison_summary = pd.DataFrame(
    [
        {
            "representation":
                "frozen_am_jepa_context",

            "pretraining_epoch":
                selected_pretraining_epoch,

            "campaign_f1":
                campaign_metrics["f1"],

            "campaign_f1_delta":
                campaign_metrics["f1"]
                - BASELINE_CAMPAIGN_F1,

            "campaign_auroc":
                campaign_metrics["auroc"],

            "campaign_auroc_delta":
                campaign_metrics["auroc"]
                - BASELINE_CAMPAIGN_AUROC,

            "campaign_auprc":
                campaign_metrics["auprc"],

            "campaign_auprc_delta":
                campaign_metrics["auprc"]
                - BASELINE_CAMPAIGN_AUPRC,

            **aspect_summary,

            "core3_macro_f1_delta":
                aspect_summary[
                    "core3_macro_f1"
                ]
                - BASELINE_CORE3_MACRO_F1,

            "mean_latent_dimension_std":
                mean_dimension_std,

            "near_constant_dimensions":
                near_constant_dimensions,
        }
    ]
)


# -----------------------------------------------------------------------------
# Modality-weight summary
# -----------------------------------------------------------------------------

modality_weight_summary = pd.DataFrame(
    {
        "modality": [
            "network",
            "temporal",
            "text",
        ],

        "train_mean_weight": (
            mean_modality_weights[
                train_indices
            ].mean(axis=0)
        ),

        "validation_mean_weight": (
            mean_modality_weights[
                validation_indices
            ].mean(axis=0)
        ),

        "test_mean_weight": (
            mean_modality_weights[
                test_indices
            ].mean(axis=0)
        ),
    }
)

modality_weight_summary.to_csv(
    AMJEPA_DIR
    / "am_jepa_modality_weight_summary.csv",
    index=False,
)


# -----------------------------------------------------------------------------
# Save results
# -----------------------------------------------------------------------------

campaign_results.to_csv(
    CAMPAIGN_RESULT_PATH,
    index=False,
)

aspect_results.to_csv(
    ASPECT_RESULT_PATH,
    index=False,
)

comparison_summary.to_csv(
    SUMMARY_PATH,
    index=False,
)


# -----------------------------------------------------------------------------
# Display
# -----------------------------------------------------------------------------

print("\n" + "=" * 80)
print("CAMPAIGN FROZEN LINEAR PROBE")
print("=" * 80)

display(
    campaign_results.round(6)
)

print("\n" + "=" * 80)
print("ASPECT FROZEN LINEAR PROBES")
print("=" * 80)

display(
    aspect_results[
        [
            "aspect",
            "threshold",
            "precision",
            "recall",
            "f1",
            "mcc",
            "auroc",
            "auprc",
        ]
    ].round(6)
)

print("\n" + "=" * 80)
print("AM-JEPA VERSUS STRONGEST POOLED BASELINES")
print("=" * 80)

display(
    comparison_summary.round(6)
)

print("\nMean learned modality weights:")
display(
    modality_weight_summary.round(6)
)

print("\nInterpretation criteria:")

print(
    "Campaign F1 target:",
    BASELINE_CAMPAIGN_F1,
)

print(
    "Campaign AUROC target:",
    BASELINE_CAMPAIGN_AUROC,
)

print(
    "Campaign AUPRC target:",
    BASELINE_CAMPAIGN_AUPRC,
)

print(
    "Core-three macro-F1 target:",
    BASELINE_CORE3_MACRO_F1,
)

print("\nSaved frozen representations:")
print(REPRESENTATION_PATH)

print("\nSaved linear-probe summary:")
print(SUMMARY_PATH)

In [ ]:
# =============================================================================
# CELL 22: AM-JEPA OBJECTIVE ABLATION AND FIVE-SEED EVALUATION
# =============================================================================

from pathlib import Path
import random
import time

import numpy as np
import pandas as pd
import torch

from torch.utils.data import DataLoader
from sklearn.multiclass import OneVsRestClassifier


# -----------------------------------------------------------------------------
# Configuration
# -----------------------------------------------------------------------------

ABLATION_EPOCHS = 10
ABLATION_BATCH_SIZE = 64

ABLATION_DIR = (
    AMJEPA_DIR
    / "objective_ablation"
)

ABLATION_DIR.mkdir(
    parents=True,
    exist_ok=True,
)

ABLATION_SUMMARY_PATH = (
    ABLATION_DIR
    / "am_jepa_objective_ablation.csv"
)

ABLATION_ASPECT_PATH = (
    ABLATION_DIR
    / "am_jepa_objective_ablation_aspects.csv"
)

MULTISEED_SUMMARY_PATH = (
    ABLATION_DIR
    / "am_jepa_full_multiseed_summary.csv"
)

MULTISEED_AGGREGATE_PATH = (
    ABLATION_DIR
    / "am_jepa_full_multiseed_aggregate.csv"
)

FULL_SEEDS = [
    42,
    52,
    62,
    72,
    82,
]


# -----------------------------------------------------------------------------
# Experiment definitions
# -----------------------------------------------------------------------------

# The variance term is retained across partial-objective variants
# so that collapse control remains constant. It is removed only
# in the explicit no-variance experiment.
EXPERIMENTS = [
    {
        "variant": "random_encoder",
        "seed": 42,
        "epochs": 0,
        "evidence_mask_rate": 0.0,
        "modality_drop_rate": 0.0,
        "lambda_evidence": 0.0,
        "lambda_modality": 0.0,
        "lambda_future": 0.0,
        "lambda_variance": 0.0,
    },
    {
        "variant": "masked_evidence_only",
        "seed": 42,
        "epochs": ABLATION_EPOCHS,
        "evidence_mask_rate": 0.30,
        "modality_drop_rate": 0.0,
        "lambda_evidence": 1.0,
        "lambda_modality": 0.0,
        "lambda_future": 0.0,
        "lambda_variance": 0.05,
    },
    {
        "variant": "evidence_plus_modality",
        "seed": 42,
        "epochs": ABLATION_EPOCHS,
        "evidence_mask_rate": 0.30,
        "modality_drop_rate": 0.25,
        "lambda_evidence": 1.0,
        "lambda_modality": 1.0,
        "lambda_future": 0.0,
        "lambda_variance": 0.05,
    },
    {
        "variant": "evidence_plus_future",
        "seed": 42,
        "epochs": ABLATION_EPOCHS,
        "evidence_mask_rate": 0.30,
        "modality_drop_rate": 0.0,
        "lambda_evidence": 1.0,
        "lambda_modality": 0.0,
        "lambda_future": 0.5,
        "lambda_variance": 0.05,
    },
    {
        "variant": "full_without_variance",
        "seed": 42,
        "epochs": ABLATION_EPOCHS,
        "evidence_mask_rate": 0.30,
        "modality_drop_rate": 0.25,
        "lambda_evidence": 1.0,
        "lambda_modality": 1.0,
        "lambda_future": 0.5,
        "lambda_variance": 0.0,
    },
]

for seed in FULL_SEEDS:
    EXPERIMENTS.append(
        {
            "variant": "full_am_jepa",
            "seed": seed,
            "epochs": ABLATION_EPOCHS,
            "evidence_mask_rate": 0.30,
            "modality_drop_rate": 0.25,
            "lambda_evidence": 1.0,
            "lambda_modality": 1.0,
            "lambda_future": 0.5,
            "lambda_variance": 0.05,
        }
    )


# -----------------------------------------------------------------------------
# Required-object verification
# -----------------------------------------------------------------------------

required_objects = [
    "AMJEPA",
    "run_pretraining_epoch",
    "train_pretraining_dataset",
    "validation_pretraining_loader",
    "calculate_metrics",
    "select_validation_threshold",
    "build_linear_probe",
]

missing_objects = [
    name
    for name in required_objects
    if name not in globals()
]

if missing_objects:
    raise NameError(
        "Missing required objects:\n"
        + "\n".join(missing_objects)
        + "\nRun Cells 19, 20, and 21 first."
    )


# -----------------------------------------------------------------------------
# Reproducibility helper
# -----------------------------------------------------------------------------

def set_experiment_seed(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)

    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

        torch.backends.cudnn.deterministic = True
        torch.backends.cudnn.benchmark = False


def make_seeded_train_loader(seed):
    generator = torch.Generator()
    generator.manual_seed(seed)

    return DataLoader(
        train_pretraining_dataset,
        batch_size=ABLATION_BATCH_SIZE,
        shuffle=True,
        num_workers=0,
        pin_memory=(
            DEVICE.type == "cuda"
        ),
        drop_last=False,
        generator=generator,
    )


# -----------------------------------------------------------------------------
# Frozen representation extraction
# -----------------------------------------------------------------------------

def extract_context_representations(
    trained_model,
    batch_size=256,
):
    trained_model.context_encoder.eval()

    representations = np.zeros(
        (
            X_network.shape[0],
            LATENT_DIMENSION,
        ),
        dtype=np.float32,
    )

    modality_weights = np.zeros(
        (
            X_network.shape[0],
            3,
        ),
        dtype=np.float32,
    )

    with torch.no_grad():
        for start in range(
            0,
            X_network.shape[0],
            batch_size,
        ):
            end = min(
                start + batch_size,
                X_network.shape[0],
            )

            indices = np.arange(
                start,
                end,
            )

            network_batch = (
                torch.from_numpy(
                    X_network[indices]
                )
                .float()
                .to(DEVICE)
            )

            temporal_batch = (
                torch.from_numpy(
                    X_temporal[indices]
                )
                .float()
                .to(DEVICE)
            )

            text_batch = (
                torch.from_numpy(
                    text_embedding_lookup[
                        window_text_ids[
                            indices
                        ]
                    ]
                )
                .float()
                .to(DEVICE)
            )

            mask_batch = (
                torch.from_numpy(
                    attention_mask[
                        indices
                    ]
                )
                .bool()
                .to(DEVICE)
            )

            output = (
                trained_model
                .context_encoder(
                    network=network_batch,
                    temporal=temporal_batch,
                    text=text_batch,
                    attention_mask=
                        mask_batch,
                    modality_keep_mask=None,
                    evidence_mask=None,
                )
            )

            representations[
                start:end
            ] = (
                output["pooled"]
                .cpu()
                .numpy()
            )

            expanded_mask = (
                mask_batch
                .float()
                .unsqueeze(-1)
            )

            pooled_weights = (
                (
                    output[
                        "modality_weights"
                    ]
                    * expanded_mask
                ).sum(dim=1)
                / expanded_mask
                .sum(dim=1)
                .clamp_min(1.0)
            )

            modality_weights[
                start:end
            ] = (
                pooled_weights
                .cpu()
                .numpy()
            )

    return representations, modality_weights


# -----------------------------------------------------------------------------
# Frozen linear-probe evaluation
# -----------------------------------------------------------------------------

def evaluate_frozen_representation(
    representations,
    variant_name,
    seed,
):
    X_train_probe = representations[
        train_indices
    ]

    X_validation_probe = (
        representations[
            validation_indices
        ]
    )

    X_test_probe = representations[
        test_indices
    ]

    y_campaign_train = (
        campaign_targets[
            train_indices
        ]
    )

    y_campaign_validation = (
        campaign_targets[
            validation_indices
        ]
    )

    y_campaign_test = (
        campaign_targets[
            test_indices
        ]
    )

    y_aspect_train = aspect_targets[
        train_indices
    ]

    y_aspect_validation = (
        aspect_targets[
            validation_indices
        ]
    )

    y_aspect_test = aspect_targets[
        test_indices
    ]

    # -----------------------------------------------------------------
    # Campaign probe
    # -----------------------------------------------------------------

    campaign_probe = (
        build_linear_probe()
    )

    campaign_probe.fit(
        X_train_probe,
        y_campaign_train,
    )

    validation_probabilities = (
        campaign_probe.predict_proba(
            X_validation_probe
        )[:, 1]
    )

    test_probabilities = (
        campaign_probe.predict_proba(
            X_test_probe
        )[:, 1]
    )

    campaign_threshold = (
        select_validation_threshold(
            y_campaign_validation,
            validation_probabilities,
        )
    )

    campaign_metrics = (
        calculate_metrics(
            y_campaign_test,
            test_probabilities,
            campaign_threshold,
        )
    )

    # -----------------------------------------------------------------
    # Aspect probes
    # -----------------------------------------------------------------

    aspect_probe = (
        OneVsRestClassifier(
            build_linear_probe(),
            n_jobs=-1,
        )
    )

    aspect_probe.fit(
        X_train_probe,
        y_aspect_train,
    )

    aspect_validation_probabilities = (
        aspect_probe.predict_proba(
            X_validation_probe
        )
    )

    aspect_test_probabilities = (
        aspect_probe.predict_proba(
            X_test_probe
        )
    )

    aspect_rows = []

    for aspect_index, aspect_name in enumerate(
        ASPECT_NAMES
    ):
        validation_labels = (
            y_aspect_validation[
                :,
                aspect_index,
            ]
        )

        test_labels = (
            y_aspect_test[
                :,
                aspect_index,
            ]
        )

        if int(
            validation_labels.sum()
        ) >= 5:
            threshold = (
                select_validation_threshold(
                    validation_labels,
                    aspect_validation_probabilities[
                        :,
                        aspect_index,
                    ],
                )
            )

            threshold_status = (
                "validation_calibrated"
            )

        else:
            threshold = 0.5

            threshold_status = (
                "fixed_insufficient_positives"
            )

        metrics = calculate_metrics(
            test_labels,
            aspect_test_probabilities[
                :,
                aspect_index,
            ],
            threshold,
        )

        aspect_rows.append(
            {
                "variant": variant_name,
                "seed": seed,
                "aspect": aspect_name,
                "threshold_status":
                    threshold_status,
                **metrics,
            }
        )

    aspect_table = pd.DataFrame(
        aspect_rows
    )

    core_three = aspect_table[
        aspect_table["aspect"]
        != "data_exfiltration"
    ]

    training_std = (
        X_train_probe.std(axis=0)
    )

    summary = {
        "variant": variant_name,
        "seed": seed,

        "campaign_threshold":
            campaign_threshold,

        "campaign_accuracy":
            campaign_metrics["accuracy"],

        "campaign_balanced_accuracy":
            campaign_metrics[
                "balanced_accuracy"
            ],

        "campaign_precision":
            campaign_metrics["precision"],

        "campaign_recall":
            campaign_metrics["recall"],

        "campaign_f1":
            campaign_metrics["f1"],

        "campaign_mcc":
            campaign_metrics["mcc"],

        "campaign_auroc":
            campaign_metrics["auroc"],

        "campaign_auprc":
            campaign_metrics["auprc"],

        "four_aspect_macro_f1":
            aspect_table["f1"].mean(),

        "four_aspect_macro_mcc":
            aspect_table["mcc"].mean(),

        "core3_macro_f1":
            core_three["f1"].mean(),

        "core3_macro_mcc":
            core_three["mcc"].mean(),

        "core3_macro_auroc":
            core_three["auroc"].mean(),

        "core3_macro_auprc":
            core_three["auprc"].mean(),

        "mean_latent_std":
            float(
                training_std.mean()
            ),

        "minimum_latent_std":
            float(
                training_std.min()
            ),

        "near_constant_dimensions":
            int(
                (
                    training_std < 1e-3
                ).sum()
            ),
    }

    return summary, aspect_table


# -----------------------------------------------------------------------------
# Run all objective and seed experiments
# -----------------------------------------------------------------------------

experiment_rows = []
all_aspect_rows = []

print("=" * 80)
print("AM-JEPA OBJECTIVE ABLATION")
print("=" * 80)

for experiment_number, experiment in enumerate(
    EXPERIMENTS,
    start=1,
):
    variant_name = experiment["variant"]
    seed = experiment["seed"]
    number_of_epochs = (
        experiment["epochs"]
    )

    print("\n" + "-" * 80)

    print(
        f"Experiment "
        f"{experiment_number}/"
        f"{len(EXPERIMENTS)}"
    )

    print(
        "Variant:",
        variant_name,
        "| Seed:",
        seed,
        "| Epochs:",
        number_of_epochs,
    )

    set_experiment_seed(seed)

    # Update the global objective configuration used by
    # run_pretraining_epoch from Cell 20.
    EVIDENCE_MASK_RATE = (
        experiment[
            "evidence_mask_rate"
        ]
    )

    MODALITY_DROP_RATE = (
        experiment[
            "modality_drop_rate"
        ]
    )

    LAMBDA_EVIDENCE = (
        experiment[
            "lambda_evidence"
        ]
    )

    LAMBDA_MODALITY = (
        experiment[
            "lambda_modality"
        ]
    )

    LAMBDA_FUTURE = (
        experiment[
            "lambda_future"
        ]
    )

    LAMBDA_VARIANCE = (
        experiment[
            "lambda_variance"
        ]
    )

    variant_model = AMJEPA(
        network_dimension=79,
        temporal_dimension=3,
        text_dimension=128,
        latent_dimension=
            LATENT_DIMENSION,
        hidden_dimension=
            HIDDEN_DIMENSION,
        number_of_layers=
            NUMBER_OF_GRU_LAYERS,
        dropout_rate=
            DROPOUT_RATE,
    ).to(DEVICE)

    final_training_metrics = {
        "total_loss": np.nan,
        "evidence_loss": np.nan,
        "modality_loss": np.nan,
        "future_loss": np.nan,
        "variance_loss": np.nan,
        "latent_std": np.nan,
    }

    final_validation_metrics = (
        final_training_metrics.copy()
    )

    training_seconds = 0.0

    if number_of_epochs > 0:
        train_loader = (
            make_seeded_train_loader(
                seed
            )
        )

        trainable_parameters = [
            parameter
            for parameter
            in variant_model.parameters()
            if parameter.requires_grad
        ]

        variant_optimizer = (
            torch.optim.AdamW(
                trainable_parameters,
                lr=LEARNING_RATE,
                weight_decay=
                    WEIGHT_DECAY,
            )
        )

        total_steps = (
            number_of_epochs
            * len(train_loader)
        )

        variant_scheduler = (
            torch.optim.lr_scheduler
            .CosineAnnealingLR(
                variant_optimizer,
                T_max=max(
                    total_steps,
                    1,
                ),
                eta_min=1e-5,
            )
        )

        global_step = 0
        start_time = time.perf_counter()

        for epoch in range(
            1,
            number_of_epochs + 1,
        ):
            (
                final_training_metrics,
                global_step,
            ) = run_pretraining_epoch(
                model=variant_model,
                data_loader=train_loader,
                optimizer=
                    variant_optimizer,
                scheduler=
                    variant_scheduler,
                global_step=
                    global_step,
                total_steps=
                    total_steps,
                training=True,
            )

            (
                final_validation_metrics,
                _,
            ) = run_pretraining_epoch(
                model=variant_model,
                data_loader=
                    validation_pretraining_loader,
                optimizer=None,
                scheduler=None,
                global_step=
                    global_step,
                total_steps=
                    total_steps,
                training=False,
            )

            print(
                f"Epoch {epoch:02d}/"
                f"{number_of_epochs:02d} | "
                f"train="
                f"{final_training_metrics['total_loss']:.5f} | "
                f"val="
                f"{final_validation_metrics['total_loss']:.5f} | "
                f"latent_std="
                f"{final_validation_metrics['latent_std']:.5f}"
            )

        training_seconds = (
            time.perf_counter()
            - start_time
        )

    representations, modality_weights = (
        extract_context_representations(
            variant_model
        )
    )

    summary, aspect_table = (
        evaluate_frozen_representation(
            representations=
                representations,
            variant_name=
                variant_name,
            seed=seed,
        )
    )

    summary.update(
        {
            "epochs":
                number_of_epochs,

            "evidence_mask_rate":
                EVIDENCE_MASK_RATE,

            "modality_drop_rate":
                MODALITY_DROP_RATE,

            "lambda_evidence":
                LAMBDA_EVIDENCE,

            "lambda_modality":
                LAMBDA_MODALITY,

            "lambda_future":
                LAMBDA_FUTURE,

            "lambda_variance":
                LAMBDA_VARIANCE,

            "final_train_loss":
                final_training_metrics[
                    "total_loss"
                ],

            "final_validation_loss":
                final_validation_metrics[
                    "total_loss"
                ],

            "training_seconds":
                training_seconds,

            "network_weight":
                modality_weights[
                    validation_indices,
                    0,
                ].mean(),

            "temporal_weight":
                modality_weights[
                    validation_indices,
                    1,
                ].mean(),

            "text_weight":
                modality_weights[
                    validation_indices,
                    2,
                ].mean(),
        }
    )

    experiment_rows.append(summary)
    all_aspect_rows.append(
        aspect_table
    )

    checkpoint_filename = (
        f"{variant_name}_seed_{seed}.pt"
    )

    torch.save(
        {
            "variant":
                variant_name,

            "seed":
                seed,

            "epochs":
                number_of_epochs,

            "model_state_dict":
                variant_model.state_dict(),

            "experiment":
                experiment,

            "summary":
                summary,
        },
        ABLATION_DIR
        / checkpoint_filename,
    )

    np.save(
        ABLATION_DIR
        / (
            f"{variant_name}"
            f"_seed_{seed}"
            "_representations.npy"
        ),
        representations,
    )

    print(
        "Campaign F1:",
        round(
            summary["campaign_f1"],
            6,
        ),
        "| Core-three macro-F1:",
        round(
            summary[
                "core3_macro_f1"
            ],
            6,
        ),
    )

    # Release GPU memory before the next run.
    del variant_model

    if DEVICE.type == "cuda":
        torch.cuda.empty_cache()


# -----------------------------------------------------------------------------
# Assemble and save detailed results
# -----------------------------------------------------------------------------

ablation_results = pd.DataFrame(
    experiment_rows
)

ablation_aspect_results = pd.concat(
    all_aspect_rows,
    ignore_index=True,
)

ablation_results.to_csv(
    ABLATION_SUMMARY_PATH,
    index=False,
)

ablation_aspect_results.to_csv(
    ABLATION_ASPECT_PATH,
    index=False,
)


# -----------------------------------------------------------------------------
# Objective ablation: seed 42 only
# -----------------------------------------------------------------------------

objective_ablation = (
    ablation_results[
        ablation_results["seed"] == 42
    ]
    .copy()
    .sort_values(
        by="campaign_f1",
        ascending=False,
    )
)


# -----------------------------------------------------------------------------
# Five-seed full AM-JEPA summary
# -----------------------------------------------------------------------------

full_multiseed = (
    ablation_results[
        ablation_results["variant"]
        == "full_am_jepa"
    ]
    .copy()
    .sort_values("seed")
)

full_multiseed.to_csv(
    MULTISEED_SUMMARY_PATH,
    index=False,
)

aggregate_metrics = [
    "campaign_accuracy",
    "campaign_balanced_accuracy",
    "campaign_precision",
    "campaign_recall",
    "campaign_f1",
    "campaign_mcc",
    "campaign_auroc",
    "campaign_auprc",
    "four_aspect_macro_f1",
    "four_aspect_macro_mcc",
    "core3_macro_f1",
    "core3_macro_mcc",
    "core3_macro_auroc",
    "core3_macro_auprc",
    "mean_latent_std",
]

aggregate_rows = []

for metric in aggregate_metrics:
    values = full_multiseed[
        metric
    ].to_numpy(dtype=np.float64)

    aggregate_rows.append(
        {
            "metric": metric,
            "mean": values.mean(),
            "standard_deviation":
                values.std(ddof=1),
            "minimum": values.min(),
            "maximum": values.max(),
            "seeds": len(values),
        }
    )

multiseed_aggregate = pd.DataFrame(
    aggregate_rows
)

multiseed_aggregate.to_csv(
    MULTISEED_AGGREGATE_PATH,
    index=False,
)


# -----------------------------------------------------------------------------
# Improvement relative to random encoder
# -----------------------------------------------------------------------------

random_result = (
    ablation_results[
        ablation_results["variant"]
        == "random_encoder"
    ]
    .iloc[0]
)

objective_ablation[
    "campaign_f1_over_random"
] = (
    objective_ablation[
        "campaign_f1"
    ]
    - random_result[
        "campaign_f1"
    ]
)

objective_ablation[
    "core3_f1_over_random"
] = (
    objective_ablation[
        "core3_macro_f1"
    ]
    - random_result[
        "core3_macro_f1"
    ]
)


# -----------------------------------------------------------------------------
# Display results
# -----------------------------------------------------------------------------

print("\n" + "=" * 80)
print("OBJECTIVE ABLATION: SEED 42")
print("=" * 80)

display(
    objective_ablation[
        [
            "variant",
            "epochs",
            "campaign_f1",
            "campaign_auroc",
            "campaign_auprc",
            "core3_macro_f1",
            "core3_macro_mcc",
            "campaign_f1_over_random",
            "core3_f1_over_random",
            "mean_latent_std",
            "near_constant_dimensions",
            "training_seconds",
        ]
    ].round(6)
)

print("\n" + "=" * 80)
print("FULL AM-JEPA: FIVE-SEED RESULTS")
print("=" * 80)

display(
    full_multiseed[
        [
            "seed",
            "campaign_f1",
            "campaign_mcc",
            "campaign_auroc",
            "campaign_auprc",
            "core3_macro_f1",
            "core3_macro_mcc",
            "core3_macro_auroc",
            "core3_macro_auprc",
            "mean_latent_std",
        ]
    ].round(6)
)

print("\n" + "=" * 80)
print("FULL AM-JEPA: MEAN AND STANDARD DEVIATION")
print("=" * 80)

display(
    multiseed_aggregate.round(6)
)

print("\nSaved objective-ablation results:")
print(ABLATION_SUMMARY_PATH)

print("\nSaved five-seed aggregate:")
print(MULTISEED_AGGREGATE_PATH)

In [ ]:
# =============================================================================
# CELL 23: RECOVER AND VERIFY SAVED AM-JEPA ABLATION RESULTS
# =============================================================================

from pathlib import Path
import gc

import numpy as np
import pandas as pd


DATA_DIR = Path(
    r"."
)

ABLATION_DIR = (
    DATA_DIR
    / "outputs"
    / "am_jepa_experiment"
    / "objective_ablation"
)

ABLATION_SUMMARY_PATH = (
    ABLATION_DIR
    / "am_jepa_objective_ablation.csv"
)

ABLATION_ASPECT_PATH = (
    ABLATION_DIR
    / "am_jepa_objective_ablation_aspects.csv"
)

MULTISEED_SUMMARY_PATH = (
    ABLATION_DIR
    / "am_jepa_full_multiseed_summary.csv"
)

MULTISEED_AGGREGATE_PATH = (
    ABLATION_DIR
    / "am_jepa_full_multiseed_aggregate.csv"
)


required_files = [
    ABLATION_SUMMARY_PATH,
    ABLATION_ASPECT_PATH,
    MULTISEED_SUMMARY_PATH,
    MULTISEED_AGGREGATE_PATH,
]

missing_files = [
    str(path)
    for path in required_files
    if not path.exists()
]

if missing_files:
    raise FileNotFoundError(
        "Missing result files:\n"
        + "\n".join(missing_files)
    )


ablation_results = pd.read_csv(
    ABLATION_SUMMARY_PATH
)

ablation_aspect_results = pd.read_csv(
    ABLATION_ASPECT_PATH
)

full_multiseed = pd.read_csv(
    MULTISEED_SUMMARY_PATH
)

multiseed_aggregate = pd.read_csv(
    MULTISEED_AGGREGATE_PATH
)


# -----------------------------------------------------------------------------
# Integrity checks
# -----------------------------------------------------------------------------

assert len(ablation_results) == 10, (
    "Expected 10 completed experiment rows."
)

assert len(ablation_aspect_results) == 40, (
    "Expected four aspect rows for each of "
    "the 10 experiments."
)

assert len(full_multiseed) == 5, (
    "Expected five full AM-JEPA seeds."
)

assert set(
    full_multiseed["seed"].astype(int)
) == {
    42,
    52,
    62,
    72,
    82,
}

assert not ablation_results[
    [
        "campaign_f1",
        "campaign_auroc",
        "campaign_auprc",
        "core3_macro_f1",
    ]
].isna().any().any()

assert not full_multiseed[
    [
        "campaign_f1",
        "campaign_auroc",
        "campaign_auprc",
        "core3_macro_f1",
    ]
].isna().any().any()


# -----------------------------------------------------------------------------
# Display recovered results
# -----------------------------------------------------------------------------

print("=" * 80)
print("RECOVERED AM-JEPA OBJECTIVE ABLATION")
print("=" * 80)

objective_ablation = (
    ablation_results[
        ablation_results["seed"] == 42
    ]
    .sort_values(
        by="campaign_f1",
        ascending=False,
    )
)

display(
    objective_ablation[
        [
            "variant",
            "epochs",
            "campaign_f1",
            "campaign_mcc",
            "campaign_auroc",
            "campaign_auprc",
            "core3_macro_f1",
            "core3_macro_mcc",
            "mean_latent_std",
            "near_constant_dimensions",
        ]
    ].round(6)
)

print("\n" + "=" * 80)
print("RECOVERED FIVE-SEED FULL AM-JEPA RESULTS")
print("=" * 80)

display(
    full_multiseed[
        [
            "seed",
            "campaign_f1",
            "campaign_mcc",
            "campaign_auroc",
            "campaign_auprc",
            "core3_macro_f1",
            "core3_macro_mcc",
            "core3_macro_auroc",
            "core3_macro_auprc",
        ]
    ].round(6)
)

print("\n" + "=" * 80)
print("RECOVERED MULTISEED AGGREGATE")
print("=" * 80)

display(
    multiseed_aggregate.round(6)
)

print("\nIntegrity verification passed.")
print("No experiment rerun is required.")


# -----------------------------------------------------------------------------
# Conservative memory cleanup
# -----------------------------------------------------------------------------

gc.collect()

try:
    import torch

    if torch.cuda.is_available():
        torch.cuda.empty_cache()

        print(
            "\nCUDA cache cleared:",
            round(
                torch.cuda.memory_allocated()
                / 1024**3,
                4,
            ),
            "GB allocated",
        )

except ImportError:
    pass

In [ ]:
required_objects = [
    "AMJEPA",
    "X_network",
    "X_temporal",
    "text_embedding_lookup",
    "window_text_ids",
    "attention_mask",
    "train_indices",
    "validation_indices",
    "test_indices",
    "campaign_targets",
    "aspect_targets",
    "ASPECT_NAMES",
    "build_linear_probe",
    "select_validation_threshold",
    "calculate_metrics",
    "DEVICE",
    "LATENT_DIMENSION",
    "HIDDEN_DIMENSION",
    "NUMBER_OF_GRU_LAYERS",
    "DROPOUT_RATE",
]

missing_objects = [
    name
    for name in required_objects
    if name not in globals()
]

if missing_objects:
    print("Still missing:")
    print("\n".join(missing_objects))
else:
    print("All required objects are available.")
    print("You can now run Cell 24.")

In [ ]:
# -----------------------------------------------------------------------------
# Representation extraction — corrected Boolean modality mask
# -----------------------------------------------------------------------------

def extract_representations(
    model,
    sample_indices,
    keep_mask=None,
    batch_size=256,
):
    sample_indices = np.asarray(
        sample_indices,
        dtype=np.int64,
    )

    number_of_samples = len(
        sample_indices
    )

    representations = np.zeros(
        (
            number_of_samples,
            LATENT_DIMENSION,
        ),
        dtype=np.float32,
    )

    if keep_mask is not None:
        # The encoder uses ~modality_keep_mask,
        # so this mask must be Boolean.
        keep_mask = np.asarray(
            keep_mask,
            dtype=np.bool_,
        )

        if keep_mask.ndim == 1:
            keep_mask = np.repeat(
                keep_mask[None, :],
                number_of_samples,
                axis=0,
            )

        expected_shape = (
            number_of_samples,
            3,
        )

        if keep_mask.shape != expected_shape:
            raise ValueError(
                "keep_mask must have shape "
                f"{expected_shape}, but received "
                f"{keep_mask.shape}."
            )

        if np.any(
            keep_mask.sum(axis=1) == 0
        ):
            raise ValueError(
                "Every sample must retain at least "
                "one modality."
            )

    with torch.no_grad():
        for start in range(
            0,
            number_of_samples,
            batch_size,
        ):
            end = min(
                start + batch_size,
                number_of_samples,
            )

            batch_indices = sample_indices[
                start:end
            ]

            network_batch = (
                torch.from_numpy(
                    X_network[
                        batch_indices
                    ]
                )
                .float()
                .to(DEVICE)
            )

            temporal_batch = (
                torch.from_numpy(
                    X_temporal[
                        batch_indices
                    ]
                )
                .float()
                .to(DEVICE)
            )

            text_batch = (
                torch.from_numpy(
                    text_embedding_lookup[
                        window_text_ids[
                            batch_indices
                        ]
                    ]
                )
                .float()
                .to(DEVICE)
            )

            sequence_mask_batch = (
                torch.from_numpy(
                    attention_mask[
                        batch_indices
                    ]
                )
                .bool()
                .to(DEVICE)
            )

            if keep_mask is None:
                modality_keep_batch = None
            else:
                modality_keep_batch = (
                    torch.from_numpy(
                        keep_mask[
                            start:end
                        ]
                    )
                    .bool()
                    .to(DEVICE)
                )

            output = model.context_encoder(
                network=network_batch,
                temporal=temporal_batch,
                text=text_batch,
                attention_mask=
                    sequence_mask_batch,
                modality_keep_mask=
                    modality_keep_batch,
                evidence_mask=None,
            )

            representations[
                start:end
            ] = (
                output["pooled"]
                .detach()
                .cpu()
                .numpy()
            )

    return representations

In [ ]:
# =============================================================================
# CELL 24B: CORRECTED MISSING-MODALITY ROBUSTNESS
# =============================================================================

import gc
import random

import numpy as np
import pandas as pd
import torch


# -----------------------------------------------------------------------------
# Verify objects created by the previous setup
# -----------------------------------------------------------------------------

required_objects = [
    "MODEL_CONFIGURATIONS",
    "FIXED_SCENARIOS",
    "STOCHASTIC_DROP_RATES",
    "COMMON_STOCHASTIC_MASKS",
    "load_checkpoint_model",
    "train_clean_probes",
    "evaluate_scenario",
    "X_network",
    "X_temporal",
    "text_embedding_lookup",
    "window_text_ids",
    "attention_mask",
    "train_indices",
    "validation_indices",
    "test_indices",
    "LATENT_DIMENSION",
    "DEVICE",
    "DETAILED_RESULTS_PATH",
    "AGGREGATE_RESULTS_PATH",
    "ASPECT_RESULTS_PATH",
]

missing_objects = [
    name
    for name in required_objects
    if name not in globals()
]

if missing_objects:
    raise NameError(
        "Missing required objects:\n"
        + "\n".join(missing_objects)
        + "\n\nRun the setup portion of Cell 24 first."
    )


# -----------------------------------------------------------------------------
# Reproducibility
# -----------------------------------------------------------------------------

def set_seed(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)

    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)


# -----------------------------------------------------------------------------
# Corrected representation extraction
#
# modality_keep_mask shape:
# [batch, sequence_length, 3]
# -----------------------------------------------------------------------------

def extract_representations(
    model,
    sample_indices,
    keep_mask=None,
    batch_size=256,
):
    sample_indices = np.asarray(
        sample_indices,
        dtype=np.int64,
    )

    number_of_samples = len(
        sample_indices
    )

    representations = np.zeros(
        (
            number_of_samples,
            LATENT_DIMENSION,
        ),
        dtype=np.float32,
    )

    if keep_mask is not None:
        keep_mask = np.asarray(
            keep_mask,
            dtype=np.bool_,
        )

        if keep_mask.ndim == 1:
            keep_mask = np.repeat(
                keep_mask[None, :],
                number_of_samples,
                axis=0,
            )

        expected_shape = (
            number_of_samples,
            3,
        )

        if keep_mask.shape != expected_shape:
            raise ValueError(
                "keep_mask must have shape "
                f"{expected_shape}, but received "
                f"{keep_mask.shape}."
            )

        if np.any(
            keep_mask.sum(axis=1) == 0
        ):
            raise ValueError(
                "Every sample must retain at least "
                "one modality."
            )

    with torch.no_grad():
        for start in range(
            0,
            number_of_samples,
            batch_size,
        ):
            end = min(
                start + batch_size,
                number_of_samples,
            )

            batch_indices = sample_indices[
                start:end
            ]

            network_batch = (
                torch.from_numpy(
                    X_network[
                        batch_indices
                    ]
                )
                .float()
                .to(DEVICE)
            )

            temporal_batch = (
                torch.from_numpy(
                    X_temporal[
                        batch_indices
                    ]
                )
                .float()
                .to(DEVICE)
            )

            text_batch = (
                torch.from_numpy(
                    text_embedding_lookup[
                        window_text_ids[
                            batch_indices
                        ]
                    ]
                )
                .float()
                .to(DEVICE)
            )

            sequence_mask_batch = (
                torch.from_numpy(
                    attention_mask[
                        batch_indices
                    ]
                )
                .bool()
                .to(DEVICE)
            )

            if keep_mask is None:
                modality_keep_batch = None

            else:
                # Shape before expansion:
                # [batch, 3]
                modality_keep_batch = (
                    torch.from_numpy(
                        keep_mask[
                            start:end
                        ]
                    )
                    .bool()
                    .to(DEVICE)
                )

                # Expand to:
                # [batch, sequence_length, 3]
                modality_keep_batch = (
                    modality_keep_batch
                    .unsqueeze(1)
                    .expand(
                        -1,
                        sequence_mask_batch.shape[1],
                        -1,
                    )
                )

                # Remove all modalities at padded positions.
                modality_keep_batch = (
                    modality_keep_batch
                    & sequence_mask_batch.unsqueeze(-1)
                )

            output = model.context_encoder(
                network=network_batch,
                temporal=temporal_batch,
                text=text_batch,
                attention_mask=
                    sequence_mask_batch,
                modality_keep_mask=
                    modality_keep_batch,
                evidence_mask=None,
            )

            representations[
                start:end
            ] = (
                output["pooled"]
                .detach()
                .cpu()
                .numpy()
            )

    return representations


# -----------------------------------------------------------------------------
# Reset results from previous failed attempts
# -----------------------------------------------------------------------------

result_rows = []
aspect_result_tables = []


# -----------------------------------------------------------------------------
# Display common corruption masks
# -----------------------------------------------------------------------------

print("=" * 80)
print("COMMON STOCHASTIC CORRUPTION MASKS")
print("=" * 80)

for drop_rate, keep_mask in (
    COMMON_STOCHASTIC_MASKS.items()
):
    realized_drop_rate = float(
        1.0
        - keep_mask.mean()
    )

    print(
        f"Nominal drop rate: "
        f"{drop_rate:.2f} | "
        f"Realized modality drop rate: "
        f"{realized_drop_rate:.4f}"
    )


# -----------------------------------------------------------------------------
# Execute robustness evaluation
# -----------------------------------------------------------------------------

print("\n" + "=" * 80)
print("AM-JEPA MISSING-MODALITY ROBUSTNESS")
print("=" * 80)

for configuration_number, configuration in enumerate(
    MODEL_CONFIGURATIONS,
    start=1,
):
    variant = configuration[
        "variant"
    ]

    seed = configuration[
        "seed"
    ]

    checkpoint_path = configuration[
        "checkpoint"
    ]

    print("\n" + "-" * 80)

    print(
        f"Model {configuration_number}/"
        f"{len(MODEL_CONFIGURATIONS)} | "
        f"{variant} | seed={seed}"
    )

    set_seed(seed)

    model = load_checkpoint_model(
        checkpoint_path
    )

    # -------------------------------------------------------------------------
    # Train probes on complete training and validation representations
    # -------------------------------------------------------------------------

    training_representations = (
        extract_representations(
            model=model,
            sample_indices=
                train_indices,
            keep_mask=None,
        )
    )

    validation_representations = (
        extract_representations(
            model=model,
            sample_indices=
                validation_indices,
            keep_mask=None,
        )
    )

    probes = train_clean_probes(
        training_representations,
        validation_representations,
    )

    model_rows = []

    # -------------------------------------------------------------------------
    # Fixed missing-modality scenarios
    # -------------------------------------------------------------------------

    for scenario, keep_vector in (
        FIXED_SCENARIOS.items()
    ):
        if scenario == "complete":
            scenario_keep_mask = None
            nominal_drop_rate = 0.0
            realized_drop_rate = 0.0

        else:
            scenario_keep_mask = np.asarray(
                keep_vector,
                dtype=np.bool_,
            )

            nominal_drop_rate = float(
                1.0
                - scenario_keep_mask.mean()
            )

            realized_drop_rate = (
                nominal_drop_rate
            )

        test_representations = (
            extract_representations(
                model=model,
                sample_indices=
                    test_indices,
                keep_mask=
                    scenario_keep_mask,
            )
        )

        summary, aspect_table = (
            evaluate_scenario(
                representations=
                    test_representations,
                probes=probes,
                variant=variant,
                seed=seed,
                scenario=scenario,
                nominal_drop_rate=
                    nominal_drop_rate,
                realized_drop_rate=
                    realized_drop_rate,
            )
        )

        result_rows.append(
            summary
        )

        model_rows.append(
            summary
        )

        aspect_result_tables.append(
            aspect_table
        )

    # -------------------------------------------------------------------------
    # Random per-window missing-modality scenarios
    # -------------------------------------------------------------------------

    for drop_rate in (
        STOCHASTIC_DROP_RATES
    ):
        scenario_name = (
            f"random_drop_"
            f"{int(drop_rate * 100)}pct"
        )

        # The same corruption mask is used for every model seed.
        stochastic_keep_mask = np.asarray(
            COMMON_STOCHASTIC_MASKS[
                drop_rate
            ],
            dtype=np.bool_,
        )

        realized_drop_rate = float(
            1.0
            - stochastic_keep_mask.mean()
        )

        test_representations = (
            extract_representations(
                model=model,
                sample_indices=
                    test_indices,
                keep_mask=
                    stochastic_keep_mask,
            )
        )

        summary, aspect_table = (
            evaluate_scenario(
                representations=
                    test_representations,
                probes=probes,
                variant=variant,
                seed=seed,
                scenario=
                    scenario_name,
                nominal_drop_rate=
                    drop_rate,
                realized_drop_rate=
                    realized_drop_rate,
            )
        )

        result_rows.append(
            summary
        )

        model_rows.append(
            summary
        )

        aspect_result_tables.append(
            aspect_table
        )

    # -------------------------------------------------------------------------
    # Immediate model-level output
    # -------------------------------------------------------------------------

    model_table = pd.DataFrame(
        model_rows
    )

    clean_row = model_table[
        model_table["scenario"]
        == "complete"
    ].iloc[0]

    model_table[
        "campaign_f1_drop"
    ] = (
        clean_row[
            "campaign_f1"
        ]
        - model_table[
            "campaign_f1"
        ]
    )

    model_table[
        "core3_f1_drop"
    ] = (
        clean_row[
            "core3_macro_f1"
        ]
        - model_table[
            "core3_macro_f1"
        ]
    )

    print(
        model_table[
            [
                "scenario",
                "realized_drop_rate",
                "campaign_f1",
                "campaign_auroc",
                "core3_macro_f1",
                "campaign_f1_drop",
                "core3_f1_drop",
            ]
        ]
        .round(6)
        .to_string(
            index=False
        )
    )

    del model
    del probes
    del training_representations
    del validation_representations

    gc.collect()

    if DEVICE.type == "cuda":
        torch.cuda.empty_cache()


# -----------------------------------------------------------------------------
# Assemble detailed results
# -----------------------------------------------------------------------------

detailed_results = pd.DataFrame(
    result_rows
)

aspect_results = pd.concat(
    aspect_result_tables,
    ignore_index=True,
)

clean_reference = (
    detailed_results[
        detailed_results["scenario"]
        == "complete"
    ][
        [
            "variant",
            "seed",
            "campaign_f1",
            "campaign_mcc",
            "core3_macro_f1",
            "core3_macro_mcc",
        ]
    ]
    .rename(
        columns={
            "campaign_f1":
                "clean_campaign_f1",
            "campaign_mcc":
                "clean_campaign_mcc",
            "core3_macro_f1":
                "clean_core3_macro_f1",
            "core3_macro_mcc":
                "clean_core3_macro_mcc",
        }
    )
)

detailed_results = (
    detailed_results.merge(
        clean_reference,
        on=[
            "variant",
            "seed",
        ],
        how="left",
        validate="many_to_one",
    )
)

detailed_results[
    "campaign_f1_drop"
] = (
    detailed_results[
        "clean_campaign_f1"
    ]
    - detailed_results[
        "campaign_f1"
    ]
)

detailed_results[
    "campaign_mcc_drop"
] = (
    detailed_results[
        "clean_campaign_mcc"
    ]
    - detailed_results[
        "campaign_mcc"
    ]
)

detailed_results[
    "core3_f1_drop"
] = (
    detailed_results[
        "clean_core3_macro_f1"
    ]
    - detailed_results[
        "core3_macro_f1"
    ]
)

detailed_results[
    "core3_mcc_drop"
] = (
    detailed_results[
        "clean_core3_macro_mcc"
    ]
    - detailed_results[
        "core3_macro_mcc"
    ]
)

detailed_results.to_csv(
    DETAILED_RESULTS_PATH,
    index=False,
)

aspect_results.to_csv(
    ASPECT_RESULTS_PATH,
    index=False,
)


# -----------------------------------------------------------------------------
# Five-seed AM-JEPA aggregation
# -----------------------------------------------------------------------------

full_results = detailed_results[
    detailed_results["variant"]
    == "full_am_jepa"
].copy()

aggregate_metrics = [
    "campaign_f1",
    "campaign_mcc",
    "campaign_auroc",
    "campaign_auprc",
    "four_aspect_macro_f1",
    "core3_macro_f1",
    "core3_macro_mcc",
    "core3_macro_auroc",
    "core3_macro_auprc",
    "campaign_f1_drop",
    "campaign_mcc_drop",
    "core3_f1_drop",
    "core3_mcc_drop",
]

aggregate_rows = []

for scenario in full_results[
    "scenario"
].unique():
    scenario_table = full_results[
        full_results["scenario"]
        == scenario
    ]

    for metric in aggregate_metrics:
        values = scenario_table[
            metric
        ].to_numpy(
            dtype=np.float64
        )

        aggregate_rows.append(
            {
                "scenario": scenario,
                "metric": metric,
                "mean": values.mean(),
                "standard_deviation":
                    values.std(ddof=1),
                "minimum": values.min(),
                "maximum": values.max(),
                "seeds": len(values),
            }
        )

aggregate_results = pd.DataFrame(
    aggregate_rows
)

aggregate_results.to_csv(
    AGGREGATE_RESULTS_PATH,
    index=False,
)


# -----------------------------------------------------------------------------
# Primary summary table
# -----------------------------------------------------------------------------

primary_metrics = [
    "campaign_f1",
    "campaign_auroc",
    "core3_macro_f1",
    "campaign_f1_drop",
    "core3_f1_drop",
]

primary_rows = []

for scenario in full_results[
    "scenario"
].unique():
    scenario_table = full_results[
        full_results["scenario"]
        == scenario
    ]

    row = {
        "scenario": scenario,
        "realized_drop_rate":
            scenario_table[
                "realized_drop_rate"
            ].iloc[0],
    }

    for metric in primary_metrics:
        values = scenario_table[
            metric
        ].to_numpy(
            dtype=np.float64
        )

        row[
            f"{metric}_mean"
        ] = values.mean()

        row[
            f"{metric}_sd"
        ] = values.std(
            ddof=1
        )

    primary_rows.append(
        row
    )

primary_summary = pd.DataFrame(
    primary_rows
)

scenario_order = (
    list(
        FIXED_SCENARIOS.keys()
    )
    + [
        "random_drop_25pct",
        "random_drop_50pct",
        "random_drop_75pct",
    ]
)

primary_summary[
    "scenario"
] = pd.Categorical(
    primary_summary[
        "scenario"
    ],
    categories=
        scenario_order,
    ordered=True,
)

primary_summary = (
    primary_summary
    .sort_values(
        "scenario"
    )
    .reset_index(
        drop=True
    )
)


# -----------------------------------------------------------------------------
# Display final results
# -----------------------------------------------------------------------------

print("\n" + "=" * 80)
print("FULL AM-JEPA ROBUSTNESS: FIVE-SEED SUMMARY")
print("=" * 80)

display(
    primary_summary.round(6)
)

print("\n" + "=" * 80)
print("SEED-42 RANDOM-ENCODER REFERENCE")
print("=" * 80)

display(
    detailed_results[
        detailed_results["variant"]
        == "random_encoder"
    ][
        [
            "scenario",
            "realized_drop_rate",
            "campaign_f1",
            "campaign_auroc",
            "core3_macro_f1",
            "campaign_f1_drop",
            "core3_f1_drop",
        ]
    ].round(6)
)

print("\nSaved detailed results:")
print(DETAILED_RESULTS_PATH)

print("\nSaved aggregate results:")
print(AGGREGATE_RESULTS_PATH)

print("\nSaved aspect-level results:")
print(ASPECT_RESULTS_PATH)

print(
    "\nMissing-modality robustness "
    "evaluation completed successfully."
)

In [ ]:
# =============================================================================
# CELL 25A: DAPT2020-MAST THREAT-STANCE ANNOTATION PACKAGE
# =============================================================================

from pathlib import Path
import hashlib
import json
import math
import re

import numpy as np
import pandas as pd


# -----------------------------------------------------------------------------
# Configuration
# -----------------------------------------------------------------------------

if "DATA_DIR" not in globals():
    DATA_DIR = Path(
        r"."
    )
else:
    DATA_DIR = Path(DATA_DIR)

OUTPUT_DIR = DATA_DIR / "outputs"

ANNOTATION_DIR = (
    OUTPUT_DIR
    / "dapt2020_mast_annotation"
)

ANNOTATION_DIR.mkdir(
    parents=True,
    exist_ok=True,
)

WINDOWS_PATH = (
    OUTPUT_DIR
    / "dapt2020_mast_windows.parquet"
)

MEMBERSHIP_PATH = (
    OUTPUT_DIR
    / "dapt2020_mast_window_membership_enriched.parquet"
)

ASPECT_PAIRS_PATH = (
    OUTPUT_DIR
    / "dapt2020_mast_aspect_pairs.parquet"
)

# Training is sampled for expert supervision. Every validation and test
# window-aspect pair is retained to avoid evaluation-sampling bias.
TRAINING_REVIEW_TARGET = 1200
ANNOTATION_BATCH_SIZE = 240
ANNOTATION_SEED = 20260922

ASPECT_ORDER = [
    "reconnaissance",
    "initial_access_foothold",
    "lateral_movement",
    "data_exfiltration",
]

SPLIT_ORDER = [
    "train",
    "validation",
    "test",
]

VALID_STANCE_LABELS = [
    "SUPPORTING",
    "CONTRADICTING",
    "NEUTRAL",
]

ASPECT_GUIDE = {
    "reconnaissance": {
        "definition": (
            "Evidence indicating discovery, probing, enumeration, or "
            "information gathering that can prepare a later intrusion."
        ),
        "include": (
            "Scanning, service discovery, host discovery, systematic probing, "
            "or repeated exploratory communication."
        ),
        "exclude": (
            "Routine isolated service contact without an exploratory pattern."
        ),
    },
    "initial_access_foothold": {
        "definition": (
            "Evidence indicating an attempt to obtain or maintain an initial "
            "presence in the target environment."
        ),
        "include": (
            "Exploit-like access behavior, suspicious authentication or login "
            "activity, and communication consistent with foothold creation."
        ),
        "exclude": (
            "Ordinary authorized access or unrelated discovery behavior."
        ),
    },
    "lateral_movement": {
        "definition": (
            "Evidence indicating movement, access, or propagation from one "
            "internal system or trust boundary to another."
        ),
        "include": (
            "Abnormal internal-to-internal access, remote-service use, or "
            "multi-endpoint propagation consistent with campaign progression."
        ),
        "exclude": (
            "Expected internal service communication without movement evidence."
        ),
    },
    "data_exfiltration": {
        "definition": (
            "Evidence indicating unauthorized transfer, staging, or removal of "
            "data from the monitored environment."
        ),
        "include": (
            "Outbound transfer behavior, unusual volume or asymmetry, or a "
            "campaign-consistent channel for removing data."
        ),
        "exclude": (
            "Routine outbound communication without evidence of data removal."
        ),
    },
}


# -----------------------------------------------------------------------------
# Required-artifact checks
# -----------------------------------------------------------------------------

required_paths = [
    WINDOWS_PATH,
    MEMBERSHIP_PATH,
    ASPECT_PAIRS_PATH,
]

missing_paths = [
    str(path)
    for path in required_paths
    if not path.exists()
]

if missing_paths:
    raise FileNotFoundError(
        "Missing required DAPT2020-MAST artifacts:\n"
        + "\n".join(missing_paths)
        + "\n\nRun the dataset-construction and verbalization cells first."
    )


# -----------------------------------------------------------------------------
# Load and validate source tables
# -----------------------------------------------------------------------------

windows = pd.read_parquet(WINDOWS_PATH)
membership = pd.read_parquet(MEMBERSHIP_PATH)
aspect_pairs = pd.read_parquet(ASPECT_PAIRS_PATH)

required_window_columns = {
    "window_id",
    "split",
    "day",
    "evidence_count",
}

required_membership_columns = {
    "window_id",
    "position",
    "evidence_id",
    "timestamp_parsed",
    "network_scope",
    "verbalized_event",
}

required_pair_columns = {
    "window_id",
    "split",
    "day",
    "aspect",
    "weak_stance",
    "support_count",
    "other_attack_count",
    "benign_count",
    "insufficient_context",
    "requires_expert_review",
}

for table_name, table, required_columns in [
    ("windows", windows, required_window_columns),
    ("membership", membership, required_membership_columns),
    ("aspect_pairs", aspect_pairs, required_pair_columns),
]:
    missing_columns = sorted(
        required_columns - set(table.columns)
    )

    if missing_columns:
        raise ValueError(
            f"{table_name} is missing columns: {missing_columns}"
        )

assert windows["window_id"].is_unique
assert aspect_pairs[["window_id", "aspect"]].duplicated().sum() == 0
assert set(aspect_pairs["aspect"]) == set(ASPECT_ORDER)
assert set(aspect_pairs["split"]).issubset(set(SPLIT_ORDER))
assert membership["verbalized_event"].notna().all()

membership["timestamp_parsed"] = pd.to_datetime(
    membership["timestamp_parsed"],
    errors="raise",
)


# -----------------------------------------------------------------------------
# Candidate-context categories used only for balanced sampling
# -----------------------------------------------------------------------------

def candidate_context(row):
    if int(row["support_count"]) > 0:
        return "EXACT_ASPECT_SUPPORT"

    if int(row["other_attack_count"]) > 0:
        return "OTHER_ATTACK_CONTEXT"

    return "BENIGN_ONLY_CONTEXT"


aspect_pairs = aspect_pairs.copy()

aspect_pairs["candidate_context"] = aspect_pairs.apply(
    candidate_context,
    axis=1,
)


# -----------------------------------------------------------------------------
# Deterministic identifiers and balanced training sampling
# -----------------------------------------------------------------------------

def stable_digest(value, length=20):
    return hashlib.sha256(
        str(value).encode("utf-8")
    ).hexdigest()[:length].upper()


def deterministic_balanced_sample(
    dataframe,
    target_size,
    seed,
):
    """
    Round-robin sampling across aspect, candidate context, and review flag.
    This keeps rare strata represented without using evaluation outcomes.
    """
    dataframe = dataframe.copy()

    if target_size is None or target_size >= len(dataframe):
        return dataframe.copy()

    dataframe["sampling_stratum"] = (
        dataframe["aspect"].astype(str)
        + "|"
        + dataframe["candidate_context"].astype(str)
        + "|review="
        + dataframe["requires_expert_review"].astype(str)
    )

    dataframe["sampling_rank"] = [
        stable_digest(
            f"{seed}|{window_id}|{aspect}",
            length=64,
        )
        for window_id, aspect in zip(
            dataframe["window_id"],
            dataframe["aspect"],
        )
    ]

    groups = {
        stratum: group.sort_values("sampling_rank").copy()
        for stratum, group in dataframe.groupby(
            "sampling_stratum",
            sort=True,
        )
    }

    selected_indices = []
    offsets = {
        stratum: 0
        for stratum in groups
    }

    while len(selected_indices) < target_size:
        added = False

        for stratum in sorted(groups):
            offset = offsets[stratum]
            group = groups[stratum]

            if offset < len(group):
                selected_indices.append(
                    group.index[offset]
                )
                offsets[stratum] += 1
                added = True

                if len(selected_indices) == target_size:
                    break

        if not added:
            break

    return dataframe.loc[selected_indices].copy()


training_pairs = aspect_pairs[
    aspect_pairs["split"] == "train"
].copy()

validation_pairs = aspect_pairs[
    aspect_pairs["split"] == "validation"
].copy()

test_pairs = aspect_pairs[
    aspect_pairs["split"] == "test"
].copy()

selected_training_pairs = deterministic_balanced_sample(
    dataframe=training_pairs,
    target_size=min(
        TRAINING_REVIEW_TARGET,
        len(training_pairs),
    ),
    seed=ANNOTATION_SEED,
)

# Preserve every evaluation pair. The final model-development workflow must
# keep the test annotations sealed until all design choices are frozen.
selected_pairs = pd.concat(
    [
        selected_training_pairs,
        validation_pairs,
        test_pairs,
    ],
    ignore_index=True,
)

assert selected_pairs[["window_id", "aspect"]].duplicated().sum() == 0

selected_pairs["annotation_id"] = [
    "MAST-A-" + stable_digest(
        f"{ANNOTATION_SEED}|{window_id}|{aspect}",
        length=20,
    )
    for window_id, aspect in zip(
        selected_pairs["window_id"],
        selected_pairs["aspect"],
    )
]

assert selected_pairs["annotation_id"].is_unique


# -----------------------------------------------------------------------------
# Build blinded evidence packets
# -----------------------------------------------------------------------------

membership_groups = {
    window_id: group.sort_values(
        ["position", "timestamp_parsed", "evidence_id"]
    ).copy()
    for window_id, group in membership.groupby(
        "window_id",
        sort=False,
    )
}


def build_evidence_packet(window_id):
    if window_id not in membership_groups:
        raise KeyError(
            f"No evidence membership found for {window_id}."
        )

    group = membership_groups[window_id]
    packet = []

    for _, row in group.iterrows():
        packet.append(
            {
                "position": int(row["position"]),
                "evidence_id": str(row["evidence_id"]),
                "timestamp": row["timestamp_parsed"].isoformat(),
                "network_scope": str(row["network_scope"]),
                "evidence_text": str(row["verbalized_event"]),
            }
        )

    return json.dumps(
        packet,
        ensure_ascii=False,
        separators=(",", ":"),
    )


annotation_rows = []

for _, row in selected_pairs.iterrows():
    guide = ASPECT_GUIDE[row["aspect"]]

    annotation_rows.append(
        {
            "annotation_id": row["annotation_id"],
            "aspect": row["aspect"],
            "aspect_definition": guide["definition"],
            "inclusion_criteria": guide["include"],
            "exclusion_criteria": guide["exclude"],
            "evidence_packet_json": build_evidence_packet(
                row["window_id"]
            ),
            "annotator_a_stance": "",
            "annotator_a_insufficient_context": "",
            "annotator_a_confidence_1_to_5": "",
            "annotator_a_decisive_evidence_ids": "",
            "annotator_a_notes": "",
            "annotator_b_stance": "",
            "annotator_b_insufficient_context": "",
            "annotator_b_confidence_1_to_5": "",
            "annotator_b_decisive_evidence_ids": "",
            "annotator_b_notes": "",
        }
    )

annotation_table = pd.DataFrame(annotation_rows)

# Deterministic blind ordering prevents split and weak-label blocks from being
# visible to annotators.
annotation_table["blind_order"] = [
    stable_digest(
        f"blind|{ANNOTATION_SEED}|{annotation_id}",
        length=64,
    )
    for annotation_id in annotation_table["annotation_id"]
]

annotation_table = (
    annotation_table
    .sort_values("blind_order")
    .drop(columns="blind_order")
    .reset_index(drop=True)
)


# -----------------------------------------------------------------------------
# Leakage and integrity audit
# -----------------------------------------------------------------------------

forbidden_export_columns = {
    "split",
    "day",
    "window_id",
    "weak_stance",
    "support_count",
    "other_attack_count",
    "benign_count",
    "campaign_label",
    "aspect_label",
    "stage_label",
    "activity_label",
}

assert forbidden_export_columns.isdisjoint(
    set(annotation_table.columns)
)

assert len(annotation_table) == len(selected_pairs)
assert annotation_table["annotation_id"].is_unique

packet_text = "\n".join(
    annotation_table["evidence_packet_json"].astype(str)
)

# These patterns detect accidental field-name leakage, not ordinary words
# appearing naturally in controlled descriptions.
forbidden_field_patterns = [
    r"Activity_[A-Za-z0-9_]+",
    r"Stage_[A-Za-z0-9_]+",
    r"campaign_label",
    r"aspect_label",
    r"stage_label",
    r"activity_label",
]

detected_patterns = [
    pattern
    for pattern in forbidden_field_patterns
    if re.search(pattern, packet_text, flags=re.IGNORECASE)
]

if detected_patterns:
    raise ValueError(
        "Potential label-field leakage detected in annotation packets: "
        f"{detected_patterns}"
    )

# Every evidence identifier in every annotation packet must be immutable and
# traceable to the membership table.
known_evidence_ids = set(
    membership["evidence_id"].astype(str)
)

for packet in annotation_table["evidence_packet_json"]:
    for item in json.loads(packet):
        if item["evidence_id"] not in known_evidence_ids:
            raise ValueError(
                "Unknown evidence identifier in annotation packet: "
                f"{item['evidence_id']}"
            )


# -----------------------------------------------------------------------------
# Hidden provenance key and adjudication template
# -----------------------------------------------------------------------------

key_columns = [
    "annotation_id",
    "window_id",
    "split",
    "day",
    "aspect",
    "candidate_context",
    "weak_stance",
    "support_count",
    "other_attack_count",
    "benign_count",
    "insufficient_context",
    "requires_expert_review",
]

annotation_key = selected_pairs[key_columns].copy()

adjudication_template = annotation_table[
    [
        "annotation_id",
        "aspect",
    ]
].copy()

adjudication_template["adjudicated_stance"] = ""
adjudication_template["adjudicated_insufficient_context"] = ""
adjudication_template["adjudicated_decisive_evidence_ids"] = ""
adjudication_template["adjudication_notes"] = ""


# -----------------------------------------------------------------------------
# Batch creation
# -----------------------------------------------------------------------------

number_of_batches = math.ceil(
    len(annotation_table)
    / ANNOTATION_BATCH_SIZE
)

batch_manifest_rows = []

for batch_index in range(number_of_batches):
    start = batch_index * ANNOTATION_BATCH_SIZE
    stop = min(
        start + ANNOTATION_BATCH_SIZE,
        len(annotation_table),
    )

    batch = annotation_table.iloc[start:stop].copy()
    batch_number = batch_index + 1

    shared_columns = [
        "annotation_id",
        "aspect",
        "aspect_definition",
        "inclusion_criteria",
        "exclusion_criteria",
        "evidence_packet_json",
    ]

    annotator_a_columns = shared_columns + [
        column
        for column in batch.columns
        if column.startswith("annotator_a_")
    ]

    annotator_b_columns = shared_columns + [
        column
        for column in batch.columns
        if column.startswith("annotator_b_")
    ]

    batch_a_name = (
        f"annotator_A_batch_{batch_number:03d}.csv"
    )
    batch_b_name = (
        f"annotator_B_batch_{batch_number:03d}.csv"
    )

    batch[annotator_a_columns].to_csv(
        ANNOTATION_DIR / batch_a_name,
        index=False,
        encoding="utf-8-sig",
    )

    batch[annotator_b_columns].to_csv(
        ANNOTATION_DIR / batch_b_name,
        index=False,
        encoding="utf-8-sig",
    )

    batch_manifest_rows.append(
        {
            "batch_number": batch_number,
            "annotator_a_file": batch_a_name,
            "annotator_b_file": batch_b_name,
            "records": len(batch),
            "first_annotation_id": batch.iloc[0]["annotation_id"],
            "last_annotation_id": batch.iloc[-1]["annotation_id"],
        }
    )

batch_manifest = pd.DataFrame(
    batch_manifest_rows
)


# -----------------------------------------------------------------------------
# Annotation guidelines
# -----------------------------------------------------------------------------

guidelines = {
    "task": (
        "Determine the evidential stance of the complete window toward the "
        "specified cybersecurity aspect. Threat stance is not emotion."
    ),
    "labels": {
        "SUPPORTING": (
            "The observed evidence materially supports that the target aspect "
            "is present in this window."
        ),
        "CONTRADICTING": (
            "The observed evidence materially supports a benign or alternative "
            "interpretation that conflicts with the target aspect. Mere absence "
            "of attack evidence is not contradiction."
        ),
        "NEUTRAL": (
            "The evidence is unrelated, non-decisive, or compatible with both "
            "the target aspect and an alternative interpretation."
        ),
    },
    "insufficient_context": (
        "Mark true when the available window does not permit a defensible stance. "
        "This flag is separate from the three semantic labels."
    ),
    "decisive_evidence_ids": (
        "List only evidence identifiers that directly justify the selected stance, "
        "separated by semicolons. Do not create new identifiers."
    ),
    "rules": [
        "Judge only the supplied evidence and aspect definition.",
        "Do not infer an attack solely from unusual volume or timing.",
        "Do not label a benign-only window as CONTRADICTING solely because the "
        "target aspect is absent.",
        "Use NEUTRAL when evidence concerns a different attack aspect.",
        "Set insufficient_context=true when a reliable decision requires evidence "
        "outside the supplied window.",
        "Confidence must be an integer from 1 (very uncertain) to 5 (very certain).",
    ],
    "aspect_guide": ASPECT_GUIDE,
    "allowed_stance_labels": VALID_STANCE_LABELS,
}


# -----------------------------------------------------------------------------
# Save annotation artifacts
# -----------------------------------------------------------------------------

MASTER_ANNOTATION_PATH = (
    ANNOTATION_DIR
    / "dapt2020_mast_blinded_annotation_master.csv"
)

HIDDEN_KEY_PATH = (
    ANNOTATION_DIR
    / "dapt2020_mast_annotation_hidden_key.parquet"
)

ADJUDICATION_PATH = (
    ANNOTATION_DIR
    / "dapt2020_mast_adjudication_template.csv"
)

BATCH_MANIFEST_PATH = (
    ANNOTATION_DIR
    / "dapt2020_mast_annotation_batch_manifest.csv"
)

GUIDELINES_PATH = (
    ANNOTATION_DIR
    / "dapt2020_mast_annotation_guidelines.json"
)

PACKAGE_MANIFEST_PATH = (
    ANNOTATION_DIR
    / "dapt2020_mast_annotation_package_manifest.json"
)

annotation_table.to_csv(
    MASTER_ANNOTATION_PATH,
    index=False,
    encoding="utf-8-sig",
)

annotation_key.to_parquet(
    HIDDEN_KEY_PATH,
    index=False,
)

adjudication_template.to_csv(
    ADJUDICATION_PATH,
    index=False,
    encoding="utf-8-sig",
)

batch_manifest.to_csv(
    BATCH_MANIFEST_PATH,
    index=False,
)

with open(
    GUIDELINES_PATH,
    "w",
    encoding="utf-8",
) as file:
    json.dump(
        guidelines,
        file,
        ensure_ascii=False,
        indent=2,
    )

package_manifest = {
    "seed": ANNOTATION_SEED,
    "training_review_target": TRAINING_REVIEW_TARGET,
    "validation_policy": "all_window_aspect_pairs",
    "test_policy": "all_window_aspect_pairs_sealed_until_final_evaluation",
    "annotation_batch_size": ANNOTATION_BATCH_SIZE,
    "total_annotation_units": int(len(annotation_table)),
    "number_of_batches": int(number_of_batches),
    "stance_labels": VALID_STANCE_LABELS,
    "insufficient_context_is_separate_flag": True,
    "contradiction_is_never_automatically_inferred": True,
    "weak_labels_exposed_to_annotators": False,
    "split_exposed_to_annotators": False,
    "source_artifacts": {
        "windows": str(WINDOWS_PATH),
        "membership": str(MEMBERSHIP_PATH),
        "aspect_pairs": str(ASPECT_PAIRS_PATH),
    },
}

with open(
    PACKAGE_MANIFEST_PATH,
    "w",
    encoding="utf-8",
) as file:
    json.dump(
        package_manifest,
        file,
        ensure_ascii=False,
        indent=2,
    )


# -----------------------------------------------------------------------------
# Report package statistics without exposing test-label distributions
# -----------------------------------------------------------------------------

selection_summary = (
    annotation_key
    .groupby("split", observed=True)
    .size()
    .reindex(SPLIT_ORDER)
    .rename("annotation_units")
    .reset_index()
)

aspect_summary = (
    annotation_key
    .groupby(
        ["split", "aspect"],
        observed=True,
    )
    .size()
    .rename("annotation_units")
    .reset_index()
)

integrity_summary = pd.DataFrame(
    [
        {
            "annotation_units": len(annotation_table),
            "unique_annotation_ids": annotation_table[
                "annotation_id"
            ].nunique(),
            "batches": number_of_batches,
            "forbidden_export_columns_present": len(
                forbidden_export_columns.intersection(
                    annotation_table.columns
                )
            ),
            "forbidden_field_patterns_detected": len(
                detected_patterns
            ),
            "contradiction_auto_labels": 0,
        }
    ]
)

print("=" * 80)
print("DAPT2020-MAST THREAT-STANCE ANNOTATION PACKAGE")
print("=" * 80)

print("\nSelection summary:")
display(selection_summary)

print("\nAspect coverage:")
display(aspect_summary)

print("\nIntegrity summary:")
display(integrity_summary)

print("\nAnnotation batches:")
display(batch_manifest)

print("\nSaved blinded annotation master:")
print(MASTER_ANNOTATION_PATH)

print("\nSaved hidden provenance key:")
print(HIDDEN_KEY_PATH)

print("\nSaved adjudication template:")
print(ADJUDICATION_PATH)

print("\nSaved annotation guidelines:")
print(GUIDELINES_PATH)

print(
    "\nCell 25A completed successfully. "
    "Begin with annotation_batch_001.csv as the pilot batch. "
    "Do not open the hidden key while annotating."
)


In [ ]:
# =============================================================================
# CELL 26: SILVER-LABEL THREAT-STANCE EVALUATION
# =============================================================================

from pathlib import Path
import json
import random
import warnings

import numpy as np
import pandas as pd

from sklearn.dummy import DummyClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    confusion_matrix,
    f1_score,
    log_loss,
    matthews_corrcoef,
    precision_recall_fscore_support,
    roc_auc_score,
)
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler, label_binarize

try:
    from scipy.optimize import minimize_scalar
except ImportError as error:
    raise ImportError(
        "Cell 26 requires SciPy for validation-only temperature scaling."
    ) from error


# -----------------------------------------------------------------------------
# Configuration
# -----------------------------------------------------------------------------

STANCE_SEED = 42
BOOTSTRAP_REPETITIONS = 2000

STANCE_CLASSES = [
    "CONTRADICTING",
    "NEUTRAL",
    "SUPPORTING",
]

ASPECT_ORDER = [
    "reconnaissance",
    "initial_access_foothold",
    "lateral_movement",
    "data_exfiltration",
]

C_GRID = [
    0.01,
    0.1,
    1.0,
    10.0,
    100.0,
]

TARGET_COVERAGES = [
    1.00,
    0.90,
    0.80,
]

random.seed(STANCE_SEED)
np.random.seed(STANCE_SEED)

if "DATA_DIR" not in globals():
    DATA_DIR = Path(
        r"."
    )
else:
    DATA_DIR = Path(DATA_DIR)

OUTPUT_DIR = DATA_DIR / "outputs"
TENSOR_DIR = OUTPUT_DIR / "am_jepa_tensors"
AMJEPA_DIR = OUTPUT_DIR / "am_jepa_experiment"
ANNOTATION_DIR = OUTPUT_DIR / "dapt2020_mast_annotation"

STANCE_OUTPUT_DIR = (
    AMJEPA_DIR
    / "silver_stance_evaluation"
)

STANCE_OUTPUT_DIR.mkdir(
    parents=True,
    exist_ok=True,
)

HIDDEN_KEY_PATH = (
    ANNOTATION_DIR
    / "dapt2020_mast_annotation_hidden_key.parquet"
)

REPRESENTATION_PATH = (
    AMJEPA_DIR
    / "am_jepa_frozen_representations.npy"
)

WINDOW_IDS_PATH = (
    TENSOR_DIR
    / "window_ids.npy"
)


def locate_resolved_labels():
    filename = (
        "DAPT2020_MAST_batch_001_"
        "provisional_resolved_labels.xlsx"
    )

    candidates = [
        ANNOTATION_DIR / filename,
        OUTPUT_DIR / "provisional_resolved_labels" / filename,
        DATA_DIR / filename,
    ]

    for candidate in candidates:
        if candidate.exists():
            return candidate

    recursive_matches = list(
        OUTPUT_DIR.rglob(filename)
    )

    if len(recursive_matches) == 1:
        return recursive_matches[0]

    if len(recursive_matches) > 1:
        raise RuntimeError(
            "Multiple resolved-label workbooks were found. "
            "Keep one authoritative copy or set RESOLVED_LABEL_PATH "
            "manually.\n"
            + "\n".join(str(path) for path in recursive_matches)
        )

    raise FileNotFoundError(
        "Resolved-label workbook was not found. Expected:\n"
        + "\n".join(str(path) for path in candidates)
    )


RESOLVED_LABEL_PATH = locate_resolved_labels()

required_paths = [
    HIDDEN_KEY_PATH,
    REPRESENTATION_PATH,
    WINDOW_IDS_PATH,
    TENSOR_DIR / "X_network.npy",
    TENSOR_DIR / "X_temporal_final.npy",
    TENSOR_DIR / "attention_mask.npy",
    TENSOR_DIR / "text_embedding_lookup.npy",
    TENSOR_DIR / "window_text_ids.npy",
]

missing_paths = [
    str(path)
    for path in required_paths
    if not path.exists()
]

if missing_paths:
    raise FileNotFoundError(
        "Missing required stance-evaluation artifacts:\n"
        + "\n".join(missing_paths)
        + "\n\nRun Cells 19–21 and 25A first, then place the "
          "resolved pilot workbook in the annotation output directory."
    )


# -----------------------------------------------------------------------------
# Load resolved labels and hidden provenance
# -----------------------------------------------------------------------------

resolved = pd.read_excel(
    RESOLVED_LABEL_PATH,
    sheet_name="Provisional labels",
)

resolved = resolved.rename(
    columns={
        "Annotation ID": "annotation_id",
        "Aspect": "aspect",
        "Resolved stance": "stance",
        "Resolved insufficient context": "insufficient_context",
        "Resolved confidence (1–5)": "confidence",
        "Resolved decisive evidence IDs": "decisive_evidence_ids",
        "Resolution source": "resolution_source",
        "Dataset status": "dataset_status",
    }
)

required_label_columns = {
    "annotation_id",
    "aspect",
    "stance",
    "insufficient_context",
    "confidence",
    "decisive_evidence_ids",
    "resolution_source",
    "dataset_status",
}

missing_label_columns = sorted(
    required_label_columns
    - set(resolved.columns)
)

if missing_label_columns:
    raise ValueError(
        "Resolved-label workbook is missing columns: "
        f"{missing_label_columns}"
    )

resolved["annotation_id"] = (
    resolved["annotation_id"]
    .astype(str)
    .str.strip()
)

resolved["aspect"] = (
    resolved["aspect"]
    .astype(str)
    .str.strip()
    .str.lower()
)

resolved["stance"] = (
    resolved["stance"]
    .astype(str)
    .str.strip()
    .str.upper()
)

resolved["insufficient_context"] = (
    resolved["insufficient_context"]
    .astype(str)
    .str.strip()
    .str.upper()
    .map({
        "TRUE": True,
        "FALSE": False,
    })
)

resolved["confidence"] = pd.to_numeric(
    resolved["confidence"],
    errors="raise",
).astype(np.int64)

if resolved["annotation_id"].duplicated().any():
    raise ValueError(
        "Resolved labels contain duplicate annotation IDs."
    )

if not set(resolved["stance"]).issubset(
    set(STANCE_CLASSES)
):
    raise ValueError(
        "Unexpected stance labels: "
        f"{sorted(set(resolved['stance']) - set(STANCE_CLASSES))}"
    )

if not set(resolved["aspect"]).issubset(
    set(ASPECT_ORDER)
):
    raise ValueError(
        "Unexpected aspects: "
        f"{sorted(set(resolved['aspect']) - set(ASPECT_ORDER))}"
    )

if resolved["insufficient_context"].isna().any():
    raise ValueError(
        "Invalid insufficient-context value detected."
    )

if not resolved["confidence"].between(1, 5).all():
    raise ValueError(
        "Confidence must be an integer from 1 through 5."
    )

hidden_key = pd.read_parquet(
    HIDDEN_KEY_PATH
)

required_key_columns = {
    "annotation_id",
    "window_id",
    "split",
    "aspect",
}

missing_key_columns = sorted(
    required_key_columns
    - set(hidden_key.columns)
)

if missing_key_columns:
    raise ValueError(
        "Hidden provenance key is missing columns: "
        f"{missing_key_columns}"
    )

hidden_key["annotation_id"] = (
    hidden_key["annotation_id"]
    .astype(str)
    .str.strip()
)

hidden_key["window_id"] = (
    hidden_key["window_id"]
    .astype(str)
)

hidden_key["aspect"] = (
    hidden_key["aspect"]
    .astype(str)
    .str.strip()
    .str.lower()
)

hidden_key["split"] = (
    hidden_key["split"]
    .astype(str)
    .str.strip()
    .str.lower()
)

stance_table = resolved.merge(
    hidden_key[
        [
            "annotation_id",
            "window_id",
            "split",
            "aspect",
        ]
    ],
    on=[
        "annotation_id",
        "aspect",
    ],
    how="left",
    validate="one_to_one",
)

if stance_table[
    [
        "window_id",
        "split",
    ]
].isna().any().any():
    missing_ids = stance_table.loc[
        stance_table["window_id"].isna(),
        "annotation_id",
    ].tolist()

    raise ValueError(
        "Some resolved labels do not match the hidden key: "
        f"{missing_ids[:10]}"
    )

if len(stance_table) != 240:
    warnings.warn(
        "The pilot contains "
        f"{len(stance_table)} records rather than the expected 240."
    )


# -----------------------------------------------------------------------------
# Recover tensor order and build representations
# -----------------------------------------------------------------------------

try:
    window_ids = np.load(
        WINDOW_IDS_PATH,
        allow_pickle=False,
    )
except ValueError:
    window_ids = np.load(
        WINDOW_IDS_PATH,
        allow_pickle=True,
    )

window_ids = np.asarray(
    window_ids
).astype(str)

window_to_tensor_index = {
    window_id: index
    for index, window_id in enumerate(window_ids)
}

stance_table["tensor_index"] = (
    stance_table["window_id"]
    .map(window_to_tensor_index)
)

if stance_table["tensor_index"].isna().any():
    raise ValueError(
        "Some resolved windows were not found in window_ids.npy."
    )

stance_table["tensor_index"] = (
    stance_table["tensor_index"]
    .astype(np.int64)
)

frozen_representations = np.load(
    REPRESENTATION_PATH,
    allow_pickle=False,
).astype(np.float32)

X_network_stance = np.load(
    TENSOR_DIR / "X_network.npy",
    allow_pickle=False,
).astype(np.float32)

X_temporal_stance = np.load(
    TENSOR_DIR / "X_temporal_final.npy",
    allow_pickle=False,
).astype(np.float32)

attention_mask_stance = np.load(
    TENSOR_DIR / "attention_mask.npy",
    allow_pickle=False,
).astype(bool)

text_embedding_lookup_stance = np.load(
    TENSOR_DIR / "text_embedding_lookup.npy",
    allow_pickle=False,
).astype(np.float32)

window_text_ids_stance = np.load(
    TENSOR_DIR / "window_text_ids.npy",
    allow_pickle=False,
).astype(np.int64)

number_of_windows = len(window_ids)

if frozen_representations.shape[0] != number_of_windows:
    raise ValueError(
        "Frozen representation count does not match window_ids.npy."
    )


def masked_mean(array, mask):
    expanded_mask = mask.astype(
        np.float32
    )[..., None]

    denominator = expanded_mask.sum(
        axis=1
    ).clip(min=1.0)

    return (
        array
        * expanded_mask
    ).sum(axis=1) / denominator


network_pooled = masked_mean(
    X_network_stance,
    attention_mask_stance,
)

temporal_pooled = masked_mean(
    X_temporal_stance,
    attention_mask_stance,
)

text_sequence_stance = (
    text_embedding_lookup_stance[
        window_text_ids_stance
    ]
)

text_pooled = masked_mean(
    text_sequence_stance,
    attention_mask_stance,
)

aspect_to_index = {
    aspect: index
    for index, aspect in enumerate(
        ASPECT_ORDER
    )
}

aspect_one_hot = np.zeros(
    (
        len(stance_table),
        len(ASPECT_ORDER),
    ),
    dtype=np.float32,
)

aspect_one_hot[
    np.arange(len(stance_table)),
    stance_table["aspect"]
    .map(aspect_to_index)
    .to_numpy(dtype=np.int64),
] = 1.0

tensor_indices = stance_table[
    "tensor_index"
].to_numpy(dtype=np.int64)

representation_sets = {
    "aspect_prior": aspect_one_hot,
    "pooled_multimodal": np.concatenate(
        [
            network_pooled[tensor_indices],
            temporal_pooled[tensor_indices],
            text_pooled[tensor_indices],
            aspect_one_hot,
        ],
        axis=1,
    ),
    "frozen_am_jepa": np.concatenate(
        [
            frozen_representations[tensor_indices],
            aspect_one_hot,
        ],
        axis=1,
    ),
}

stance_to_index = {
    stance: index
    for index, stance in enumerate(
        STANCE_CLASSES
    )
}

y_stance = stance_table[
    "stance"
].map(stance_to_index).to_numpy(
    dtype=np.int64
)

sample_weights = (
    stance_table["confidence"]
    .to_numpy(dtype=np.float64)
    / 5.0
)

sample_weights *= np.where(
    stance_table[
        "insufficient_context"
    ].to_numpy(dtype=bool),
    0.5,
    1.0,
)


# -----------------------------------------------------------------------------
# Fixed leakage-aware partitions
# -----------------------------------------------------------------------------

split_masks = {
    split_name: (
        stance_table["split"]
        .to_numpy()
        == split_name
    )
    for split_name in [
        "train",
        "validation",
        "test",
    ]
}

for split_name, split_mask in split_masks.items():
    if split_mask.sum() == 0:
        raise ValueError(
            f"The pilot has no {split_name} labels."
        )

train_window_ids = set(
    stance_table.loc[
        split_masks["train"],
        "window_id",
    ]
)

validation_window_ids = set(
    stance_table.loc[
        split_masks["validation"],
        "window_id",
    ]
)

test_window_ids = set(
    stance_table.loc[
        split_masks["test"],
        "window_id",
    ]
)

if (
    train_window_ids & validation_window_ids
    or train_window_ids & test_window_ids
    or validation_window_ids & test_window_ids
):
    raise AssertionError(
        "A window appears in more than one partition."
    )

distribution_rows = []

for split_name, split_mask in split_masks.items():
    for stance_name in STANCE_CLASSES:
        distribution_rows.append(
            {
                "split": split_name,
                "stance": stance_name,
                "count": int(
                    (
                        stance_table.loc[
                            split_mask,
                            "stance",
                        ]
                        == stance_name
                    ).sum()
                ),
                "unique_windows": int(
                    stance_table.loc[
                        split_mask,
                        "window_id",
                    ].nunique()
                ),
                "insufficient_context": int(
                    stance_table.loc[
                        split_mask,
                        "insufficient_context",
                    ].sum()
                ),
            }
        )

split_distribution = pd.DataFrame(
    distribution_rows
)

for split_name in [
    "train",
    "validation",
    "test",
]:
    present_classes = set(
        stance_table.loc[
            split_masks[split_name],
            "stance",
        ]
    )

    if present_classes != set(STANCE_CLASSES):
        raise ValueError(
            f"The {split_name} pilot partition does not contain "
            "all three stance classes. This 240-unit pilot is too "
            "small for a fixed-split three-class evaluation. "
            "Complete additional blinded batches before proceeding. "
            f"Present classes: {sorted(present_classes)}"
        )


# -----------------------------------------------------------------------------
# Metrics and validation-only calibration
# -----------------------------------------------------------------------------

def build_probe(C_value):
    return Pipeline(
        steps=[
            (
                "standardizer",
                StandardScaler(),
            ),
            (
                "classifier",
                LogisticRegression(
                    C=C_value,
                    class_weight="balanced",
                    solver="lbfgs",
                    max_iter=10000,
                    random_state=STANCE_SEED,
                ),
            ),
        ]
    )


def align_probabilities(probabilities, observed_classes):
    aligned = np.full(
        (
            probabilities.shape[0],
            len(STANCE_CLASSES),
        ),
        1e-12,
        dtype=np.float64,
    )

    for source_column, class_index in enumerate(
        observed_classes
    ):
        aligned[
            :,
            int(class_index),
        ] = probabilities[
            :,
            source_column,
        ]

    aligned /= aligned.sum(
        axis=1,
        keepdims=True,
    )

    return aligned


def expected_calibration_error(
    labels,
    probabilities,
    number_of_bins=10,
):
    confidence = probabilities.max(
        axis=1
    )

    predictions = probabilities.argmax(
        axis=1
    )

    correct = (
        predictions == labels
    ).astype(np.float64)

    edges = np.linspace(
        0.0,
        1.0,
        number_of_bins + 1,
    )

    ece = 0.0

    for bin_index in range(
        number_of_bins
    ):
        lower = edges[bin_index]
        upper = edges[bin_index + 1]

        if bin_index == 0:
            selected = (
                confidence >= lower
            ) & (
                confidence <= upper
            )
        else:
            selected = (
                confidence > lower
            ) & (
                confidence <= upper
            )

        if selected.any():
            ece += (
                selected.mean()
                * abs(
                    correct[selected].mean()
                    - confidence[selected].mean()
                )
            )

    return float(ece)


def multiclass_brier(labels, probabilities):
    one_hot = label_binarize(
        labels,
        classes=np.arange(
            len(STANCE_CLASSES)
        ),
    )

    return float(
        np.mean(
            np.sum(
                (
                    probabilities
                    - one_hot
                ) ** 2,
                axis=1,
            )
        )
    )


def fit_temperature(labels, probabilities):
    logits = np.log(
        np.clip(
            probabilities,
            1e-12,
            1.0,
        )
    )

    def objective(log_temperature):
        temperature = np.exp(
            log_temperature
        )

        scaled_logits = logits / temperature

        scaled_logits -= scaled_logits.max(
            axis=1,
            keepdims=True,
        )

        scaled_probabilities = np.exp(
            scaled_logits
        )

        scaled_probabilities /= (
            scaled_probabilities.sum(
                axis=1,
                keepdims=True,
            )
        )

        return log_loss(
            labels,
            scaled_probabilities,
            labels=np.arange(
                len(STANCE_CLASSES)
            ),
        )

    result = minimize_scalar(
        objective,
        bounds=(
            np.log(0.05),
            np.log(20.0),
        ),
        method="bounded",
    )

    return float(
        np.exp(result.x)
    )


def apply_temperature(probabilities, temperature):
    logits = np.log(
        np.clip(
            probabilities,
            1e-12,
            1.0,
        )
    )

    scaled_logits = logits / temperature

    scaled_logits -= scaled_logits.max(
        axis=1,
        keepdims=True,
    )

    scaled_probabilities = np.exp(
        scaled_logits
    )

    scaled_probabilities /= (
        scaled_probabilities.sum(
            axis=1,
            keepdims=True,
        )
    )

    return scaled_probabilities


def calculate_stance_metrics(
    labels,
    probabilities,
):
    predictions = probabilities.argmax(
        axis=1
    )

    result = {
        "samples": int(len(labels)),
        "accuracy": accuracy_score(
            labels,
            predictions,
        ),
        "balanced_accuracy": balanced_accuracy_score(
            labels,
            predictions,
        ),
        "macro_f1": f1_score(
            labels,
            predictions,
            average="macro",
            zero_division=0,
        ),
        "weighted_f1": f1_score(
            labels,
            predictions,
            average="weighted",
            zero_division=0,
        ),
        "mcc": matthews_corrcoef(
            labels,
            predictions,
        ),
        "nll": log_loss(
            labels,
            probabilities,
            labels=np.arange(
                len(STANCE_CLASSES)
            ),
        ),
        "brier": multiclass_brier(
            labels,
            probabilities,
        ),
        "ece": expected_calibration_error(
            labels,
            probabilities,
        ),
    }

    try:
        result["macro_ovr_auroc"] = (
            roc_auc_score(
                labels,
                probabilities,
                multi_class="ovr",
                average="macro",
                labels=np.arange(
                    len(STANCE_CLASSES)
                ),
            )
        )
    except ValueError:
        result["macro_ovr_auroc"] = np.nan

    return result


# -----------------------------------------------------------------------------
# Train on training labels, tune/calibrate on validation, test once
# -----------------------------------------------------------------------------

main_rows = []
per_class_rows = []
selective_rows = []
prediction_tables = []

train_mask = split_masks["train"]
validation_mask = split_masks["validation"]
test_mask = split_masks["test"]

for representation_name, features in (
    representation_sets.items()
):
    validation_search_rows = []

    if representation_name == "aspect_prior":
        candidate_C_values = C_GRID
    else:
        candidate_C_values = C_GRID

    for C_value in candidate_C_values:
        probe = build_probe(C_value)

        probe.fit(
            features[train_mask],
            y_stance[train_mask],
            classifier__sample_weight=
                sample_weights[train_mask],
        )

        validation_probabilities = (
            align_probabilities(
                probe.predict_proba(
                    features[
                        validation_mask
                    ]
                ),
                probe.named_steps[
                    "classifier"
                ].classes_,
            )
        )

        validation_metrics = (
            calculate_stance_metrics(
                y_stance[validation_mask],
                validation_probabilities,
            )
        )

        validation_search_rows.append(
            {
                "C": C_value,
                **validation_metrics,
            }
        )

    validation_search = pd.DataFrame(
        validation_search_rows
    )

    best_validation_row = (
        validation_search
        .sort_values(
            by=[
                "macro_f1",
                "mcc",
                "balanced_accuracy",
                "nll",
                "C",
            ],
            ascending=[
                False,
                False,
                False,
                True,
                True,
            ],
        )
        .iloc[0]
    )

    best_C = float(
        best_validation_row["C"]
    )

    final_probe = build_probe(
        best_C
    )

    final_probe.fit(
        features[train_mask],
        y_stance[train_mask],
        classifier__sample_weight=
            sample_weights[train_mask],
    )

    validation_probabilities = (
        align_probabilities(
            final_probe.predict_proba(
                features[validation_mask]
            ),
            final_probe.named_steps[
                "classifier"
            ].classes_,
        )
    )

    temperature = fit_temperature(
        y_stance[validation_mask],
        validation_probabilities,
    )

    test_probabilities_uncalibrated = (
        align_probabilities(
            final_probe.predict_proba(
                features[test_mask]
            ),
            final_probe.named_steps[
                "classifier"
            ].classes_,
        )
    )

    test_probabilities = apply_temperature(
        test_probabilities_uncalibrated,
        temperature,
    )

    evaluation_masks = {
        "all_test": np.ones(
            test_mask.sum(),
            dtype=bool,
        ),
        "sufficient_context_test": (
            ~stance_table.loc[
                test_mask,
                "insufficient_context",
            ].to_numpy(dtype=bool)
        ),
    }

    test_labels = y_stance[
        test_mask
    ]

    for evaluation_set, local_mask in (
        evaluation_masks.items()
    ):
        if local_mask.sum() == 0:
            continue

        metrics = calculate_stance_metrics(
            test_labels[local_mask],
            test_probabilities[local_mask],
        )

        main_rows.append(
            {
                "representation": representation_name,
                "evaluation_set": evaluation_set,
                "best_C": best_C,
                "temperature": temperature,
                "validation_macro_f1": float(
                    best_validation_row[
                        "macro_f1"
                    ]
                ),
                **metrics,
            }
        )

        local_predictions = (
            test_probabilities[
                local_mask
            ].argmax(axis=1)
        )

        precision, recall, f1, support = (
            precision_recall_fscore_support(
                test_labels[local_mask],
                local_predictions,
                labels=np.arange(
                    len(STANCE_CLASSES)
                ),
                zero_division=0,
            )
        )

        for class_index, class_name in enumerate(
            STANCE_CLASSES
        ):
            per_class_rows.append(
                {
                    "representation": representation_name,
                    "evaluation_set": evaluation_set,
                    "stance": class_name,
                    "precision": precision[class_index],
                    "recall": recall[class_index],
                    "f1": f1[class_index],
                    "support": int(
                        support[class_index]
                    ),
                }
            )

    validation_confidence = (
        apply_temperature(
            validation_probabilities,
            temperature,
        ).max(axis=1)
    )

    test_confidence = test_probabilities.max(
        axis=1
    )

    for target_coverage in TARGET_COVERAGES:
        if target_coverage >= 1.0:
            threshold = 0.0
        else:
            threshold = float(
                np.quantile(
                    validation_confidence,
                    1.0 - target_coverage,
                    method="lower",
                )
            )

        retained = (
            test_confidence
            >= threshold
        )

        if retained.sum() == 0:
            continue

        retained_metrics = (
            calculate_stance_metrics(
                test_labels[retained],
                test_probabilities[retained],
            )
        )

        selective_rows.append(
            {
                "representation": representation_name,
                "target_validation_coverage": target_coverage,
                "confidence_threshold": threshold,
                "test_coverage": retained.mean(),
                "abstention_rate": 1.0 - retained.mean(),
                **retained_metrics,
            }
        )

    prediction_table = stance_table.loc[
        test_mask,
        [
            "annotation_id",
            "window_id",
            "aspect",
            "stance",
            "insufficient_context",
            "confidence",
        ],
    ].copy()

    prediction_table[
        "representation"
    ] = representation_name

    prediction_table[
        "predicted_stance"
    ] = [
        STANCE_CLASSES[index]
        for index in test_probabilities.argmax(
            axis=1
        )
    ]

    prediction_table[
        "predictive_confidence"
    ] = test_confidence

    prediction_table[
        "predictive_entropy"
    ] = -np.sum(
        test_probabilities
        * np.log(
            np.clip(
                test_probabilities,
                1e-12,
                1.0,
            )
        ),
        axis=1,
    )

    for class_index, class_name in enumerate(
        STANCE_CLASSES
    ):
        prediction_table[
            "probability_"
            + class_name.lower()
        ] = test_probabilities[
            :,
            class_index,
        ]

    prediction_tables.append(
        prediction_table
    )


# -----------------------------------------------------------------------------
# Majority-class baseline
# -----------------------------------------------------------------------------

dummy = DummyClassifier(
    strategy="prior",
    random_state=STANCE_SEED,
)

dummy.fit(
    aspect_one_hot[train_mask],
    y_stance[train_mask],
    sample_weight=sample_weights[
        train_mask
    ],
)

dummy_test_probabilities = (
    align_probabilities(
        dummy.predict_proba(
            aspect_one_hot[test_mask]
        ),
        dummy.classes_,
    )
)

dummy_metrics = calculate_stance_metrics(
    y_stance[test_mask],
    dummy_test_probabilities,
)

main_rows.append(
    {
        "representation": "majority_prior",
        "evaluation_set": "all_test",
        "best_C": np.nan,
        "temperature": 1.0,
        "validation_macro_f1": np.nan,
        **dummy_metrics,
    }
)


# -----------------------------------------------------------------------------
# Group bootstrap confidence intervals for primary AM-JEPA test result
# -----------------------------------------------------------------------------

all_predictions = pd.concat(
    prediction_tables,
    ignore_index=True,
)

primary_predictions = all_predictions[
    all_predictions["representation"]
    == "frozen_am_jepa"
].copy()

bootstrap_source = primary_predictions[
    ~primary_predictions[
        "insufficient_context"
    ].astype(bool)
].copy()

bootstrap_groups = (
    bootstrap_source["window_id"]
    .drop_duplicates()
    .to_numpy()
)

rng = np.random.default_rng(
    STANCE_SEED
)

bootstrap_rows = []

for repetition in range(
    BOOTSTRAP_REPETITIONS
):
    sampled_groups = rng.choice(
        bootstrap_groups,
        size=len(bootstrap_groups),
        replace=True,
    )

    sampled_parts = [
        bootstrap_source[
            bootstrap_source["window_id"]
            == group_id
        ]
        for group_id in sampled_groups
    ]

    sampled = pd.concat(
        sampled_parts,
        ignore_index=True,
    )

    sampled_true = sampled[
        "stance"
    ].map(stance_to_index).to_numpy(
        dtype=np.int64
    )

    sampled_predicted = sampled[
        "predicted_stance"
    ].map(stance_to_index).to_numpy(
        dtype=np.int64
    )

    bootstrap_rows.append(
        {
            "repetition": repetition,
            "macro_f1": f1_score(
                sampled_true,
                sampled_predicted,
                average="macro",
                zero_division=0,
            ),
            "mcc": matthews_corrcoef(
                sampled_true,
                sampled_predicted,
            ),
        }
    )

bootstrap_results = pd.DataFrame(
    bootstrap_rows
)

bootstrap_summary = pd.DataFrame(
    [
        {
            "metric": metric,
            "mean": bootstrap_results[
                metric
            ].mean(),
            "ci_2_5": bootstrap_results[
                metric
            ].quantile(0.025),
            "ci_97_5": bootstrap_results[
                metric
            ].quantile(0.975),
            "bootstrap_repetitions": (
                BOOTSTRAP_REPETITIONS
            ),
            "resampling_unit": "window_id",
            "evaluation_set": (
                "sufficient_context_test"
            ),
        }
        for metric in [
            "macro_f1",
            "mcc",
        ]
    ]
)


# -----------------------------------------------------------------------------
# Decisive-evidence sparsity audit
# -----------------------------------------------------------------------------

def count_decisive_ids(value):
    text = str(value).strip()

    if (
        not text
        or text.upper() in {
            "NAN",
            "NONE",
            "NONE_RECORDED",
        }
    ):
        return 0

    return len(
        [
            item
            for item in text.split(";")
            if item.strip()
        ]
    )


stance_table[
    "decisive_evidence_count"
] = stance_table[
    "decisive_evidence_ids"
].map(count_decisive_ids)

sparsity_rows = []

for aspect_name in [
    "ALL",
    *ASPECT_ORDER,
]:
    if aspect_name == "ALL":
        subset = stance_table
    else:
        subset = stance_table[
            stance_table["aspect"]
            == aspect_name
        ]

    sparsity_rows.append(
        {
            "aspect": aspect_name,
            "records": len(subset),
            "records_with_decisive_evidence": int(
                (
                    subset[
                        "decisive_evidence_count"
                    ]
                    > 0
                ).sum()
            ),
            "decisive_evidence_coverage": float(
                (
                    subset[
                        "decisive_evidence_count"
                    ]
                    > 0
                ).mean()
            ),
            "mean_decisive_evidence_count": float(
                subset[
                    "decisive_evidence_count"
                ].mean()
            ),
            "median_decisive_evidence_count": float(
                subset[
                    "decisive_evidence_count"
                ].median()
            ),
            "mean_retained_fraction_of_32": float(
                (
                    subset[
                        "decisive_evidence_count"
                    ]
                    / 32.0
                ).mean()
            ),
        }
    )

sparsity_audit = pd.DataFrame(
    sparsity_rows
)


# -----------------------------------------------------------------------------
# Save results and manifest
# -----------------------------------------------------------------------------

main_results = pd.DataFrame(
    main_rows
)

per_class_results = pd.DataFrame(
    per_class_rows
)

selective_results = pd.DataFrame(
    selective_rows
)

main_results.to_csv(
    STANCE_OUTPUT_DIR
    / "stance_main_results.csv",
    index=False,
)

per_class_results.to_csv(
    STANCE_OUTPUT_DIR
    / "stance_per_class_results.csv",
    index=False,
)

selective_results.to_csv(
    STANCE_OUTPUT_DIR
    / "stance_selective_results.csv",
    index=False,
)

all_predictions.to_csv(
    STANCE_OUTPUT_DIR
    / "stance_test_predictions.csv",
    index=False,
)

split_distribution.to_csv(
    STANCE_OUTPUT_DIR
    / "stance_split_distribution.csv",
    index=False,
)

bootstrap_results.to_csv(
    STANCE_OUTPUT_DIR
    / "stance_bootstrap_samples.csv",
    index=False,
)

bootstrap_summary.to_csv(
    STANCE_OUTPUT_DIR
    / "stance_bootstrap_summary.csv",
    index=False,
)

sparsity_audit.to_csv(
    STANCE_OUTPUT_DIR
    / "decisive_evidence_sparsity_audit.csv",
    index=False,
)

manifest = {
    "status": "exploratory_silver_label_evaluation",
    "records": int(len(stance_table)),
    "unique_annotation_ids": int(
        stance_table["annotation_id"].nunique()
    ),
    "unique_windows": int(
        stance_table["window_id"].nunique()
    ),
    "classes": STANCE_CLASSES,
    "aspects": ASPECT_ORDER,
    "split_policy": (
        "Original leakage-aware train/validation/test partition "
        "recovered from the hidden provenance key."
    ),
    "validation_policy": (
        "Regularization and temperature selected using validation only."
    ),
    "test_policy": (
        "The test labels are evaluated once after model selection."
    ),
    "context_policy": (
        "Training weights are halved for insufficient-context records; "
        "test metrics are reported both with and without them."
    ),
    "label_status": (
        "AI-assisted, single-reviewer-resolved silver labels; "
        "not an independent human gold standard."
    ),
    "resolved_label_path": str(
        RESOLVED_LABEL_PATH
    ),
    "frozen_representation_path": str(
        REPRESENTATION_PATH
    ),
    "seed": STANCE_SEED,
    "bootstrap_repetitions": (
        BOOTSTRAP_REPETITIONS
    ),
}

with open(
    STANCE_OUTPUT_DIR
    / "stance_evaluation_manifest.json",
    "w",
    encoding="utf-8",
) as file:
    json.dump(
        manifest,
        file,
        indent=2,
    )


# -----------------------------------------------------------------------------
# Display publication-facing summaries
# -----------------------------------------------------------------------------

print("=" * 80)
print("DAPT2020-MAST SILVER-LABEL THREAT-STANCE EVALUATION")
print("=" * 80)

print("\nResolved labels:")
print(RESOLVED_LABEL_PATH)

print("\nLabel status:")
print(manifest["label_status"])

print("\nLeakage-aware split distribution:")
display(
    split_distribution.pivot_table(
        index="split",
        columns="stance",
        values="count",
        aggfunc="sum",
        fill_value=0,
    )
)

print("\nPrimary and baseline results:")
display(
    main_results[
        [
            "representation",
            "evaluation_set",
            "samples",
            "accuracy",
            "balanced_accuracy",
            "macro_f1",
            "weighted_f1",
            "mcc",
            "macro_ovr_auroc",
            "nll",
            "brier",
            "ece",
        ]
    ]
    .sort_values(
        [
            "evaluation_set",
            "macro_f1",
        ],
        ascending=[
            True,
            False,
        ],
    )
    .round(6)
)

print("\nPer-class test performance:")
display(
    per_class_results[
        per_class_results[
            "evaluation_set"
        ] == "all_test"
    ]
    .round(6)
)

print("\nValidation-selected abstention results:")
display(
    selective_results[
        [
            "representation",
            "target_validation_coverage",
            "confidence_threshold",
            "test_coverage",
            "macro_f1",
            "mcc",
            "ece",
        ]
    ].round(6)
)

print("\nAM-JEPA group-bootstrap confidence intervals:")
display(
    bootstrap_summary.round(6)
)

print("\nDecisive-evidence sparsity audit:")
display(
    sparsity_audit.round(6)
)

print("\nSaved stance-evaluation results to:")
print(STANCE_OUTPUT_DIR)

print(
    "\nCell 26 completed successfully. "
    "Interpret these results as exploratory silver-label evidence only; "
    "do not describe them as human gold-standard performance."
)


In [ ]:
# =============================================================================
# CELL 27: ASPECT-CONDITIONED SPARSE EVIDENCE SELECTOR
# =============================================================================

from pathlib import Path
import copy
import json
import random

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset

from sklearn.preprocessing import StandardScaler


# -----------------------------------------------------------------------------
# Required Cell 26 objects
# -----------------------------------------------------------------------------

required_objects = [
    "DATA_DIR",
    "OUTPUT_DIR",
    "AMJEPA_DIR",
    "TENSOR_DIR",
    "STANCE_OUTPUT_DIR",
    "stance_table",
    "ASPECT_ORDER",
    "X_network_stance",
    "X_temporal_stance",
    "attention_mask_stance",
    "text_embedding_lookup_stance",
    "window_text_ids_stance",
    "window_to_tensor_index",
]

missing_objects = [
    name
    for name in required_objects
    if name not in globals()
]

if missing_objects:
    raise NameError(
        "Missing objects from Cell 26:\n"
        + "\n".join(missing_objects)
        + "\n\nRun Cell 26 before Cell 27."
    )

DATA_DIR = Path(DATA_DIR)
OUTPUT_DIR = Path(OUTPUT_DIR)

MEMBERSHIP_PATH = (
    OUTPUT_DIR
    / "dapt2020_mast_window_membership_enriched.parquet"
)

SELECTOR_OUTPUT_DIR = (
    Path(AMJEPA_DIR)
    / "silver_sparse_selector"
)

SELECTOR_OUTPUT_DIR.mkdir(
    parents=True,
    exist_ok=True,
)

if not MEMBERSHIP_PATH.exists():
    raise FileNotFoundError(
        f"Missing evidence membership table: {MEMBERSHIP_PATH}"
    )


# -----------------------------------------------------------------------------
# Configuration
# -----------------------------------------------------------------------------

SELECTOR_SEEDS = [
    42,
    52,
    62,
    72,
    82,
]

EVIDENCE_BUDGETS = [
    1,
    2,
    4,
    8,
]

HIDDEN_DIMENSION = 64
MAXIMUM_EPOCHS = 300
EARLY_STOPPING_PATIENCE = 30
BATCH_SIZE = 256
LEARNING_RATE = 1e-3
WEIGHT_DECAY = 1e-4
RANDOM_BASELINE_REPETITIONS = 200

DEVICE_SELECTOR = torch.device(
    "cuda"
    if torch.cuda.is_available()
    else "cpu"
)


def set_selector_seed(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)

    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)


# -----------------------------------------------------------------------------
# Select only records with explicit decisive-evidence supervision
# -----------------------------------------------------------------------------

def parse_evidence_ids(value):
    text = str(value).strip()

    if (
        not text
        or text.upper() in {
            "NAN",
            "NONE",
            "NONE_RECORDED",
        }
    ):
        return []

    return [
        item.strip()
        for item in text.split(";")
        if item.strip()
    ]


selector_records = stance_table.copy()

selector_records[
    "decisive_id_list"
] = selector_records[
    "decisive_evidence_ids"
].map(parse_evidence_ids)

selector_records[
    "decisive_evidence_count"
] = selector_records[
    "decisive_id_list"
].map(len)

selector_records = selector_records[
    selector_records[
        "decisive_evidence_count"
    ] > 0
].copy()

selector_records = selector_records.reset_index(
    drop=True
)

if len(selector_records) != 94:
    print(
        "Warning: expected 94 rationale-annotated records, "
        f"but found {len(selector_records)}."
    )

if len(selector_records) < 30:
    raise ValueError(
        "Too few records contain decisive evidence for a "
        "meaningful selector experiment."
    )


# -----------------------------------------------------------------------------
# Recover evidence ID to tensor-position mapping
# -----------------------------------------------------------------------------

membership = pd.read_parquet(
    MEMBERSHIP_PATH
)

required_membership_columns = {
    "window_id",
    "position",
    "evidence_id",
}

missing_membership_columns = sorted(
    required_membership_columns
    - set(membership.columns)
)

if missing_membership_columns:
    raise ValueError(
        "Membership table is missing columns: "
        f"{missing_membership_columns}"
    )

membership["window_id"] = (
    membership["window_id"]
    .astype(str)
)

membership["evidence_id"] = (
    membership["evidence_id"]
    .astype(str)
    .str.strip()
)

membership["position"] = pd.to_numeric(
    membership["position"],
    errors="raise",
).astype(np.int64)

if membership[
    [
        "window_id",
        "position",
    ]
].duplicated().any():
    raise ValueError(
        "Membership contains duplicate window-position pairs."
    )

membership_groups = {
    window_id: group.sort_values(
        "position"
    ).copy()
    for window_id, group in membership.groupby(
        "window_id",
        sort=False,
    )
}


# -----------------------------------------------------------------------------
# Event-level multimodal representations
# -----------------------------------------------------------------------------

text_event_features = (
    text_embedding_lookup_stance[
        window_text_ids_stance
    ]
)

feature_sets = {
    "network_temporal": np.concatenate(
        [
            X_network_stance,
            X_temporal_stance,
        ],
        axis=-1,
    ).astype(np.float32),
    "text_only": text_event_features.astype(
        np.float32
    ),
    "full_multimodal": np.concatenate(
        [
            X_network_stance,
            X_temporal_stance,
            text_event_features,
        ],
        axis=-1,
    ).astype(np.float32),
}

aspect_to_index_selector = {
    aspect: index
    for index, aspect in enumerate(
        ASPECT_ORDER
    )
}


# -----------------------------------------------------------------------------
# Construct candidate-event supervision
# -----------------------------------------------------------------------------

event_rows = []

for record_index, record in (
    selector_records.iterrows()
):
    window_id = str(
        record["window_id"]
    )

    if window_id not in membership_groups:
        raise ValueError(
            f"No membership rows found for {window_id}."
        )

    tensor_index = int(
        window_to_tensor_index[
            window_id
        ]
    )

    group = membership_groups[
        window_id
    ]

    decisive_ids = set(
        record["decisive_id_list"]
    )

    available_ids = set(
        group["evidence_id"]
    )

    unknown_ids = sorted(
        decisive_ids
        - available_ids
    )

    if unknown_ids:
        raise ValueError(
            "Decisive evidence is not present in its source window. "
            f"Annotation {record['annotation_id']}: {unknown_ids}"
        )

    valid_positions = np.where(
        attention_mask_stance[
            tensor_index
        ]
    )[0]

    group_positions = set(
        group["position"].astype(int)
    )

    for position in valid_positions:
        if int(position) not in group_positions:
            raise ValueError(
                "A valid tensor position has no evidence identifier: "
                f"{window_id}, position={position}."
            )

    evidence_by_position = dict(
        zip(
            group["position"].astype(int),
            group["evidence_id"],
        )
    )

    positive_count = len(
        decisive_ids
    )

    negative_count = (
        len(valid_positions)
        - positive_count
    )

    if negative_count <= 0:
        raise ValueError(
            "A selector record has no non-decisive candidates: "
            f"{record['annotation_id']}"
        )

    for position in valid_positions:
        evidence_id = evidence_by_position[
            int(position)
        ]

        target = int(
            evidence_id in decisive_ids
        )

        # Each annotation unit receives equal total weight. Within a unit,
        # positive and non-positive candidate sets each receive half the weight.
        if target == 1:
            within_window_weight = (
                0.5 / positive_count
            )
        else:
            within_window_weight = (
                0.5 / negative_count
            )

        annotation_weight = (
            float(record["confidence"])
            / 5.0
        )

        if bool(
            record["insufficient_context"]
        ):
            annotation_weight *= 0.5

        event_rows.append(
            {
                "record_index": record_index,
                "annotation_id": record[
                    "annotation_id"
                ],
                "window_id": window_id,
                "split": record["split"],
                "aspect": record["aspect"],
                "aspect_index": (
                    aspect_to_index_selector[
                        record["aspect"]
                    ]
                ),
                "tensor_index": tensor_index,
                "position": int(position),
                "evidence_id": evidence_id,
                "target": target,
                "sample_weight": (
                    within_window_weight
                    * annotation_weight
                ),
            }
        )

event_table = pd.DataFrame(
    event_rows
)

if event_table.empty:
    raise ValueError(
        "No selector candidates were constructed."
    )

positive_audit = (
    event_table
    .groupby("annotation_id")[
        "target"
    ]
    .sum()
)

if (positive_audit <= 0).any():
    raise AssertionError(
        "Every selector record must contain at least one positive event."
    )


# -----------------------------------------------------------------------------
# Split audit: no annotation or window may cross partitions
# -----------------------------------------------------------------------------

split_order_selector = [
    "train",
    "validation",
    "test",
]

split_audit_rows = []

for split_name in split_order_selector:
    split_events = event_table[
        event_table["split"]
        == split_name
    ]

    split_records = selector_records[
        selector_records["split"]
        == split_name
    ]

    if split_records.empty:
        raise ValueError(
            f"No rationale-annotated records occur in {split_name}."
        )

    split_audit_rows.append(
        {
            "split": split_name,
            "records": len(split_records),
            "unique_windows": int(
                split_records[
                    "window_id"
                ].nunique()
            ),
            "candidate_events": len(
                split_events
            ),
            "decisive_events": int(
                split_events[
                    "target"
                ].sum()
            ),
            "positive_prevalence": float(
                split_events[
                    "target"
                ].mean()
            ),
        }
    )

split_audit = pd.DataFrame(
    split_audit_rows
)

window_split_count = (
    event_table
    .groupby("window_id")["split"]
    .nunique()
)

if (window_split_count > 1).any():
    raise AssertionError(
        "At least one window crosses selector partitions."
    )


# -----------------------------------------------------------------------------
# Aspect-conditioned multiplicative selector
# -----------------------------------------------------------------------------

class AspectConditionedSelector(nn.Module):
    def __init__(
        self,
        input_dimension,
        number_of_aspects,
        hidden_dimension,
    ):
        super().__init__()

        self.event_projection = nn.Sequential(
            nn.Linear(
                input_dimension,
                hidden_dimension,
            ),
            nn.GELU(),
            nn.Dropout(0.10),
        )

        self.aspect_embedding = nn.Embedding(
            number_of_aspects,
            hidden_dimension,
        )

        self.score_head = nn.Sequential(
            nn.Linear(
                hidden_dimension * 3,
                hidden_dimension,
            ),
            nn.GELU(),
            nn.Dropout(0.10),
            nn.Linear(
                hidden_dimension,
                1,
            ),
        )

    def forward(
        self,
        event_features,
        aspect_indices,
    ):
        event_state = self.event_projection(
            event_features
        )

        aspect_state = self.aspect_embedding(
            aspect_indices
        )

        interaction = (
            event_state
            * aspect_state
        )

        combined = torch.cat(
            [
                event_state,
                aspect_state,
                interaction,
            ],
            dim=-1,
        )

        return self.score_head(
            combined
        ).squeeze(-1)


# -----------------------------------------------------------------------------
# Ranking metrics
# -----------------------------------------------------------------------------

def average_precision_from_ranking(
    targets,
    scores,
):
    order = np.argsort(-scores)
    ranked_targets = targets[order]
    positive_total = int(
        ranked_targets.sum()
    )

    if positive_total == 0:
        return np.nan

    cumulative = np.cumsum(
        ranked_targets
    )

    precision_at_rank = (
        cumulative
        / np.arange(
            1,
            len(ranked_targets) + 1,
        )
    )

    return float(
        (
            precision_at_rank
            * ranked_targets
        ).sum()
        / positive_total
    )


def reciprocal_rank(
    targets,
    scores,
):
    order = np.argsort(-scores)
    ranked_targets = targets[order]
    positive_positions = np.where(
        ranked_targets == 1
    )[0]

    if len(positive_positions) == 0:
        return np.nan

    return float(
        1.0
        / (
            positive_positions[0]
            + 1
        )
    )


def ndcg_at_k(
    targets,
    scores,
    budget,
):
    budget = min(
        budget,
        len(targets),
    )

    order = np.argsort(-scores)[
        :budget
    ]

    gains = targets[order]

    discounts = 1.0 / np.log2(
        np.arange(
            2,
            budget + 2,
        )
    )

    dcg = float(
        (gains * discounts).sum()
    )

    ideal_positive_count = min(
        int(targets.sum()),
        budget,
    )

    if ideal_positive_count == 0:
        return np.nan

    idcg = float(
        discounts[
            :ideal_positive_count
        ].sum()
    )

    return dcg / idcg


def evaluate_rankings(
    table,
    score_column,
    method_name,
    seed,
):
    rows = []

    for annotation_id, group in table.groupby(
        "annotation_id",
        sort=False,
    ):
        targets = group[
            "target"
        ].to_numpy(dtype=np.int64)

        scores = group[
            score_column
        ].to_numpy(dtype=np.float64)

        positive_total = int(
            targets.sum()
        )

        ap = average_precision_from_ranking(
            targets,
            scores,
        )

        mrr = reciprocal_rank(
            targets,
            scores,
        )

        for budget in EVIDENCE_BUDGETS:
            effective_budget = min(
                budget,
                len(group),
            )

            selected = np.argsort(
                -scores
            )[:effective_budget]

            true_positive = int(
                targets[selected].sum()
            )

            precision = (
                true_positive
                / effective_budget
            )

            recall = (
                true_positive
                / positive_total
            )

            if precision + recall > 0:
                f1 = (
                    2.0
                    * precision
                    * recall
                    / (
                        precision
                        + recall
                    )
                )
            else:
                f1 = 0.0

            rows.append(
                {
                    "method": method_name,
                    "seed": seed,
                    "annotation_id": annotation_id,
                    "window_id": group[
                        "window_id"
                    ].iloc[0],
                    "aspect": group[
                        "aspect"
                    ].iloc[0],
                    "budget": budget,
                    "candidate_events": len(
                        group
                    ),
                    "decisive_events": (
                        positive_total
                    ),
                    "selected_events": (
                        effective_budget
                    ),
                    "true_positive": (
                        true_positive
                    ),
                    "precision_at_k": (
                        precision
                    ),
                    "recall_at_k": recall,
                    "f1_at_k": f1,
                    "hit_at_k": float(
                        true_positive > 0
                    ),
                    "ndcg_at_k": ndcg_at_k(
                        targets,
                        scores,
                        budget,
                    ),
                    "average_precision": ap,
                    "reciprocal_rank": mrr,
                    "retained_fraction": (
                        effective_budget
                        / len(group)
                    ),
                }
            )

    return pd.DataFrame(rows)


def mean_average_precision(
    table,
    score_column,
):
    values = []

    for _, group in table.groupby(
        "annotation_id",
        sort=False,
    ):
        values.append(
            average_precision_from_ranking(
                group["target"].to_numpy(
                    dtype=np.int64
                ),
                group[score_column].to_numpy(
                    dtype=np.float64
                ),
            )
        )

    return float(
        np.nanmean(values)
    )


# -----------------------------------------------------------------------------
# Feature assembly
# -----------------------------------------------------------------------------

def gather_event_features(
    feature_tensor,
    table,
):
    return feature_tensor[
        table["tensor_index"].to_numpy(
            dtype=np.int64
        ),
        table["position"].to_numpy(
            dtype=np.int64
        ),
    ]


def predict_selector_scores(
    model,
    features,
    aspects,
    batch_size=1024,
):
    model.eval()
    score_parts = []

    with torch.no_grad():
        for start in range(
            0,
            len(features),
            batch_size,
        ):
            end = min(
                start + batch_size,
                len(features),
            )

            batch_features = torch.from_numpy(
                features[start:end]
            ).float().to(
                DEVICE_SELECTOR
            )

            batch_aspects = torch.from_numpy(
                aspects[start:end]
            ).long().to(
                DEVICE_SELECTOR
            )

            scores = model(
                batch_features,
                batch_aspects,
            )

            score_parts.append(
                scores.detach().cpu().numpy()
            )

    return np.concatenate(
        score_parts
    )


# -----------------------------------------------------------------------------
# Train five seeds for each modality setting
# -----------------------------------------------------------------------------

all_record_results = []
training_history_rows = []
validation_selection_rows = []
event_prediction_tables = []

for feature_name, feature_tensor in (
    feature_sets.items()
):
    all_features = gather_event_features(
        feature_tensor,
        event_table,
    ).astype(np.float32)

    train_event_mask = (
        event_table["split"].to_numpy()
        == "train"
    )

    validation_event_mask = (
        event_table["split"].to_numpy()
        == "validation"
    )

    test_event_mask = (
        event_table["split"].to_numpy()
        == "test"
    )

    scaler = StandardScaler()

    scaler.fit(
        all_features[
            train_event_mask
        ]
    )

    scaled_features = scaler.transform(
        all_features
    ).astype(np.float32)

    aspects = event_table[
        "aspect_index"
    ].to_numpy(dtype=np.int64)

    targets = event_table[
        "target"
    ].to_numpy(dtype=np.float32)

    weights = event_table[
        "sample_weight"
    ].to_numpy(dtype=np.float32)

    train_dataset = TensorDataset(
        torch.from_numpy(
            scaled_features[
                train_event_mask
            ]
        ),
        torch.from_numpy(
            aspects[
                train_event_mask
            ]
        ),
        torch.from_numpy(
            targets[
                train_event_mask
            ]
        ),
        torch.from_numpy(
            weights[
                train_event_mask
            ]
        ),
    )

    validation_table = event_table[
        validation_event_mask
    ].copy()

    test_table_base = event_table[
        test_event_mask
    ].copy()

    for seed in SELECTOR_SEEDS:
        set_selector_seed(seed)

        generator = torch.Generator()
        generator.manual_seed(seed)

        train_loader = DataLoader(
            train_dataset,
            batch_size=BATCH_SIZE,
            shuffle=True,
            num_workers=0,
            generator=generator,
        )

        model = AspectConditionedSelector(
            input_dimension=
                scaled_features.shape[1],
            number_of_aspects=
                len(ASPECT_ORDER),
            hidden_dimension=
                HIDDEN_DIMENSION,
        ).to(DEVICE_SELECTOR)

        optimizer = torch.optim.AdamW(
            model.parameters(),
            lr=LEARNING_RATE,
            weight_decay=WEIGHT_DECAY,
        )

        loss_function = (
            nn.BCEWithLogitsLoss(
                reduction="none"
            )
        )

        best_validation_map = -np.inf
        best_epoch = 0
        best_state = None
        epochs_without_improvement = 0

        for epoch in range(
            1,
            MAXIMUM_EPOCHS + 1,
        ):
            model.train()
            epoch_loss_sum = 0.0
            epoch_weight_sum = 0.0

            for (
                batch_features,
                batch_aspects,
                batch_targets,
                batch_weights,
            ) in train_loader:
                batch_features = (
                    batch_features
                    .float()
                    .to(DEVICE_SELECTOR)
                )

                batch_aspects = (
                    batch_aspects
                    .long()
                    .to(DEVICE_SELECTOR)
                )

                batch_targets = (
                    batch_targets
                    .float()
                    .to(DEVICE_SELECTOR)
                )

                batch_weights = (
                    batch_weights
                    .float()
                    .to(DEVICE_SELECTOR)
                )

                optimizer.zero_grad(
                    set_to_none=True
                )

                logits = model(
                    batch_features,
                    batch_aspects,
                )

                losses = loss_function(
                    logits,
                    batch_targets,
                )

                loss = (
                    losses
                    * batch_weights
                ).sum() / batch_weights.sum().clamp_min(
                    1e-8
                )

                loss.backward()

                torch.nn.utils.clip_grad_norm_(
                    model.parameters(),
                    max_norm=5.0,
                )

                optimizer.step()

                epoch_loss_sum += float(
                    (
                        losses
                        * batch_weights
                    ).sum().item()
                )

                epoch_weight_sum += float(
                    batch_weights.sum().item()
                )

            validation_scores = (
                predict_selector_scores(
                    model,
                    scaled_features[
                        validation_event_mask
                    ],
                    aspects[
                        validation_event_mask
                    ],
                )
            )

            validation_scored = (
                validation_table.copy()
            )

            validation_scored[
                "selector_score"
            ] = validation_scores

            validation_map = (
                mean_average_precision(
                    validation_scored,
                    "selector_score",
                )
            )

            training_history_rows.append(
                {
                    "feature_set": feature_name,
                    "seed": seed,
                    "epoch": epoch,
                    "train_weighted_bce": (
                        epoch_loss_sum
                        / max(
                            epoch_weight_sum,
                            1e-8,
                        )
                    ),
                    "validation_map": (
                        validation_map
                    ),
                }
            )

            if (
                validation_map
                > best_validation_map
                + 1e-8
            ):
                best_validation_map = (
                    validation_map
                )

                best_epoch = epoch
                best_state = copy.deepcopy(
                    model.state_dict()
                )

                epochs_without_improvement = 0
            else:
                epochs_without_improvement += 1

            if (
                epochs_without_improvement
                >= EARLY_STOPPING_PATIENCE
            ):
                break

        if best_state is None:
            raise RuntimeError(
                "Selector training did not produce a valid checkpoint."
            )

        model.load_state_dict(
            best_state
        )

        validation_selection_rows.append(
            {
                "feature_set": feature_name,
                "seed": seed,
                "best_epoch": best_epoch,
                "best_validation_map": (
                    best_validation_map
                ),
            }
        )

        test_scores = predict_selector_scores(
            model,
            scaled_features[
                test_event_mask
            ],
            aspects[
                test_event_mask
            ],
        )

        test_scored = test_table_base.copy()
        test_scored["selector_score"] = (
            test_scores
        )
        test_scored["feature_set"] = (
            feature_name
        )
        test_scored["seed"] = seed

        event_prediction_tables.append(
            test_scored
        )

        torch.save(
            {
                "model_state_dict": best_state,
                "feature_set": feature_name,
                "input_dimension": int(
                    scaled_features.shape[1]
                ),
                "number_of_aspects": len(
                    ASPECT_ORDER
                ),
                "hidden_dimension": (
                    HIDDEN_DIMENSION
                ),
                "aspect_order": ASPECT_ORDER,
                "seed": seed,
                "best_epoch": best_epoch,
                "best_validation_map": (
                    best_validation_map
                ),
                "scaler_mean": (
                    scaler.mean_.astype(
                        np.float32
                    )
                ),
                "scaler_scale": (
                    scaler.scale_.astype(
                        np.float32
                    )
                ),
            },
            SELECTOR_OUTPUT_DIR
            / (
                f"selector_{feature_name}_"
                f"seed_{seed}.pt"
            ),
        )

        seed_results = evaluate_rankings(
            test_scored,
            score_column="selector_score",
            method_name=feature_name,
            seed=seed,
        )

        all_record_results.append(
            seed_results
        )

        del model

        if torch.cuda.is_available():
            torch.cuda.empty_cache()


# -----------------------------------------------------------------------------
# Non-learned test baselines
# -----------------------------------------------------------------------------

full_features = gather_event_features(
    feature_sets[
        "full_multimodal"
    ],
    event_table,
).astype(np.float32)

train_event_mask = (
    event_table["split"].to_numpy()
    == "train"
)

test_event_mask = (
    event_table["split"].to_numpy()
    == "test"
)

baseline_scaler = StandardScaler()
baseline_scaler.fit(
    full_features[
        train_event_mask
    ]
)

standardized_full_features = (
    baseline_scaler.transform(
        full_features
    )
)

baseline_test_table = event_table[
    test_event_mask
].copy()

baseline_test_table[
    "magnitude_score"
] = np.linalg.norm(
    standardized_full_features[
        test_event_mask
    ],
    axis=1,
)

all_record_results.append(
    evaluate_rankings(
        baseline_test_table,
        score_column="magnitude_score",
        method_name="magnitude_baseline",
        seed=-1,
    )
)

random_result_parts = []

for repetition in range(
    RANDOM_BASELINE_REPETITIONS
):
    rng = np.random.default_rng(
        10000 + repetition
    )

    random_table = (
        baseline_test_table.copy()
    )

    random_table[
        "random_score"
    ] = rng.random(
        len(random_table)
    )

    random_result_parts.append(
        evaluate_rankings(
            random_table,
            score_column="random_score",
            method_name="random_baseline",
            seed=repetition,
        )
    )

random_record_results = pd.concat(
    random_result_parts,
    ignore_index=True,
)

all_record_results.append(
    random_record_results
)


# -----------------------------------------------------------------------------
# Aggregate test results
# -----------------------------------------------------------------------------

record_results = pd.concat(
    all_record_results,
    ignore_index=True,
)

metric_columns = [
    "precision_at_k",
    "recall_at_k",
    "f1_at_k",
    "hit_at_k",
    "ndcg_at_k",
    "average_precision",
    "reciprocal_rank",
    "retained_fraction",
]

seed_level_results = (
    record_results
    .groupby(
        [
            "method",
            "seed",
            "budget",
        ],
        as_index=False,
    )[metric_columns]
    .mean()
)

aggregate_rows = []

for (
    method_name,
    budget,
), group in seed_level_results.groupby(
    [
        "method",
        "budget",
    ],
    sort=False,
):
    for metric in metric_columns:
        values = group[
            metric
        ].to_numpy(dtype=np.float64)

        aggregate_rows.append(
            {
                "method": method_name,
                "budget": int(budget),
                "metric": metric,
                "mean": float(
                    np.nanmean(values)
                ),
                "standard_deviation": (
                    float(
                        np.nanstd(
                            values,
                            ddof=1,
                        )
                    )
                    if len(values) > 1
                    else 0.0
                ),
                "minimum": float(
                    np.nanmin(values)
                ),
                "maximum": float(
                    np.nanmax(values)
                ),
                "seeds": int(
                    len(values)
                ),
            }
        )

aggregate_results = pd.DataFrame(
    aggregate_rows
)

primary_summary = (
    seed_level_results[
        [
            "method",
            "seed",
            "budget",
            "precision_at_k",
            "recall_at_k",
            "f1_at_k",
            "hit_at_k",
            "ndcg_at_k",
            "average_precision",
            "reciprocal_rank",
            "retained_fraction",
        ]
    ]
    .groupby(
        [
            "method",
            "budget",
        ],
        as_index=False,
    )
    .agg(
        precision_at_k_mean=(
            "precision_at_k",
            "mean",
        ),
        recall_at_k_mean=(
            "recall_at_k",
            "mean",
        ),
        f1_at_k_mean=(
            "f1_at_k",
            "mean",
        ),
        hit_at_k_mean=(
            "hit_at_k",
            "mean",
        ),
        ndcg_at_k_mean=(
            "ndcg_at_k",
            "mean",
        ),
        map_mean=(
            "average_precision",
            "mean",
        ),
        mrr_mean=(
            "reciprocal_rank",
            "mean",
        ),
        retained_fraction_mean=(
            "retained_fraction",
            "mean",
        ),
    )
)


# -----------------------------------------------------------------------------
# Per-aspect audit for the full multimodal selector
# -----------------------------------------------------------------------------

full_selector_records = record_results[
    record_results["method"]
    == "full_multimodal"
].copy()

per_aspect_results = (
    full_selector_records
    .groupby(
        [
            "aspect",
            "budget",
        ],
        as_index=False,
    )[
        [
            "precision_at_k",
            "recall_at_k",
            "f1_at_k",
            "hit_at_k",
            "ndcg_at_k",
            "average_precision",
            "reciprocal_rank",
            "retained_fraction",
        ]
    ]
    .mean()
)


# -----------------------------------------------------------------------------
# Save results
# -----------------------------------------------------------------------------

record_results.to_csv(
    SELECTOR_OUTPUT_DIR
    / "selector_record_level_results.csv",
    index=False,
)

seed_level_results.to_csv(
    SELECTOR_OUTPUT_DIR
    / "selector_seed_level_results.csv",
    index=False,
)

aggregate_results.to_csv(
    SELECTOR_OUTPUT_DIR
    / "selector_aggregate_results.csv",
    index=False,
)

primary_summary.to_csv(
    SELECTOR_OUTPUT_DIR
    / "selector_primary_summary.csv",
    index=False,
)

per_aspect_results.to_csv(
    SELECTOR_OUTPUT_DIR
    / "selector_per_aspect_results.csv",
    index=False,
)

split_audit.to_csv(
    SELECTOR_OUTPUT_DIR
    / "selector_split_audit.csv",
    index=False,
)

pd.DataFrame(
    training_history_rows
).to_csv(
    SELECTOR_OUTPUT_DIR
    / "selector_training_history.csv",
    index=False,
)

pd.DataFrame(
    validation_selection_rows
).to_csv(
    SELECTOR_OUTPUT_DIR
    / "selector_validation_selection.csv",
    index=False,
)

pd.concat(
    event_prediction_tables,
    ignore_index=True,
).to_csv(
    SELECTOR_OUTPUT_DIR
    / "selector_test_event_scores.csv",
    index=False,
)

selector_manifest = {
    "status": (
        "exploratory_silver_label_rationale_retrieval"
    ),
    "supervised_records": int(
        len(selector_records)
    ),
    "unlabeled_records_excluded": int(
        len(stance_table)
        - len(selector_records)
    ),
    "unlabeled_policy": (
        "Records without decisive evidence identifiers were excluded "
        "from selector supervision and were not treated as negatives."
    ),
    "within_record_negative_policy": (
        "Within explicitly rationale-annotated records, valid events "
        "not selected as decisive were treated as non-decisive ranking "
        "candidates."
    ),
    "split_policy": (
        "Original hidden-key train/validation/test split; no window "
        "crosses partitions."
    ),
    "selection_policy": (
        "Early stopping used validation MAP; test rankings were "
        "computed once for each frozen seed checkpoint."
    ),
    "feature_sets": {
        name: int(tensor.shape[-1])
        for name, tensor in feature_sets.items()
    },
    "seeds": SELECTOR_SEEDS,
    "budgets": EVIDENCE_BUDGETS,
    "random_baseline_repetitions": (
        RANDOM_BASELINE_REPETITIONS
    ),
    "device": str(DEVICE_SELECTOR),
    "label_limitation": (
        "AI-assisted, single-reviewer-resolved silver rationales; "
        "not independent human gold-standard evidence selection."
    ),
}

with open(
    SELECTOR_OUTPUT_DIR
    / "selector_manifest.json",
    "w",
    encoding="utf-8",
) as file:
    json.dump(
        selector_manifest,
        file,
        indent=2,
    )


# -----------------------------------------------------------------------------
# Display publication-facing results
# -----------------------------------------------------------------------------

print("=" * 80)
print("DAPT2020-MAST ASPECT-CONDITIONED SPARSE EVIDENCE SELECTOR")
print("=" * 80)

print("\nDevice:", DEVICE_SELECTOR)
print(
    "Rationale-annotated records:",
    len(selector_records),
)
print(
    "Records without decisive IDs excluded as unlabeled:",
    len(stance_table)
    - len(selector_records),
)

print("\nLeakage-aware selector split audit:")
display(
    split_audit.round(6)
)

print("\nPrimary sparse-retrieval summary:")
display(
    primary_summary
    .sort_values(
        [
            "budget",
            "map_mean",
        ],
        ascending=[
            True,
            False,
        ],
    )
    .round(6)
)

print("\nFull-multimodal selector by aspect:")
display(
    per_aspect_results.round(6)
)

print("\nValidation-selected checkpoints:")
display(
    pd.DataFrame(
        validation_selection_rows
    ).round(6)
)

print("\nSaved selector results to:")
print(SELECTOR_OUTPUT_DIR)

print(
    "\nCell 27 completed successfully. These are exploratory "
    "silver-rationale results. Do not describe them as independently "
    "human-verified evidence-selection performance."
)


In [ ]:
# =============================================================================
# CELL 28: DUAL SUPPORTING/CONTRADICTING EVIDENCE SELECTORS
# =============================================================================

from pathlib import Path
import copy
import json
import random

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset

from sklearn.preprocessing import StandardScaler


# -----------------------------------------------------------------------------
# Required Cell 27 objects
# -----------------------------------------------------------------------------

required_objects = [
    "AMJEPA_DIR",
    "ASPECT_ORDER",
    "selector_records",
    "event_table",
    "feature_sets",
    "EVIDENCE_BUDGETS",
    "SELECTOR_SEEDS",
    "evaluate_rankings",
    "average_precision_from_ranking",
    "gather_event_features",
    "record_results",
]

missing_objects = [
    name
    for name in required_objects
    if name not in globals()
]

if missing_objects:
    raise NameError(
        "Missing objects from Cell 27:\n"
        + "\n".join(missing_objects)
        + "\n\nRun Cells 26 and 27 before Cell 28."
    )

DUAL_OUTPUT_DIR = (
    Path(AMJEPA_DIR)
    / "silver_dual_selector"
)

DUAL_OUTPUT_DIR.mkdir(
    parents=True,
    exist_ok=True,
)


# -----------------------------------------------------------------------------
# Configuration
# -----------------------------------------------------------------------------

DUAL_CHANNELS = [
    "SUPPORTING",
    "CONTRADICTING",
]

DUAL_FEATURE_SETS = [
    "text_only",
    "full_multimodal",
]

DUAL_HIDDEN_DIMENSION = 64
DUAL_MAXIMUM_EPOCHS = 300
DUAL_PATIENCE = 30
DUAL_BATCH_SIZE = 256
DUAL_LEARNING_RATE = 1e-3
DUAL_WEIGHT_DECAY = 1e-4

DEVICE_DUAL = torch.device(
    "cuda"
    if torch.cuda.is_available()
    else "cpu"
)

channel_to_index = {
    channel: index
    for index, channel in enumerate(
        DUAL_CHANNELS
    )
}


def set_dual_seed(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)

    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)


# -----------------------------------------------------------------------------
# Verify that the decisive-ID records support the dual-channel interpretation
# -----------------------------------------------------------------------------

observed_rationale_stances = set(
    selector_records["stance"]
)

unexpected_rationale_stances = sorted(
    observed_rationale_stances
    - set(DUAL_CHANNELS)
)

if unexpected_rationale_stances:
    raise ValueError(
        "Some decisive-evidence records have a neutral or unknown stance. "
        "They cannot be assigned automatically to E+ or E-. "
        f"Unexpected labels: {unexpected_rationale_stances}"
    )

expected_channel_counts = {
    "SUPPORTING": 56,
    "CONTRADICTING": 38,
}

observed_channel_counts = (
    selector_records["stance"]
    .value_counts()
    .to_dict()
)

if observed_channel_counts != expected_channel_counts:
    print(
        "Warning: rationale-channel counts differ from the audited pilot."
    )
    print("Expected:", expected_channel_counts)
    print("Observed:", observed_channel_counts)

channel_lookup = (
    selector_records
    .set_index("annotation_id")[
        "stance"
    ]
    .to_dict()
)

dual_event_table = event_table.copy()

dual_event_table["channel"] = (
    dual_event_table["annotation_id"]
    .map(channel_lookup)
)

if dual_event_table["channel"].isna().any():
    raise ValueError(
        "At least one event could not be assigned to a dual channel."
    )

dual_event_table["channel_index"] = (
    dual_event_table["channel"]
    .map(channel_to_index)
    .astype(np.int64)
)


# -----------------------------------------------------------------------------
# Split and channel audit
# -----------------------------------------------------------------------------

dual_audit_rows = []

for split_name in [
    "train",
    "validation",
    "test",
]:
    for channel_name in DUAL_CHANNELS:
        subset = dual_event_table[
            (
                dual_event_table["split"]
                == split_name
            )
            & (
                dual_event_table["channel"]
                == channel_name
            )
        ]

        records = int(
            subset["annotation_id"]
            .nunique()
        )

        if records == 0:
            raise ValueError(
                f"No {channel_name} rationale records occur in "
                f"the {split_name} partition."
            )

        dual_audit_rows.append(
            {
                "split": split_name,
                "channel": channel_name,
                "records": records,
                "unique_windows": int(
                    subset["window_id"]
                    .nunique()
                ),
                "candidate_events": int(
                    len(subset)
                ),
                "decisive_events": int(
                    subset["target"].sum()
                ),
                "positive_prevalence": float(
                    subset["target"].mean()
                ),
            }
        )

dual_split_audit = pd.DataFrame(
    dual_audit_rows
)


# -----------------------------------------------------------------------------
# Shared encoder with separate positive and negative scoring heads
# -----------------------------------------------------------------------------

class DualAspectConditionedSelector(nn.Module):
    def __init__(
        self,
        input_dimension,
        number_of_aspects,
        hidden_dimension,
    ):
        super().__init__()

        self.event_projection = nn.Sequential(
            nn.Linear(
                input_dimension,
                hidden_dimension,
            ),
            nn.GELU(),
            nn.Dropout(0.10),
        )

        self.aspect_embedding = nn.Embedding(
            number_of_aspects,
            hidden_dimension,
        )

        interaction_dimension = (
            hidden_dimension * 3
        )

        self.supporting_head = nn.Sequential(
            nn.Linear(
                interaction_dimension,
                hidden_dimension,
            ),
            nn.GELU(),
            nn.Dropout(0.10),
            nn.Linear(
                hidden_dimension,
                1,
            ),
        )

        self.contradicting_head = nn.Sequential(
            nn.Linear(
                interaction_dimension,
                hidden_dimension,
            ),
            nn.GELU(),
            nn.Dropout(0.10),
            nn.Linear(
                hidden_dimension,
                1,
            ),
        )

    def encode(
        self,
        event_features,
        aspect_indices,
    ):
        event_state = self.event_projection(
            event_features
        )

        aspect_state = self.aspect_embedding(
            aspect_indices
        )

        interaction = (
            event_state
            * aspect_state
        )

        return torch.cat(
            [
                event_state,
                aspect_state,
                interaction,
            ],
            dim=-1,
        )

    def forward_all_channels(
        self,
        event_features,
        aspect_indices,
    ):
        combined = self.encode(
            event_features,
            aspect_indices,
        )

        supporting_scores = (
            self.supporting_head(
                combined
            ).squeeze(-1)
        )

        contradicting_scores = (
            self.contradicting_head(
                combined
            ).squeeze(-1)
        )

        return torch.stack(
            [
                supporting_scores,
                contradicting_scores,
            ],
            dim=1,
        )

    def forward(
        self,
        event_features,
        aspect_indices,
        channel_indices,
    ):
        all_scores = self.forward_all_channels(
            event_features,
            aspect_indices,
        )

        return all_scores.gather(
            1,
            channel_indices.unsqueeze(1),
        ).squeeze(1)


# -----------------------------------------------------------------------------
# Prediction and validation helpers
# -----------------------------------------------------------------------------

def predict_dual_scores(
    model,
    features,
    aspects,
    channels,
    batch_size=1024,
):
    model.eval()
    selected_parts = []
    all_channel_parts = []

    with torch.no_grad():
        for start in range(
            0,
            len(features),
            batch_size,
        ):
            end = min(
                start + batch_size,
                len(features),
            )

            batch_features = torch.from_numpy(
                features[start:end]
            ).float().to(DEVICE_DUAL)

            batch_aspects = torch.from_numpy(
                aspects[start:end]
            ).long().to(DEVICE_DUAL)

            batch_channels = torch.from_numpy(
                channels[start:end]
            ).long().to(DEVICE_DUAL)

            all_scores = model.forward_all_channels(
                batch_features,
                batch_aspects,
            )

            selected_scores = all_scores.gather(
                1,
                batch_channels.unsqueeze(1),
            ).squeeze(1)

            selected_parts.append(
                selected_scores.cpu().numpy()
            )

            all_channel_parts.append(
                all_scores.cpu().numpy()
            )

    return (
        np.concatenate(selected_parts),
        np.concatenate(all_channel_parts),
    )


def calculate_channel_map(
    table,
    score_column,
):
    rows = []

    for channel_name in DUAL_CHANNELS:
        channel_table = table[
            table["channel"]
            == channel_name
        ]

        values = []

        for _, group in channel_table.groupby(
            "annotation_id",
            sort=False,
        ):
            values.append(
                average_precision_from_ranking(
                    group["target"].to_numpy(
                        dtype=np.int64
                    ),
                    group[score_column].to_numpy(
                        dtype=np.float64
                    ),
                )
            )

        rows.append(
            {
                "channel": channel_name,
                "records": int(
                    channel_table[
                        "annotation_id"
                    ].nunique()
                ),
                "map": float(
                    np.nanmean(values)
                ),
            }
        )

    result = pd.DataFrame(rows)

    return (
        result,
        float(result["map"].mean()),
    )


# -----------------------------------------------------------------------------
# Five-seed training for text and full multimodal dual selectors
# -----------------------------------------------------------------------------

dual_record_result_parts = []
dual_training_history_rows = []
dual_validation_rows = []
dual_event_score_parts = []

train_mask_dual = (
    dual_event_table["split"].to_numpy()
    == "train"
)

validation_mask_dual = (
    dual_event_table["split"].to_numpy()
    == "validation"
)

test_mask_dual = (
    dual_event_table["split"].to_numpy()
    == "test"
)

aspects_dual = dual_event_table[
    "aspect_index"
].to_numpy(dtype=np.int64)

channels_dual = dual_event_table[
    "channel_index"
].to_numpy(dtype=np.int64)

targets_dual = dual_event_table[
    "target"
].to_numpy(dtype=np.float32)

weights_dual = dual_event_table[
    "sample_weight"
].to_numpy(dtype=np.float32)

for feature_name in DUAL_FEATURE_SETS:
    raw_features = gather_event_features(
        feature_sets[feature_name],
        dual_event_table,
    ).astype(np.float32)

    scaler = StandardScaler()
    scaler.fit(
        raw_features[
            train_mask_dual
        ]
    )

    scaled_features = scaler.transform(
        raw_features
    ).astype(np.float32)

    train_dataset = TensorDataset(
        torch.from_numpy(
            scaled_features[
                train_mask_dual
            ]
        ),
        torch.from_numpy(
            aspects_dual[
                train_mask_dual
            ]
        ),
        torch.from_numpy(
            channels_dual[
                train_mask_dual
            ]
        ),
        torch.from_numpy(
            targets_dual[
                train_mask_dual
            ]
        ),
        torch.from_numpy(
            weights_dual[
                train_mask_dual
            ]
        ),
    )

    validation_table_base = dual_event_table[
        validation_mask_dual
    ].copy()

    test_table_base = dual_event_table[
        test_mask_dual
    ].copy()

    for seed in SELECTOR_SEEDS:
        set_dual_seed(seed)

        generator = torch.Generator()
        generator.manual_seed(seed)

        train_loader = DataLoader(
            train_dataset,
            batch_size=DUAL_BATCH_SIZE,
            shuffle=True,
            num_workers=0,
            generator=generator,
        )

        model = DualAspectConditionedSelector(
            input_dimension=
                scaled_features.shape[1],
            number_of_aspects=
                len(ASPECT_ORDER),
            hidden_dimension=
                DUAL_HIDDEN_DIMENSION,
        ).to(DEVICE_DUAL)

        optimizer = torch.optim.AdamW(
            model.parameters(),
            lr=DUAL_LEARNING_RATE,
            weight_decay=DUAL_WEIGHT_DECAY,
        )

        loss_function = nn.BCEWithLogitsLoss(
            reduction="none"
        )

        best_macro_map = -np.inf
        best_epoch = 0
        best_state = None
        epochs_without_improvement = 0

        for epoch in range(
            1,
            DUAL_MAXIMUM_EPOCHS + 1,
        ):
            model.train()
            weighted_loss_sum = 0.0
            weight_sum = 0.0

            for (
                batch_features,
                batch_aspects,
                batch_channels,
                batch_targets,
                batch_weights,
            ) in train_loader:
                batch_features = (
                    batch_features
                    .float()
                    .to(DEVICE_DUAL)
                )

                batch_aspects = (
                    batch_aspects
                    .long()
                    .to(DEVICE_DUAL)
                )

                batch_channels = (
                    batch_channels
                    .long()
                    .to(DEVICE_DUAL)
                )

                batch_targets = (
                    batch_targets
                    .float()
                    .to(DEVICE_DUAL)
                )

                batch_weights = (
                    batch_weights
                    .float()
                    .to(DEVICE_DUAL)
                )

                optimizer.zero_grad(
                    set_to_none=True
                )

                logits = model(
                    batch_features,
                    batch_aspects,
                    batch_channels,
                )

                losses = loss_function(
                    logits,
                    batch_targets,
                )

                loss = (
                    losses
                    * batch_weights
                ).sum() / batch_weights.sum().clamp_min(
                    1e-8
                )

                loss.backward()

                torch.nn.utils.clip_grad_norm_(
                    model.parameters(),
                    max_norm=5.0,
                )

                optimizer.step()

                weighted_loss_sum += float(
                    (
                        losses
                        * batch_weights
                    ).sum().item()
                )

                weight_sum += float(
                    batch_weights.sum().item()
                )

            validation_scores, _ = (
                predict_dual_scores(
                    model,
                    scaled_features[
                        validation_mask_dual
                    ],
                    aspects_dual[
                        validation_mask_dual
                    ],
                    channels_dual[
                        validation_mask_dual
                    ],
                )
            )

            validation_scored = (
                validation_table_base.copy()
            )

            validation_scored[
                "dual_score"
            ] = validation_scores

            channel_map_table, macro_map = (
                calculate_channel_map(
                    validation_scored,
                    "dual_score",
                )
            )

            dual_training_history_rows.append(
                {
                    "feature_set": feature_name,
                    "seed": seed,
                    "epoch": epoch,
                    "train_weighted_bce": (
                        weighted_loss_sum
                        / max(
                            weight_sum,
                            1e-8,
                        )
                    ),
                    "validation_supporting_map": float(
                        channel_map_table.loc[
                            channel_map_table[
                                "channel"
                            ] == "SUPPORTING",
                            "map",
                        ].iloc[0]
                    ),
                    "validation_contradicting_map": float(
                        channel_map_table.loc[
                            channel_map_table[
                                "channel"
                            ] == "CONTRADICTING",
                            "map",
                        ].iloc[0]
                    ),
                    "validation_macro_map": (
                        macro_map
                    ),
                }
            )

            if macro_map > best_macro_map + 1e-8:
                best_macro_map = macro_map
                best_epoch = epoch
                best_state = copy.deepcopy(
                    model.state_dict()
                )
                epochs_without_improvement = 0
            else:
                epochs_without_improvement += 1

            if (
                epochs_without_improvement
                >= DUAL_PATIENCE
            ):
                break

        if best_state is None:
            raise RuntimeError(
                "Dual selector failed to produce a checkpoint."
            )

        model.load_state_dict(
            best_state
        )

        best_history_row = next(
            row
            for row in dual_training_history_rows
            if (
                row["feature_set"]
                == feature_name
                and row["seed"] == seed
                and row["epoch"] == best_epoch
            )
        )

        dual_validation_rows.append(
            {
                "feature_set": feature_name,
                "seed": seed,
                "best_epoch": best_epoch,
                "best_validation_supporting_map": (
                    best_history_row[
                        "validation_supporting_map"
                    ]
                ),
                "best_validation_contradicting_map": (
                    best_history_row[
                        "validation_contradicting_map"
                    ]
                ),
                "best_validation_macro_map": (
                    best_macro_map
                ),
            }
        )

        test_scores, all_test_scores = (
            predict_dual_scores(
                model,
                scaled_features[
                    test_mask_dual
                ],
                aspects_dual[
                    test_mask_dual
                ],
                channels_dual[
                    test_mask_dual
                ],
            )
        )

        test_scored = test_table_base.copy()
        test_scored["dual_score"] = (
            test_scores
        )
        test_scored[
            "supporting_head_score"
        ] = all_test_scores[:, 0]
        test_scored[
            "contradicting_head_score"
        ] = all_test_scores[:, 1]
        test_scored["feature_set"] = (
            feature_name
        )
        test_scored["seed"] = seed

        dual_event_score_parts.append(
            test_scored
        )

        seed_results = evaluate_rankings(
            test_scored,
            score_column="dual_score",
            method_name=(
                "dual_" + feature_name
            ),
            seed=seed,
        )

        seed_results["channel"] = (
            seed_results[
                "annotation_id"
            ].map(channel_lookup)
        )

        dual_record_result_parts.append(
            seed_results
        )

        torch.save(
            {
                "model_state_dict": best_state,
                "feature_set": feature_name,
                "input_dimension": int(
                    scaled_features.shape[1]
                ),
                "hidden_dimension": (
                    DUAL_HIDDEN_DIMENSION
                ),
                "aspect_order": ASPECT_ORDER,
                "channel_order": DUAL_CHANNELS,
                "seed": seed,
                "best_epoch": best_epoch,
                "best_validation_macro_map": (
                    best_macro_map
                ),
                "scaler_mean": (
                    scaler.mean_.astype(
                        np.float32
                    )
                ),
                "scaler_scale": (
                    scaler.scale_.astype(
                        np.float32
                    )
                ),
            },
            DUAL_OUTPUT_DIR
            / (
                f"dual_selector_{feature_name}_"
                f"seed_{seed}.pt"
            ),
        )

        del model

        if torch.cuda.is_available():
            torch.cuda.empty_cache()


# -----------------------------------------------------------------------------
# Add the corresponding single-selector results from Cell 27
# -----------------------------------------------------------------------------

single_comparison = record_results[
    record_results["method"].isin(
        DUAL_FEATURE_SETS
    )
].copy()

single_comparison["channel"] = (
    single_comparison["annotation_id"]
    .map(channel_lookup)
)

single_comparison["method"] = (
    "single_"
    + single_comparison["method"]
)

if single_comparison["channel"].isna().any():
    raise ValueError(
        "Cell 27 results could not be aligned with dual channels."
    )

dual_record_results = pd.concat(
    dual_record_result_parts,
    ignore_index=True,
)

comparison_record_results = pd.concat(
    [
        dual_record_results,
        single_comparison,
    ],
    ignore_index=True,
)


# -----------------------------------------------------------------------------
# Channel-specific, seed-level, and macro-channel summaries
# -----------------------------------------------------------------------------

ranking_metrics = [
    "precision_at_k",
    "recall_at_k",
    "f1_at_k",
    "hit_at_k",
    "ndcg_at_k",
    "average_precision",
    "reciprocal_rank",
    "retained_fraction",
]

channel_seed_results = (
    comparison_record_results
    .groupby(
        [
            "method",
            "seed",
            "channel",
            "budget",
        ],
        as_index=False,
    )[ranking_metrics]
    .mean()
)

macro_seed_results = (
    channel_seed_results
    .groupby(
        [
            "method",
            "seed",
            "budget",
        ],
        as_index=False,
    )[ranking_metrics]
    .mean()
)


def aggregate_seed_table(
    table,
    group_columns,
):
    rows = []

    for keys, group in table.groupby(
        group_columns,
        sort=False,
    ):
        if not isinstance(keys, tuple):
            keys = (keys,)

        key_values = dict(
            zip(
                group_columns,
                keys,
            )
        )

        for metric in ranking_metrics:
            values = group[
                metric
            ].to_numpy(dtype=np.float64)

            rows.append(
                {
                    **key_values,
                    "metric": metric,
                    "mean": float(
                        np.nanmean(values)
                    ),
                    "standard_deviation": (
                        float(
                            np.nanstd(
                                values,
                                ddof=1,
                            )
                        )
                        if len(values) > 1
                        else 0.0
                    ),
                    "minimum": float(
                        np.nanmin(values)
                    ),
                    "maximum": float(
                        np.nanmax(values)
                    ),
                    "seeds": int(
                        len(values)
                    ),
                }
            )

    return pd.DataFrame(rows)


channel_aggregate = aggregate_seed_table(
    channel_seed_results,
    [
        "method",
        "channel",
        "budget",
    ],
)

macro_aggregate = aggregate_seed_table(
    macro_seed_results,
    [
        "method",
        "budget",
    ],
)


# -----------------------------------------------------------------------------
# Direct dual-versus-single deltas at matched seed and budget
# -----------------------------------------------------------------------------

delta_rows = []

for feature_name in DUAL_FEATURE_SETS:
    dual_method = (
        "dual_" + feature_name
    )
    single_method = (
        "single_" + feature_name
    )

    dual_subset = macro_seed_results[
        macro_seed_results["method"]
        == dual_method
    ]

    single_subset = macro_seed_results[
        macro_seed_results["method"]
        == single_method
    ]

    matched = dual_subset.merge(
        single_subset,
        on=[
            "seed",
            "budget",
        ],
        suffixes=(
            "_dual",
            "_single",
        ),
        validate="one_to_one",
    )

    for _, row in matched.iterrows():
        delta_rows.append(
            {
                "feature_set": feature_name,
                "seed": int(row["seed"]),
                "budget": int(row["budget"]),
                "map_delta": float(
                    row[
                        "average_precision_dual"
                    ]
                    - row[
                        "average_precision_single"
                    ]
                ),
                "mrr_delta": float(
                    row[
                        "reciprocal_rank_dual"
                    ]
                    - row[
                        "reciprocal_rank_single"
                    ]
                ),
                "precision_at_k_delta": float(
                    row[
                        "precision_at_k_dual"
                    ]
                    - row[
                        "precision_at_k_single"
                    ]
                ),
                "recall_at_k_delta": float(
                    row[
                        "recall_at_k_dual"
                    ]
                    - row[
                        "recall_at_k_single"
                    ]
                ),
                "f1_at_k_delta": float(
                    row[
                        "f1_at_k_dual"
                    ]
                    - row[
                        "f1_at_k_single"
                    ]
                ),
                "ndcg_at_k_delta": float(
                    row[
                        "ndcg_at_k_dual"
                    ]
                    - row[
                        "ndcg_at_k_single"
                    ]
                ),
            }
        )

dual_single_deltas = pd.DataFrame(
    delta_rows
)

delta_summary = (
    dual_single_deltas
    .groupby(
        [
            "feature_set",
            "budget",
        ],
        as_index=False,
    )
    .agg(
        map_delta_mean=(
            "map_delta",
            "mean",
        ),
        map_delta_sd=(
            "map_delta",
            "std",
        ),
        mrr_delta_mean=(
            "mrr_delta",
            "mean",
        ),
        precision_at_k_delta_mean=(
            "precision_at_k_delta",
            "mean",
        ),
        recall_at_k_delta_mean=(
            "recall_at_k_delta",
            "mean",
        ),
        f1_at_k_delta_mean=(
            "f1_at_k_delta",
            "mean",
        ),
        ndcg_at_k_delta_mean=(
            "ndcg_at_k_delta",
            "mean",
        ),
    )
)


# -----------------------------------------------------------------------------
# Publication-facing wide tables
# -----------------------------------------------------------------------------

macro_primary = (
    macro_seed_results
    .groupby(
        [
            "method",
            "budget",
        ],
        as_index=False,
    )
    .agg(
        precision_at_k_mean=(
            "precision_at_k",
            "mean",
        ),
        recall_at_k_mean=(
            "recall_at_k",
            "mean",
        ),
        f1_at_k_mean=(
            "f1_at_k",
            "mean",
        ),
        hit_at_k_mean=(
            "hit_at_k",
            "mean",
        ),
        ndcg_at_k_mean=(
            "ndcg_at_k",
            "mean",
        ),
        macro_map_mean=(
            "average_precision",
            "mean",
        ),
        macro_map_sd=(
            "average_precision",
            "std",
        ),
        macro_mrr_mean=(
            "reciprocal_rank",
            "mean",
        ),
        retained_fraction_mean=(
            "retained_fraction",
            "mean",
        ),
    )
)

channel_primary = (
    channel_seed_results
    .groupby(
        [
            "method",
            "channel",
            "budget",
        ],
        as_index=False,
    )
    .agg(
        precision_at_k_mean=(
            "precision_at_k",
            "mean",
        ),
        recall_at_k_mean=(
            "recall_at_k",
            "mean",
        ),
        f1_at_k_mean=(
            "f1_at_k",
            "mean",
        ),
        ndcg_at_k_mean=(
            "ndcg_at_k",
            "mean",
        ),
        map_mean=(
            "average_precision",
            "mean",
        ),
        map_sd=(
            "average_precision",
            "std",
        ),
        mrr_mean=(
            "reciprocal_rank",
            "mean",
        ),
    )
)


# -----------------------------------------------------------------------------
# Save results and manifest
# -----------------------------------------------------------------------------

dual_record_results.to_csv(
    DUAL_OUTPUT_DIR
    / "dual_selector_record_results.csv",
    index=False,
)

comparison_record_results.to_csv(
    DUAL_OUTPUT_DIR
    / "dual_single_comparison_records.csv",
    index=False,
)

channel_seed_results.to_csv(
    DUAL_OUTPUT_DIR
    / "dual_selector_channel_seed_results.csv",
    index=False,
)

macro_seed_results.to_csv(
    DUAL_OUTPUT_DIR
    / "dual_selector_macro_seed_results.csv",
    index=False,
)

channel_aggregate.to_csv(
    DUAL_OUTPUT_DIR
    / "dual_selector_channel_aggregate.csv",
    index=False,
)

macro_aggregate.to_csv(
    DUAL_OUTPUT_DIR
    / "dual_selector_macro_aggregate.csv",
    index=False,
)

dual_single_deltas.to_csv(
    DUAL_OUTPUT_DIR
    / "dual_single_seed_deltas.csv",
    index=False,
)

delta_summary.to_csv(
    DUAL_OUTPUT_DIR
    / "dual_single_delta_summary.csv",
    index=False,
)

macro_primary.to_csv(
    DUAL_OUTPUT_DIR
    / "dual_selector_primary_summary.csv",
    index=False,
)

channel_primary.to_csv(
    DUAL_OUTPUT_DIR
    / "dual_selector_channel_summary.csv",
    index=False,
)

dual_split_audit.to_csv(
    DUAL_OUTPUT_DIR
    / "dual_selector_split_audit.csv",
    index=False,
)

pd.DataFrame(
    dual_training_history_rows
).to_csv(
    DUAL_OUTPUT_DIR
    / "dual_selector_training_history.csv",
    index=False,
)

pd.DataFrame(
    dual_validation_rows
).to_csv(
    DUAL_OUTPUT_DIR
    / "dual_selector_validation_selection.csv",
    index=False,
)

pd.concat(
    dual_event_score_parts,
    ignore_index=True,
).to_csv(
    DUAL_OUTPUT_DIR
    / "dual_selector_test_event_scores.csv",
    index=False,
)

dual_manifest = {
    "status": (
        "exploratory_silver_label_dual_rationale_retrieval"
    ),
    "channels": DUAL_CHANNELS,
    "channel_semantics": {
        "SUPPORTING": "E+ supporting-evidence selector",
        "CONTRADICTING": "E- contradicting-evidence selector",
    },
    "rationale_records": int(
        len(selector_records)
    ),
    "channel_record_counts": {
        key: int(value)
        for key, value in observed_channel_counts.items()
    },
    "neutral_policy": (
        "Neutral records contain no decisive evidence identifiers "
        "in this pilot and are excluded rather than treated as negatives."
    ),
    "architecture": (
        "Shared event projection and aspect embedding with separate "
        "supporting and contradicting scoring heads."
    ),
    "checkpoint_selection": (
        "Validation macro-MAP averaged equally across E+ and E-."
    ),
    "feature_sets": DUAL_FEATURE_SETS,
    "seeds": SELECTOR_SEEDS,
    "budgets": EVIDENCE_BUDGETS,
    "device": str(DEVICE_DUAL),
    "label_limitation": (
        "AI-assisted, single-reviewer-resolved silver rationales; "
        "not independent human gold-standard evidence channels."
    ),
}

with open(
    DUAL_OUTPUT_DIR
    / "dual_selector_manifest.json",
    "w",
    encoding="utf-8",
) as file:
    json.dump(
        dual_manifest,
        file,
        indent=2,
    )


# -----------------------------------------------------------------------------
# Display publication-facing results
# -----------------------------------------------------------------------------

print("=" * 80)
print("DAPT2020-MAST DUAL E+/E- SPARSE EVIDENCE SELECTORS")
print("=" * 80)

print("\nDevice:", DEVICE_DUAL)

print("\nChannel interpretation:")
print("SUPPORTING   -> E+ selector")
print("CONTRADICTING -> E- selector")

print("\nLeakage-aware dual-channel audit:")
display(
    dual_split_audit.round(6)
)

print("\nDual versus single selector: macro-channel results")
display(
    macro_primary
    .sort_values(
        [
            "budget",
            "macro_map_mean",
        ],
        ascending=[
            True,
            False,
        ],
    )
    .round(6)
)

print("\nDual-selector channel-specific results:")
display(
    channel_primary[
        channel_primary["method"]
        .str.startswith("dual_")
    ]
    .sort_values(
        [
            "method",
            "channel",
            "budget",
        ]
    )
    .round(6)
)

print("\nMatched-seed dual-minus-single deltas:")
display(
    delta_summary.round(6)
)

print("\nValidation-selected dual checkpoints:")
display(
    pd.DataFrame(
        dual_validation_rows
    ).round(6)
)

print("\nSaved dual-selector results to:")
print(DUAL_OUTPUT_DIR)

print(
    "\nCell 28 completed successfully. Positive deltas favor the "
    "dual E+/E- architecture. Interpret all results as exploratory "
    "silver-rationale evidence, not independent human gold-standard "
    "performance."
)


In [ ]:
# =============================================================================
# CELL 29: FINAL SPARSE STANCE, CALIBRATION, AND ABSTENTION EVALUATION
# =============================================================================

from pathlib import Path
import gc
import json
import math
import random

import numpy as np
import pandas as pd
import torch

from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score,
    average_precision_score,
    balanced_accuracy_score,
    f1_score,
    log_loss,
    matthews_corrcoef,
    precision_recall_fscore_support,
    roc_auc_score,
)
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler, label_binarize


# -----------------------------------------------------------------------------
# Required objects from Cells 26--28
# -----------------------------------------------------------------------------

required_objects = [
    "DATA_DIR",
    "AMJEPA_DIR",
    "STANCE_CLASSES",
    "ASPECT_ORDER",
    "C_GRID",
    "stance_table",
    "y_stance",
    "sample_weights",
    "split_masks",
    "frozen_representations",
    "aspect_one_hot",
    "window_to_tensor_index",
    "attention_mask_stance",
    "feature_sets",
    "membership",
    "membership_groups",
    "aspect_to_index_selector",
    "SELECTOR_SEEDS",
    "EVIDENCE_BUDGETS",
    "DUAL_FEATURE_SETS",
    "DUAL_OUTPUT_DIR",
    "DUAL_CHANNELS",
    "DUAL_HIDDEN_DIMENSION",
    "DualAspectConditionedSelector",
    "predict_dual_scores",
    "align_probabilities",
    "fit_temperature",
    "apply_temperature",
    "expected_calibration_error",
    "multiclass_brier",
]

missing_objects = [
    name
    for name in required_objects
    if name not in globals()
]

if missing_objects:
    raise NameError(
        "Missing objects required by Cell 29:\n"
        + "\n".join(missing_objects)
        + "\n\nRun Cells 26, 27, and 28 first."
    )


# -----------------------------------------------------------------------------
# Configuration
# -----------------------------------------------------------------------------

FINAL_OUTPUT_DIR = (
    Path(AMJEPA_DIR)
    / "silver_final_stance"
)

FINAL_OUTPUT_DIR.mkdir(
    parents=True,
    exist_ok=True,
)

FINAL_SEEDS = list(SELECTOR_SEEDS)
FINAL_BUDGETS = sorted(
    {
        int(value)
        for value in EVIDENCE_BUDGETS
    }
)

FINAL_TARGET_COVERAGES = [
    1.00,
    0.90,
    0.80,
    0.70,
]

FINAL_METHODS = [
    "dense_am_jepa",
    "dual_text_only",
    "dual_full_multimodal",
]

DEEPSEEK_TARGET_COVERAGE = 0.80
FINAL_DEVICE = torch.device(
    "cuda"
    if torch.cuda.is_available()
    else "cpu"
)


def set_final_seed(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)

    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)


def build_final_probe(C_value, seed):
    return Pipeline(
        steps=[
            (
                "standardizer",
                StandardScaler(),
            ),
            (
                "classifier",
                LogisticRegression(
                    C=float(C_value),
                    class_weight="balanced",
                    solver="lbfgs",
                    max_iter=10000,
                    random_state=int(seed),
                ),
            ),
        ]
    )


def calculate_final_metrics(labels, probabilities):
    labels = np.asarray(
        labels,
        dtype=np.int64,
    )

    probabilities = np.asarray(
        probabilities,
        dtype=np.float64,
    )

    predictions = probabilities.argmax(
        axis=1
    )

    one_hot = label_binarize(
        labels,
        classes=np.arange(
            len(STANCE_CLASSES)
        ),
    )

    result = {
        "samples": int(len(labels)),
        "accuracy": float(
            accuracy_score(
                labels,
                predictions,
            )
        ),
        "balanced_accuracy": float(
            balanced_accuracy_score(
                labels,
                predictions,
            )
        ),
        "macro_f1": float(
            f1_score(
                labels,
                predictions,
                average="macro",
                zero_division=0,
            )
        ),
        "weighted_f1": float(
            f1_score(
                labels,
                predictions,
                average="weighted",
                zero_division=0,
            )
        ),
        "mcc": float(
            matthews_corrcoef(
                labels,
                predictions,
            )
        ),
        "nll": float(
            log_loss(
                labels,
                probabilities,
                labels=np.arange(
                    len(STANCE_CLASSES)
                ),
            )
        ),
        "brier": float(
            multiclass_brier(
                labels,
                probabilities,
            )
        ),
        "ece": float(
            expected_calibration_error(
                labels,
                probabilities,
            )
        ),
    }

    try:
        result["macro_ovr_auroc"] = float(
            roc_auc_score(
                labels,
                probabilities,
                multi_class="ovr",
                average="macro",
                labels=np.arange(
                    len(STANCE_CLASSES)
                ),
            )
        )
    except ValueError:
        result["macro_ovr_auroc"] = np.nan

    try:
        result["macro_ovr_auprc"] = float(
            average_precision_score(
                one_hot,
                probabilities,
                average="macro",
            )
        )
    except ValueError:
        result["macro_ovr_auprc"] = np.nan

    return result


def normalized_entropy(probabilities):
    probabilities = np.asarray(
        probabilities,
        dtype=np.float64,
    )

    entropy = -np.sum(
        probabilities
        * np.log(
            np.clip(
                probabilities,
                1e-12,
                1.0,
            )
        ),
        axis=1,
    )

    return entropy / math.log(
        probabilities.shape[1]
    )


def confidence_threshold_for_coverage(
    validation_confidence,
    target_coverage,
):
    if target_coverage >= 1.0:
        return 0.0

    return float(
        np.quantile(
            validation_confidence,
            1.0 - target_coverage,
            method="lower",
        )
    )


# -----------------------------------------------------------------------------
# Construct candidate events for every stance-annotation unit
# -----------------------------------------------------------------------------

all_event_rows = []

for record_index, record in stance_table.reset_index(
    drop=True
).iterrows():
    window_id = str(
        record["window_id"]
    )

    if window_id not in membership_groups:
        raise ValueError(
            f"No membership rows were found for {window_id}."
        )

    tensor_index = int(
        window_to_tensor_index[
            window_id
        ]
    )

    group = membership_groups[
        window_id
    ]

    evidence_by_position = dict(
        zip(
            group["position"].astype(int),
            group["evidence_id"].astype(str),
        )
    )

    valid_positions = np.where(
        attention_mask_stance[
            tensor_index
        ]
    )[0]

    for position in valid_positions:
        if int(position) not in evidence_by_position:
            raise ValueError(
                "A valid tensor position has no evidence ID: "
                f"{window_id}, position={position}."
            )

        all_event_rows.append(
            {
                "record_index": int(record_index),
                "annotation_id": record[
                    "annotation_id"
                ],
                "window_id": window_id,
                "split": record["split"],
                "aspect": record["aspect"],
                "aspect_index": int(
                    aspect_to_index_selector[
                        record["aspect"]
                    ]
                ),
                "tensor_index": tensor_index,
                "position": int(position),
                "evidence_id": evidence_by_position[
                    int(position)
                ],
            }
        )

all_stance_events = pd.DataFrame(
    all_event_rows
)

if all_stance_events.empty:
    raise ValueError(
        "No candidate events were constructed for Cell 29."
    )

record_event_counts = (
    all_stance_events
    .groupby("record_index")
    .size()
)

if len(record_event_counts) != len(stance_table):
    raise AssertionError(
        "At least one stance record has no candidate events."
    )

if (
    all_stance_events
    .groupby("window_id")["split"]
    .nunique()
    > 1
).any():
    raise AssertionError(
        "A window crosses the fixed train/validation/test partitions."
    )


# -----------------------------------------------------------------------------
# Apply frozen dual-selector checkpoints to every annotation unit
# -----------------------------------------------------------------------------

event_feature_cache = {}
selector_score_cache = {}

event_tensor_indices = all_stance_events[
    "tensor_index"
].to_numpy(dtype=np.int64)

event_positions = all_stance_events[
    "position"
].to_numpy(dtype=np.int64)

event_aspects = all_stance_events[
    "aspect_index"
].to_numpy(dtype=np.int64)

dummy_channels = np.zeros(
    len(all_stance_events),
    dtype=np.int64,
)

for feature_name in DUAL_FEATURE_SETS:
    raw_event_features = feature_sets[
        feature_name
    ][
        event_tensor_indices,
        event_positions,
    ].astype(np.float32)

    event_feature_cache[
        feature_name
    ] = raw_event_features

    for seed in FINAL_SEEDS:
        checkpoint_path = (
            Path(DUAL_OUTPUT_DIR)
            / (
                f"dual_selector_{feature_name}_"
                f"seed_{seed}.pt"
            )
        )

        if not checkpoint_path.exists():
            raise FileNotFoundError(
                "Missing Cell 28 checkpoint: "
                f"{checkpoint_path}"
            )

        checkpoint = torch.load(
            checkpoint_path,
            map_location=FINAL_DEVICE,
            weights_only=False,
        )

        scaler_mean = np.asarray(
            checkpoint["scaler_mean"],
            dtype=np.float32,
        )

        scaler_scale = np.asarray(
            checkpoint["scaler_scale"],
            dtype=np.float32,
        )

        scaler_scale = np.where(
            scaler_scale > 0,
            scaler_scale,
            1.0,
        )

        scaled_event_features = (
            (
                raw_event_features
                - scaler_mean
            )
            / scaler_scale
        ).astype(np.float32)

        model = DualAspectConditionedSelector(
            input_dimension=int(
                checkpoint[
                    "input_dimension"
                ]
            ),
            number_of_aspects=len(
                ASPECT_ORDER
            ),
            hidden_dimension=int(
                checkpoint[
                    "hidden_dimension"
                ]
            ),
        ).to(FINAL_DEVICE)

        model.load_state_dict(
            checkpoint[
                "model_state_dict"
            ],
            strict=True,
        )

        _, all_logits = predict_dual_scores(
            model,
            scaled_event_features,
            event_aspects,
            dummy_channels,
        )

        all_probabilities = 1.0 / (
            1.0
            + np.exp(
                -np.clip(
                    all_logits,
                    -30.0,
                    30.0,
                )
            )
        )

        selector_score_cache[
            (
                feature_name,
                int(seed),
            )
        ] = all_probabilities.astype(
            np.float32
        )

        del model
        del checkpoint

        if torch.cuda.is_available():
            torch.cuda.empty_cache()


# -----------------------------------------------------------------------------
# Create sparse E+/E- stance representations
# -----------------------------------------------------------------------------

dense_context = np.concatenate(
    [
        frozen_representations[
            stance_table[
                "tensor_index"
            ].to_numpy(dtype=np.int64)
        ],
        aspect_one_hot,
    ],
    axis=1,
).astype(np.float32)

sparse_feature_cache = {}
selection_rows = []


def build_sparse_stance_features(
    feature_name,
    seed,
    budget,
    save_selections=False,
):
    cache_key = (
        feature_name,
        int(seed),
        int(budget),
    )

    if (
        cache_key in sparse_feature_cache
        and not save_selections
    ):
        return sparse_feature_cache[
            cache_key
        ]

    raw_features = event_feature_cache[
        feature_name
    ]

    scores = selector_score_cache[
        (
            feature_name,
            int(seed),
        )
    ]

    feature_rows = []

    for record_index in range(
        len(stance_table)
    ):
        event_indices = np.where(
            all_stance_events[
                "record_index"
            ].to_numpy(dtype=np.int64)
            == record_index
        )[0]

        local_features = raw_features[
            event_indices
        ]

        local_scores = scores[
            event_indices
        ]

        number_to_keep = min(
            int(budget),
            len(event_indices),
        )

        supporting_order = np.argsort(
            -local_scores[:, 0],
            kind="stable",
        )[:number_to_keep]

        contradicting_order = np.argsort(
            -local_scores[:, 1],
            kind="stable",
        )[:number_to_keep]

        supporting_features = local_features[
            supporting_order
        ].mean(axis=0)

        contradicting_features = local_features[
            contradicting_order
        ].mean(axis=0)

        supporting_scores = local_scores[
            supporting_order,
            0,
        ]

        contradicting_scores = local_scores[
            contradicting_order,
            1,
        ]

        overlap = len(
            set(
                supporting_order.tolist()
            )
            & set(
                contradicting_order.tolist()
            )
        ) / max(
            number_to_keep,
            1,
        )

        score_features = np.asarray(
            [
                supporting_scores.mean(),
                supporting_scores.max(),
                supporting_scores.std(),
                contradicting_scores.mean(),
                contradicting_scores.max(),
                contradicting_scores.std(),
                supporting_scores.mean()
                - contradicting_scores.mean(),
                overlap,
                number_to_keep
                / max(
                    len(event_indices),
                    1,
                ),
            ],
            dtype=np.float32,
        )

        feature_rows.append(
            np.concatenate(
                [
                    dense_context[
                        record_index
                    ],
                    supporting_features,
                    contradicting_features,
                    supporting_features
                    - contradicting_features,
                    score_features,
                ]
            )
        )

        if save_selections:
            record_events = all_stance_events.iloc[
                event_indices
            ]

            for channel_index, (
                channel_name,
                selected_order,
            ) in enumerate(
                [
                    (
                        "SUPPORTING",
                        supporting_order,
                    ),
                    (
                        "CONTRADICTING",
                        contradicting_order,
                    ),
                ]
            ):
                for rank, local_index in enumerate(
                    selected_order,
                    start=1,
                ):
                    selected_event = record_events.iloc[
                        int(local_index)
                    ]

                    selection_rows.append(
                        {
                            "method": (
                                "dual_"
                                + feature_name
                            ),
                            "feature_set": feature_name,
                            "seed": int(seed),
                            "budget": int(budget),
                            "record_index": int(
                                record_index
                            ),
                            "annotation_id": selected_event[
                                "annotation_id"
                            ],
                            "window_id": selected_event[
                                "window_id"
                            ],
                            "split": selected_event[
                                "split"
                            ],
                            "aspect": selected_event[
                                "aspect"
                            ],
                            "channel": channel_name,
                            "rank": int(rank),
                            "position": int(
                                selected_event[
                                    "position"
                                ]
                            ),
                            "evidence_id": selected_event[
                                "evidence_id"
                            ],
                            "selector_score": float(
                                local_scores[
                                    int(local_index),
                                    channel_index,
                                ]
                            ),
                        }
                    )

    features = np.stack(
        feature_rows
    ).astype(np.float32)

    sparse_feature_cache[
        cache_key
    ] = features

    return features


# -----------------------------------------------------------------------------
# Validation-only model and evidence-budget selection
# -----------------------------------------------------------------------------

train_mask = split_masks["train"]
validation_mask = split_masks[
    "validation"
]
test_mask = split_masks["test"]

validation_search_rows = []
selected_configuration_rows = []
test_metric_rows = []
per_class_rows = []
selective_rows = []
prediction_parts = []

for method in FINAL_METHODS:
    if method == "dense_am_jepa":
        feature_name = None
        candidate_budgets = [0]
    else:
        feature_name = method.replace(
            "dual_",
            "",
            1,
        )

        if feature_name not in DUAL_FEATURE_SETS:
            raise ValueError(
                f"Unsupported dual feature set: {feature_name}"
            )

        candidate_budgets = FINAL_BUDGETS

    for seed in FINAL_SEEDS:
        set_final_seed(seed)
        seed_search_rows = []

        for budget in candidate_budgets:
            if method == "dense_am_jepa":
                features = dense_context
            else:
                features = build_sparse_stance_features(
                    feature_name,
                    seed,
                    budget,
                )

            for C_value in C_GRID:
                probe = build_final_probe(
                    C_value,
                    seed,
                )

                probe.fit(
                    features[train_mask],
                    y_stance[train_mask],
                    classifier__sample_weight=
                        sample_weights[
                            train_mask
                        ],
                )

                validation_probabilities = (
                    align_probabilities(
                        probe.predict_proba(
                            features[
                                validation_mask
                            ]
                        ),
                        probe.named_steps[
                            "classifier"
                        ].classes_,
                    )
                )

                validation_metrics = (
                    calculate_final_metrics(
                        y_stance[
                            validation_mask
                        ],
                        validation_probabilities,
                    )
                )

                search_row = {
                    "method": method,
                    "feature_set": (
                        "frozen_am_jepa"
                        if feature_name is None
                        else feature_name
                    ),
                    "seed": int(seed),
                    "budget": int(budget),
                    "C": float(C_value),
                    **validation_metrics,
                }

                validation_search_rows.append(
                    search_row
                )

                seed_search_rows.append(
                    search_row
                )

        best_validation = (
            pd.DataFrame(
                seed_search_rows
            )
            .sort_values(
                by=[
                    "macro_f1",
                    "mcc",
                    "balanced_accuracy",
                    "macro_ovr_auroc",
                    "nll",
                    "budget",
                    "C",
                ],
                ascending=[
                    False,
                    False,
                    False,
                    False,
                    True,
                    True,
                    True,
                ],
            )
            .iloc[0]
        )

        selected_budget = int(
            best_validation[
                "budget"
            ]
        )

        selected_C = float(
            best_validation["C"]
        )

        if method == "dense_am_jepa":
            selected_features = dense_context
        else:
            selected_features = (
                build_sparse_stance_features(
                    feature_name,
                    seed,
                    selected_budget,
                    save_selections=True,
                )
            )

        final_probe = build_final_probe(
            selected_C,
            seed,
        )

        final_probe.fit(
            selected_features[
                train_mask
            ],
            y_stance[train_mask],
            classifier__sample_weight=
                sample_weights[train_mask],
        )

        validation_probabilities_raw = (
            align_probabilities(
                final_probe.predict_proba(
                    selected_features[
                        validation_mask
                    ]
                ),
                final_probe.named_steps[
                    "classifier"
                ].classes_,
            )
        )

        temperature = fit_temperature(
            y_stance[validation_mask],
            validation_probabilities_raw,
        )

        validation_probabilities = (
            apply_temperature(
                validation_probabilities_raw,
                temperature,
            )
        )

        test_probabilities_raw = (
            align_probabilities(
                final_probe.predict_proba(
                    selected_features[
                        test_mask
                    ]
                ),
                final_probe.named_steps[
                    "classifier"
                ].classes_,
            )
        )

        test_probabilities = apply_temperature(
            test_probabilities_raw,
            temperature,
        )

        calibrated_validation_metrics = (
            calculate_final_metrics(
                y_stance[validation_mask],
                validation_probabilities,
            )
        )

        selected_configuration_rows.append(
            {
                "method": method,
                "feature_set": (
                    "frozen_am_jepa"
                    if feature_name is None
                    else feature_name
                ),
                "seed": int(seed),
                "selected_budget": int(
                    selected_budget
                ),
                "selected_C": selected_C,
                "temperature": temperature,
                "validation_macro_f1": (
                    calibrated_validation_metrics[
                        "macro_f1"
                    ]
                ),
                "validation_mcc": (
                    calibrated_validation_metrics[
                        "mcc"
                    ]
                ),
                "validation_macro_ovr_auroc": (
                    calibrated_validation_metrics[
                        "macro_ovr_auroc"
                    ]
                ),
                "validation_macro_ovr_auprc": (
                    calibrated_validation_metrics[
                        "macro_ovr_auprc"
                    ]
                ),
                "validation_ece": (
                    calibrated_validation_metrics[
                        "ece"
                    ]
                ),
            }
        )

        evaluation_masks = {
            "all_test": np.ones(
                int(test_mask.sum()),
                dtype=bool,
            ),
            "sufficient_context_test": (
                ~stance_table.loc[
                    test_mask,
                    "insufficient_context",
                ].to_numpy(dtype=bool)
            ),
        }

        test_labels = y_stance[
            test_mask
        ]

        for evaluation_set, local_mask in (
            evaluation_masks.items()
        ):
            if local_mask.sum() == 0:
                continue

            metrics = calculate_final_metrics(
                test_labels[local_mask],
                test_probabilities[local_mask],
            )

            test_metric_rows.append(
                {
                    "method": method,
                    "feature_set": (
                        "frozen_am_jepa"
                        if feature_name is None
                        else feature_name
                    ),
                    "seed": int(seed),
                    "selected_budget": int(
                        selected_budget
                    ),
                    "selected_C": selected_C,
                    "temperature": temperature,
                    "evaluation_set": evaluation_set,
                    **metrics,
                }
            )

            predictions = test_probabilities[
                local_mask
            ].argmax(axis=1)

            precision, recall, f1, support = (
                precision_recall_fscore_support(
                    test_labels[local_mask],
                    predictions,
                    labels=np.arange(
                        len(STANCE_CLASSES)
                    ),
                    zero_division=0,
                )
            )

            for class_index, class_name in enumerate(
                STANCE_CLASSES
            ):
                per_class_rows.append(
                    {
                        "method": method,
                        "seed": int(seed),
                        "evaluation_set": evaluation_set,
                        "stance": class_name,
                        "precision": float(
                            precision[
                                class_index
                            ]
                        ),
                        "recall": float(
                            recall[
                                class_index
                            ]
                        ),
                        "f1": float(
                            f1[class_index]
                        ),
                        "support": int(
                            support[
                                class_index
                            ]
                        ),
                    }
                )

        validation_confidence = (
            validation_probabilities.max(
                axis=1
            )
        )

        test_confidence = (
            test_probabilities.max(
                axis=1
            )
        )

        for target_coverage in (
            FINAL_TARGET_COVERAGES
        ):
            threshold = (
                confidence_threshold_for_coverage(
                    validation_confidence,
                    target_coverage,
                )
            )

            retained = (
                test_confidence
                >= threshold
            )

            if retained.sum() == 0:
                continue

            retained_metrics = (
                calculate_final_metrics(
                    test_labels[retained],
                    test_probabilities[
                        retained
                    ],
                )
            )

            selective_rows.append(
                {
                    "method": method,
                    "seed": int(seed),
                    "selected_budget": int(
                        selected_budget
                    ),
                    "target_validation_coverage": float(
                        target_coverage
                    ),
                    "confidence_threshold": threshold,
                    "test_coverage": float(
                        retained.mean()
                    ),
                    "abstention_rate": float(
                        1.0
                        - retained.mean()
                    ),
                    "selective_risk": float(
                        1.0
                        - retained_metrics[
                            "accuracy"
                        ]
                    ),
                    **retained_metrics,
                }
            )

        for split_name, local_mask, probabilities in [
            (
                "validation",
                validation_mask,
                validation_probabilities,
            ),
            (
                "test",
                test_mask,
                test_probabilities,
            ),
        ]:
            prediction_table = stance_table.loc[
                local_mask,
                [
                    "annotation_id",
                    "window_id",
                    "aspect",
                    "stance",
                    "insufficient_context",
                    "confidence",
                ],
            ].copy()

            prediction_table["split"] = (
                split_name
            )
            prediction_table["method"] = method
            prediction_table["seed"] = int(
                seed
            )
            prediction_table[
                "selected_budget"
            ] = int(selected_budget)
            prediction_table[
                "predicted_stance"
            ] = [
                STANCE_CLASSES[index]
                for index in probabilities.argmax(
                    axis=1
                )
            ]
            prediction_table[
                "predictive_confidence"
            ] = probabilities.max(axis=1)
            prediction_table[
                "predictive_entropy"
            ] = normalized_entropy(
                probabilities
            )

            for class_index, class_name in enumerate(
                STANCE_CLASSES
            ):
                prediction_table[
                    "probability_"
                    + class_name.lower()
                ] = probabilities[
                    :,
                    class_index,
                ]

            prediction_parts.append(
                prediction_table
            )

        del final_probe


# -----------------------------------------------------------------------------
# Assemble and aggregate five-seed results
# -----------------------------------------------------------------------------

validation_search = pd.DataFrame(
    validation_search_rows
)

selected_configurations = pd.DataFrame(
    selected_configuration_rows
)

test_seed_results = pd.DataFrame(
    test_metric_rows
)

per_class_results = pd.DataFrame(
    per_class_rows
)

selective_results = pd.DataFrame(
    selective_rows
)

prediction_results = pd.concat(
    prediction_parts,
    ignore_index=True,
)

aggregate_metric_names = [
    "accuracy",
    "balanced_accuracy",
    "macro_f1",
    "weighted_f1",
    "mcc",
    "macro_ovr_auroc",
    "macro_ovr_auprc",
    "nll",
    "brier",
    "ece",
]

aggregate_rows = []

for (
    method,
    evaluation_set,
), group in test_seed_results.groupby(
    [
        "method",
        "evaluation_set",
    ],
    sort=False,
):
    row = {
        "method": method,
        "evaluation_set": evaluation_set,
        "seeds": int(
            group["seed"].nunique()
        ),
        "selected_budget_mode": int(
            group[
                "selected_budget"
            ].mode().iloc[0]
        ),
    }

    for metric in aggregate_metric_names:
        values = group[metric].to_numpy(
            dtype=np.float64
        )

        row[
            metric + "_mean"
        ] = float(values.mean())

        row[
            metric + "_sd"
        ] = float(
            values.std(ddof=1)
            if len(values) > 1
            else 0.0
        )

    aggregate_rows.append(row)

aggregate_results = pd.DataFrame(
    aggregate_rows
)

selective_aggregate_rows = []

for (
    method,
    target_coverage,
), group in selective_results.groupby(
    [
        "method",
        "target_validation_coverage",
    ],
    sort=False,
):
    row = {
        "method": method,
        "target_validation_coverage": float(
            target_coverage
        ),
        "seeds": int(
            group["seed"].nunique()
        ),
    }

    for metric in [
        "test_coverage",
        "abstention_rate",
        "accuracy",
        "macro_f1",
        "mcc",
        "selective_risk",
        "ece",
    ]:
        values = group[metric].to_numpy(
            dtype=np.float64
        )

        row[metric + "_mean"] = float(
            values.mean()
        )

        row[metric + "_sd"] = float(
            values.std(ddof=1)
            if len(values) > 1
            else 0.0
        )

    selective_aggregate_rows.append(row)

selective_aggregate = pd.DataFrame(
    selective_aggregate_rows
)


# -----------------------------------------------------------------------------
# Select the DeepSeek routing configuration using validation results only
# -----------------------------------------------------------------------------

dual_validation_summary = (
    selected_configurations[
        selected_configurations[
            "method"
        ].str.startswith("dual_")
    ]
    .groupby("method", as_index=False)
    .agg(
        validation_macro_f1_mean=(
            "validation_macro_f1",
            "mean",
        ),
        validation_mcc_mean=(
            "validation_mcc",
            "mean",
        ),
        validation_auroc_mean=(
            "validation_macro_ovr_auroc",
            "mean",
        ),
        validation_ece_mean=(
            "validation_ece",
            "mean",
        ),
    )
)

primary_validation_row = (
    dual_validation_summary
    .sort_values(
        by=[
            "validation_macro_f1_mean",
            "validation_mcc_mean",
            "validation_auroc_mean",
            "validation_ece_mean",
            "method",
        ],
        ascending=[
            False,
            False,
            False,
            True,
            True,
        ],
    )
    .iloc[0]
)

primary_method = str(
    primary_validation_row[
        "method"
    ]
)

primary_feature_set = primary_method.replace(
    "dual_",
    "",
    1,
)

primary_seed_configs = selected_configurations[
    selected_configurations["method"]
    == primary_method
].copy()

primary_budget = int(
    primary_seed_configs[
        "selected_budget"
    ].mode().iloc[0]
)


def build_ensemble_table(split_name):
    subset = prediction_results[
        (
            prediction_results["method"]
            == primary_method
        )
        & (
            prediction_results["split"]
            == split_name
        )
    ].copy()

    probability_columns = [
        "probability_"
        + class_name.lower()
        for class_name in STANCE_CLASSES
    ]

    metadata_columns = [
        "annotation_id",
        "window_id",
        "aspect",
        "stance",
        "insufficient_context",
        "confidence",
    ]

    metadata = (
        subset[
            metadata_columns
        ]
        .drop_duplicates(
            "annotation_id"
        )
        .set_index("annotation_id")
    )

    probabilities = (
        subset
        .groupby("annotation_id")[
            probability_columns
        ]
        .mean()
    )

    ensemble = metadata.join(
        probabilities,
        how="inner",
    ).reset_index()

    probability_array = ensemble[
        probability_columns
    ].to_numpy(dtype=np.float64)

    ensemble[
        "predicted_stance"
    ] = [
        STANCE_CLASSES[index]
        for index in probability_array.argmax(
            axis=1
        )
    ]

    ensemble[
        "predictive_confidence"
    ] = probability_array.max(axis=1)

    ensemble[
        "predictive_entropy"
    ] = normalized_entropy(
        probability_array
    )

    ensemble["split"] = split_name

    return ensemble


validation_ensemble = build_ensemble_table(
    "validation"
)

test_ensemble = build_ensemble_table(
    "test"
)

deepseek_threshold = (
    confidence_threshold_for_coverage(
        validation_ensemble[
            "predictive_confidence"
        ].to_numpy(dtype=np.float64),
        DEEPSEEK_TARGET_COVERAGE,
    )
)

test_ensemble["route_to_deepseek"] = (
    test_ensemble[
        "predictive_confidence"
    ]
    < deepseek_threshold
)


# -----------------------------------------------------------------------------
# Build consensus sparse evidence IDs for the label-blind DeepSeek queue
# -----------------------------------------------------------------------------

primary_scores = np.mean(
    np.stack(
        [
            selector_score_cache[
                (
                    primary_feature_set,
                    int(seed),
                )
            ]
            for seed in FINAL_SEEDS
        ],
        axis=0,
    ),
    axis=0,
)

consensus_selection_rows = []

for record_index in range(
    len(stance_table)
):
    event_indices = np.where(
        all_stance_events[
            "record_index"
        ].to_numpy(dtype=np.int64)
        == record_index
    )[0]

    local_scores = primary_scores[
        event_indices
    ]

    record_events = all_stance_events.iloc[
        event_indices
    ]

    number_to_keep = min(
        primary_budget,
        len(event_indices),
    )

    for channel_index, channel_name in enumerate(
        DUAL_CHANNELS
    ):
        order = np.argsort(
            -local_scores[
                :,
                channel_index,
            ],
            kind="stable",
        )[:number_to_keep]

        for rank, local_index in enumerate(
            order,
            start=1,
        ):
            event = record_events.iloc[
                int(local_index)
            ]

            consensus_selection_rows.append(
                {
                    "record_index": int(
                        record_index
                    ),
                    "annotation_id": event[
                        "annotation_id"
                    ],
                    "window_id": event[
                        "window_id"
                    ],
                    "split": event["split"],
                    "aspect": event["aspect"],
                    "channel": channel_name,
                    "rank": int(rank),
                    "position": int(
                        event["position"]
                    ),
                    "evidence_id": event[
                        "evidence_id"
                    ],
                    "selector_score": float(
                        local_scores[
                            int(local_index),
                            channel_index,
                        ]
                    ),
                }
            )

consensus_selections = pd.DataFrame(
    consensus_selection_rows
)

test_consensus = consensus_selections[
    consensus_selections["split"]
    == "test"
].copy()

evidence_lists = (
    test_consensus
    .sort_values(
        [
            "annotation_id",
            "channel",
            "rank",
        ]
    )
    .groupby(
        [
            "annotation_id",
            "channel",
        ]
    )["evidence_id"]
    .apply(
        lambda values: ";".join(
            values.astype(str)
        )
    )
    .unstack("channel")
    .reset_index()
    .rename(
        columns={
            "SUPPORTING": (
                "selected_supporting_evidence_ids"
            ),
            "CONTRADICTING": (
                "selected_contradicting_evidence_ids"
            ),
        }
    )
)

deepseek_blind_queue = (
    test_ensemble[
        test_ensemble[
            "route_to_deepseek"
        ]
    ][
        [
            "annotation_id",
            "window_id",
            "aspect",
            "predicted_stance",
            "predictive_confidence",
            "predictive_entropy",
        ]
    ]
    .merge(
        evidence_lists,
        on="annotation_id",
        how="left",
        validate="one_to_one",
    )
)

deepseek_blind_queue.insert(
    0,
    "request_id",
    [
        f"DEEPSEEK-{index:04d}"
        for index in range(
            1,
            len(deepseek_blind_queue) + 1,
        )
    ],
)

# Deliberately exclude the reference stance and all correctness fields.
for forbidden_column in [
    "stance",
    "gold_stance",
    "correct",
    "label",
]:
    if forbidden_column in deepseek_blind_queue.columns:
        raise AssertionError(
            "The label-blind DeepSeek queue contains a forbidden field: "
            f"{forbidden_column}"
        )

deepseek_evaluation_key = (
    test_ensemble[
        test_ensemble[
            "route_to_deepseek"
        ]
    ][
        [
            "annotation_id",
            "stance",
            "insufficient_context",
            "confidence",
        ]
    ]
    .merge(
        deepseek_blind_queue[
            [
                "request_id",
                "annotation_id",
            ]
        ],
        on="annotation_id",
        how="inner",
        validate="one_to_one",
    )
)


# -----------------------------------------------------------------------------
# Save all outputs
# -----------------------------------------------------------------------------

validation_search.to_csv(
    FINAL_OUTPUT_DIR
    / "stance_validation_search.csv",
    index=False,
)

selected_configurations.to_csv(
    FINAL_OUTPUT_DIR
    / "stance_selected_configurations.csv",
    index=False,
)

test_seed_results.to_csv(
    FINAL_OUTPUT_DIR
    / "stance_test_seed_results.csv",
    index=False,
)

aggregate_results.to_csv(
    FINAL_OUTPUT_DIR
    / "stance_test_aggregate.csv",
    index=False,
)

per_class_results.to_csv(
    FINAL_OUTPUT_DIR
    / "stance_per_class_results.csv",
    index=False,
)

selective_results.to_csv(
    FINAL_OUTPUT_DIR
    / "stance_selective_seed_results.csv",
    index=False,
)

selective_aggregate.to_csv(
    FINAL_OUTPUT_DIR
    / "stance_selective_aggregate.csv",
    index=False,
)

prediction_results.to_csv(
    FINAL_OUTPUT_DIR
    / "stance_validation_test_predictions.csv",
    index=False,
)

pd.DataFrame(selection_rows).to_csv(
    FINAL_OUTPUT_DIR
    / "stance_selected_evidence_by_seed.csv",
    index=False,
)

consensus_selections.to_csv(
    FINAL_OUTPUT_DIR
    / "stance_consensus_selected_evidence.csv",
    index=False,
)

validation_ensemble.to_csv(
    FINAL_OUTPUT_DIR
    / "primary_validation_ensemble.csv",
    index=False,
)

test_ensemble.to_csv(
    FINAL_OUTPUT_DIR
    / "primary_test_ensemble_with_reference.csv",
    index=False,
)

deepseek_blind_queue.to_csv(
    FINAL_OUTPUT_DIR
    / "deepseek_blind_request_queue.csv",
    index=False,
)

deepseek_evaluation_key.to_csv(
    FINAL_OUTPUT_DIR
    / "deepseek_evaluation_key_DO_NOT_LOAD_BEFORE_RESPONSES.csv",
    index=False,
)

manifest = {
    "study_type": (
        "exploratory_ai_assisted_silver_label_evaluation"
    ),
    "stance_classes": STANCE_CLASSES,
    "aspect_order": ASPECT_ORDER,
    "seeds": FINAL_SEEDS,
    "candidate_budgets": FINAL_BUDGETS,
    "methods": FINAL_METHODS,
    "selection_policy": (
        "budget and C selected independently per seed using validation "
        "macro-F1, MCC, balanced accuracy, AUROC, NLL, then smaller "
        "budget and C"
    ),
    "calibration_policy": (
        "temperature fitted on validation predictions only"
    ),
    "abstention_policy": (
        "confidence thresholds selected on validation for fixed target "
        "coverage; test retained when confidence >= threshold"
    ),
    "primary_method_selection": (
        "dual method selected using five-seed validation means only"
    ),
    "primary_method": primary_method,
    "primary_feature_set": primary_feature_set,
    "primary_budget_mode": primary_budget,
    "deepseek_target_validation_coverage": (
        DEEPSEEK_TARGET_COVERAGE
    ),
    "deepseek_confidence_threshold": (
        deepseek_threshold
    ),
    "deepseek_requests": int(
        len(deepseek_blind_queue)
    ),
    "deepseek_queue_is_label_blind": True,
    "test_used_for_configuration_selection": False,
    "limitations": [
        "The 240-unit pilot uses AI-assisted single-reviewer silver labels.",
        "The rationale-supervised dual selectors use only 94 annotated units.",
        "Results are exploratory and are not independent human gold-standard estimates.",
    ],
}

with open(
    FINAL_OUTPUT_DIR
    / "stance_experiment_manifest.json",
    "w",
    encoding="utf-8",
) as file:
    json.dump(
        manifest,
        file,
        indent=2,
    )


# -----------------------------------------------------------------------------
# Display concise primary results
# -----------------------------------------------------------------------------

print("=" * 80)
print("SPARSESENTINEL-APT FINAL STANCE AND UNCERTAINTY EVALUATION")
print("=" * 80)

print(
    "\nLabel status: AI-assisted single-reviewer silver pilot "
    "(not an independent human gold standard)."
)

print("\nValidation-selected configurations:")

display(
    selected_configurations[
        [
            "method",
            "seed",
            "selected_budget",
            "selected_C",
            "temperature",
            "validation_macro_f1",
            "validation_mcc",
            "validation_ece",
        ]
    ].round(6)
)

print("\nFive-seed test summary:")

display(
    aggregate_results[
        aggregate_results[
            "evaluation_set"
        ] == "all_test"
    ][
        [
            "method",
            "selected_budget_mode",
            "accuracy_mean",
            "accuracy_sd",
            "macro_f1_mean",
            "macro_f1_sd",
            "mcc_mean",
            "mcc_sd",
            "macro_ovr_auroc_mean",
            "macro_ovr_auprc_mean",
            "ece_mean",
            "brier_mean",
        ]
    ]
    .sort_values(
        "macro_f1_mean",
        ascending=False,
    )
    .round(6)
)

print("\nSelective prediction summary:")

display(
    selective_aggregate[
        [
            "method",
            "target_validation_coverage",
            "test_coverage_mean",
            "accuracy_mean",
            "macro_f1_mean",
            "mcc_mean",
            "selective_risk_mean",
            "ece_mean",
        ]
    ]
    .sort_values(
        [
            "method",
            "target_validation_coverage",
        ],
        ascending=[
            True,
            False,
        ],
    )
    .round(6)
)

print("\nDeepSeek routing package:")
print("Primary method:", primary_method)
print("Evidence budget:", primary_budget)
print(
    "Validation-selected confidence threshold:",
    round(deepseek_threshold, 6),
)
print(
    "Test cases routed to DeepSeek:",
    len(deepseek_blind_queue),
    "/",
    len(test_ensemble),
)

print("\nSaved Cell 29 results to:")
print(FINAL_OUTPUT_DIR)

print(
    "\nCell 29 completed successfully. The DeepSeek request queue is "
    "label-blind; do not load the evaluation key until all API responses "
    "have been cached."
)

gc.collect()

if torch.cuda.is_available():
    torch.cuda.empty_cache()


In [ ]:
# =============================================================================
# CELL 30: FINAL ABLATION, ROBUSTNESS, AND DETERMINISTIC REPORTING
# =============================================================================

from pathlib import Path
import hashlib
import json

import numpy as np
import pandas as pd


# -----------------------------------------------------------------------------
# Configuration
# -----------------------------------------------------------------------------

if "DATA_DIR" not in globals():
    DATA_DIR = Path(
        r"."
    )
else:
    DATA_DIR = Path(DATA_DIR)

OUTPUT_DIR = DATA_DIR / "outputs"
AMJEPA_DIR = OUTPUT_DIR / "am_jepa_experiment"

OBJECTIVE_DIR = AMJEPA_DIR / "objective_ablation"
ROBUSTNESS_DIR = AMJEPA_DIR / "missing_modality_robustness"
SELECTOR_DIR = AMJEPA_DIR / "silver_sparse_selector"
DUAL_DIR = AMJEPA_DIR / "silver_dual_selector"
FINAL_DIR = AMJEPA_DIR / "silver_final_stance"

CELL30_DIR = AMJEPA_DIR / "final_ablation_robustness"
CELL30_DIR.mkdir(parents=True, exist_ok=True)

TARGET_VALIDATION_COVERAGE = 0.80
EXPECTED_TEST_RECORDS = 89
EXPECTED_CANDIDATES_PER_RECORD = 32
REPORT_SCHEMA_VERSION = "1.0"


# -----------------------------------------------------------------------------
# Helpers
# -----------------------------------------------------------------------------

def require_file(path):
    path = Path(path)
    if not path.exists():
        raise FileNotFoundError(
            f"Required preserved output is missing:\n{path}\n"
            "Run the relevant completed cell or restore its saved output."
        )
    return path


def read_csv(path, required_columns=None):
    path = require_file(path)
    table = pd.read_csv(path)

    if required_columns is not None:
        missing = sorted(set(required_columns) - set(table.columns))
        if missing:
            raise ValueError(
                f"{path.name} is missing columns: {missing}"
            )

    return table


def confidence_threshold_for_fixed_coverage(confidences, target_coverage):
    """Return a deterministic validation-only confidence threshold."""
    values = np.asarray(confidences, dtype=np.float64)
    values = values[np.isfinite(values)]

    if len(values) == 0:
        raise ValueError("No finite validation confidences were found.")

    number_to_retain = int(
        np.ceil(float(target_coverage) * len(values))
    )
    number_to_retain = min(max(number_to_retain, 1), len(values))

    return float(
        np.sort(values)[::-1][number_to_retain - 1]
    )


def split_identifiers(value):
    if pd.isna(value):
        return []

    return [
        item.strip()
        for item in str(value).split(";")
        if item.strip()
    ]


def stable_report_id(annotation_id):
    digest = hashlib.sha256(
        str(annotation_id).encode("utf-8")
    ).hexdigest()[:16].upper()
    return f"SSR-{digest}"


def validate_identifier_list(value, ledger):
    identifiers = split_identifiers(value)
    invalid = [item for item in identifiers if item not in ledger]
    return len(invalid) == 0, invalid


# -----------------------------------------------------------------------------
# Load only preserved experiment outputs
# -----------------------------------------------------------------------------

objective_results = read_csv(
    OBJECTIVE_DIR / "am_jepa_objective_ablation.csv",
    [
        "variant",
        "campaign_f1",
        "campaign_auroc",
        "core3_macro_f1",
    ],
)

missing_modality_detailed = read_csv(
    ROBUSTNESS_DIR / "missing_modality_detailed_results.csv",
    [
        "variant",
        "seed",
        "scenario",
        "campaign_f1",
        "campaign_auroc",
        "core3_macro_f1",
        "campaign_f1_drop",
        "core3_f1_drop",
    ],
)

selector_primary = read_csv(
    SELECTOR_DIR / "selector_primary_summary.csv"
)

dual_delta_summary = read_csv(
    DUAL_DIR / "dual_single_delta_summary.csv"
)

stance_aggregate = read_csv(
    FINAL_DIR / "stance_test_aggregate.csv",
    [
        "method",
        "evaluation_set",
        "macro_f1_mean",
        "mcc_mean",
        "macro_ovr_auroc_mean",
        "ece_mean",
    ],
)

selective_aggregate = read_csv(
    FINAL_DIR / "stance_selective_aggregate.csv",
    [
        "method",
        "target_validation_coverage",
        "test_coverage_mean",
        "accuracy_mean",
        "macro_f1_mean",
        "mcc_mean",
        "selective_risk_mean",
    ],
)

validation_ensemble = read_csv(
    FINAL_DIR / "primary_validation_ensemble.csv",
    [
        "annotation_id",
        "predictive_confidence",
        "predicted_stance",
    ],
)

test_ensemble = read_csv(
    FINAL_DIR / "primary_test_ensemble_with_reference.csv",
    [
        "annotation_id",
        "window_id",
        "aspect",
        "predicted_stance",
        "predictive_confidence",
        "predictive_entropy",
    ],
)

consensus_evidence = read_csv(
    FINAL_DIR / "stance_consensus_selected_evidence.csv",
    [
        "annotation_id",
        "split",
        "channel",
        "rank",
        "evidence_id",
    ],
)

if len(test_ensemble) != EXPECTED_TEST_RECORDS:
    raise AssertionError(
        f"Expected {EXPECTED_TEST_RECORDS} test records, "
        f"but found {len(test_ensemble)}."
    )


# -----------------------------------------------------------------------------
# A. Objective ablation
# -----------------------------------------------------------------------------

objective_columns = [
    column
    for column in [
        "variant",
        "epochs",
        "campaign_f1",
        "campaign_mcc",
        "campaign_auroc",
        "campaign_auprc",
        "core3_macro_f1",
        "core3_macro_mcc",
        "mean_latent_std",
        "near_constant_dimensions",
    ]
    if column in objective_results.columns
]

objective_ablation_table = (
    objective_results[objective_columns]
    .sort_values("campaign_f1", ascending=False)
    .reset_index(drop=True)
)


# -----------------------------------------------------------------------------
# B. Five-seed missing-modality robustness
# -----------------------------------------------------------------------------

full_robustness = missing_modality_detailed[
    missing_modality_detailed["variant"] == "full_am_jepa"
].copy()

if full_robustness["seed"].nunique() != 5:
    raise AssertionError(
        "Missing-modality robustness does not contain five AM-JEPA seeds."
    )

robustness_rows = []

for scenario, group in full_robustness.groupby(
    "scenario",
    sort=False,
):
    row = {
        "scenario": scenario,
        "seeds": int(group["seed"].nunique()),
    }

    if "realized_drop_rate" in group.columns:
        row["realized_drop_rate"] = float(
            group["realized_drop_rate"].mean()
        )

    for metric in [
        "campaign_f1",
        "campaign_auroc",
        "core3_macro_f1",
        "campaign_f1_drop",
        "core3_f1_drop",
    ]:
        values = group[metric].to_numpy(dtype=np.float64)
        row[metric + "_mean"] = float(values.mean())
        row[metric + "_sd"] = float(values.std(ddof=1))

    robustness_rows.append(row)

missing_modality_summary = pd.DataFrame(robustness_rows)

scenario_order = [
    "complete",
    "missing_network",
    "missing_temporal",
    "missing_text",
    "network_only",
    "temporal_only",
    "text_only",
    "random_drop_25pct",
    "random_drop_50pct",
    "random_drop_75pct",
]

missing_modality_summary["scenario"] = pd.Categorical(
    missing_modality_summary["scenario"],
    categories=scenario_order,
    ordered=True,
)

missing_modality_summary = (
    missing_modality_summary
    .sort_values("scenario")
    .reset_index(drop=True)
)


# -----------------------------------------------------------------------------
# C. Sparse-selector and final stance ablations
# -----------------------------------------------------------------------------

stance_ablation_table = (
    stance_aggregate[
        stance_aggregate["evaluation_set"] == "all_test"
    ]
    .copy()
    .sort_values("macro_f1_mean", ascending=False)
    .reset_index(drop=True)
)

selective_robustness_table = (
    selective_aggregate
    .copy()
    .sort_values(
        [
            "method",
            "target_validation_coverage",
        ],
        ascending=[True, False],
    )
    .reset_index(drop=True)
)


# -----------------------------------------------------------------------------
# D. Deterministic evidence-linked reports and abstention
# -----------------------------------------------------------------------------

confidence_threshold = confidence_threshold_for_fixed_coverage(
    validation_ensemble["predictive_confidence"],
    TARGET_VALIDATION_COVERAGE,
)

test_reports = test_ensemble.drop(
    columns=[
        column
        for column in test_ensemble.columns
        if column.lower() in {
            "route_to_deepseek",
            "deepseek_request",
        }
    ],
    errors="ignore",
).copy()

test_evidence = consensus_evidence[
    consensus_evidence["split"] == "test"
].copy()

evidence_lists = (
    test_evidence
    .sort_values(
        [
            "annotation_id",
            "channel",
            "rank",
        ]
    )
    .groupby(
        [
            "annotation_id",
            "channel",
        ]
    )["evidence_id"]
    .apply(lambda values: ";".join(values.astype(str)))
    .unstack("channel")
    .reset_index()
    .rename(
        columns={
            "SUPPORTING": "supporting_evidence_ids",
            "CONTRADICTING": "contradicting_evidence_ids",
        }
    )
)

for column in [
    "supporting_evidence_ids",
    "contradicting_evidence_ids",
]:
    if column not in evidence_lists.columns:
        evidence_lists[column] = ""

test_reports = test_reports.merge(
    evidence_lists,
    on="annotation_id",
    how="left",
    validate="one_to_one",
)

test_reports[
    [
        "supporting_evidence_ids",
        "contradicting_evidence_ids",
    ]
] = test_reports[
    [
        "supporting_evidence_ids",
        "contradicting_evidence_ids",
    ]
].fillna("")

test_reports["abstained"] = (
    test_reports["predictive_confidence"]
    < confidence_threshold
)

test_reports["decision"] = np.where(
    test_reports["abstained"],
    "ABSTAIN",
    "REPORT_STANCE",
)

test_reports["reason_code"] = np.where(
    test_reports["abstained"],
    "LOW_VALIDATION_CALIBRATED_CONFIDENCE",
    "SUFFICIENT_VALIDATION_CALIBRATED_CONFIDENCE",
)

test_reports["report_id"] = test_reports[
    "annotation_id"
].map(stable_report_id)

test_reports["report_schema_version"] = REPORT_SCHEMA_VERSION
test_reports["confidence_threshold"] = confidence_threshold
test_reports["target_validation_coverage"] = TARGET_VALIDATION_COVERAGE


# Build the immutable identifier ledger from the complete candidate-event table.
if "all_stance_events" in globals():
    evidence_ledger = set(
        all_stance_events["evidence_id"].astype(str)
    )
else:
    # Safe fallback for a restarted kernel: validate against every identifier
    # preserved in the consensus-evidence file.
    evidence_ledger = set(
        consensus_evidence["evidence_id"].astype(str)
    )

support_validity = []
contradiction_validity = []
invalid_identifier_lists = []

for row in test_reports.itertuples(index=False):
    support_valid, invalid_support = validate_identifier_list(
        row.supporting_evidence_ids,
        evidence_ledger,
    )
    contradiction_valid, invalid_contradiction = validate_identifier_list(
        row.contradicting_evidence_ids,
        evidence_ledger,
    )

    support_validity.append(support_valid)
    contradiction_validity.append(contradiction_valid)
    invalid_identifier_lists.append(
        ";".join(invalid_support + invalid_contradiction)
    )

test_reports["supporting_ids_valid"] = support_validity
test_reports["contradicting_ids_valid"] = contradiction_validity
test_reports["invalid_evidence_ids"] = invalid_identifier_lists
test_reports["identifier_integrity_passed"] = (
    test_reports["supporting_ids_valid"]
    & test_reports["contradicting_ids_valid"]
)

# Fail closed: any ledger violation becomes an abstention.
integrity_failure = ~test_reports["identifier_integrity_passed"]
test_reports.loc[integrity_failure, "abstained"] = True
test_reports.loc[integrity_failure, "decision"] = "ABSTAIN"
test_reports.loc[
    integrity_failure,
    "reason_code",
] = "EVIDENCE_IDENTIFIER_INTEGRITY_FAILURE"


# -----------------------------------------------------------------------------
# E. Integrity and tamper-detection tests
# -----------------------------------------------------------------------------

required_report_columns = {
    "report_id",
    "annotation_id",
    "window_id",
    "aspect",
    "predicted_stance",
    "predictive_confidence",
    "predictive_entropy",
    "decision",
    "reason_code",
    "supporting_evidence_ids",
    "contradicting_evidence_ids",
    "identifier_integrity_passed",
}

schema_passed = required_report_columns.issubset(test_reports.columns)
unique_report_ids_passed = test_reports["report_id"].is_unique
finite_confidence_passed = np.isfinite(
    test_reports["predictive_confidence"]
).all()
finite_entropy_passed = np.isfinite(
    test_reports["predictive_entropy"]
).all()
identifier_integrity_passed = bool(
    test_reports["identifier_integrity_passed"].all()
)

# Deliberately inject one nonexistent identifier. The validator must reject it.
tampered_value = "NONEXISTENT-EVIDENCE-ID"
tamper_was_rejected = not validate_identifier_list(
    tampered_value,
    evidence_ledger,
)[0]

if not all(
    [
        schema_passed,
        unique_report_ids_passed,
        finite_confidence_passed,
        finite_entropy_passed,
        identifier_integrity_passed,
        tamper_was_rejected,
    ]
):
    raise AssertionError(
        "Deterministic report integrity verification failed."
    )

report_integrity_summary = pd.DataFrame(
    [
        {
            "records": int(len(test_reports)),
            "confidence_threshold": confidence_threshold,
            "target_validation_coverage": TARGET_VALIDATION_COVERAGE,
            "retained_records": int((~test_reports["abstained"]).sum()),
            "abstained_records": int(test_reports["abstained"].sum()),
            "realized_coverage": float((~test_reports["abstained"]).mean()),
            "schema_passed": bool(schema_passed),
            "unique_report_ids_passed": bool(unique_report_ids_passed),
            "finite_confidence_passed": bool(finite_confidence_passed),
            "finite_entropy_passed": bool(finite_entropy_passed),
            "identifier_integrity_passed": bool(identifier_integrity_passed),
            "tamper_detection_passed": bool(tamper_was_rejected),
            "external_verifier_used": False,
        }
    ]
)


# -----------------------------------------------------------------------------
# Save Cell 30 outputs
# -----------------------------------------------------------------------------

objective_ablation_table.to_csv(
    CELL30_DIR / "objective_ablation_publication_table.csv",
    index=False,
)

missing_modality_summary.to_csv(
    CELL30_DIR / "missing_modality_robustness_publication_table.csv",
    index=False,
)

selector_primary.to_csv(
    CELL30_DIR / "single_selector_ablation_publication_table.csv",
    index=False,
)

dual_delta_summary.to_csv(
    CELL30_DIR / "dual_selector_ablation_publication_table.csv",
    index=False,
)

stance_ablation_table.to_csv(
    CELL30_DIR / "stance_architecture_ablation_publication_table.csv",
    index=False,
)

selective_robustness_table.to_csv(
    CELL30_DIR / "selective_prediction_robustness_table.csv",
    index=False,
)

test_reports.to_csv(
    CELL30_DIR / "deterministic_evidence_linked_test_reports.csv",
    index=False,
)

report_integrity_summary.to_csv(
    CELL30_DIR / "deterministic_report_integrity_summary.csv",
    index=False,
)

cell30_manifest = {
    "cell": 30,
    "status": "completed",
    "policy": (
        "Synthesis of preserved Cells 22--29 outputs; no retraining, "
        "external verification, or post-hoc label correction."
    ),
    "target_validation_coverage": TARGET_VALIDATION_COVERAGE,
    "validation_selected_confidence_threshold": confidence_threshold,
    "test_records": int(len(test_reports)),
    "retained_records": int((~test_reports["abstained"]).sum()),
    "abstained_records": int(test_reports["abstained"].sum()),
    "report_schema_version": REPORT_SCHEMA_VERSION,
    "integrity_checks_passed": True,
    "label_limitation": (
        "AI-assisted single-reviewer silver pilot; not an independent "
        "human gold standard."
    ),
}

with open(
    CELL30_DIR / "cell30_manifest.json",
    "w",
    encoding="utf-8",
) as file:
    json.dump(cell30_manifest, file, indent=2)


# -----------------------------------------------------------------------------
# Display concise publication-facing output
# -----------------------------------------------------------------------------

print("=" * 80)
print("SPARSESENTINEL-APT FINAL ABLATION AND ROBUSTNESS")
print("=" * 80)

print("\nObjective ablation:")
display(objective_ablation_table.round(6))

print("\nFive-seed missing-modality robustness:")
display(missing_modality_summary.round(6))

print("\nFinal stance architecture ablation:")
display(stance_ablation_table.round(6))

print("\nSelective-prediction robustness:")
display(selective_robustness_table.round(6))

print("\nDeterministic report integrity:")
display(report_integrity_summary.round(6))

print("\nCell 30 completed successfully.")
print("No model was retrained and no external verifier was used.")
print("Saved outputs to:")
print(CELL30_DIR)


In [ ]:
# =============================================================================
# CELL 31: STATISTICS, INTEGRITY CHECKS, FIGURES, AND FINAL EXPORTS
# =============================================================================

from pathlib import Path
import hashlib
import json
import math
import zipfile

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patheffects as path_effects

from matplotlib.colors import TwoSlopeNorm
from matplotlib.lines import Line2D

from scipy.stats import wilcoxon


# -----------------------------------------------------------------------------
# Configuration
# -----------------------------------------------------------------------------

if "DATA_DIR" not in globals():
    DATA_DIR = Path(
        r"."
    )
else:
    DATA_DIR = Path(DATA_DIR)

OUTPUT_DIR = DATA_DIR / "outputs"
AMJEPA_DIR = OUTPUT_DIR / "am_jepa_experiment"

OBJECTIVE_DIR = AMJEPA_DIR / "objective_ablation"
ROBUSTNESS_DIR = AMJEPA_DIR / "missing_modality_robustness"
SELECTOR_DIR = AMJEPA_DIR / "silver_sparse_selector"
DUAL_DIR = AMJEPA_DIR / "silver_dual_selector"
FINAL_DIR = AMJEPA_DIR / "silver_final_stance"
CELL30_DIR = AMJEPA_DIR / "final_ablation_robustness"

CELL31_DIR = AMJEPA_DIR / "final_statistics_exports"
FIGURE_DIR = CELL31_DIR / "figures"
TABLE_DIR = CELL31_DIR / "tables"

CELL31_DIR.mkdir(parents=True, exist_ok=True)
FIGURE_DIR.mkdir(parents=True, exist_ok=True)
TABLE_DIR.mkdir(parents=True, exist_ok=True)

ALPHA = 0.05
FIGURE_DPI = 300
EXPECTED_SEEDS = [42, 52, 62, 72, 82]
EXPECTED_TEST_REPORTS = 89
EXPECTED_ABSTENTIONS = 12


# -----------------------------------------------------------------------------
# Helpers
# -----------------------------------------------------------------------------

def require_file(path):
    path = Path(path)
    if not path.exists():
        raise FileNotFoundError(
            f"Required output is missing:\n{path}"
        )
    return path


def load_csv(path, required=None):
    path = require_file(path)
    table = pd.read_csv(path)

    if required:
        missing = sorted(set(required) - set(table.columns))
        if missing:
            raise ValueError(
                f"{path.name} is missing columns: {missing}"
            )

    return table


def sha256_file(path):
    digest = hashlib.sha256()
    with open(path, "rb") as file:
        for block in iter(lambda: file.read(1024 * 1024), b""):
            digest.update(block)
    return digest.hexdigest()


def paired_effect_size(differences):
    differences = np.asarray(differences, dtype=np.float64)
    standard_deviation = differences.std(ddof=1)

    if len(differences) < 2 or standard_deviation == 0:
        return np.nan

    return float(differences.mean() / standard_deviation)


def exact_wilcoxon(differences):
    differences = np.asarray(differences, dtype=np.float64)
    differences = differences[np.isfinite(differences)]

    if len(differences) == 0 or np.allclose(differences, 0.0):
        return 0.0, 1.0

    result = wilcoxon(
        differences,
        zero_method="wilcox",
        alternative="two-sided",
        method="auto",
    )

    return float(result.statistic), float(result.pvalue)


def holm_adjust(p_values):
    values = np.asarray(p_values, dtype=np.float64)
    order = np.argsort(values)
    adjusted = np.empty(len(values), dtype=np.float64)
    running_maximum = 0.0
    number = len(values)

    for rank, index in enumerate(order):
        candidate = min(
            1.0,
            (number - rank) * values[index],
        )
        running_maximum = max(running_maximum, candidate)
        adjusted[index] = running_maximum

    return adjusted


def save_figure(fig, stem):
    png_path = FIGURE_DIR / f"{stem}.png"
    pdf_path = FIGURE_DIR / f"{stem}.pdf"
    fig.savefig(png_path, dpi=FIGURE_DPI, bbox_inches="tight")
    fig.savefig(pdf_path, bbox_inches="tight")
    plt.close(fig)
    return png_path, pdf_path


def write_latex_table(table, path, caption, label, columns=None):
    export = table.copy()
    if columns is not None:
        export = export[columns]

    latex = export.to_latex(
        index=False,
        escape=True,
        float_format=lambda value: f"{value:.4f}",
        caption=caption,
        label=label,
        position="t",
    )

    with open(path, "w", encoding="utf-8") as file:
        file.write(latex)


# -----------------------------------------------------------------------------
# Load frozen results
# -----------------------------------------------------------------------------

objective_raw = load_csv(
    OBJECTIVE_DIR / "am_jepa_objective_ablation.csv",
    ["variant", "campaign_f1", "core3_macro_f1"],
)

multiseed_summary = load_csv(
    OBJECTIVE_DIR / "am_jepa_full_multiseed_summary.csv",
    ["seed", "campaign_f1", "core3_macro_f1"],
)

missing_detailed = load_csv(
    ROBUSTNESS_DIR / "missing_modality_detailed_results.csv",
    [
        "variant",
        "seed",
        "scenario",
        "campaign_f1",
        "core3_macro_f1",
    ],
)

selector_primary = load_csv(
    SELECTOR_DIR / "selector_primary_summary.csv"
)

dual_seed_deltas = load_csv(
    DUAL_DIR / "dual_single_seed_deltas.csv",
    ["feature_set", "seed", "budget", "map_delta"],
)

dual_primary = load_csv(
    DUAL_DIR / "dual_selector_primary_summary.csv"
)

stance_seed = load_csv(
    FINAL_DIR / "stance_test_seed_results.csv",
    ["method", "seed", "evaluation_set", "macro_f1", "mcc"],
)

stance_aggregate = load_csv(
    FINAL_DIR / "stance_test_aggregate.csv"
)

selective_aggregate = load_csv(
    FINAL_DIR / "stance_selective_aggregate.csv",
    [
        "method",
        "target_validation_coverage",
        "test_coverage_mean",
        "macro_f1_mean",
        "selective_risk_mean",
    ],
)

test_reports = load_csv(
    CELL30_DIR / "deterministic_evidence_linked_test_reports.csv",
    [
        "report_id",
        "annotation_id",
        "decision",
        "reason_code",
        "abstained",
        "identifier_integrity_passed",
    ],
)

report_integrity = load_csv(
    CELL30_DIR / "deterministic_report_integrity_summary.csv"
)

cell30_manifest_path = require_file(
    CELL30_DIR / "cell30_manifest.json"
)

with open(cell30_manifest_path, encoding="utf-8") as file:
    cell30_manifest = json.load(file)


# -----------------------------------------------------------------------------
# Corrected seed-42 objective-ablation table
# -----------------------------------------------------------------------------

if "seed" in objective_raw.columns:
    objective_seed42 = objective_raw[
        objective_raw["seed"] == 42
    ].copy()
else:
    # The objective variants were written first, followed by four extra
    # full-AM-JEPA seeds. Preserve the first occurrence of each variant.
    objective_seed42 = objective_raw.drop_duplicates(
        "variant",
        keep="first",
    ).copy()

if objective_seed42["variant"].nunique() != 6:
    raise AssertionError(
        "The corrected objective ablation must contain six seed-42 variants."
    )

objective_seed42 = (
    objective_seed42
    .sort_values("campaign_f1", ascending=False)
    .reset_index(drop=True)
)

objective_seed42.to_csv(
    TABLE_DIR / "objective_ablation_seed42_corrected.csv",
    index=False,
)

multiseed_summary.to_csv(
    TABLE_DIR / "full_am_jepa_five_seed_results.csv",
    index=False,
)


# -----------------------------------------------------------------------------
# Statistical test 1: paired stance-method comparisons across five seeds
# -----------------------------------------------------------------------------

stance_all = stance_seed[
    stance_seed["evaluation_set"] == "all_test"
].copy()

stance_methods = [
    "dense_am_jepa",
    "dual_text_only",
    "dual_full_multimodal",
]

stance_comparisons = [
    ("dual_text_only", "dense_am_jepa"),
    ("dual_full_multimodal", "dense_am_jepa"),
    ("dual_text_only", "dual_full_multimodal"),
]

stance_test_rows = []

for metric in [
    "macro_f1",
    "mcc",
    "macro_ovr_auroc",
]:
    if metric not in stance_all.columns:
        continue

    pivot = stance_all.pivot(
        index="seed",
        columns="method",
        values=metric,
    )

    for method_a, method_b in stance_comparisons:
        if not {method_a, method_b}.issubset(pivot.columns):
            continue

        paired = pivot[[method_a, method_b]].dropna()
        differences = (
            paired[method_a] - paired[method_b]
        ).to_numpy(dtype=np.float64)
        statistic, p_value = exact_wilcoxon(differences)

        stance_test_rows.append(
            {
                "family": "stance_method",
                "metric": metric,
                "comparison": f"{method_a} - {method_b}",
                "pairs": len(differences),
                "mean_difference": float(differences.mean()),
                "median_difference": float(np.median(differences)),
                "cohen_dz": paired_effect_size(differences),
                "wilcoxon_statistic": statistic,
                "p_value": p_value,
            }
        )

stance_tests = pd.DataFrame(stance_test_rows)


# -----------------------------------------------------------------------------
# Statistical test 2: missing-modality degradation versus complete input
# -----------------------------------------------------------------------------

full_missing = missing_detailed[
    missing_detailed["variant"] == "full_am_jepa"
].copy()

missing_test_rows = []

for metric in ["campaign_f1", "core3_macro_f1"]:
    pivot = full_missing.pivot(
        index="seed",
        columns="scenario",
        values=metric,
    )

    if "complete" not in pivot.columns:
        raise AssertionError("Complete-input robustness reference is missing.")

    for scenario in pivot.columns:
        if scenario == "complete":
            continue

        paired = pivot[["complete", scenario]].dropna()
        differences = (
            paired[scenario] - paired["complete"]
        ).to_numpy(dtype=np.float64)
        statistic, p_value = exact_wilcoxon(differences)

        missing_test_rows.append(
            {
                "family": "missing_modality",
                "metric": metric,
                "comparison": f"{scenario} - complete",
                "pairs": len(differences),
                "mean_difference": float(differences.mean()),
                "median_difference": float(np.median(differences)),
                "cohen_dz": paired_effect_size(differences),
                "wilcoxon_statistic": statistic,
                "p_value": p_value,
            }
        )

missing_tests = pd.DataFrame(missing_test_rows)


# -----------------------------------------------------------------------------
# Statistical test 3: matched dual-versus-single selector deltas
# -----------------------------------------------------------------------------

selector_test_rows = []

for (feature_set, budget), group in dual_seed_deltas.groupby(
    ["feature_set", "budget"],
    sort=False,
):
    for metric in [
        "map_delta",
        "mrr_delta",
        "f1_at_k_delta",
        "ndcg_at_k_delta",
    ]:
        if metric not in group.columns:
            continue

        differences = group[metric].to_numpy(dtype=np.float64)
        statistic, p_value = exact_wilcoxon(differences)

        selector_test_rows.append(
            {
                "family": "dual_vs_single_selector",
                "metric": metric,
                "comparison": f"{feature_set}, K={int(budget)}",
                "pairs": len(differences),
                "mean_difference": float(differences.mean()),
                "median_difference": float(np.median(differences)),
                "cohen_dz": paired_effect_size(differences),
                "wilcoxon_statistic": statistic,
                "p_value": p_value,
            }
        )

selector_tests = pd.DataFrame(selector_test_rows)


# -----------------------------------------------------------------------------
# Holm correction within each hypothesis family
# -----------------------------------------------------------------------------

all_tests = pd.concat(
    [stance_tests, missing_tests, selector_tests],
    ignore_index=True,
)

all_tests["p_holm"] = np.nan

for family, indices in all_tests.groupby("family").groups.items():
    index_array = np.asarray(list(indices), dtype=np.int64)
    adjusted = holm_adjust(
        all_tests.loc[index_array, "p_value"].to_numpy()
    )
    all_tests.loc[index_array, "p_holm"] = adjusted

all_tests["significant_holm_0_05"] = (
    all_tests["p_holm"] < ALPHA
)

all_tests.to_csv(
    TABLE_DIR / "paired_statistical_tests_holm.csv",
    index=False,
)


# -----------------------------------------------------------------------------
# Integrity audit
# -----------------------------------------------------------------------------

forbidden_report_columns = {
    "route_to_deepseek",
    "deepseek_request",
    "verifier_response",
    "corrected_label",
}

integrity_checks = {
    "cell30_manifest_completed": (
        cell30_manifest.get("status") == "completed"
    ),
    "expected_test_report_count": (
        len(test_reports) == EXPECTED_TEST_REPORTS
    ),
    "expected_abstention_count": (
        int(test_reports["abstained"].astype(bool).sum())
        == EXPECTED_ABSTENTIONS
    ),
    "report_ids_unique": test_reports["report_id"].is_unique,
    "annotation_ids_unique": test_reports["annotation_id"].is_unique,
    "all_identifier_checks_passed": bool(
        test_reports["identifier_integrity_passed"].astype(bool).all()
    ),
    "no_forbidden_verifier_columns": not bool(
        forbidden_report_columns.intersection(test_reports.columns)
    ),
    "external_verifier_false": (
        "external_verifier_used" in report_integrity.columns
        and not bool(
            report_integrity["external_verifier_used"].astype(bool).any()
        )
    ),
    "five_full_am_jepa_seeds": (
        sorted(multiseed_summary["seed"].astype(int).tolist())
        == EXPECTED_SEEDS
    ),
    "six_seed42_objective_variants": (
        objective_seed42["variant"].nunique() == 6
    ),
    "finite_statistical_outputs": bool(
        np.isfinite(all_tests["p_value"]).all()
        and np.isfinite(all_tests["p_holm"]).all()
    ),
}

integrity_table = pd.DataFrame(
    [
        {
            "check": key,
            "passed": bool(value),
        }
        for key, value in integrity_checks.items()
    ]
)

if not integrity_table["passed"].all():
    failed = integrity_table.loc[
        ~integrity_table["passed"],
        "check",
    ].tolist()
    raise AssertionError(
        "Final integrity audit failed: " + ", ".join(failed)
    )

integrity_table.to_csv(
    TABLE_DIR / "final_integrity_checks.csv",
    index=False,
)


# -----------------------------------------------------------------------------
# Publication visual system
# -----------------------------------------------------------------------------

plt.rcParams.update(
    {
        "font.family": "serif",
        "font.serif": ["Times New Roman", "DejaVu Serif"],
        "font.size": 9.5,
        "axes.titlesize": 11,
        "axes.labelsize": 10,
        "axes.linewidth": 0.8,
        "xtick.labelsize": 8.5,
        "ytick.labelsize": 8.5,
        "legend.fontsize": 8.5,
        "figure.facecolor": "white",
        "axes.facecolor": "white",
        "savefig.facecolor": "white",
        "pdf.fonttype": 42,
        "ps.fonttype": 42,
    }
)

COLORS = {
    "navy": "#173F5F",
    "blue": "#20639B",
    "teal": "#2A9D8F",
    "gold": "#E9C46A",
    "orange": "#F4A261",
    "red": "#D1495B",
    "gray": "#6C757D",
    "light": "#E9ECEF",
}


def polish_axis(ax, grid_axis="both"):
    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)
    ax.grid(
        axis=grid_axis,
        color="#D9DEE3",
        linewidth=0.6,
        alpha=0.7,
        zorder=0,
    )
    ax.tick_params(length=3, width=0.7)


figure_catalog_rows = []


# -----------------------------------------------------------------------------
# Figure 1: objective trade-space with collapse diagnostic
# -----------------------------------------------------------------------------

fig, ax = plt.subplots(figsize=(7.8, 5.6))

latent_values = objective_seed42.get(
    "mean_latent_std",
    pd.Series(np.ones(len(objective_seed42))),
).to_numpy(dtype=np.float64)

latent_min = float(latent_values.min())
latent_range = max(float(latent_values.max() - latent_min), 1e-12)
sizes = 90 + 260 * (latent_values - latent_min) / latent_range

for row_number, row in objective_seed42.reset_index(drop=True).iterrows():
    variant = str(row["variant"])
    is_full = variant == "full_am_jepa"
    is_random = variant == "random_encoder"
    marker = "*" if is_full else ("X" if is_random else "o")
    color = (
        COLORS["teal"]
        if is_full
        else COLORS["gray"]
        if is_random
        else COLORS["blue"]
    )

    point = ax.scatter(
        row["campaign_f1"],
        row["core3_macro_f1"],
        s=sizes[row_number],
        c=color,
        marker=marker,
        edgecolor="white",
        linewidth=1.0,
        alpha=0.95,
        zorder=4,
    )
    point.set_path_effects(
        [path_effects.withStroke(linewidth=2.2, foreground="white")]
    )

    label = variant.replace("_", " ")
    offsets = {
        "full_am_jepa": (7, -13),
        "random_encoder": (7, -2),
        "masked_evidence_only": (7, 5),
        "evidence_plus_modality": (7, -12),
        "evidence_plus_future": (7, 6),
        "full_without_variance": (-105, -14),
    }
    offset = offsets.get(variant, (6, 5))
    ax.annotate(
        label,
        (row["campaign_f1"], row["core3_macro_f1"]),
        xytext=offset,
        textcoords="offset points",
        fontsize=8.2,
        color="#263238",
        arrowprops=(
            dict(arrowstyle="-", color="#9AA0A6", lw=0.6)
            if abs(offset[0]) > 20
            else None
        ),
    )

random_row = objective_seed42[
    objective_seed42["variant"] == "random_encoder"
].iloc[0]

ax.axvline(
    random_row["campaign_f1"],
    color=COLORS["gray"],
    linestyle="--",
    linewidth=0.9,
    alpha=0.65,
)
ax.axhline(
    random_row["core3_macro_f1"],
    color=COLORS["gray"],
    linestyle="--",
    linewidth=0.9,
    alpha=0.65,
)

ax.text(
    0.012,
    0.975,
    "Upper-right is better\nBubble area $\\propto$ latent dispersion",
    transform=ax.transAxes,
    va="top",
    ha="left",
    fontsize=8.3,
    bbox=dict(
        boxstyle="round,pad=0.35",
        facecolor="white",
        edgecolor="#CFD8DC",
        alpha=0.95,
    ),
)

ax.set_xlabel("Campaign F1")
ax.set_ylabel("Core-three aspect macro-F1")
ax.set_title("Objective Trade-space and Representation Dispersion")
ax.set_xlim(
    objective_seed42["campaign_f1"].min() - 0.006,
    objective_seed42["campaign_f1"].max() + 0.007,
)
ax.set_ylim(
    objective_seed42["core3_macro_f1"].min() - 0.012,
    objective_seed42["core3_macro_f1"].max() + 0.014,
)
polish_axis(ax)
fig.tight_layout()
save_figure(fig, "figure_1_objective_trade_space")
figure_catalog_rows.append(
    {
        "figure": "Figure 1",
        "file_stem": "figure_1_objective_trade_space",
        "design": "Annotated bubble trade-space",
        "message": "Objective balance and latent-dispersion behavior",
    }
)


# -----------------------------------------------------------------------------
# Figure 2: missing-modality degradation fingerprint
# -----------------------------------------------------------------------------

robustness_summary = load_csv(
    CELL30_DIR / "missing_modality_robustness_publication_table.csv"
)

degradation = robustness_summary[
    robustness_summary["scenario"] != "complete"
].copy()

degradation_labels = (
    degradation["scenario"]
    .str.replace("_", " ", regex=False)
    .str.replace("pct", "%", regex=False)
)

drop_matrix = degradation[
    ["campaign_f1_drop_mean", "core3_f1_drop_mean"]
].to_numpy(dtype=np.float64)

fig = plt.figure(figsize=(10.4, 6.2))
grid = fig.add_gridspec(
    1,
    2,
    width_ratios=[1.25, 1.0],
    wspace=0.34,
)

ax_heat = fig.add_subplot(grid[0, 0])
image = ax_heat.imshow(
    drop_matrix,
    cmap="YlOrRd",
    aspect="auto",
    vmin=0.0,
    vmax=max(0.56, float(drop_matrix.max())),
)

ax_heat.set_yticks(np.arange(len(degradation)))
ax_heat.set_yticklabels(degradation_labels)
ax_heat.set_xticks([0, 1])
ax_heat.set_xticklabels(["Campaign F1", "Core-three F1"])
ax_heat.set_title("A. Degradation fingerprint")

for row in range(drop_matrix.shape[0]):
    for column in range(drop_matrix.shape[1]):
        value = drop_matrix[row, column]
        ax_heat.text(
            column,
            row,
            f"$-$ {value:.3f}",
            ha="center",
            va="center",
            color="white" if value > 0.27 else "#202124",
            fontsize=8.5,
            fontweight="bold" if value > 0.25 else "normal",
        )

colorbar = fig.colorbar(
    image,
    ax=ax_heat,
    fraction=0.046,
    pad=0.04,
)
colorbar.set_label("Mean loss from complete input")

ax_profile = fig.add_subplot(grid[0, 1])
y = np.arange(len(degradation))

ax_profile.errorbar(
    degradation["campaign_auroc_mean"],
    y - 0.12,
    xerr=degradation["campaign_auroc_sd"],
    fmt="o",
    color=COLORS["blue"],
    ecolor="#90A4AE",
    capsize=2.5,
    label="Campaign AUROC",
    zorder=3,
)
ax_profile.errorbar(
    degradation["core3_macro_f1_mean"],
    y + 0.12,
    xerr=degradation["core3_macro_f1_sd"],
    fmt="D",
    color=COLORS["orange"],
    ecolor="#B0BEC5",
    capsize=2.5,
    label="Core-three F1",
    zorder=3,
)

ax_profile.set_yticks(y)
ax_profile.set_yticklabels([])
ax_profile.invert_yaxis()
ax_profile.set_xlabel("Retained performance (mean $\\pm$ SD)")
ax_profile.set_title("B. Residual capability")
ax_profile.legend(frameon=False, loc="lower right")
polish_axis(ax_profile, grid_axis="x")

fig.suptitle(
    "Missing-Modality Robustness: Degradation and Residual Capability",
    y=1.01,
)
fig.subplots_adjust(left=0.19, right=0.97, bottom=0.10, top=0.90)
save_figure(fig, "figure_2_missing_modality_fingerprint")
figure_catalog_rows.append(
    {
        "figure": "Figure 2",
        "file_stem": "figure_2_missing_modality_fingerprint",
        "design": "Annotated heatmap plus estimation profile",
        "message": "Which missing inputs damage which task",
    }
)


# -----------------------------------------------------------------------------
# Figure 3: seed-aware selective risk--coverage frontier
# -----------------------------------------------------------------------------

selective_seed = load_csv(
    FINAL_DIR / "stance_selective_seed_results.csv",
    [
        "method",
        "seed",
        "target_validation_coverage",
        "test_coverage",
        "selective_risk",
    ],
)

method_styles = {
    "dense_am_jepa": (COLORS["navy"], "o"),
    "dual_text_only": (COLORS["teal"], "s"),
    "dual_full_multimodal": (COLORS["orange"], "D"),
}

fig, ax = plt.subplots(figsize=(8.2, 5.6))

for method, group in selective_seed.groupby("method", sort=False):
    color, marker = method_styles.get(method, (COLORS["gray"], "o"))

    for seed, seed_group in group.groupby("seed"):
        ordered_seed = seed_group.sort_values("test_coverage")
        ax.plot(
            ordered_seed["test_coverage"],
            ordered_seed["selective_risk"],
            color=color,
            linewidth=0.75,
            alpha=0.16,
            zorder=1,
        )

    summary = (
        group.groupby("target_validation_coverage", as_index=False)
        .agg(
            coverage_mean=("test_coverage", "mean"),
            coverage_sd=("test_coverage", "std"),
            risk_mean=("selective_risk", "mean"),
            risk_sd=("selective_risk", "std"),
        )
        .sort_values("coverage_mean")
    )

    ax.errorbar(
        summary["coverage_mean"],
        summary["risk_mean"],
        xerr=summary["coverage_sd"].fillna(0.0),
        yerr=summary["risk_sd"].fillna(0.0),
        marker=marker,
        markersize=6.0,
        linewidth=2.2,
        capsize=2.5,
        color=color,
        label=method.replace("_", " "),
        zorder=4,
    )

primary_point = selective_aggregate[
    (selective_aggregate["method"] == "dual_text_only")
    & np.isclose(
        selective_aggregate["target_validation_coverage"],
        0.7,
    )
].iloc[0]

ax.scatter(
    primary_point["test_coverage_mean"],
    primary_point["selective_risk_mean"],
    marker="*",
    s=220,
    color=COLORS["gold"],
    edgecolor="#5D4037",
    linewidth=0.8,
    zorder=6,
)
ax.annotate(
    "Best selective macro-F1\ncoverage 0.692, risk 0.272",
    (
        primary_point["test_coverage_mean"],
        primary_point["selective_risk_mean"],
    ),
    xytext=(34, 28),
    textcoords="offset points",
    arrowprops=dict(
        arrowstyle="->",
        color="#5D4037",
        lw=0.8,
    ),
    fontsize=8.4,
    bbox=dict(
        boxstyle="round,pad=0.3",
        facecolor="white",
        edgecolor="#CFD8DC",
    ),
)

ax.annotate(
    "preferred direction",
    xy=(0.57, 0.16),
    xytext=(0.72, 0.23),
    arrowprops=dict(
        arrowstyle="-|>",
        color=COLORS["gray"],
        lw=1.0,
    ),
    color=COLORS["gray"],
    fontsize=8.2,
)

ax.set_xlabel("Realized test coverage")
ax.set_ylabel("Selective risk (lower is better)")
ax.set_xlim(0.50, 1.025)
ax.set_ylim(0.12, 0.44)
ax.set_title("Seed-Aware Selective Risk--Coverage Frontier")
ax.legend(frameon=False, loc="upper left")
polish_axis(ax)
fig.tight_layout()
save_figure(fig, "figure_3_selective_risk_frontier")
figure_catalog_rows.append(
    {
        "figure": "Figure 3",
        "file_stem": "figure_3_selective_risk_frontier",
        "design": "Seed trajectories and uncertainty-aware frontier",
        "message": "Reliability gained by abstention",
    }
)


# -----------------------------------------------------------------------------
# Figure 4: multi-metric estimation plot for stance architectures
# -----------------------------------------------------------------------------

stance_plot = stance_aggregate[
    stance_aggregate["evaluation_set"] == "all_test"
].copy()

method_order = [
    "dense_am_jepa",
    "dual_text_only",
    "dual_full_multimodal",
]
method_labels = {
    "dense_am_jepa": "Dense AM-JEPA",
    "dual_text_only": "Dual text sparse",
    "dual_full_multimodal": "Dual multimodal sparse",
}
method_colors = {
    "dense_am_jepa": COLORS["navy"],
    "dual_text_only": COLORS["teal"],
    "dual_full_multimodal": COLORS["orange"],
}

metric_panels = [
    ("accuracy", "Accuracy", True),
    ("macro_f1", "Macro-F1", True),
    ("mcc", "MCC", True),
    ("macro_ovr_auroc", "Macro-AUROC", True),
    ("ece", "ECE", False),
]

fig, axes = plt.subplots(
    1,
    len(metric_panels),
    figsize=(13.4, 4.2),
    sharey=True,
)
y_positions = np.arange(len(method_order))[::-1]

for panel_number, (metric, title, higher_is_better) in enumerate(metric_panels):
    ax = axes[panel_number]

    for y_position, method in zip(y_positions, method_order):
        row = stance_plot[stance_plot["method"] == method].iloc[0]
        mean = float(row[f"{metric}_mean"])
        sd = float(row[f"{metric}_sd"])
        ax.errorbar(
            mean,
            y_position,
            xerr=sd,
            fmt="o",
            markersize=6.5,
            capsize=3,
            elinewidth=1.5,
            color=method_colors[method],
            zorder=3,
        )
        ax.text(
            mean,
            y_position + 0.18,
            f"{mean:.3f}",
            ha="center",
            va="bottom",
            fontsize=7.6,
            color=method_colors[method],
        )

    ax.set_title(title)
    ax.set_xlabel("higher $\\rightarrow$" if higher_is_better else "$\\leftarrow$ lower")
    polish_axis(ax, grid_axis="x")

axes[0].set_yticks(y_positions)
axes[0].set_yticklabels([method_labels[item] for item in method_order])
axes[0].set_ylim(-0.55, len(method_order) - 0.42)

fig.suptitle("Five-Seed Stance Architecture Estimation Plot", y=1.02)
fig.tight_layout()
save_figure(fig, "figure_4_stance_estimation_forest")
figure_catalog_rows.append(
    {
        "figure": "Figure 4",
        "file_stem": "figure_4_stance_estimation_forest",
        "design": "Five-panel estimation/forest plot",
        "message": "Accuracy, discrimination, association, and calibration",
    }
)


# -----------------------------------------------------------------------------
# Figure 5: dual-selector gain map
# -----------------------------------------------------------------------------

gain_map = (
    dual_seed_deltas
    .groupby(["feature_set", "budget"])["map_delta"]
    .mean()
    .unstack("budget")
    .sort_index()
)

gain_sd = (
    dual_seed_deltas
    .groupby(["feature_set", "budget"])["map_delta"]
    .std()
    .unstack("budget")
    .reindex(index=gain_map.index, columns=gain_map.columns)
)

values = gain_map.to_numpy(dtype=np.float64)
limit = max(abs(float(np.nanmin(values))), abs(float(np.nanmax(values))), 0.01)
normalization = TwoSlopeNorm(vmin=-limit, vcenter=0.0, vmax=limit)

fig, ax = plt.subplots(figsize=(7.7, 3.8))
image = ax.imshow(
    values,
    cmap="RdBu_r",
    norm=normalization,
    aspect="auto",
)

ax.set_xticks(np.arange(len(gain_map.columns)))
ax.set_xticklabels([f"K={int(item)}" for item in gain_map.columns])
ax.set_yticks(np.arange(len(gain_map.index)))
ax.set_yticklabels(
    [item.replace("_", " ") for item in gain_map.index]
)

for row in range(values.shape[0]):
    for column in range(values.shape[1]):
        mean_value = values[row, column]
        sd_value = gain_sd.iloc[row, column]
        ax.text(
            column,
            row,
            f"{mean_value:+.3f}\n$\\pm${sd_value:.3f}",
            ha="center",
            va="center",
            fontsize=8.5,
            color=("white" if abs(mean_value) > 0.65 * limit else "#202124"),
            fontweight="bold" if mean_value > 0 else "normal",
        )

colorbar = fig.colorbar(image, ax=ax, pad=0.025)
colorbar.set_label("Dual minus matched single-selector MAP")
ax.set_title("Where Does Explicit E$^+$/E$^-$ Separation Help?")
ax.set_xlabel("Evidence budget")
ax.set_ylabel("Feature view")
fig.tight_layout()
save_figure(fig, "figure_5_dual_selector_gain_map")
figure_catalog_rows.append(
    {
        "figure": "Figure 5",
        "file_stem": "figure_5_dual_selector_gain_map",
        "design": "Diverging gain heatmap",
        "message": "Budget- and modality-dependent dual-selector gains",
    }
)


figure_catalog = pd.DataFrame(figure_catalog_rows)
figure_catalog.to_csv(
    TABLE_DIR / "publication_figure_catalog.csv",
    index=False,
)


# -----------------------------------------------------------------------------
# LaTeX-ready tables
# -----------------------------------------------------------------------------

objective_latex_columns = [
    column
    for column in [
        "variant",
        "campaign_f1",
        "campaign_auroc",
        "campaign_auprc",
        "core3_macro_f1",
        "core3_macro_mcc",
        "mean_latent_std",
    ]
    if column in objective_seed42.columns
]

write_latex_table(
    objective_seed42,
    TABLE_DIR / "objective_ablation_seed42.tex",
    "Seed-42 AM-JEPA objective ablation.",
    "tab:objective-ablation",
    objective_latex_columns,
)

write_latex_table(
    robustness_summary,
    TABLE_DIR / "missing_modality_robustness.tex",
    "Five-seed missing-modality robustness.",
    "tab:missing-modality",
    [
        "scenario",
        "campaign_f1_mean",
        "campaign_f1_sd",
        "core3_macro_f1_mean",
        "core3_macro_f1_sd",
    ],
)

write_latex_table(
    stance_plot,
    TABLE_DIR / "stance_architecture_ablation.tex",
    "Five-seed stance architecture comparison.",
    "tab:stance-architecture",
    [
        "method",
        "macro_f1_mean",
        "macro_f1_sd",
        "mcc_mean",
        "mcc_sd",
        "macro_ovr_auroc_mean",
        "ece_mean",
    ],
)


# -----------------------------------------------------------------------------
# Final result summary and reproducibility manifest
# -----------------------------------------------------------------------------

dense_row = stance_plot[
    stance_plot["method"] == "dense_am_jepa"
].iloc[0]

dual_text_row = stance_plot[
    stance_plot["method"] == "dual_text_only"
].iloc[0]

selective_primary = selective_aggregate[
    (selective_aggregate["method"] == "dual_text_only")
    & np.isclose(
        selective_aggregate["target_validation_coverage"],
        0.7,
    )
].iloc[0]

result_summary = {
    "label_status": (
        "AI-assisted single-reviewer silver pilot; not independent "
        "human gold-standard evidence."
    ),
    "campaign_f1_five_seed_mean": float(
        multiseed_summary["campaign_f1"].mean()
    ),
    "campaign_f1_five_seed_sd": float(
        multiseed_summary["campaign_f1"].std(ddof=1)
    ),
    "core3_macro_f1_five_seed_mean": float(
        multiseed_summary["core3_macro_f1"].mean()
    ),
    "core3_macro_f1_five_seed_sd": float(
        multiseed_summary["core3_macro_f1"].std(ddof=1)
    ),
    "dense_stance_macro_f1": float(dense_row["macro_f1_mean"]),
    "dual_text_full_coverage_macro_f1_mean": float(
        dual_text_row["macro_f1_mean"]
    ),
    "dual_text_full_coverage_macro_f1_sd": float(
        dual_text_row["macro_f1_sd"]
    ),
    "dual_text_selective_macro_f1": float(
        selective_primary["macro_f1_mean"]
    ),
    "dual_text_selective_test_coverage": float(
        selective_primary["test_coverage_mean"]
    ),
    "deterministic_reports": int(len(test_reports)),
    "deterministic_abstentions": int(
        test_reports["abstained"].astype(bool).sum()
    ),
    "external_verifier_used": False,
    "statistical_note": (
        "Five-seed Wilcoxon tests have low power; exact effect estimates "
        "and Holm-adjusted p-values are reported without overstating "
        "non-significance."
    ),
}

with open(
    CELL31_DIR / "final_result_summary.json",
    "w",
    encoding="utf-8",
) as file:
    json.dump(result_summary, file, indent=2)


# Hash every final table and figure before packaging.
export_candidates = sorted(
    list(TABLE_DIR.glob("*"))
    + list(FIGURE_DIR.glob("*"))
    + [
        CELL30_DIR / "deterministic_evidence_linked_test_reports.csv",
        CELL30_DIR / "deterministic_report_integrity_summary.csv",
        CELL31_DIR / "final_result_summary.json",
    ]
)

hash_rows = []

for path in export_candidates:
    require_file(path)
    hash_rows.append(
        {
            "file": path.name,
            "relative_path": str(path.relative_to(AMJEPA_DIR)),
            "bytes": int(path.stat().st_size),
            "sha256": sha256_file(path),
        }
    )

hash_manifest = pd.DataFrame(hash_rows)
hash_manifest.to_csv(
    CELL31_DIR / "final_export_hash_manifest.csv",
    index=False,
)


final_manifest = {
    "cell": 31,
    "status": "completed",
    "alpha": ALPHA,
    "multiple_testing": "Holm adjustment within hypothesis family",
    "paired_test": "two-sided Wilcoxon signed-rank",
    "seeds": EXPECTED_SEEDS,
    "integrity_checks": integrity_checks,
    "figures": [path.name for path in sorted(FIGURE_DIR.glob("*.png"))],
    "tables": [path.name for path in sorted(TABLE_DIR.glob("*"))],
    "external_verifier_used": False,
    "test_used_for_model_selection": False,
}

with open(
    CELL31_DIR / "cell31_final_manifest.json",
    "w",
    encoding="utf-8",
) as file:
    json.dump(final_manifest, file, indent=2)


# -----------------------------------------------------------------------------
# Final publication package
# -----------------------------------------------------------------------------

zip_path = CELL31_DIR / "SparseSentinel_APT_final_results_package.zip"

package_files = (
    export_candidates
    + [
        CELL31_DIR / "final_export_hash_manifest.csv",
        CELL31_DIR / "cell31_final_manifest.json",
    ]
)

with zipfile.ZipFile(
    zip_path,
    mode="w",
    compression=zipfile.ZIP_DEFLATED,
) as archive:
    for path in package_files:
        # Do not include legacy routing queues or protected evaluation keys.
        lowered = path.name.lower()
        if "deepseek" in lowered or "do_not_load" in lowered:
            raise AssertionError(
                f"Forbidden legacy artifact entered final package: {path.name}"
            )

        archive.write(
            path,
            arcname=str(path.relative_to(AMJEPA_DIR)),
        )


# -----------------------------------------------------------------------------
# Display final results
# -----------------------------------------------------------------------------

print("=" * 80)
print("SPARSESENTINEL-APT FINAL STATISTICS AND EXPORT")
print("=" * 80)

print("\nCorrected seed-42 objective ablation:")
display(objective_seed42.round(6))

print("\nHolm-adjusted paired statistical tests:")
display(
    all_tests[
        [
            "family",
            "metric",
            "comparison",
            "pairs",
            "mean_difference",
            "cohen_dz",
            "p_value",
            "p_holm",
            "significant_holm_0_05",
        ]
    ].round(6)
)

print("\nFinal integrity checks:")
display(integrity_table)

print("\nFinal result summary:")
display(pd.DataFrame([result_summary]).T.rename(columns={0: "value"}))

print("\nCell 31 completed successfully.")
print("Final package:")
print(zip_path)
print("\nFinal package SHA-256:")
print(sha256_file(zip_path))
